In [1]:
import os
import re
import copy
import joblib
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torch.multiprocessing as mp
from torch.utils.data import DataLoader, TensorDataset, WeightedRandomSampler
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

mp.set_sharing_strategy("file_system")

# =========================
# Configuration
# =========================
BASE_DIR = "./"

DATA_PATHS = [
    os.path.join(BASE_DIR, "data_random_with_random_variances_total_v2.csv"),
    os.path.join(BASE_DIR, "data_physics_with_variances_total.csv"),
]

NROWS_PER_DATASET = 5_000_000

BEST_MODEL_SAVE_PATH = os.path.join(
    BASE_DIR,
    "random_extra_multitask_best2_normthr_monoord.pth",
)
LAST_MODEL_SAVE_PATH = os.path.join(
    BASE_DIR,
    "random_extra_multitask_last2_normthr_monoord.pth",
)
SCALER_SAVE_PATH = os.path.join(
    BASE_DIR,
    "random_extra_multitask_scalers2_normthr_monoord.pkl",
)

EPS = 1e-12
LOG10_HALF = float(np.log10(0.5))

# =========================
# Threshold / numerical feature options
# =========================
# "normalized_logit" is the recommended default:
#     threshold_feature = logit(clip(threshold / sum(valid scaled taps), eps, 1-eps))
# This puts every threshold sweep on a scenario-normalized axis and removes
# scenario magnitude from the threshold input. Magnitude is still available
# through the scaled tap token features.
# Other supported modes: "normalized", "log10_raw".
THRESHOLD_FEATURE_MODE = "normalized_logit"
THRESHOLD_NORM_CLIP_EPS = 1e-5
HARMONIC_DENOM_EPS = 1e-8
FEATURE_CLIP_ABS = 1e6

ORDINAL_THRESHOLDS = [
    1e-6, 1e-5, 1e-4, 1e-3, 1e-2, 1e-1,
    0.15, 0.20, 0.25, 0.30, 0.35, 0.40, 0.45,
]
NUM_ORDINAL_THRESHOLDS = len(ORDINAL_THRESHOLDS)
NUM_REGION_CLASSES = NUM_ORDINAL_THRESHOLDS + 1


# =========================
# Utilities
# =========================
def get_sorted_seq_cols(columns, prefix):
    pattern = re.compile(rf"^{re.escape(prefix)}_(\d+)$")
    matched = []
    for col in columns:
        m = pattern.match(col)
        if m:
            matched.append((int(m.group(1)), col))
    matched.sort(key=lambda x: x[0])
    return [col for _, col in matched]


def make_strat_bins(y_log, n_bins=10):
    y_flat = y_log.reshape(-1)
    quantiles = np.linspace(0, 1, n_bins + 1)
    edges = np.quantile(y_flat, quantiles)
    edges = np.unique(edges)

    if len(edges) < 3:
        return None

    bins = np.digitize(y_flat, edges[1:-1], right=True)
    counts = np.bincount(bins)
    if np.any(counts < 2):
        return None
    return bins


def has_nonfinite_tensor(x):
    return not torch.isfinite(x).all().item()


def validate_required_columns(df, file_path):
    if "mem_len" not in df.columns:
        raise ValueError(f"Required column 'mem_len' not found in {file_path}")
    if "N" not in df.columns:
        raise ValueError(f"Required column 'N' not found in {file_path}")

    tap_cols = get_sorted_seq_cols(df.columns, "tap")
    var_cols = get_sorted_seq_cols(df.columns, "var")

    if not tap_cols:
        raise ValueError(f"No tap_* columns found in {file_path}")
    if not var_cols:
        raise ValueError(f"No var_* columns found in {file_path}")
    if len(tap_cols) != len(var_cols):
        raise ValueError(
            f"tap/var length mismatch in {file_path}: "
            f"{len(tap_cols)} tap cols vs {len(var_cols)} var cols"
        )

    required_cols = tap_cols + var_cols + ["threshold", "BER", "N"]
    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        raise ValueError(f"Missing required columns in {file_path}: {missing}")

    return tap_cols, var_cols


def load_and_merge_data(csv_paths, nrows_per_dataset):
    dfs = []
    reference_tap_cols = None
    reference_var_cols = None

    for path in csv_paths:
        if not os.path.exists(path):
            raise FileNotFoundError(f"Dataset not found: {path}")

        print(f"Loading up to {nrows_per_dataset:,} rows from: {path}")
        df = pd.read_csv(path, nrows=nrows_per_dataset)

        tap_cols, var_cols = validate_required_columns(df, path)

        if reference_tap_cols is None:
            reference_tap_cols = tap_cols
            reference_var_cols = var_cols
        else:
            if tap_cols != reference_tap_cols:
                raise ValueError("tap columns do not match across files.")
            if var_cols != reference_var_cols:
                raise ValueError("var columns do not match across files.")

        df["source_dataset"] = os.path.basename(path)
        dfs.append(df)

    merged = pd.concat(dfs, ignore_index=True)
    print(f"Combined rows before filtering: {len(merged):,}")

    return merged, reference_tap_cols, reference_var_cols


def y_log_to_raw_np(y_log):
    return np.clip(10 ** y_log, EPS, 0.5)


def raw_to_region_labels_np(y_raw):
    y = np.asarray(y_raw).reshape(-1)
    labels = np.searchsorted(np.asarray(ORDINAL_THRESHOLDS), y, side="right")
    return labels.astype(np.int64)


def region_labels_to_ordinal_targets_np(labels, num_thresholds=NUM_ORDINAL_THRESHOLDS):
    labels = np.asarray(labels).reshape(-1)
    thresholds = np.arange(1, num_thresholds + 1, dtype=np.int64)
    ordinal = (labels[:, None] >= thresholds[None, :]).astype(np.float32)
    return ordinal


def ordinal_logits_to_region_labels_torch(logits):
    probs = torch.sigmoid(logits)
    passed = (probs >= 0.5).sum(dim=1)
    return passed.long()


def compute_region_class_weights(labels_np, num_classes=NUM_REGION_CLASSES, max_weight=8.0):
    counts = np.bincount(labels_np.reshape(-1), minlength=num_classes).astype(np.float64)
    counts = np.maximum(counts, 1.0)
    weights = counts.sum() / (num_classes * counts)
    weights = np.sqrt(weights)
    weights = np.clip(weights, 1.0, max_weight)
    return weights.astype(np.float32)


def compute_ordinal_pos_weights_from_region_labels(
    region_labels_np,
    num_thresholds=NUM_ORDINAL_THRESHOLDS,
    max_weight=20.0,
):
    ordinal_targets = region_labels_to_ordinal_targets_np(region_labels_np, num_thresholds)
    pos_counts = ordinal_targets.sum(axis=0)
    neg_counts = ordinal_targets.shape[0] - pos_counts
    pos_counts = np.maximum(pos_counts, 1.0)
    pos_weight = neg_counts / pos_counts
    pos_weight = np.clip(pos_weight, 1.0, max_weight)
    return pos_weight.astype(np.float32)


# =========================
# Position-Independent Scaling
# =========================
class SharedFeatureScaler:
    """A scaler that stores a single (mean, std) per feature channel,
    shared across all sequence positions.

    For the first token (always position 0) we keep a separate scaler,
    since the current-symbol tap has a genuinely different distribution
    than the ISI taps at positions 1+.

    For all past positions we pool every valid entry into one distribution
    and fit a single mean/std.  This makes the scaler independent of
    sequence length — at inference time, any number of past taps can be
    scaled with the same parameters.
    """

    def __init__(self):
        self.first_mean = None  # shape [n_features]
        self.first_std = None
        self.past_mean = None   # shape [n_features]
        self.past_std = None

    def fit(self, first_data_list, past_data_list, past_valid_list):
        """
        Args:
            first_data_list: list of arrays, each [N, n_features] for the
                             first-token features (taps[:,0], vars[:,0], ...).
                             Concatenated across splits if desired, or just train.
            past_data_list:  list of arrays, each [N, L_past, n_features] or
                             [N, L_past] for a single feature channel.
            past_valid_list: list of bool arrays, each [N, L_past], True = valid.
        """
        # --- First token ---
        first_all = np.concatenate(first_data_list, axis=0)  # [N_total, F]
        self.first_mean = first_all.mean(axis=0).astype(np.float64)
        self.first_std = first_all.std(axis=0).astype(np.float64)
        self.first_std = np.maximum(self.first_std, 1e-12)

        # --- Past tokens (pool all valid entries per feature) ---
        valid_entries = []
        for data, valid in zip(past_data_list, past_valid_list):
            if data.ndim == 2:
                # single feature: [N, L] -> expand to [N, L, 1]
                data = data[:, :, None]
            # data: [N, L, F], valid: [N, L]
            valid_expanded = valid[:, :, None]  # [N, L, 1]
            # Gather valid entries: [?, F]
            valid_entries.append(data[np.broadcast_to(valid_expanded, data.shape)].reshape(-1, data.shape[-1]))

        pooled = np.concatenate(valid_entries, axis=0)  # [total_valid, F]
        self.past_mean = pooled.mean(axis=0).astype(np.float64)
        self.past_std = pooled.std(axis=0).astype(np.float64)
        self.past_std = np.maximum(self.past_std, 1e-12)

    def transform_first(self, first_data):
        """first_data: [N, F] -> scaled [N, F]"""
        return ((first_data - self.first_mean) / self.first_std).astype(np.float32)

    def transform_past(self, past_data, past_lens):
        """past_data: [N, L, F] or [N, L] -> scaled + re-zeroed [N, L, ...].
        past_lens: [N] int, number of valid past positions per row."""
        shape = past_data.shape
        if past_data.ndim == 2:
            scaled = ((past_data - self.past_mean[0]) / self.past_std[0]).astype(np.float32)
            L = shape[1]
        else:
            scaled = ((past_data - self.past_mean) / self.past_std).astype(np.float32)
            L = shape[1]

        # Re-zero padding positions
        valid = (np.arange(L)[None, :] < past_lens[:, None])  # [N, L]
        if scaled.ndim == 3:
            valid = valid[:, :, None]
        scaled = scaled * valid.astype(np.float32)
        return scaled


def fit_shared_scalar_scaler(train_data, train_valid_mask):
    """Fit a single-feature StandardScaler on pooled valid entries.
    train_data: [N, L], train_valid_mask: [N, L] bool.
    Returns (mean, std) as floats."""
    entries = train_data[train_valid_mask].reshape(-1)
    if len(entries) == 0:
        entries = train_data.reshape(-1)
    mean = float(entries.mean())
    std = float(max(entries.std(), 1e-12))
    return mean, std


def apply_shared_scale(data, mean, std, valid_mask=None):
    """Scale data with a single (mean, std), optionally re-zero invalid positions.
    data: [N, L] or [N, 1], valid_mask: [N, L] bool or None."""
    scaled = ((data - mean) / std).astype(np.float32)
    if valid_mask is not None and scaled.ndim == 2 and valid_mask.shape == scaled.shape:
        scaled = scaled * valid_mask.astype(np.float32)
    return scaled


def sanitize_np_feature(x, name, clip_abs=FEATURE_CLIP_ABS):
    """Replace NaN/inf with 0 and clip extreme engineered features."""
    arr = np.asarray(x, dtype=np.float32)
    bad = ~np.isfinite(arr)
    if np.any(bad):
        print(f"Warning: {name} contained {int(bad.sum())} non-finite values; replacing with 0.")
    arr = np.nan_to_num(arr, nan=0.0, posinf=0.0, neginf=0.0)
    arr = np.clip(arr, -clip_abs, clip_abs).astype(np.float32)
    return arr


def stable_harmonic_pair_np(a, b, denom_eps=HARMONIC_DENOM_EPS):
    """Stable 2ab/(a+b), with near-zero denominators mapped to zero."""
    a = np.asarray(a, dtype=np.float32)
    b = np.asarray(b, dtype=np.float32)
    denom = (a + b).astype(np.float32)
    numer = (2.0 * a * b).astype(np.float32)
    out = np.divide(
        numer,
        denom,
        out=np.zeros_like(numer, dtype=np.float32),
        where=np.abs(denom) > denom_eps,
    ).astype(np.float32)
    return sanitize_np_feature(out, "stable_harmonic_pair")


def compute_threshold_feature_np(
    threshold_raw,
    total_scaled_taps,
    mode=THRESHOLD_FEATURE_MODE,
):
    """Build the scalar threshold input used by the threshold branch.

    The recommended mode is normalized_logit:
        logit(clip(threshold / sum(valid tap_i * N), eps, 1-eps))

    This normalizes every threshold sweep to a common [0, 1] scenario axis.
    """
    thr = np.asarray(threshold_raw, dtype=np.float32).reshape(-1, 1)
    total = np.asarray(total_scaled_taps, dtype=np.float32).reshape(-1, 1)
    denom = np.maximum(total, EPS).astype(np.float32)

    if mode == "normalized":
        feat = (thr / denom).astype(np.float32)
    elif mode == "normalized_logit":
        u = (thr / denom).astype(np.float32)
        u = np.clip(u, THRESHOLD_NORM_CLIP_EPS, 1.0 - THRESHOLD_NORM_CLIP_EPS)
        feat = np.log(u / (1.0 - u)).astype(np.float32)
    elif mode == "log10_raw":
        feat = np.log10(thr + EPS).astype(np.float32)
    else:
        raise ValueError(f"Unknown THRESHOLD_FEATURE_MODE: {mode}")

    return sanitize_np_feature(feat, f"threshold_feature_{mode}")


# =========================
# Targeted Regression Loss
# =========================
class StableMultiObjectiveBERBoundedLogLoss(nn.Module):
    def __init__(
        self,
        log_delta=0.35,
        raw_delta=0.006,
        rel_delta=0.03,
        alpha_log=0.45,
        beta_raw=0.35,
        gamma_rel=0.20,
        use_regime_weights=True,
    ):
        super().__init__()
        self.log_delta = log_delta
        self.raw_delta = raw_delta
        self.rel_delta = rel_delta
        self.alpha_log = alpha_log
        self.beta_raw = beta_raw
        self.gamma_rel = gamma_rel
        self.use_regime_weights = use_regime_weights

    @staticmethod
    def huber_elementwise(pred, target, delta):
        err = pred - target
        abs_err = err.abs()
        return torch.where(
            abs_err < delta,
            0.5 * err * err,
            delta * (abs_err - 0.5 * delta),
        )

    @staticmethod
    def raw_to_pred_log(pred_raw_unconstrained):
        log10_half = torch.log10(
            torch.tensor(
                0.5,
                device=pred_raw_unconstrained.device,
                dtype=pred_raw_unconstrained.dtype,
            )
        )
        return log10_half - F.softplus(pred_raw_unconstrained)

    @staticmethod
    def raw_to_pred_ber(pred_raw_unconstrained):
        pred_log = StableMultiObjectiveBERBoundedLogLoss.raw_to_pred_log(
            pred_raw_unconstrained
        )
        return torch.pow(10.0, pred_log).clamp(min=EPS, max=0.5)

    def forward(self, pred_raw_unconstrained, target_log):
        pred_log = self.raw_to_pred_log(pred_raw_unconstrained)
        pred_raw = torch.pow(10.0, pred_log).clamp(min=EPS, max=0.5)

        target_raw = torch.pow(10.0, target_log).clamp(min=EPS, max=0.5)
        target_log_for_loss = torch.log10(target_raw)

        log_loss = self.huber_elementwise(pred_log, target_log_for_loss, self.log_delta)
        raw_loss = self.huber_elementwise(pred_raw, target_raw, self.raw_delta)
        rel_err = (pred_raw - target_raw) / torch.clamp(target_raw, min=1e-6)
        rel_loss = self.huber_elementwise(rel_err, torch.zeros_like(rel_err), self.rel_delta)

        total = (
            self.alpha_log * log_loss
            + self.beta_raw * raw_loss
            + self.gamma_rel * rel_loss
        )

        if self.use_regime_weights:
            weights = torch.ones_like(target_raw)
            weights = torch.where(
                (target_raw >= 1e-2) & (target_raw < 0.1),
                torch.full_like(weights, 1.5), weights,
            )
            weights = torch.where(
                (target_raw >= 0.1) & (target_raw < 0.15),
                torch.full_like(weights, 2.5), weights,
            )
            weights = torch.where(
                (target_raw >= 0.15) & (target_raw < 0.2),
                torch.full_like(weights, 3.0), weights,
            )
            weights = torch.where(
                (target_raw >= 0.2) & (target_raw < 0.3),
                torch.full_like(weights, 4.0), weights,
            )
            weights = torch.where(
                (target_raw >= 0.3) & (target_raw < 0.4),
                torch.full_like(weights, 4.5), weights,
            )
            weights = torch.where(
                (target_raw >= 0.4) & (target_raw < 0.45),
                torch.full_like(weights, 3.0), weights,
            )
            weights = torch.where(
                target_raw >= 0.45,
                torch.full_like(weights, 2.5), weights,
            )
            total = total * weights

        return total.mean()


class OrdinalBCELoss(nn.Module):
    def __init__(self, pos_weight=None, reduction="mean"):
        super().__init__()
        if pos_weight is not None and not isinstance(pos_weight, torch.Tensor):
            pos_weight = torch.tensor(pos_weight, dtype=torch.float32)
        self.register_buffer(
            "pos_weight", pos_weight if pos_weight is not None else None
        )
        self.reduction = reduction

    def forward(self, logits, ordinal_targets):
        return F.binary_cross_entropy_with_logits(
            logits,
            ordinal_targets,
            pos_weight=self.pos_weight,
            reduction=self.reduction,
        )


class MultiTaskBERLoss(nn.Module):
    def __init__(self, reg_loss, ord_loss, lambda_ord=0.25):
        super().__init__()
        self.reg_loss = reg_loss
        self.ord_loss = ord_loss
        self.lambda_ord = lambda_ord

    def forward(self, pred_raw_unconstrained, ord_logits, target_log, target_ord):
        reg = self.reg_loss(pred_raw_unconstrained, target_log)
        ordl = self.ord_loss(ord_logits, target_ord)
        total = reg + self.lambda_ord * ordl
        return total, reg.detach(), ordl.detach()


# =========================
# Set Transformer Blocks
# =========================
class SetSelfAttentionBlock(nn.Module):
    def __init__(self, d_model, num_heads, mlp_ratio=4.0, dropout=0.1):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model)
        self.attn = nn.MultiheadAttention(
            embed_dim=d_model,
            num_heads=num_heads,
            dropout=dropout,
            batch_first=True,
        )
        self.norm2 = nn.LayerNorm(d_model)

        hidden = int(d_model * mlp_ratio)
        self.mlp = nn.Sequential(
            nn.Linear(d_model, hidden),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, d_model),
            nn.Dropout(dropout),
        )

    def forward(self, x, key_padding_mask=None):
        """
        Args:
            x: [B, L, D]
            key_padding_mask: [B, L] bool, True = padding (ignore)
        """
        y = self.norm1(x)
        attn_out, _ = self.attn(
            y, y, y,
            need_weights=False,
            key_padding_mask=key_padding_mask,
        )
        x = x + attn_out
        x = x + self.mlp(self.norm2(x))
        return x


class ConditionalFeatureModulation(nn.Module):
    def __init__(self, cond_dim, feat_dim, hidden_dim=256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(cond_dim, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, 2 * feat_dim),
        )
        self.scale = nn.Parameter(torch.tensor(0.25, dtype=torch.float32))

    def forward(self, x, cond):
        gamma_beta = self.net(cond)
        gamma, beta = torch.chunk(gamma_beta, 2, dim=-1)

        if x.dim() == 3:
            gamma = gamma.unsqueeze(1)
            beta = beta.unsqueeze(1)

        return x * (1.0 + self.scale * gamma) + self.scale * beta


class SetAttentionPooling(nn.Module):
    def __init__(self, d_model, hidden_dim=128):
        super().__init__()
        self.score = nn.Sequential(
            nn.Linear(d_model, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, 1),
        )

    def forward(self, x, key_padding_mask=None):
        """
        Args:
            x: [B, L, D]
            key_padding_mask: [B, L] bool, True = padding
        Returns:
            pooled: [B, D]
        """
        logits = self.score(x)  # [B, L, 1]

        if key_padding_mask is not None:
            logits = logits.masked_fill(
                key_padding_mask.unsqueeze(-1), float("-inf")
            )

        weights = torch.softmax(logits, dim=1)  # [B, L, 1]
        # Safety: all-masked rows produce NaN from softmax(-inf); replace with 0
        weights = torch.nan_to_num(weights, nan=0.0)

        pooled = (weights * x).sum(dim=1)  # [B, D]
        return pooled


# =========================
# Monotonic Ordinal Head
# =========================
class MonotonicOrdinalHead(nn.Module):
    """Ordinal head with monotonic probabilities by construction.

    For ordered BER thresholds t_1 < ... < t_K, the ordinal targets are
    P(BER >= t_k). These probabilities must be non-increasing in k.
    This head enforces that by constructing non-increasing logits:

        logit_1 = free
        logit_k = logit_{k-1} - softplus(delta_k)

    Since sigmoid is monotone, sigmoid(logit_k) is also non-increasing.
    """
    def __init__(self, head_in, num_thresholds):
        super().__init__()
        self.num_thresholds = num_thresholds
        self.base = nn.Sequential(
            nn.Linear(head_in, 256),
            nn.GELU(),
            nn.Dropout(0.20),
            nn.Linear(256, 128),
            nn.GELU(),
            nn.Dropout(0.15),
        )
        self.first_logit = nn.Linear(128, 1)
        self.deltas = nn.Linear(128, max(num_thresholds - 1, 1))

    def forward(self, x):
        h = self.base(x)
        first = self.first_logit(h)
        if self.num_thresholds == 1:
            return first
        deltas = -F.softplus(self.deltas(h)[:, : self.num_thresholds - 1])
        return torch.cat([first, deltas], dim=-1).cumsum(dim=-1)


# =========================
# Model
# =========================
class BERFirstTokenSetTransformerBoundedLogHalfMultiTask(nn.Module):
    def __init__(
        self,
        max_past_seq_len,
        token_dim=4,
        first_token_dim=4,
        threshold_dim=1,
        global_dim=7,
        d_model=128,
        num_set_layers=4,
        num_heads=4,
        mlp_ratio=4.0,
        dropout=0.10,
        num_ordinal_thresholds=NUM_ORDINAL_THRESHOLDS,
    ):
        super().__init__()

        self.max_past_seq_len = max_past_seq_len
        self.token_dim = token_dim
        self.first_token_dim = first_token_dim
        self.d_model = d_model
        self.num_ordinal_thresholds = num_ordinal_thresholds

        self.first_proj = nn.Sequential(
            nn.Linear(first_token_dim, d_model),
            nn.LayerNorm(d_model),
            nn.GELU(),
            nn.Dropout(dropout),
        )

        self.token_proj = nn.Sequential(
            nn.Linear(token_dim, d_model),
            nn.LayerNorm(d_model),
            nn.GELU(),
            nn.Dropout(dropout),
        )

        self.thr_embed = nn.Sequential(
            nn.Linear(threshold_dim, 32),
            nn.GELU(),
            nn.Linear(32, 32),
            nn.GELU(),
        )

        self.global_embed = nn.Sequential(
            nn.Linear(global_dim, 96),
            nn.GELU(),
            nn.Linear(96, 96),
            nn.GELU(),
        )

        cond_dim = 32 + 96

        self.first_cond_mod = ConditionalFeatureModulation(
            cond_dim=cond_dim, feat_dim=d_model, hidden_dim=256,
        )
        self.set_cond_mod = ConditionalFeatureModulation(
            cond_dim=cond_dim, feat_dim=d_model, hidden_dim=256,
        )

        self.set_blocks = nn.ModuleList(
            [
                SetSelfAttentionBlock(
                    d_model=d_model,
                    num_heads=num_heads,
                    mlp_ratio=mlp_ratio,
                    dropout=dropout,
                )
                for _ in range(num_set_layers)
            ]
        )

        self.first_refine = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, d_model),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model, d_model),
        )

        self.final_set_norm = nn.LayerNorm(d_model)
        self.set_attn_pool = SetAttentionPooling(d_model=d_model, hidden_dim=128)

        set_summary_dim = 3 * d_model
        head_in = d_model + set_summary_dim + cond_dim

        self.reg_head = nn.Sequential(
            nn.Linear(head_in, 256),
            nn.GELU(),
            nn.Dropout(0.20),
            nn.Linear(256, 128),
            nn.GELU(),
            nn.Dropout(0.15),
            nn.Linear(128, 1),
        )

        self.ord_head = MonotonicOrdinalHead(
            head_in=head_in,
            num_thresholds=num_ordinal_thresholds,
        )

    @staticmethod
    def raw_to_log10ber(pred_raw_unconstrained):
        log10_half = torch.log10(
            torch.tensor(
                0.5,
                device=pred_raw_unconstrained.device,
                dtype=pred_raw_unconstrained.dtype,
            )
        )
        return log10_half - F.softplus(pred_raw_unconstrained)

    @staticmethod
    def raw_to_ber(pred_raw_unconstrained):
        pred_log = BERFirstTokenSetTransformerBoundedLogHalfMultiTask.raw_to_log10ber(
            pred_raw_unconstrained
        )
        return torch.pow(10.0, pred_log).clamp(min=EPS, max=0.5)

    def forward(self, first_token, past_tokens, global_feats, threshold, key_padding_mask=None):
        """
        Args:
            first_token:       [B, first_token_dim]
            past_tokens:       [B, L, token_dim]  (L can vary between training and inference)
            global_feats:      [B, global_dim]
            threshold:         [B, 1]
            key_padding_mask:  [B, L] bool, True = padding position
        """
        thr = self.thr_embed(threshold)
        g = self.global_embed(global_feats)
        cond = torch.cat([thr, g], dim=-1)

        # First token path
        first_x = self.first_proj(first_token)
        first_x = self.first_cond_mod(first_x, cond)
        first_x = first_x + self.first_refine(first_x)

        # Past tokens path with mask
        set_x = self.token_proj(past_tokens)
        set_x = self.set_cond_mod(set_x, cond)

        for block in self.set_blocks:
            set_x = block(set_x, key_padding_mask=key_padding_mask)

        set_x = self.final_set_norm(set_x)

        # --- Masked pooling ---
        if key_padding_mask is not None:
            valid_mask = (~key_padding_mask).unsqueeze(-1).float()  # [B, L, 1]
        else:
            valid_mask = torch.ones(
                set_x.shape[0], set_x.shape[1], 1,
                device=set_x.device, dtype=set_x.dtype,
            )

        # Attention pooling
        pooled_attn = self.set_attn_pool(set_x, key_padding_mask=key_padding_mask)

        # Masked mean pooling
        valid_counts = valid_mask.sum(dim=1).clamp(min=1.0)  # [B, 1]
        pooled_mean = (set_x * valid_mask).sum(dim=1) / valid_counts

        # Masked max pooling
        if key_padding_mask is not None:
            x_for_max = set_x.masked_fill(key_padding_mask.unsqueeze(-1), float("-inf"))
        else:
            x_for_max = set_x
        pooled_max = x_for_max.amax(dim=1)
        pooled_max = torch.nan_to_num(pooled_max, nan=0.0, posinf=0.0, neginf=0.0)

        set_summary = torch.cat([pooled_attn, pooled_mean, pooled_max], dim=-1)
        fused = torch.cat([first_x, set_summary, cond], dim=-1)

        pred_raw_unconstrained = self.reg_head(fused)
        ord_logits = self.ord_head(fused)

        return pred_raw_unconstrained, ord_logits


# =========================
# Data
# =========================
def prepare_data(csv_paths, batch_size=256, nrows_per_dataset=2_500_000, num_workers=0):
    df, tap_cols, var_cols = load_and_merge_data(csv_paths, nrows_per_dataset)

    df = df[df["mem_len"] != 1].copy()

    df[tap_cols] = df[tap_cols].fillna(0.0)
    df[var_cols] = df[var_cols].fillna(0.0)
    df["threshold"] = df["threshold"].fillna(0.0)
    df["BER"] = df["BER"].fillna(0.0)
    df["N"] = df["N"].fillna(0.0)

    df = df[(df["threshold"] > 0) & (df["BER"] > 0) & (df["N"] > 0)].copy()
    df["BER"] = df["BER"].clip(lower=EPS, upper=0.5)

    print(f"Rows after cleaning/filtering: {len(df):,}")

    # ---- Extract raw arrays ----
    X_taps_raw = df[tap_cols].to_numpy(dtype=np.float32)
    X_vars_raw = df[var_cols].to_numpy(dtype=np.float32)
    num_molecules = df["N"].to_numpy(dtype=np.float32).reshape(-1, 1)
    mem_len = df["mem_len"].to_numpy(dtype=np.int64)

    X_thr_raw = df["threshold"].to_numpy(dtype=np.float32).reshape(-1, 1)
    y_raw = df["BER"].to_numpy(dtype=np.float32).reshape(-1, 1)

    # X_thr is the model threshold feature. It is computed below after
    # valid-memory masking because normalized threshold needs sum(valid tap_i * N).
    y_log = np.log10(y_raw + EPS).astype(np.float32)

    # ---- Feature engineering ----
    X_taps_feat = (X_taps_raw * num_molecules).astype(np.float32)

    if np.any(X_vars_raw < 0):
        raise ValueError("Variance columns contain negative values.")

    X_vars_feat = X_vars_raw.astype(np.float32)
    abs_taps_raw = np.abs(X_taps_feat).astype(np.float32)
    snr_raw = np.log10((X_taps_feat ** 2) / (X_vars_raw + EPS) + EPS).astype(np.float32)

    L_full = X_taps_feat.shape[1]

    # ---- Validity masks ----
    mem_len_clamped = np.minimum(mem_len, L_full)
    valid_full = (np.arange(L_full)[None, :] < mem_len_clamped[:, None])  # [N, L_full]
    valid_full_float = valid_full.astype(np.float32)
    valid_past = valid_full[:, 1:]  # [N, L_full-1]
    valid_past_float = valid_past.astype(np.float32)
    past_lens = np.maximum(mem_len_clamped - 1, 0).astype(np.int64)

    # Scenario-normalized threshold feature. This replaces raw log10(threshold)
    # as the model threshold input. It makes threshold sweeps comparable across N.
    total_scaled_taps_raw = np.sum(
        X_taps_feat * valid_full_float,
        axis=1,
        keepdims=True,
    ).astype(np.float32)
    X_thr = compute_threshold_feature_np(
        X_thr_raw,
        total_scaled_taps_raw,
        mode=THRESHOLD_FEATURE_MODE,
    )

    # ---- Mask-aware global features ----
    first_mean = X_taps_feat[:, 0:1]
    past_means = X_taps_feat[:, 1:]
    first_var = X_vars_feat[:, 0:1]
    past_vars = X_vars_feat[:, 1:]

    past_means_masked = past_means * valid_past_float
    past_vars_masked = past_vars * valid_past_float

    mu0_raw = (0.5 * np.sum(past_means_masked, axis=1, keepdims=True)).astype(np.float32)
    mu1_raw = (first_mean + mu0_raw).astype(np.float32)

    var0_raw = (0.5 * np.sum(past_vars_masked, axis=1, keepdims=True)).astype(np.float32)
    var1_raw = (first_var + var0_raw).astype(np.float32)

    std0_raw = np.sqrt(np.maximum(var0_raw, EPS)).astype(np.float32)
    std1_raw = np.sqrt(np.maximum(var1_raw, EPS)).astype(np.float32)

    z0_raw = ((X_thr_raw - mu0_raw) / (std0_raw + EPS)).astype(np.float32)
    z1_raw = ((mu1_raw - X_thr_raw) / (std1_raw + EPS)).astype(np.float32)

    harmonic_side_z_raw = stable_harmonic_pair_np(z0_raw, z1_raw)
    abs_diff_side_z_raw = np.abs(z0_raw - z1_raw).astype(np.float32)
    harmonic_minus_gap_raw = (
        harmonic_side_z_raw - 0.25 * abs_diff_side_z_raw
    ).astype(np.float32)
    harmonic_minus_gap_raw = sanitize_np_feature(
        harmonic_minus_gap_raw,
        "harmonic_minus_gap_raw",
    )

    # ---- Scenario-level features (threshold-INDEPENDENT) ----
    # These capture intrinsic difficulty of the physical scenario.

    # 1. NSID: Normalized Signal-Interference Difference, range ~ [-1, +1]
    signal_raw = first_mean  # P_0 * N, shape [N, 1]
    isi_raw = np.sum(past_means_masked, axis=1, keepdims=True)  # [N, 1]
    nsid_raw = ((signal_raw - isi_raw) / (signal_raw + isi_raw + EPS)).astype(np.float32)

    # 2. Log Harmonic Discriminability (threshold-free)
    gap_raw = signal_raw  # mu1 - mu0 = P_0 * N
    d0_raw = (gap_raw / (std0_raw + EPS)).astype(np.float32)
    d1_raw = (gap_raw / (std1_raw + EPS)).astype(np.float32)
    harmonic_discrim_raw = (2.0 * d0_raw * d1_raw / (d0_raw + d1_raw + EPS)).astype(np.float32)
    log_harmonic_discrim_raw = np.log10(harmonic_discrim_raw + EPS).astype(np.float32)

    # 3. Discriminability Asymmetry
    discrim_asymmetry_raw = (np.abs(d0_raw - d1_raw) / (d0_raw + d1_raw + EPS)).astype(np.float32)

    # 4. Herfindahl Index of ISI concentration [1/n_past, 1]
    past_taps_sum = np.sum(past_means_masked, axis=1, keepdims=True)  # [N, 1]
    past_shares = past_means_masked / (past_taps_sum + EPS)           # [N, L_past]
    herfindahl_raw = np.sum(past_shares ** 2, axis=1, keepdims=True).astype(np.float32)  # [N, 1]

    # ---- Labels ----
    region_labels = raw_to_region_labels_np(y_raw)
    ordinal_targets = region_labels_to_ordinal_targets_np(region_labels)
    strat_labels = make_strat_bins(y_log, n_bins=10)

    # ---- Train / temp split ----
    split_args = dict(test_size=0.30, random_state=42)
    if strat_labels is not None:
        split_args["stratify"] = strat_labels

    (
        t_taps_feat, temp_taps_feat,
        t_vars_feat, temp_vars_feat,
        t_abs_raw, temp_abs_raw,
        t_snr_raw, temp_snr_raw,
        t_z0_raw, temp_z0_raw,
        t_z1_raw, temp_z1_raw,
        t_hmg_raw, temp_hmg_raw,
        t_nsid_raw, temp_nsid_raw,
        t_log_hd_raw, temp_log_hd_raw,
        t_da_raw, temp_da_raw,
        t_herf_raw, temp_herf_raw,
        t_thr, temp_thr,
        t_y_log, temp_y_log,
        t_region, temp_region,
        t_ord, temp_ord,
        t_past_lens, temp_past_lens,
    ) = train_test_split(
        X_taps_feat, X_vars_feat, abs_taps_raw, snr_raw,
        z0_raw, z1_raw, harmonic_minus_gap_raw,
        nsid_raw, log_harmonic_discrim_raw, discrim_asymmetry_raw, herfindahl_raw,
        X_thr,
        y_log, region_labels, ordinal_targets, past_lens,
        **split_args,
    )

    # ---- temp -> val / test split ----
    temp_strat = make_strat_bins(temp_y_log, n_bins=6)
    split_args2 = dict(test_size=0.50, random_state=42)
    if temp_strat is not None:
        split_args2["stratify"] = temp_strat

    (
        v_taps_feat, te_taps_feat,
        v_vars_feat, te_vars_feat,
        v_abs_raw, te_abs_raw,
        v_snr_raw, te_snr_raw,
        v_z0_raw, te_z0_raw,
        v_z1_raw, te_z1_raw,
        v_hmg_raw, te_hmg_raw,
        v_nsid_raw, te_nsid_raw,
        v_log_hd_raw, te_log_hd_raw,
        v_da_raw, te_da_raw,
        v_herf_raw, te_herf_raw,
        v_thr, te_thr,
        v_y_log, te_y_log,
        v_region, te_region,
        v_ord, te_ord,
        v_past_lens, te_past_lens,
    ) = train_test_split(
        temp_taps_feat, temp_vars_feat, temp_abs_raw, temp_snr_raw,
        temp_z0_raw, temp_z1_raw, temp_hmg_raw,
        temp_nsid_raw, temp_log_hd_raw, temp_da_raw, temp_herf_raw,
        temp_thr,
        temp_y_log, temp_region, temp_ord, temp_past_lens,
        **split_args2,
    )

    # =========================================================
    # Position-independent scaling
    # =========================================================
    # For each feature channel (taps, vars, abs, snr) we fit:
    #   - A SEPARATE mean/std for position 0 (first token)
    #   - A SHARED mean/std pooled across ALL valid past positions 1+
    #
    # This decouples the scaler from the number of columns,
    # so at inference any mem_len works with the same parameters.
    # =========================================================

    L_past = L_full - 1

    # Build per-split past validity masks
    t_valid_past = (np.arange(L_past)[None, :] < t_past_lens[:, None])

    # --- Fit first-token scalers (on train only) ---
    first_tap_mean, first_tap_std = float(t_taps_feat[:, 0].mean()), max(float(t_taps_feat[:, 0].std()), 1e-12)
    first_var_mean, first_var_std = float(t_vars_feat[:, 0].mean()), max(float(t_vars_feat[:, 0].std()), 1e-12)
    first_abs_mean, first_abs_std = float(t_abs_raw[:, 0].mean()), max(float(t_abs_raw[:, 0].std()), 1e-12)
    first_snr_mean, first_snr_std = float(t_snr_raw[:, 0].mean()), max(float(t_snr_raw[:, 0].std()), 1e-12)

    # --- Fit shared past scalers (pool all valid past entries from train) ---
    past_tap_mean, past_tap_std = fit_shared_scalar_scaler(t_taps_feat[:, 1:], t_valid_past)
    past_var_mean, past_var_std = fit_shared_scalar_scaler(t_vars_feat[:, 1:], t_valid_past)
    past_abs_mean, past_abs_std = fit_shared_scalar_scaler(t_abs_raw[:, 1:], t_valid_past)
    past_snr_mean, past_snr_std = fit_shared_scalar_scaler(t_snr_raw[:, 1:], t_valid_past)

    # --- Global / threshold feature scalers (standard, shape [N, 1]) ---
    z0_scaler = StandardScaler().fit(t_z0_raw)
    z1_scaler = StandardScaler().fit(t_z1_raw)
    hmg_scaler = StandardScaler().fit(t_hmg_raw)
    nsid_scaler = StandardScaler().fit(t_nsid_raw)
    log_hd_scaler = StandardScaler().fit(t_log_hd_raw)
    da_scaler = StandardScaler().fit(t_da_raw)
    herf_scaler = StandardScaler().fit(t_herf_raw)
    thr_scaler = StandardScaler().fit(t_thr)

    # ---- Helper: build first_token and past_tokens arrays ----
    def build_tokens(taps_feat, vars_feat, abs_raw, snr_raw, p_lens):
        """Returns first_token [N, 4] and past_tokens [N, L_past, 4], both scaled + re-zeroed."""
        N = taps_feat.shape[0]

        # Scale first token
        ft_tap = ((taps_feat[:, 0] - first_tap_mean) / first_tap_std).astype(np.float32)
        ft_var = ((vars_feat[:, 0] - first_var_mean) / first_var_std).astype(np.float32)
        ft_abs = ((abs_raw[:, 0] - first_abs_mean) / first_abs_std).astype(np.float32)
        ft_snr = ((snr_raw[:, 0] - first_snr_mean) / first_snr_std).astype(np.float32)
        first_token = np.stack([ft_tap, ft_var, ft_abs, ft_snr], axis=1).astype(np.float32)

        # Scale past tokens (shared scaler) + re-zero
        pt_tap = apply_shared_scale(taps_feat[:, 1:], past_tap_mean, past_tap_std,
                                    (np.arange(L_past)[None, :] < p_lens[:, None]))
        pt_var = apply_shared_scale(vars_feat[:, 1:], past_var_mean, past_var_std,
                                    (np.arange(L_past)[None, :] < p_lens[:, None]))
        pt_abs = apply_shared_scale(abs_raw[:, 1:], past_abs_mean, past_abs_std,
                                    (np.arange(L_past)[None, :] < p_lens[:, None]))
        pt_snr = apply_shared_scale(snr_raw[:, 1:], past_snr_mean, past_snr_std,
                                    (np.arange(L_past)[None, :] < p_lens[:, None]))

        past_tokens = np.stack([pt_tap, pt_var, pt_abs, pt_snr], axis=2).astype(np.float32)

        return first_token, past_tokens

    def build_global(z0, z1, hmg, nsid, log_hd, da, herf):
        return np.concatenate([
            z0_scaler.transform(z0),
            z1_scaler.transform(z1),
            hmg_scaler.transform(hmg),
            nsid_scaler.transform(nsid),
            log_hd_scaler.transform(log_hd),
            da_scaler.transform(da),
            herf_scaler.transform(herf),
        ], axis=1).astype(np.float32)

    def build_padding_mask(p_lens, L_max):
        return (np.arange(L_max)[None, :] >= p_lens[:, None])  # True = padding

    # ---- Build all splits ----
    t_first, t_past = build_tokens(t_taps_feat, t_vars_feat, t_abs_raw, t_snr_raw, t_past_lens)
    v_first, v_past = build_tokens(v_taps_feat, v_vars_feat, v_abs_raw, v_snr_raw, v_past_lens)
    te_first, te_past = build_tokens(te_taps_feat, te_vars_feat, te_abs_raw, te_snr_raw, te_past_lens)

    t_global = build_global(t_z0_raw, t_z1_raw, t_hmg_raw,
                            t_nsid_raw, t_log_hd_raw, t_da_raw, t_herf_raw)
    v_global = build_global(v_z0_raw, v_z1_raw, v_hmg_raw,
                            v_nsid_raw, v_log_hd_raw, v_da_raw, v_herf_raw)
    te_global = build_global(te_z0_raw, te_z1_raw, te_hmg_raw,
                             te_nsid_raw, te_log_hd_raw, te_da_raw, te_herf_raw)

    t_thr_s = thr_scaler.transform(t_thr).astype(np.float32)
    v_thr_s = thr_scaler.transform(v_thr).astype(np.float32)
    te_thr_s = thr_scaler.transform(te_thr).astype(np.float32)

    L_max_past = L_past
    t_mask = build_padding_mask(t_past_lens, L_max_past)
    v_mask = build_padding_mask(v_past_lens, L_max_past)
    te_mask = build_padding_mask(te_past_lens, L_max_past)

    # ---- TensorDatasets ----
    train_ds = TensorDataset(
        torch.from_numpy(t_first),
        torch.from_numpy(t_past),
        torch.from_numpy(t_global),
        torch.from_numpy(t_thr_s),
        torch.from_numpy(t_y_log.astype(np.float32)),
        torch.from_numpy(t_ord.astype(np.float32)),
        torch.from_numpy(t_region.astype(np.int64)),
        torch.from_numpy(t_mask),
    )
    val_ds = TensorDataset(
        torch.from_numpy(v_first),
        torch.from_numpy(v_past),
        torch.from_numpy(v_global),
        torch.from_numpy(v_thr_s),
        torch.from_numpy(v_y_log.astype(np.float32)),
        torch.from_numpy(v_ord.astype(np.float32)),
        torch.from_numpy(v_region.astype(np.int64)),
        torch.from_numpy(v_mask),
    )
    test_ds = TensorDataset(
        torch.from_numpy(te_first),
        torch.from_numpy(te_past),
        torch.from_numpy(te_global),
        torch.from_numpy(te_thr_s),
        torch.from_numpy(te_y_log.astype(np.float32)),
        torch.from_numpy(te_ord.astype(np.float32)),
        torch.from_numpy(te_region.astype(np.int64)),
        torch.from_numpy(te_mask),
    )

    # ---- Weighted sampler (region + SIR-aware) ----
    region_sample_weights = np.ones_like(t_region, dtype=np.float32)
    region_sample_weights[t_region == 6] = 2.5
    region_sample_weights[t_region == 7] = 3.0
    region_sample_weights[t_region == 8] = 5.0
    region_sample_weights[t_region == 9] = 5.0
    region_sample_weights[t_region == 10] = 5.0
    region_sample_weights[t_region == 11] = 5.0
    region_sample_weights[t_region == 12] = 3.0
    region_sample_weights[t_region == 13] = 2.5

    # SIR-aware weights (threshold-independent scenario difficulty)
    t_signal = t_taps_feat[:, 0]
    t_valid_past_for_sir = (np.arange(L_past)[None, :] < t_past_lens[:, None]).astype(np.float32)
    t_isi = np.sum(t_taps_feat[:, 1:] * t_valid_past_for_sir, axis=1)
    t_sir = t_signal / (t_isi + EPS)

    sir_sample_weights = np.ones_like(t_sir, dtype=np.float32)
    sir_sample_weights[t_sir < 0.26] = 3.0
    sir_sample_weights[(t_sir >= 0.26) & (t_sir < 0.36)] = 2.0

    combined_sample_weights = region_sample_weights * sir_sample_weights

    sampler = WeightedRandomSampler(
        weights=torch.from_numpy(combined_sample_weights),
        num_samples=len(combined_sample_weights),
        replacement=True,
    )

    pin_mem = torch.cuda.is_available()

    train_loader = DataLoader(train_ds, batch_size=batch_size, sampler=sampler,
                              shuffle=False, pin_memory=pin_mem, num_workers=num_workers)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False,
                            pin_memory=pin_mem, num_workers=num_workers)
    test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False,
                             pin_memory=pin_mem, num_workers=num_workers)

    class_weights = compute_region_class_weights(t_region, NUM_REGION_CLASSES)
    ordinal_pos_weights = compute_ordinal_pos_weights_from_region_labels(
        t_region, num_thresholds=NUM_ORDINAL_THRESHOLDS, max_weight=20.0,
    )

    # ---- Save all scaler parameters (position-independent) ----
    scalers = {
        # First-token scalers (per feature channel)
        "first_tap_mean": first_tap_mean, "first_tap_std": first_tap_std,
        "first_var_mean": first_var_mean, "first_var_std": first_var_std,
        "first_abs_mean": first_abs_mean, "first_abs_std": first_abs_std,
        "first_snr_mean": first_snr_mean, "first_snr_std": first_snr_std,
        # Shared past scalers (single mean/std per feature, position-independent)
        "past_tap_mean": past_tap_mean, "past_tap_std": past_tap_std,
        "past_var_mean": past_var_mean, "past_var_std": past_var_std,
        "past_abs_mean": past_abs_mean, "past_abs_std": past_abs_std,
        "past_snr_mean": past_snr_mean, "past_snr_std": past_snr_std,
        # Global feature scalers (sklearn StandardScaler objects)
        "z0_scaler": z0_scaler,
        "z1_scaler": z1_scaler,
        "harmonic_minus_gap_scaler": hmg_scaler,
        "nsid_scaler": nsid_scaler,
        "log_hd_scaler": log_hd_scaler,
        "da_scaler": da_scaler,
        "herf_scaler": herf_scaler,
        "thr_scaler": thr_scaler,
        "threshold_feature_mode": THRESHOLD_FEATURE_MODE,
        "threshold_feature_description": (
            "threshold branch input is StandardScaler(threshold / sum(valid tap_i*N)) "
            "or its logit, depending on threshold_feature_mode"
        ),
        "threshold_norm_clip_eps": THRESHOLD_NORM_CLIP_EPS,
        # Metadata
        "tap_cols": tap_cols,
        "var_cols": var_cols,
        "first_token_dim": 4,
        "past_token_dim": 4,
        "global_dim": 7,
        "train_max_past_seq_len": L_past,
        "scaling_strategy": "position_independent",
        "uses_positional_encoding": False,
        "permutation_invariance_post_first": True,
        "variable_length_support": True,
        "target_parameterization": "pred_log10_ber = log10(0.5) - softplus(raw_out)",
        "ordinal_thresholds": ORDINAL_THRESHOLDS,
        "num_region_classes": NUM_REGION_CLASSES,
        "class_weights": class_weights.tolist(),
        "ordinal_pos_weights": ordinal_pos_weights.tolist(),
        "data_paths": csv_paths,
        "nrows_per_dataset": nrows_per_dataset,
    }

    aux_info = {
        "class_weights": class_weights,
        "ordinal_pos_weights": ordinal_pos_weights,
        "t_region": t_region,
    }

    return train_loader, val_loader, test_loader, scalers, aux_info, L_past


# =========================
# Inference Helper
# =========================
def prepare_inference_batch(
    taps_raw_2d,
    vars_raw_2d,
    N_array,
    threshold_array,
    mem_len_array,
    scalers,
):
    """Prepare a batch for inference from raw numpy arrays.

    This handles ARBITRARY mem_len — the set can be larger than anything
    seen during training because scalers are position-independent.

    Args:
        taps_raw_2d:     [B, L] raw tap coefficients (padded to L with zeros)
        vars_raw_2d:     [B, L] raw variance values  (padded to L with zeros)
        N_array:         [B] or [B, 1] number of molecules
        threshold_array: [B] or [B, 1] raw threshold values
        mem_len_array:   [B] true memory length per sample
        scalers:         dict from joblib.load(SCALER_SAVE_PATH)

    Returns:
        first_token:      [B, 4] tensor
        past_tokens:      [B, L-1, 4] tensor
        global_feats:     [B, 7] tensor
        threshold_scaled: [B, 1] tensor
        key_padding_mask: [B, L-1] bool tensor (True = padding)
    """
    B, L = taps_raw_2d.shape
    N = N_array.reshape(-1, 1).astype(np.float32)
    thr_raw = threshold_array.reshape(-1, 1).astype(np.float32)
    mem_len = mem_len_array.reshape(-1).astype(np.int64)

    mem_len_clamped = np.minimum(mem_len, L)
    past_lens = np.maximum(mem_len_clamped - 1, 0).astype(np.int64)

    # Feature engineering (same as training)
    taps_feat = (taps_raw_2d * N).astype(np.float32)
    vars_feat = vars_raw_2d.astype(np.float32)
    abs_feat = np.abs(taps_feat).astype(np.float32)
    snr_feat = np.log10((taps_feat ** 2) / (vars_raw_2d + EPS) + EPS).astype(np.float32)

    L_past = L - 1

    # Validity masks
    valid_full = (np.arange(L)[None, :] < mem_len_clamped[:, None])
    valid_full_float = valid_full.astype(np.float32)
    valid_past = (np.arange(L_past)[None, :] < past_lens[:, None])
    valid_past_float = valid_past.astype(np.float32)

    # Global features (mask-aware)
    first_mean = taps_feat[:, 0:1]
    past_means = taps_feat[:, 1:] * valid_past_float
    first_var = vars_feat[:, 0:1]
    past_vars = vars_feat[:, 1:] * valid_past_float

    mu0 = (0.5 * past_means.sum(axis=1, keepdims=True)).astype(np.float32)
    mu1 = (first_mean + mu0).astype(np.float32)
    var0 = (0.5 * past_vars.sum(axis=1, keepdims=True)).astype(np.float32)
    var1 = (first_var + var0).astype(np.float32)

    std0 = np.sqrt(np.maximum(var0, EPS)).astype(np.float32)
    std1 = np.sqrt(np.maximum(var1, EPS)).astype(np.float32)

    z0 = ((thr_raw - mu0) / (std0 + EPS)).astype(np.float32)
    z1 = ((mu1 - thr_raw) / (std1 + EPS)).astype(np.float32)

    harmonic = stable_harmonic_pair_np(z0, z1)
    abs_diff = np.abs(z0 - z1).astype(np.float32)
    hmg = (harmonic - 0.25 * abs_diff).astype(np.float32)
    hmg = sanitize_np_feature(hmg, "inference_harmonic_minus_gap")

    # Scenario-level features (threshold-independent)
    signal = first_mean  # [B, 1]
    isi = past_means.sum(axis=1, keepdims=True)  # [B, 1]
    nsid = ((signal - isi) / (signal + isi + EPS)).astype(np.float32)

    gap = signal
    d0_inf = (gap / (std0 + EPS)).astype(np.float32)
    d1_inf = (gap / (std1 + EPS)).astype(np.float32)
    hd = (2.0 * d0_inf * d1_inf / (d0_inf + d1_inf + EPS)).astype(np.float32)
    log_hd = np.log10(hd + EPS).astype(np.float32)
    da = (np.abs(d0_inf - d1_inf) / (d0_inf + d1_inf + EPS)).astype(np.float32)

    past_taps_sum = past_means.sum(axis=1, keepdims=True)
    past_shares = past_means / (past_taps_sum + EPS)
    herf = (past_shares ** 2).sum(axis=1, keepdims=True).astype(np.float32)

    # Scale first token
    ft_tap = ((taps_feat[:, 0] - scalers["first_tap_mean"]) / scalers["first_tap_std"]).astype(np.float32)
    ft_var = ((vars_feat[:, 0] - scalers["first_var_mean"]) / scalers["first_var_std"]).astype(np.float32)
    ft_abs = ((abs_feat[:, 0] - scalers["first_abs_mean"]) / scalers["first_abs_std"]).astype(np.float32)
    ft_snr = ((snr_feat[:, 0] - scalers["first_snr_mean"]) / scalers["first_snr_std"]).astype(np.float32)
    first_token = np.stack([ft_tap, ft_var, ft_abs, ft_snr], axis=1).astype(np.float32)

    # Scale past tokens (position-independent shared scaler)
    pt_tap = apply_shared_scale(taps_feat[:, 1:], scalers["past_tap_mean"], scalers["past_tap_std"], valid_past)
    pt_var = apply_shared_scale(vars_feat[:, 1:], scalers["past_var_mean"], scalers["past_var_std"], valid_past)
    pt_abs = apply_shared_scale(abs_feat[:, 1:], scalers["past_abs_mean"], scalers["past_abs_std"], valid_past)
    pt_snr = apply_shared_scale(snr_feat[:, 1:], scalers["past_snr_mean"], scalers["past_snr_std"], valid_past)
    past_tokens = np.stack([pt_tap, pt_var, pt_abs, pt_snr], axis=2).astype(np.float32)

    # Scale globals (7 features)
    z0_s = scalers["z0_scaler"].transform(z0).astype(np.float32)
    z1_s = scalers["z1_scaler"].transform(z1).astype(np.float32)
    hmg_s = scalers["harmonic_minus_gap_scaler"].transform(hmg).astype(np.float32)
    nsid_s = scalers["nsid_scaler"].transform(nsid).astype(np.float32)
    log_hd_s = scalers["log_hd_scaler"].transform(log_hd).astype(np.float32)
    da_s = scalers["da_scaler"].transform(da).astype(np.float32)
    herf_s = scalers["herf_scaler"].transform(herf).astype(np.float32)
    global_feats = np.concatenate([z0_s, z1_s, hmg_s, nsid_s, log_hd_s, da_s, herf_s], axis=1).astype(np.float32)

    # Scale threshold using the same scenario-normalized feature as training.
    total_scaled_taps = np.sum(
        taps_feat * valid_full_float,
        axis=1,
        keepdims=True,
    ).astype(np.float32)
    threshold_mode = scalers.get("threshold_feature_mode", THRESHOLD_FEATURE_MODE)
    thr_feature = compute_threshold_feature_np(
        thr_raw,
        total_scaled_taps,
        mode=threshold_mode,
    )
    thr_s = scalers["thr_scaler"].transform(thr_feature).astype(np.float32)

    # Padding mask
    pad_mask = (np.arange(L_past)[None, :] >= past_lens[:, None])

    return (
        torch.from_numpy(first_token),
        torch.from_numpy(past_tokens),
        torch.from_numpy(global_feats),
        torch.from_numpy(thr_s),
        torch.from_numpy(pad_mask),
    )


def run_inference(model, taps_raw, vars_raw, N_arr, thr_arr, mem_len_arr, scalers, device):
    """End-to-end inference: raw arrays -> BER predictions.

    All inputs are numpy. Works with any mem_len, even values
    larger than max_past_seq_len seen during training.
    """
    first_tok, past_tok, glob, thr_s, pad_mask = prepare_inference_batch(
        taps_raw, vars_raw, N_arr, thr_arr, mem_len_arr, scalers,
    )

    model.eval()
    with torch.no_grad():
        first_tok = first_tok.to(device)
        past_tok = past_tok.to(device)
        glob = glob.to(device)
        thr_s = thr_s.to(device)
        pad_mask = pad_mask.to(device)

        pred_raw, ord_logits = model(first_tok, past_tok, glob, thr_s, key_padding_mask=pad_mask)

        pred_ber = model.raw_to_ber(pred_raw).cpu().numpy()
        pred_log = model.raw_to_log10ber(pred_raw).cpu().numpy()
        pred_region = ordinal_logits_to_region_labels_torch(ord_logits).cpu().numpy()

    return {
        "ber": pred_ber,
        "log10_ber": pred_log,
        "region": pred_region,
    }


# =========================
# Evaluation
# =========================
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0
    total_reg_loss = 0.0
    total_ord_loss = 0.0

    all_preds_log = []
    all_targets_log = []
    all_pred_regions = []
    all_true_regions = []

    with torch.no_grad():
        for b_first, b_past, b_global, b_thr, b_y_log, b_ord, b_region, b_mask in loader:
            b_first = b_first.to(device, non_blocking=True)
            b_past = b_past.to(device, non_blocking=True)
            b_global = b_global.to(device, non_blocking=True)
            b_thr = b_thr.to(device, non_blocking=True)
            b_y_log = b_y_log.to(device, non_blocking=True)
            b_ord = b_ord.to(device, non_blocking=True)
            b_region = b_region.to(device, non_blocking=True)
            b_mask = b_mask.to(device, non_blocking=True)

            pred_raw_unconstrained, ord_logits = model(
                b_first, b_past, b_global, b_thr, key_padding_mask=b_mask
            )

            if has_nonfinite_tensor(pred_raw_unconstrained):
                raise RuntimeError("Non-finite prediction during evaluation.")
            if has_nonfinite_tensor(ord_logits):
                raise RuntimeError("Non-finite ordinal logits during evaluation.")

            loss, reg_loss, ord_loss = criterion(
                pred_raw_unconstrained, ord_logits, b_y_log, b_ord,
            )

            if has_nonfinite_tensor(loss):
                raise RuntimeError("Non-finite loss during evaluation.")

            pred_log = model.raw_to_log10ber(pred_raw_unconstrained)
            pred_region = ordinal_logits_to_region_labels_torch(ord_logits)

            total_loss += loss.item()
            total_reg_loss += reg_loss.item()
            total_ord_loss += ord_loss.item()

            all_preds_log.append(pred_log.cpu().numpy())
            all_targets_log.append(b_y_log.cpu().numpy())
            all_pred_regions.append(pred_region.cpu().numpy())
            all_true_regions.append(b_region.cpu().numpy())

    n_batches = max(len(loader), 1)
    avg_loss = total_loss / n_batches
    avg_reg_loss = total_reg_loss / n_batches
    avg_ord_loss = total_ord_loss / n_batches

    preds_log = np.vstack(all_preds_log)
    targets_log = np.vstack(all_targets_log)

    preds_raw = np.clip(10 ** preds_log, EPS, 0.5)
    targets_raw = np.clip(10 ** targets_log, EPS, 0.5)

    rmse_log = float(np.sqrt(np.mean((preds_log - targets_log) ** 2)))
    mae_log = float(np.mean(np.abs(preds_log - targets_log)))
    factor_error = float(10 ** rmse_log)

    rmse_raw = float(np.sqrt(np.mean((preds_raw - targets_raw) ** 2)))
    mae_raw = float(np.mean(np.abs(preds_raw - targets_raw)))

    rel_err = (preds_raw - targets_raw) / np.maximum(targets_raw, 1e-6)
    rmse_rel = float(np.sqrt(np.mean(rel_err ** 2)))
    mae_rel = float(np.mean(np.abs(rel_err)))

    pred_regions = np.concatenate(all_pred_regions).reshape(-1)
    true_regions = np.concatenate(all_true_regions).reshape(-1)

    region_acc = float(np.mean(pred_regions == true_regions))
    region_mae = float(np.mean(np.abs(pred_regions - true_regions)))

    return {
        "loss": avg_loss,
        "reg_loss": avg_reg_loss,
        "ord_loss": avg_ord_loss,
        "rmse_log": rmse_log,
        "mae_log": mae_log,
        "factor_error": factor_error,
        "rmse_raw": rmse_raw,
        "mae_raw": mae_raw,
        "rmse_rel": rmse_rel,
        "mae_rel": mae_rel,
        "region_acc": region_acc,
        "region_mae": region_mae,
    }


def evaluate_by_target_range(model, loader, device):
    model.eval()
    all_preds_log = []
    all_targets_log = []

    with torch.no_grad():
        for b_first, b_past, b_global, b_thr, b_y_log, _, _, b_mask in loader:
            b_first = b_first.to(device, non_blocking=True)
            b_past = b_past.to(device, non_blocking=True)
            b_global = b_global.to(device, non_blocking=True)
            b_thr = b_thr.to(device, non_blocking=True)
            b_mask = b_mask.to(device, non_blocking=True)

            pred_raw_unconstrained, _ = model(
                b_first, b_past, b_global, b_thr, key_padding_mask=b_mask
            )

            if has_nonfinite_tensor(pred_raw_unconstrained):
                raise RuntimeError("Non-finite prediction during per-range evaluation.")

            pred_log = model.raw_to_log10ber(pred_raw_unconstrained)

            all_preds_log.append(pred_log.cpu().numpy())
            all_targets_log.append(b_y_log.numpy())

    preds_log = np.vstack(all_preds_log).reshape(-1)
    targets_log = np.vstack(all_targets_log).reshape(-1)

    preds_raw = np.clip(10 ** preds_log, EPS, 0.5)
    targets_raw = np.clip(10 ** targets_log, EPS, 0.5)

    ranges = {
        "low_BER(y<1e-6)": targets_raw < 1e-6,
        "mid_BER(1e-6<=y<1e-3)": (targets_raw >= 1e-6) & (targets_raw < 1e-3),
        "high_BER(1e-3<=y<0.10)": (targets_raw >= 1e-3) & (targets_raw < 0.10),
        "upper_BER(0.10<=y<0.20)": (targets_raw >= 0.10) & (targets_raw < 0.20),
        "target_BER(0.20<=y<0.40)": (targets_raw >= 0.20) & (targets_raw < 0.40),
        "very_high_BER(0.40<=y<=0.50)": targets_raw >= 0.40,
    }

    metrics = {}
    for name, mask in ranges.items():
        if np.any(mask):
            err_log = preds_log[mask] - targets_log[mask]
            err_raw = preds_raw[mask] - targets_raw[mask]
            rel_err = err_raw / np.maximum(targets_raw[mask], 1e-6)

            rmse_log = float(np.sqrt(np.mean(err_log ** 2)))
            mae_log = float(np.mean(np.abs(err_log)))
            rmse_raw = float(np.sqrt(np.mean(err_raw ** 2)))
            mae_raw = float(np.mean(np.abs(err_raw)))
            rmse_rel = float(np.sqrt(np.mean(rel_err ** 2)))
            mae_rel = float(np.mean(np.abs(rel_err)))

            metrics[name] = {
                "count": int(mask.sum()),
                "rmse_log": rmse_log,
                "mae_log": mae_log,
                "factor_error": float(10 ** rmse_log),
                "rmse_raw": rmse_raw,
                "mae_raw": mae_raw,
                "rmse_rel": rmse_rel,
                "mae_rel": mae_rel,
                "bias_raw": float(np.mean(err_raw)),
                "bias_log": float(np.mean(err_log)),
                "p90_abs_raw": float(np.percentile(np.abs(err_raw), 90)),
                "p95_abs_raw": float(np.percentile(np.abs(err_raw), 95)),
            }
        else:
            metrics[name] = None

    return metrics


# =========================
# Multi-Component Selection Scoring
# =========================
# Selection rationale:
#   BER spans 6+ orders of magnitude (1e-6 to 0.5), so multiplicative error
#   is the natural metric. We use weighted log-RMSE per region (already in log
#   space, so equivalent to weighted geometric mean of factor errors), then
#   add a bias penalty (also in log space, so multiplicative bias) and a tail
#   penalty (P95/RMSE ratio in raw space, capped, to penalize heavy tails).

# Weights for each BER region in the composite score.
# Higher weight = more important to get this region right.
SELECTION_REGION_WEIGHTS = {
    "low_BER(y<1e-6)": 0.25,
    "mid_BER(1e-6<=y<1e-3)": 0.50,
    "high_BER(1e-3<=y<0.10)": 1.00,
    "upper_BER(0.10<=y<0.20)": 2.00,
    "target_BER(0.20<=y<0.40)": 3.00,
    "very_high_BER(0.40<=y<=0.50)": 1.50,
}

# Composite weights: how much each component contributes
SELECTION_W_LOG_RMSE = 1.00   # Primary: per-region log-RMSE
SELECTION_W_BIAS = 0.50       # Secondary: bias in log space (factor bias)
SELECTION_W_TAIL = 0.05       # Tertiary: P95/RMSE ratio (heavy tails)
SELECTION_TAIL_CAP = 5.0      # Cap tail ratio so one bad region doesn't dominate


def compute_selection_score(val_range_metrics, val_metrics, min_count=50):
    """Compute a multi-component selection score, lower is better.

    Components:
      1. Weighted average of per-region RMSE(log10).
         (This is equivalent to a weighted geometric mean of factor errors.)
      2. Weighted average of |bias_log| per region (multiplicative bias).
      3. Weighted average of capped P95/RMSE ratio per region (tail penalty).

    Returns a dict with all components and the composite.
    """
    log_rmse_sum = 0.0
    bias_sum = 0.0
    tail_sum = 0.0
    total_w = 0.0

    per_region_factor = {}

    for region_name, w in SELECTION_REGION_WEIGHTS.items():
        m = val_range_metrics.get(region_name)
        if m is None or m["count"] < min_count:
            continue

        # 1. Log-RMSE component (already log-space)
        log_rmse_sum += w * m["rmse_log"]

        # 2. Bias component in log space (multiplicative bias)
        bias_sum += w * abs(m["bias_log"])

        # 3. Tail component: P95 / RMSE in raw space, capped
        if m["rmse_raw"] > 1e-9:
            tail_ratio = m["p95_abs_raw"] / (m["rmse_raw"] + 1e-9)
            tail_sum += w * min(tail_ratio, SELECTION_TAIL_CAP)
        else:
            tail_sum += w * 1.0

        total_w += w
        per_region_factor[region_name] = m["factor_error"]

    if total_w == 0:
        # Fallback to global metrics
        return {
            "composite": float(val_metrics["rmse_log"]),
            "log_rmse_weighted": float(val_metrics["rmse_log"]),
            "bias_weighted": 0.0,
            "tail_weighted": 0.0,
            "geometric_factor_error": float(val_metrics["factor_error"]),
            "fallback": True,
        }

    log_rmse_weighted = log_rmse_sum / total_w
    bias_weighted = bias_sum / total_w
    tail_weighted = tail_sum / total_w

    composite = (
        SELECTION_W_LOG_RMSE * log_rmse_weighted
        + SELECTION_W_BIAS * bias_weighted
        + SELECTION_W_TAIL * tail_weighted
    )

    # The geometric-mean factor error: 10^(weighted log_rmse)
    # Reported for human readability — "this model is off by ~Nx on average"
    geometric_factor = float(10 ** log_rmse_weighted)

    return {
        "composite": float(composite),
        "log_rmse_weighted": float(log_rmse_weighted),
        "bias_weighted": float(bias_weighted),
        "tail_weighted": float(tail_weighted),
        "geometric_factor_error": geometric_factor,
        "fallback": False,
    }


def is_acceptable_checkpoint(val_range_metrics, val_metrics):
    """Hard constraints: model must meet these to be considered for 'best'.

    These prevent pathological epochs from being selected just because
    they happen to score well on the composite (e.g., if a region has
    almost no samples and dominates by luck).
    """
    target = val_range_metrics.get("target_BER(0.20<=y<0.40)")
    if target is None or target["count"] < 100:
        return False, "target region too small"

    # No region's bias_log can exceed 0.15 (i.e., factor bias of ~1.4x)
    for region_name, m in val_range_metrics.items():
        if m is None or m["count"] < 50:
            continue
        if abs(m["bias_log"]) > 0.15:
            return False, f"{region_name} has bias_log={m['bias_log']:.3f}"

    # Overall log RMSE must be reasonable
    if val_metrics["rmse_log"] > 0.30:
        return False, f"overall rmse_log={val_metrics['rmse_log']:.3f} too high"

    return True, "ok"


# =========================
# Training
# =========================
def train_engine():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Executing on: {device}")

    train_loader, val_loader, test_loader, scalers, aux_info, max_past_seq_len = prepare_data(
        DATA_PATHS,
        batch_size=256,
        nrows_per_dataset=NROWS_PER_DATASET,
        num_workers=0,
    )

    joblib.dump(scalers, SCALER_SAVE_PATH)
    print(f"Scalers saved to: {SCALER_SAVE_PATH}")

    model = BERFirstTokenSetTransformerBoundedLogHalfMultiTask(
        max_past_seq_len=max_past_seq_len,
        token_dim=4,
        first_token_dim=4,
        threshold_dim=1,
        global_dim=7,
        d_model=128,
        num_set_layers=4,
        num_heads=4,
        mlp_ratio=4.0,
        dropout=0.10,
        num_ordinal_thresholds=NUM_ORDINAL_THRESHOLDS,
    ).to(device)

    print("Training target-focused first-token + set-transformer multitask model...")
    print(f"  Variable-length support: ENABLED (position-independent scaling)")
    print(f"  Max past sequence length (training): {max_past_seq_len}")
    print(f"  Scaling strategy: separate first-token / shared past-token scalers")
    print(f"  Threshold feature mode: {THRESHOLD_FEATURE_MODE}")
    print("  Ordinal head: monotonic cumulative logits")

    optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-5)

    reg_loss = StableMultiObjectiveBERBoundedLogLoss(
        log_delta=0.35,
        raw_delta=0.006,
        rel_delta=0.03,
        alpha_log=0.45,
        beta_raw=0.35,
        gamma_rel=0.20,
        use_regime_weights=True,
    )

    boosted_pos = aux_info["ordinal_pos_weights"].copy()
    for i, thr in enumerate(ORDINAL_THRESHOLDS):
        if 0.20 <= thr <= 0.40:
            boosted_pos[i] *= 2.5
        elif 0.15 <= thr < 0.20:
            boosted_pos[i] *= 1.5
        elif 0.40 < thr <= 0.45:
            boosted_pos[i] *= 1.5

    ord_loss = OrdinalBCELoss(
        pos_weight=torch.tensor(boosted_pos, dtype=torch.float32, device=device),
        reduction="mean",
    )

    criterion = MultiTaskBERLoss(
        reg_loss=reg_loss,
        ord_loss=ord_loss,
        lambda_ord=0.25,
    )

    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="min", patience=4, factor=0.5,
    )

    # ---- Multi-component selection state ----
    # Track multiple "best" checkpoints under different criteria.
    # We'll save all of them and the user can choose the right tradeoff at inference.
    best_checkpoints = {
        "composite": {"score": float("inf"), "state": None, "epoch": -1},   # primary: composite score
        "composite_ema": {"score": float("inf"), "state": None, "epoch": -1},  # EMA-smoothed composite
        "log_rmse": {"score": float("inf"), "state": None, "epoch": -1},    # weighted log-RMSE only
        "target_mae": {"score": float("inf"), "state": None, "epoch": -1},  # target region MAE
        "low_bias": {"score": float("inf"), "state": None, "epoch": -1},    # min weighted |bias_log|
    }

    # EMA smoothing of the composite score to reduce epoch-to-epoch noise
    EMA_ALPHA = 0.5
    ema_composite = None

    patience = 12
    wait = 0
    min_epochs_before_early_stop = 60
    max_epochs = 120

    print(f"Ordinal thresholds: {ORDINAL_THRESHOLDS}")
    print(f"Boosted ordinal pos weights: {boosted_pos}")
    print(f"Selection: weighted log-RMSE (geometric factor mean) + bias + tail penalties")
    print(f"  Region weights: {SELECTION_REGION_WEIGHTS}")
    print(f"  Composite weights: log_rmse={SELECTION_W_LOG_RMSE}, "
          f"bias={SELECTION_W_BIAS}, tail={SELECTION_W_TAIL}")
    print(f"  EMA alpha for composite smoothing: {EMA_ALPHA}")

    training_broke = False

    for epoch in range(max_epochs):
        model.train()

        for batch_idx, (b_first, b_past, b_global, b_thr, b_y_log, b_ord, _, b_mask) in enumerate(train_loader):
            b_first = b_first.to(device, non_blocking=True)
            b_past = b_past.to(device, non_blocking=True)
            b_global = b_global.to(device, non_blocking=True)
            b_thr = b_thr.to(device, non_blocking=True)
            b_y_log = b_y_log.to(device, non_blocking=True)
            b_ord = b_ord.to(device, non_blocking=True)
            b_mask = b_mask.to(device, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)

            pred_raw_unconstrained, ord_logits = model(
                b_first, b_past, b_global, b_thr, key_padding_mask=b_mask,
            )

            if has_nonfinite_tensor(pred_raw_unconstrained):
                print(f"Non-finite prediction at epoch {epoch+1}, batch {batch_idx+1}.")
                training_broke = True
                break

            if has_nonfinite_tensor(ord_logits):
                print(f"Non-finite ordinal logits at epoch {epoch+1}, batch {batch_idx+1}.")
                training_broke = True
                break

            loss, reg_part, ord_part = criterion(
                pred_raw_unconstrained, ord_logits, b_y_log, b_ord,
            )

            if has_nonfinite_tensor(loss):
                print(f"Non-finite loss at epoch {epoch+1}, batch {batch_idx+1}.")
                training_broke = True
                break

            loss.backward()

            grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            if not torch.isfinite(grad_norm):
                print(f"Non-finite gradient norm at epoch {epoch+1}, batch {batch_idx+1}.")
                training_broke = True
                break

            optimizer.step()

            bad_param = False
            for name, param in model.named_parameters():
                if param.requires_grad and param.data is not None and not torch.isfinite(param.data).all():
                    print(f"Non-finite parameter after optimizer step: {name}")
                    bad_param = True
                    break

            if bad_param:
                training_broke = True
                break

        if training_broke:
            print("Training stopped due to non-finite values.")
            break

        try:
            train_metrics = evaluate(model, train_loader, criterion, device)
            val_metrics = evaluate(model, val_loader, criterion, device)
            val_range_metrics = evaluate_by_target_range(model, val_loader, device)
        except RuntimeError as e:
            print(f"Evaluation failed at epoch {epoch+1}: {e}")
            break

        scheduler.step(val_metrics["loss"])
        current_lr = optimizer.param_groups[0]["lr"]

        # ---- Compute multi-component selection score ----
        sel = compute_selection_score(val_range_metrics, val_metrics)
        composite = sel["composite"]

        # EMA-smoothed composite (reduces noise from single bad/lucky epochs)
        if ema_composite is None:
            ema_composite = composite
        else:
            ema_composite = EMA_ALPHA * composite + (1.0 - EMA_ALPHA) * ema_composite

        # Components for individual best-checkpoints
        target_key = "target_BER(0.20<=y<0.40)"
        target_mae = (
            val_range_metrics[target_key]["mae_raw"]
            if val_range_metrics[target_key] is not None
            else float("inf")
        )

        # Acceptability gate
        is_acceptable, reason = is_acceptable_checkpoint(val_range_metrics, val_metrics)

        # ---- Logging ----
        print(
            f"Epoch {epoch+1:03d} | "
            f"LR: {current_lr:.2e} | "
            f"Train Loss: {train_metrics['loss']:.4f} | "
            f"Val Loss: {val_metrics['loss']:.4f} | "
            f"Val RMSE(log10): {val_metrics['rmse_log']:.4f} "
            f"(~{val_metrics['factor_error']:.3f}x) | "
            f"Val Region Acc: {val_metrics['region_acc']:.4f}"
        )
        print(
            f"  SELECTION | composite={composite:.5f} | "
            f"ema_composite={ema_composite:.5f} | "
            f"weighted log-RMSE={sel['log_rmse_weighted']:.5f} "
            f"(~{sel['geometric_factor_error']:.3f}x avg factor) | "
            f"weighted |bias_log|={sel['bias_weighted']:.5f} | "
            f"tail penalty={sel['tail_weighted']:.3f} | "
            f"acceptable={is_acceptable}"
            + ("" if is_acceptable else f" ({reason})")
        )

        # Per-range tail metrics
        for rng_name, rng_stats in val_range_metrics.items():
            if rng_stats is not None:
                print(
                    f"    {rng_name}: "
                    f"n={rng_stats['count']} | "
                    f"factor~{rng_stats['factor_error']:.3f}x | "
                    f"RMSE_log={rng_stats['rmse_log']:.4f} | "
                    f"MAE_raw={rng_stats['mae_raw']:.6f} | "
                    f"P90={rng_stats['p90_abs_raw']:.6f} | "
                    f"P95={rng_stats['p95_abs_raw']:.6f} | "
                    f"bias_raw={rng_stats['bias_raw']:+.6f} | "
                    f"bias_log={rng_stats['bias_log']:+.4f}"
                )

        # ---- Update each best-checkpoint (only if acceptable) ----
        improved_any = False

        if is_acceptable:
            current_state_snapshot = None  # lazy deepcopy

            checkpoint_candidates = [
                ("composite", composite),
                ("composite_ema", ema_composite),
                ("log_rmse", sel["log_rmse_weighted"]),
                ("target_mae", target_mae),
                ("low_bias", sel["bias_weighted"]),
            ]

            for ckpt_name, score in checkpoint_candidates:
                if score < best_checkpoints[ckpt_name]["score"]:
                    if current_state_snapshot is None:
                        current_state_snapshot = copy.deepcopy(model.state_dict())
                    best_checkpoints[ckpt_name] = {
                        "score": float(score),
                        "state": current_state_snapshot,
                        "epoch": epoch + 1,
                    }
                    improved_any = True
                    print(f"  -> New best [{ckpt_name}] at epoch {epoch+1}: {score:.6f}")

        # ---- Early stopping driven by EMA composite ----
        # We use the EMA score so we don't stop on a single noisy spike
        ema_best = best_checkpoints["composite_ema"]["score"]
        if ema_composite < ema_best + 1e-9 or improved_any:
            wait = 0
        else:
            if epoch + 1 >= min_epochs_before_early_stop:
                wait += 1
                if wait >= patience:
                    print(f"Early stopping triggered after {wait} non-improving epochs (EMA basis).")
                    break

    # ---- Save last model ----
    torch.save(model.state_dict(), LAST_MODEL_SAVE_PATH)
    print(f"\nLast model saved to: {LAST_MODEL_SAVE_PATH}")

    # ---- Save all best checkpoints ----
    print("\n" + "=" * 80)
    print("BEST CHECKPOINTS SUMMARY")
    print("=" * 80)
    for ckpt_name, info in best_checkpoints.items():
        if info["state"] is None:
            print(f"  [{ckpt_name:>14s}] never updated")
            continue

        path = BEST_MODEL_SAVE_PATH.replace(".pth", f"_{ckpt_name}.pth")
        torch.save(info["state"], path)
        print(
            f"  [{ckpt_name:>14s}] epoch={info['epoch']:>3d} | "
            f"score={info['score']:.6f} | saved -> {path}"
        )

    # ---- Choose primary best for evaluation: composite_ema is most stable ----
    primary_choice = "composite_ema"
    if best_checkpoints[primary_choice]["state"] is None:
        # Fall back to composite if EMA never updated (shouldn't happen but safe)
        primary_choice = "composite"
    if best_checkpoints[primary_choice]["state"] is None:
        # Fall back to log_rmse
        primary_choice = "log_rmse"

    if best_checkpoints[primary_choice]["state"] is not None:
        model.load_state_dict(best_checkpoints[primary_choice]["state"])
        torch.save(model.state_dict(), BEST_MODEL_SAVE_PATH)
        print(
            f"\nPrimary best ({primary_choice}, epoch "
            f"{best_checkpoints[primary_choice]['epoch']}) "
            f"saved to: {BEST_MODEL_SAVE_PATH}"
        )
    else:
        print("\nWarning: no valid best checkpoint was found.")

    test_metrics = evaluate(model, test_loader, criterion, device)

    print(
        f"Test Loss: {test_metrics['loss']:.4f} | "
        f"Test Reg Loss: {test_metrics['reg_loss']:.4f} | "
        f"Test Ord Loss: {test_metrics['ord_loss']:.4f} | "
        f"Test RMSE(log10): {test_metrics['rmse_log']:.4f} | "
        f"Test MAE(log10): {test_metrics['mae_log']:.4f} | "
        f"Typical multiplicative error: ~{test_metrics['factor_error']:.2f}x | "
        f"Test RMSE(raw): {test_metrics['rmse_raw']:.6f} | "
        f"Test MAE(raw): {test_metrics['mae_raw']:.6f} | "
        f"Test RMSE(rel): {test_metrics['rmse_rel']:.6f} | "
        f"Test MAE(rel): {test_metrics['mae_rel']:.6f} | "
        f"Test Region Acc: {test_metrics['region_acc']:.4f} | "
        f"Test Region MAE: {test_metrics['region_mae']:.4f}"
    )

    range_metrics = evaluate_by_target_range(model, test_loader, device)
    print("\nPer-range test diagnostics:")
    for name, stats in range_metrics.items():
        if stats is None:
            print(f"  {name}: no samples")
        else:
            print(
                f"  {name} | count={stats['count']} | "
                f"RMSE(log10)={stats['rmse_log']:.4f} | "
                f"MAE(log10)={stats['mae_log']:.4f} | "
                f"factor~{stats['factor_error']:.2f}x | "
                f"RMSE(raw)={stats['rmse_raw']:.6f} | "
                f"MAE(raw)={stats['mae_raw']:.6f} | "
                f"RMSE(rel)={stats['rmse_rel']:.6f} | "
                f"MAE(rel)={stats['mae_rel']:.6f} | "
                f"bias_raw={stats['bias_raw']:.6f} | "
                f"bias_log={stats['bias_log']:.6f} | "
                f"P90={stats['p90_abs_raw']:.6f} | "
                f"P95={stats['p95_abs_raw']:.6f}"
            )

    return model


if __name__ == "__main__":
    os.makedirs(BASE_DIR, exist_ok=True)

    missing = [p for p in DATA_PATHS if not os.path.exists(p)]
    if missing:
        print("Critical Error: Missing dataset files:")
        for p in missing:
            print(f"  - {p}")
    else:
        print("Using training from datasets:")
        for p in DATA_PATHS:
            print(f"  - {p}")
        print(f"Row cap per dataset: {NROWS_PER_DATASET:,}")

        trained_model = train_engine()

Using training from datasets:
  - ./data_random_with_random_variances_total_v2.csv
  - ./data_physics_with_variances_total.csv
Row cap per dataset: 5,000,000
Executing on: cuda
Loading up to 5,000,000 rows from: ./data_random_with_random_variances_total_v2.csv
Loading up to 5,000,000 rows from: ./data_physics_with_variances_total.csv
Combined rows before filtering: 10,000,000
Rows after cleaning/filtering: 8,220,266
Scalers saved to: ./random_extra_multitask_scalers2_normthr_monoord.pkl
Training target-focused first-token + set-transformer multitask model...
  Variable-length support: ENABLED (position-independent scaling)
  Max past sequence length (training): 13
  Scaling strategy: separate first-token / shared past-token scalers
  Threshold feature mode: normalized_logit
  Ordinal head: monotonic cumulative logits


/home/birkan/miniconda3/lib/python3.9/site-packages/torch/cuda/__init__.py:829: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")


Ordinal thresholds: [1e-06, 1e-05, 0.0001, 0.001, 0.01, 0.1, 0.15, 0.2, 0.25, 0.3, 0.35, 0.4, 0.45]
Boosted ordinal pos weights: [1.        1.        1.        1.        1.        1.        1.5
 2.5       2.5       3.1984687 4.381006  6.9339705 7.2581224]
Selection: weighted log-RMSE (geometric factor mean) + bias + tail penalties
  Region weights: {'low_BER(y<1e-6)': 0.25, 'mid_BER(1e-6<=y<1e-3)': 0.5, 'high_BER(1e-3<=y<0.10)': 1.0, 'upper_BER(0.10<=y<0.20)': 2.0, 'target_BER(0.20<=y<0.40)': 3.0, 'very_high_BER(0.40<=y<=0.50)': 1.5}
  Composite weights: log_rmse=1.0, bias=0.5, tail=0.05
  EMA alpha for composite smoothing: 0.5
Epoch 001 | LR: 1.00e-04 | Train Loss: 0.0777 | Val Loss: 0.2125 | Val RMSE(log10): 0.4069 (~2.552x) | Val Region Acc: 0.7762
  SELECTION | composite=0.26263 | ema_composite=0.26263 | weighted log-RMSE=0.15271 (~1.421x avg factor) | weighted |bias_log|=0.03612 | tail penalty=1.837 | acceptable=False (low_BER(y<1e-6) has bias_log=0.160)
    low_BER(y<1e-6): n=100

Epoch 007 | LR: 1.00e-04 | Train Loss: 0.0131 | Val Loss: 0.0352 | Val RMSE(log10): 0.3187 (~2.083x) | Val Region Acc: 0.9113
  SELECTION | composite=0.30381 | ema_composite=0.27680 | weighted log-RMSE=0.20659 (~1.609x avg factor) | weighted |bias_log|=0.02456 | tail penalty=1.699 | acceptable=False (low_BER(y<1e-6) has bias_log=0.267)
    low_BER(y<1e-6): n=100694 | factor~4.045x | RMSE_log=0.6069 | MAE_raw=0.000038 | P90=0.000000 | P95=0.000000 | bias_raw=+0.000038 | bias_log=+0.2667
    mid_BER(1e-6<=y<1e-3): n=17653 | factor~27.057x | RMSE_log=1.4323 | MAE_raw=0.001565 | P90=0.001357 | P95=0.002683 | bias_raw=+0.001481 | bias_log=-0.1901
    high_BER(1e-3<=y<0.10): n=117093 | factor~4.572x | RMSE_log=0.6601 | MAE_raw=0.009853 | P90=0.020254 | P95=0.031558 | bias_raw=+0.007790 | bias_log=+0.0080
    upper_BER(0.10<=y<0.20): n=211124 | factor~1.154x | RMSE_log=0.0621 | MAE_raw=0.007505 | P90=0.015677 | P95=0.024466 | bias_raw=+0.003635 | bias_log=+0.0098
    target_BER(0.20<=y<0.40):

Epoch 014 | LR: 1.00e-04 | Train Loss: 0.0085 | Val Loss: 0.0161 | Val RMSE(log10): 0.1414 (~1.385x) | Val Region Acc: 0.9355
  SELECTION | composite=0.16748 | ema_composite=0.16398 | weighted log-RMSE=0.07461 (~1.187x avg factor) | weighted |bias_log|=0.01909 | tail penalty=1.667 | acceptable=False (mid_BER(1e-6<=y<1e-3) has bias_log=-0.178)
    low_BER(y<1e-6): n=100694 | factor~2.612x | RMSE_log=0.4169 | MAE_raw=0.000019 | P90=0.000000 | P95=0.000000 | bias_raw=+0.000019 | bias_log=-0.0856
    mid_BER(1e-6<=y<1e-3): n=17653 | factor~3.332x | RMSE_log=0.5227 | MAE_raw=0.000208 | P90=0.000233 | P95=0.000346 | bias_raw=+0.000086 | bias_log=-0.1775
    high_BER(1e-3<=y<0.10): n=117093 | factor~1.346x | RMSE_log=0.1291 | MAE_raw=0.004206 | P90=0.009459 | P95=0.013681 | bias_raw=+0.002562 | bias_log=-0.0001
    upper_BER(0.10<=y<0.20): n=211124 | factor~1.093x | RMSE_log=0.0385 | MAE_raw=0.007890 | P90=0.013833 | P95=0.018698 | bias_raw=+0.006850 | bias_log=+0.0194
    target_BER(0.20<=y<

...

Epoch 063 | LR: 6.25e-06 | Train Loss: 0.0019 | Val Loss: 0.0022 | Val RMSE(log10): 0.0476 (~1.116x) | Val Region Acc: 0.9724
  SELECTION | composite=0.11638 | ema_composite=0.11681 | weighted log-RMSE=0.03150 (~1.075x avg factor) | weighted |bias_log|=0.01355 | tail penalty=1.562 | acceptable=True
    low_BER(y<1e-6): n=100694 | factor~1.301x | RMSE_log=0.1143 | MAE_raw=0.000000 | P90=0.000000 | P95=0.000000 | bias_raw=+0.000000 | bias_log=-0.0056
    mid_BER(1e-6<=y<1e-3): n=17653 | factor~1.696x | RMSE_log=0.2295 | MAE_raw=0.000071 | P90=0.000191 | P95=0.000238 | bias_raw=+0.000066 | bias_log=+0.1343
    high_BER(1e-3<=y<0.10): n=117093 | factor~1.160x | RMSE_log=0.0643 | MAE_raw=0.003065 | P90=0.005476 | P95=0.007237 | bias_raw=+0.002714 | bias_log=+0.0348
    upper_BER(0.10<=y<0.20): n=211124 | factor~1.028x | RMSE_log=0.0120 | MAE_raw=0.002247 | P90=0.005127 | P95=0.007884 | bias_raw=+0.000834 | bias_log=+0.0024
    target_BER(0.20<=y<0.40): n=459836 | factor~1.015x | RMSE_log=0.

Epoch 070 | LR: 6.25e-06 | Train Loss: 0.0019 | Val Loss: 0.0022 | Val RMSE(log10): 0.0439 (~1.106x) | Val Region Acc: 0.9758
  SELECTION | composite=0.11656 | ema_composite=0.11583 | weighted log-RMSE=0.02893 (~1.069x avg factor) | weighted |bias_log|=0.01744 | tail penalty=1.578 | acceptable=False (mid_BER(1e-6<=y<1e-3) has bias_log=0.173)
    low_BER(y<1e-6): n=100694 | factor~1.283x | RMSE_log=0.1084 | MAE_raw=0.000000 | P90=0.000000 | P95=0.000000 | bias_raw=+0.000000 | bias_log=-0.0072
    mid_BER(1e-6<=y<1e-3): n=17653 | factor~1.555x | RMSE_log=0.1916 | MAE_raw=0.000092 | P90=0.000282 | P95=0.000343 | bias_raw=+0.000091 | bias_log=+0.1732
    high_BER(1e-3<=y<0.10): n=117093 | factor~1.161x | RMSE_log=0.0649 | MAE_raw=0.003794 | P90=0.006339 | P95=0.007871 | bias_raw=+0.003619 | bias_log=+0.0467
    upper_BER(0.10<=y<0.20): n=211124 | factor~1.026x | RMSE_log=0.0112 | MAE_raw=0.002232 | P90=0.005031 | P95=0.007765 | bias_raw=+0.000976 | bias_log=+0.0029
    target_BER(0.20<=y<0

Epoch 077 | LR: 6.25e-06 | Train Loss: 0.0018 | Val Loss: 0.0021 | Val RMSE(log10): 0.0439 (~1.106x) | Val Region Acc: 0.9743
  SELECTION | composite=0.11455 | ema_composite=0.11395 | weighted log-RMSE=0.02977 (~1.071x avg factor) | weighted |bias_log|=0.01548 | tail penalty=1.541 | acceptable=False (mid_BER(1e-6<=y<1e-3) has bias_log=0.154)
    low_BER(y<1e-6): n=100694 | factor~1.267x | RMSE_log=0.1028 | MAE_raw=0.000000 | P90=0.000000 | P95=0.000000 | bias_raw=+0.000000 | bias_log=+0.0029
    mid_BER(1e-6<=y<1e-3): n=17653 | factor~1.642x | RMSE_log=0.2154 | MAE_raw=0.000085 | P90=0.000239 | P95=0.000290 | bias_raw=+0.000083 | bias_log=+0.1542
    high_BER(1e-3<=y<0.10): n=117093 | factor~1.149x | RMSE_log=0.0605 | MAE_raw=0.003427 | P90=0.005850 | P95=0.007693 | bias_raw=+0.003209 | bias_log=+0.0404
    upper_BER(0.10<=y<0.20): n=211124 | factor~1.027x | RMSE_log=0.0117 | MAE_raw=0.002223 | P90=0.005158 | P95=0.007877 | bias_raw=+0.001031 | bias_log=+0.0030
    target_BER(0.20<=y<0

Epoch 084 | LR: 3.13e-06 | Train Loss: 0.0018 | Val Loss: 0.0021 | Val RMSE(log10): 0.0390 (~1.094x) | Val Region Acc: 0.9747
  SELECTION | composite=0.11526 | ema_composite=0.11468 | weighted log-RMSE=0.02685 (~1.064x avg factor) | weighted |bias_log|=0.01532 | tail penalty=1.615 | acceptable=True
    low_BER(y<1e-6): n=100694 | factor~1.235x | RMSE_log=0.0915 | MAE_raw=0.000000 | P90=0.000000 | P95=0.000000 | bias_raw=+0.000000 | bias_log=+0.0062
    mid_BER(1e-6<=y<1e-3): n=17653 | factor~1.469x | RMSE_log=0.1670 | MAE_raw=0.000076 | P90=0.000242 | P95=0.000302 | bias_raw=+0.000074 | bias_log=+0.1409
    high_BER(1e-3<=y<0.10): n=117093 | factor~1.159x | RMSE_log=0.0641 | MAE_raw=0.003823 | P90=0.006178 | P95=0.007903 | bias_raw=+0.003652 | bias_log=+0.0461
    upper_BER(0.10<=y<0.20): n=211124 | factor~1.027x | RMSE_log=0.0114 | MAE_raw=0.002199 | P90=0.005001 | P95=0.007734 | bias_raw=+0.000997 | bias_log=+0.0030
    target_BER(0.20<=y<0.40): n=459836 | factor~1.015x | RMSE_log=0.

Epoch 091 | LR: 1.56e-06 | Train Loss: 0.0018 | Val Loss: 0.0021 | Val RMSE(log10): 0.0424 (~1.103x) | Val Region Acc: 0.9752
  SELECTION | composite=0.11392 | ema_composite=0.11412 | weighted log-RMSE=0.02850 (~1.068x avg factor) | weighted |bias_log|=0.01722 | tail penalty=1.536 | acceptable=False (mid_BER(1e-6<=y<1e-3) has bias_log=0.168)
    low_BER(y<1e-6): n=100694 | factor~1.264x | RMSE_log=0.1018 | MAE_raw=0.000000 | P90=0.000000 | P95=0.000000 | bias_raw=+0.000000 | bias_log=+0.0176
    mid_BER(1e-6<=y<1e-3): n=17653 | factor~1.548x | RMSE_log=0.1899 | MAE_raw=0.000087 | P90=0.000247 | P95=0.000299 | bias_raw=+0.000086 | bias_log=+0.1677
    high_BER(1e-3<=y<0.10): n=117093 | factor~1.160x | RMSE_log=0.0644 | MAE_raw=0.003696 | P90=0.006112 | P95=0.007743 | bias_raw=+0.003510 | bias_log=+0.0458
    upper_BER(0.10<=y<0.20): n=211124 | factor~1.026x | RMSE_log=0.0112 | MAE_raw=0.002170 | P90=0.004958 | P95=0.007634 | bias_raw=+0.000888 | bias_log=+0.0026
    target_BER(0.20<=y<0

In [3]:
import os
import joblib
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy.special import erfc
from itertools import product as iproduct

EPS = 1e-12
PLOT_DIR = "./plots_mixed_newdata"
os.makedirs(PLOT_DIR, exist_ok=True)

# =========================
# Paths
# =========================
MODEL_PATH = "random_extra_multitask_best2_normthr_monoord.pth"
SCALER_PATH = "random_extra_multitask_scalers2_normthr_monoord.pkl"

# =========================
# Config
# =========================
PHYSICS_MAX_MEM_LEN = 14
PHYSICS_MIN_MEM_LEN = 14
ARRIVAL_COVERAGE = 0.70
N_THRESHOLDS = 500
RANDOM_SEED = 60

# =========================
# Test-time smoothing config
# =========================
# The smoother recomputes ALL threshold-dependent features for thr-eps, thr, thr+eps.
# This is important because threshold enters both thr_t and global features such as z0/z1/HMG.
USE_TEST_TIME_SMOOTHING = True
SMOOTHING_EPS_FRAC = 0.005     # eps = SMOOTHING_EPS_FRAC * threshold sweep span
SMOOTHING_SPACE = "log"         # "log" is usually best globally; "raw" can look better at high BER
SMOOTHING_WEIGHTS = (0.25, 0.50, 0.25)

# This must match training. If the scaler file contains "threshold_feature_mode",
# that saved value is used; this default is only a fallback.
THRESHOLD_FEATURE_MODE = "normalized_logit"
THRESHOLD_NORM_CLIP_EPS = 1e-5


ORDINAL_THRESHOLDS = [
    1e-6, 1e-5, 1e-4, 1e-3, 1e-2, 1e-1,
    0.15, 0.20, 0.25, 0.30, 0.35, 0.40, 0.45,
]
NUM_ORDINAL_THRESHOLDS = len(ORDINAL_THRESHOLDS)
NUM_REGION_CLASSES = NUM_ORDINAL_THRESHOLDS + 1

REGION_LABELS = {
    0: "y < 1e-6",
    1: "1e-6 <= y < 1e-5",
    2: "1e-5 <= y < 1e-4",
    3: "1e-4 <= y < 1e-3",
    4: "1e-3 <= y < 1e-2",
    5: "1e-2 <= y < 1e-1",
    6: "0.10 <= y < 0.15",
    7: "0.15 <= y < 0.20",
    8: "0.20 <= y < 0.25",
    9: "0.25 <= y < 0.30",
    10: "0.30 <= y < 0.35",
    11: "0.35 <= y < 0.40",
    12: "0.40 <= y < 0.45",
    13: "0.45 <= y <= 0.50",
}


# =========================
# Physics helpers
# =========================
def Fhit_function(radius, distance, diffusionCoef, t):
    if t <= 0:
        return 0.0
    return (radius / (distance + radius)) * erfc(distance / np.sqrt(4 * diffusionCoef * t))


def calculate_hitting_probabilities(mem_len, radius, distance, diffusionCoef, Ts):
    P = np.zeros(mem_len)
    for i in range(mem_len):
        t_end = (i + 1) * Ts
        t_start = i * Ts
        P[i] = Fhit_function(radius, distance, diffusionCoef, t_end) - Fhit_function(
            radius, distance, diffusionCoef, t_start
        )
    return P


def calculate_ber_vectorized(mem_len, threshold, P_scaled, variances):
    P_arr = np.asarray(P_scaled, dtype=float)[:mem_len]
    vars_arr = np.asarray(variances, dtype=float)[:mem_len]

    seqs = np.array(list(iproduct([0, 1], repeat=mem_len)), dtype=np.float64)[:, ::-1]
    c_bit = seqs[:, 0]

    mu = (seqs * P_arr).sum(axis=1)
    var_total = (seqs * vars_arr).sum(axis=1)
    std = np.sqrt(np.maximum(var_total, 0.0))

    pe = np.empty_like(mu)
    zero_std = (std == 0)
    if np.any(zero_std):
        pe[zero_std & (c_bit == 1)] = np.where(
            mu[zero_std & (c_bit == 1)] < threshold, 1.0, 0.0)
        pe[zero_std & (c_bit == 0)] = np.where(
            mu[zero_std & (c_bit == 0)] >= threshold, 1.0, 0.0)
    nz = ~zero_std
    if np.any(nz):
        pe[nz & (c_bit == 1)] = 0.5 * erfc(
            (mu[nz & (c_bit == 1)] - threshold) / (std[nz & (c_bit == 1)] * np.sqrt(2)))
        pe[nz & (c_bit == 0)] = 0.5 * erfc(
            (threshold - mu[nz & (c_bit == 0)]) / (std[nz & (c_bit == 0)] * np.sqrt(2)))
    return float(np.mean(pe))


def raw_to_region_labels_np(y_raw):
    y = np.asarray(y_raw).reshape(-1)
    return np.searchsorted(np.asarray(ORDINAL_THRESHOLDS), y, side="right").astype(np.int64)


def ordinal_logits_to_region_labels_torch(logits):
    probs = torch.sigmoid(logits)
    passed = (probs >= 0.5).sum(dim=1)
    return passed.long()


# =========================
# Shared scaling helper
# =========================
def apply_shared_scale(data, mean, std, valid_mask=None):
    scaled = ((data - mean) / std).astype(np.float32)
    if valid_mask is not None and scaled.ndim == 2 and valid_mask.shape == scaled.shape:
        scaled = scaled * valid_mask.astype(np.float32)
    return scaled


def safe_harmonic_z(z0, z1, denom_eps=1e-9):
    """Finite-safe version of 2/(1/z0 + 1/z1) = 2*z0*z1/(z0+z1).

    The direct reciprocal form can create +/-inf when z0 or z1 is near zero.
    This helper uses np.divide(..., where=...) and sanitizes non-finite outputs.
    """
    z0 = np.asarray(z0, dtype=np.float32)
    z1 = np.asarray(z1, dtype=np.float32)
    num = (2.0 * z0 * z1).astype(np.float32)
    den = (z0 + z1).astype(np.float32)
    out = np.divide(
        num,
        den,
        out=np.zeros_like(num, dtype=np.float32),
        where=np.abs(den) > denom_eps,
    ).astype(np.float32)
    return np.nan_to_num(out, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)


def compute_threshold_feature_np(threshold_raw, total_scaled_taps, mode=THRESHOLD_FEATURE_MODE):
    thr = np.asarray(threshold_raw, dtype=np.float32).reshape(-1, 1)
    total = np.asarray(total_scaled_taps, dtype=np.float32).reshape(-1, 1)
    denom = np.maximum(total, EPS).astype(np.float32)

    if mode == "normalized":
        feat = (thr / denom).astype(np.float32)
    elif mode == "normalized_logit":
        u = (thr / denom).astype(np.float32)
        u = np.clip(u, THRESHOLD_NORM_CLIP_EPS, 1.0 - THRESHOLD_NORM_CLIP_EPS)
        feat = np.log(u / (1.0 - u)).astype(np.float32)
    elif mode == "log10_raw":
        feat = np.log10(thr + EPS).astype(np.float32)
    else:
        raise ValueError(f"Unknown threshold feature mode: {mode}")

    feat = np.nan_to_num(feat, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)
    return feat


# =========================
# Generate one physical scenario
# =========================
def generate_physical_case(rng, min_mem_len, max_mem_len, arrival_coverage, max_tries=5000):
    for _ in range(max_tries):
        radius = rng.uniform(3.0, 5.0)
        distance = rng.uniform(10.0, 15.0)
        diff = rng.uniform(50.0, 75.0)
        Ts = rng.uniform(0.5, 1.2)
        N = int(10 ** rng.uniform(3.0, 6.0))

        f_inf = radius / (radius + distance)
        target = arrival_coverage * f_inf

        cumsum, k = 0.0, 0
        while k < max_mem_len:
            pk = Fhit_function(radius, distance, diff, (k + 1) * Ts) - Fhit_function(
                radius, distance, diff, k * Ts)
            cumsum += pk
            k += 1
            if cumsum >= target:
                break

        if k < min_mem_len:
            continue

        P_ext = calculate_hitting_probabilities(k + 1, radius, distance, diff, Ts)
        P_main = P_ext[:k]
        P_extra = float(P_ext[k]) if k < len(P_ext) else 0.0

        P_scaled = P_main * N
        variances = N * P_main * (1.0 - P_main)

        return {
            "radius": radius, "distance": distance, "diffusion": diff,
            "Ts": Ts, "N": N, "mem_len": k, "P": P_main,
            "P_scaled": P_scaled, "variances": variances,
            "P_mem_len_extra": P_extra,
            "P_mem_len_extra_var": P_extra * (1.0 - P_extra),
        }

    raise RuntimeError(
        f"Could not generate a physical case with mem_len >= {min_mem_len} "
        f"after {max_tries} tries."
    )


# =========================
# Model (matches v3 training: global_dim=7, global_embed=96, cond_dim=128)
# =========================
class ConditionalFeatureModulation(nn.Module):
    def __init__(self, cond_dim, feat_dim, hidden_dim=256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(cond_dim, hidden_dim), nn.GELU(),
            nn.Linear(hidden_dim, 2 * feat_dim),
        )
        self.scale = nn.Parameter(torch.tensor(0.25, dtype=torch.float32))

    def forward(self, x, cond):
        gamma_beta = self.net(cond)
        gamma, beta = torch.chunk(gamma_beta, 2, dim=-1)
        if x.dim() == 3:
            gamma = gamma.unsqueeze(1)
            beta = beta.unsqueeze(1)
        return x * (1.0 + self.scale * gamma) + self.scale * beta


class SetSelfAttentionBlock(nn.Module):
    def __init__(self, d_model, num_heads, mlp_ratio=4.0, dropout=0.10):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model)
        self.attn = nn.MultiheadAttention(
            embed_dim=d_model, num_heads=num_heads,
            dropout=dropout, batch_first=True)
        self.norm2 = nn.LayerNorm(d_model)
        hidden = int(d_model * mlp_ratio)
        self.mlp = nn.Sequential(
            nn.Linear(d_model, hidden), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(hidden, d_model), nn.Dropout(dropout))

    def forward(self, x, key_padding_mask=None):
        y = self.norm1(x)
        attn_out, _ = self.attn(y, y, y, need_weights=False, key_padding_mask=key_padding_mask)
        x = x + attn_out
        x = x + self.mlp(self.norm2(x))
        return x


class SetAttentionPooling(nn.Module):
    def __init__(self, d_model, hidden_dim=128):
        super().__init__()
        self.score = nn.Sequential(
            nn.Linear(d_model, hidden_dim), nn.GELU(), nn.Linear(hidden_dim, 1))

    def forward(self, x, key_padding_mask=None):
        logits = self.score(x)
        if key_padding_mask is not None:
            logits = logits.masked_fill(key_padding_mask.unsqueeze(-1), float("-inf"))
        weights = torch.softmax(logits, dim=1)
        weights = torch.nan_to_num(weights, nan=0.0)
        return (weights * x).sum(dim=1)


class MonotonicOrdinalHead(nn.Module):
    def __init__(self, head_in, num_thresholds):
        super().__init__()
        self.num_thresholds = num_thresholds
        self.base = nn.Sequential(
            nn.Linear(head_in, 256), nn.GELU(), nn.Dropout(0.20),
            nn.Linear(256, 128), nn.GELU(), nn.Dropout(0.15),
        )
        self.first_logit = nn.Linear(128, 1)
        self.deltas = nn.Linear(128, max(num_thresholds - 1, 1))

    def forward(self, x):
        h = self.base(x)
        first = self.first_logit(h)
        if self.num_thresholds == 1:
            return first
        deltas = -F.softplus(self.deltas(h)[:, : self.num_thresholds - 1])
        return torch.cat([first, deltas], dim=-1).cumsum(dim=-1)


class BERFirstTokenSetTransformerBoundedLogHalfMultiTask(nn.Module):
    def __init__(self, max_past_seq_len, token_dim=4, first_token_dim=4,
                 threshold_dim=1, global_dim=7, d_model=128, num_set_layers=4,
                 num_heads=4, mlp_ratio=4.0, dropout=0.10,
                 num_ordinal_thresholds=NUM_ORDINAL_THRESHOLDS):
        super().__init__()
        self.max_past_seq_len = max_past_seq_len
        self.d_model = d_model
        self.num_ordinal_thresholds = num_ordinal_thresholds

        self.first_proj = nn.Sequential(
            nn.Linear(first_token_dim, d_model), nn.LayerNorm(d_model),
            nn.GELU(), nn.Dropout(dropout))
        self.token_proj = nn.Sequential(
            nn.Linear(token_dim, d_model), nn.LayerNorm(d_model),
            nn.GELU(), nn.Dropout(dropout))
        self.thr_embed = nn.Sequential(
            nn.Linear(threshold_dim, 32), nn.GELU(),
            nn.Linear(32, 32), nn.GELU())
        self.global_embed = nn.Sequential(
            nn.Linear(global_dim, 96), nn.GELU(),
            nn.Linear(96, 96), nn.GELU())

        cond_dim = 32 + 96
        self.first_cond_mod = ConditionalFeatureModulation(cond_dim, d_model, 256)
        self.set_cond_mod = ConditionalFeatureModulation(cond_dim, d_model, 256)

        self.set_blocks = nn.ModuleList([
            SetSelfAttentionBlock(d_model, num_heads, mlp_ratio, dropout)
            for _ in range(num_set_layers)])

        self.first_refine = nn.Sequential(
            nn.LayerNorm(d_model), nn.Linear(d_model, d_model),
            nn.GELU(), nn.Dropout(dropout), nn.Linear(d_model, d_model))
        self.final_set_norm = nn.LayerNorm(d_model)
        self.set_attn_pool = SetAttentionPooling(d_model, 128)

        set_summary_dim = 3 * d_model
        head_in = d_model + set_summary_dim + cond_dim

        self.reg_head = nn.Sequential(
            nn.Linear(head_in, 256), nn.GELU(), nn.Dropout(0.20),
            nn.Linear(256, 128), nn.GELU(), nn.Dropout(0.15),
            nn.Linear(128, 1))
        self.ord_head = MonotonicOrdinalHead(head_in, num_ordinal_thresholds)

    @staticmethod
    def raw_to_log10ber(pred_raw_unconstrained):
        log10_half = torch.log10(torch.tensor(
            0.5, device=pred_raw_unconstrained.device,
            dtype=pred_raw_unconstrained.dtype))
        return log10_half - F.softplus(pred_raw_unconstrained)

    @staticmethod
    def raw_to_ber(pred_raw_unconstrained):
        pred_log = BERFirstTokenSetTransformerBoundedLogHalfMultiTask.raw_to_log10ber(
            pred_raw_unconstrained)
        return torch.pow(10.0, pred_log).clamp(min=EPS, max=0.5)

    def forward(self, first_token, past_tokens, global_feats, threshold, key_padding_mask=None):
        thr = self.thr_embed(threshold)
        g = self.global_embed(global_feats)
        cond = torch.cat([thr, g], dim=-1)

        first_x = self.first_proj(first_token)
        first_x = self.first_cond_mod(first_x, cond)
        first_x = first_x + self.first_refine(first_x)

        set_x = self.token_proj(past_tokens)
        set_x = self.set_cond_mod(set_x, cond)
        for block in self.set_blocks:
            set_x = block(set_x, key_padding_mask=key_padding_mask)
        set_x = self.final_set_norm(set_x)

        if key_padding_mask is not None:
            valid_mask = (~key_padding_mask).unsqueeze(-1).float()
        else:
            valid_mask = torch.ones(
                set_x.shape[0], set_x.shape[1], 1,
                device=set_x.device, dtype=set_x.dtype)

        pooled_attn = self.set_attn_pool(set_x, key_padding_mask=key_padding_mask)
        valid_counts = valid_mask.sum(dim=1).clamp(min=1.0)
        pooled_mean = (set_x * valid_mask).sum(dim=1) / valid_counts

        if key_padding_mask is not None:
            x_for_max = set_x.masked_fill(key_padding_mask.unsqueeze(-1), float("-inf"))
        else:
            x_for_max = set_x
        pooled_max = x_for_max.amax(dim=1)
        pooled_max = torch.nan_to_num(pooled_max, nan=0.0, posinf=0.0, neginf=0.0)

        set_summary = torch.cat([pooled_attn, pooled_mean, pooled_max], dim=-1)
        fused = torch.cat([first_x, set_summary, cond], dim=-1)

        pred_raw_unconstrained = self.reg_head(fused)
        ord_logits = self.ord_head(fused)
        return pred_raw_unconstrained, ord_logits


# =========================
# Build inference inputs (7 global features, position-independent scaling)
# =========================
def prepare_inference_features(case, thresholds, scalers, device):
    """Build model inputs for a threshold sweep over a single physical case.

    Returns:
        first_t, past_t, global_t, thr_t, mask_t — all torch tensors on device
        global_t has shape [B, 7]: [z0, z1, hmg, nsid, log_hd, da, herf]
    """
    B = len(thresholds)
    mem_len = case["mem_len"]
    N = float(case["N"])

    P_raw = np.asarray(case["P"], dtype=np.float32)
    var_raw = np.asarray(case["variances"], dtype=np.float32)
    thr_raw = np.asarray(thresholds, dtype=np.float32).reshape(-1, 1)

    # Feature engineering
    taps_feat = (P_raw * N).astype(np.float32)
    vars_feat = var_raw.astype(np.float32)
    abs_feat = np.abs(taps_feat).astype(np.float32)
    snr_feat = np.log10((taps_feat ** 2) / (var_raw + EPS) + EPS).astype(np.float32)

    # Broadcast to [B, mem_len]
    taps_2d = np.broadcast_to(taps_feat[None, :], (B, mem_len)).copy()
    vars_2d = np.broadcast_to(vars_feat[None, :], (B, mem_len)).copy()
    abs_2d = np.broadcast_to(abs_feat[None, :], (B, mem_len)).copy()
    snr_2d = np.broadcast_to(snr_feat[None, :], (B, mem_len)).copy()

    L_past = mem_len - 1
    valid_past = np.ones((B, L_past), dtype=bool)

    # --- Threshold-dependent global features ---
    first_mean = taps_2d[:, 0:1]
    past_means = taps_2d[:, 1:]
    first_var = vars_2d[:, 0:1]
    past_vars = vars_2d[:, 1:]

    mu0 = (0.5 * past_means.sum(axis=1, keepdims=True)).astype(np.float32)
    mu1 = (first_mean + mu0).astype(np.float32)
    var0 = (0.5 * past_vars.sum(axis=1, keepdims=True)).astype(np.float32)
    var1 = (first_var + var0).astype(np.float32)
    std0 = np.sqrt(np.maximum(var0, EPS)).astype(np.float32)
    std1 = np.sqrt(np.maximum(var1, EPS)).astype(np.float32)

    z0 = ((thr_raw - mu0) / (std0 + EPS)).astype(np.float32)
    z1 = ((mu1 - thr_raw) / (std1 + EPS)).astype(np.float32)
    harmonic = safe_harmonic_z(z0, z1)
    abs_diff = np.abs(z0 - z1).astype(np.float32)
    hmg = (harmonic - 0.25 * abs_diff).astype(np.float32)
    hmg = np.nan_to_num(hmg, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)

    # --- Scenario-level features (threshold-independent) ---
    signal = first_mean  # [B, 1]
    isi = past_means.sum(axis=1, keepdims=True)
    nsid = ((signal - isi) / (signal + isi + EPS)).astype(np.float32)

    gap = signal
    d0_inf = (gap / (std0 + EPS)).astype(np.float32)
    d1_inf = (gap / (std1 + EPS)).astype(np.float32)
    hd = (2.0 * d0_inf * d1_inf / (d0_inf + d1_inf + EPS)).astype(np.float32)
    log_hd = np.log10(hd + EPS).astype(np.float32)
    da = (np.abs(d0_inf - d1_inf) / (d0_inf + d1_inf + EPS)).astype(np.float32)

    past_taps_sum = past_means.sum(axis=1, keepdims=True)
    past_shares = past_means / (past_taps_sum + EPS)
    herf = (past_shares ** 2).sum(axis=1, keepdims=True).astype(np.float32)

    # --- Scale first token ---
    ft_tap = ((taps_2d[:, 0] - scalers["first_tap_mean"]) / scalers["first_tap_std"]).astype(np.float32)
    ft_var = ((vars_2d[:, 0] - scalers["first_var_mean"]) / scalers["first_var_std"]).astype(np.float32)
    ft_abs = ((abs_2d[:, 0] - scalers["first_abs_mean"]) / scalers["first_abs_std"]).astype(np.float32)
    ft_snr = ((snr_2d[:, 0] - scalers["first_snr_mean"]) / scalers["first_snr_std"]).astype(np.float32)
    first_token = np.stack([ft_tap, ft_var, ft_abs, ft_snr], axis=1).astype(np.float32)

    # --- Scale past tokens ---
    pt_tap = apply_shared_scale(taps_2d[:, 1:], scalers["past_tap_mean"], scalers["past_tap_std"], valid_past)
    pt_var = apply_shared_scale(vars_2d[:, 1:], scalers["past_var_mean"], scalers["past_var_std"], valid_past)
    pt_abs = apply_shared_scale(abs_2d[:, 1:], scalers["past_abs_mean"], scalers["past_abs_std"], valid_past)
    pt_snr = apply_shared_scale(snr_2d[:, 1:], scalers["past_snr_mean"], scalers["past_snr_std"], valid_past)
    past_tokens = np.stack([pt_tap, pt_var, pt_abs, pt_snr], axis=2).astype(np.float32)

    # --- Scale globals (7 features) ---
    z0_s = scalers["z0_scaler"].transform(z0).astype(np.float32)
    z1_s = scalers["z1_scaler"].transform(z1).astype(np.float32)
    hmg_s = scalers["harmonic_minus_gap_scaler"].transform(hmg).astype(np.float32)
    nsid_s = scalers["nsid_scaler"].transform(nsid).astype(np.float32)
    log_hd_s = scalers["log_hd_scaler"].transform(log_hd).astype(np.float32)
    da_s = scalers["da_scaler"].transform(da).astype(np.float32)
    herf_s = scalers["herf_scaler"].transform(herf).astype(np.float32)
    global_feats = np.concatenate(
        [z0_s, z1_s, hmg_s, nsid_s, log_hd_s, da_s, herf_s], axis=1
    ).astype(np.float32)

    # --- Scale threshold using scenario-normalized feature from training ---
    total_scaled_taps = np.sum(taps_2d, axis=1, keepdims=True).astype(np.float32)
    threshold_mode = scalers.get("threshold_feature_mode", THRESHOLD_FEATURE_MODE)
    thr_feature = compute_threshold_feature_np(thr_raw, total_scaled_taps, mode=threshold_mode)
    thr_s = scalers["thr_scaler"].transform(thr_feature).astype(np.float32)

    # --- Padding mask (no padding needed here) ---
    pad_mask = np.zeros((B, L_past), dtype=bool)

    first_t = torch.from_numpy(first_token).to(device)
    past_t = torch.from_numpy(past_tokens).to(device)
    global_t = torch.from_numpy(global_feats).to(device)
    thr_t = torch.from_numpy(thr_s).to(device)
    mask_t = torch.from_numpy(pad_mask).to(device)

    return first_t, past_t, global_t, thr_t, mask_t


# =========================
# Prediction helpers
# =========================
def predict_ber_for_thresholds(model, case, thresholds, scalers, device):
    """Predict BER for a threshold vector.

    This recomputes threshold-dependent engineered features, so it is safe to call
    for shifted threshold vectors such as thr-eps and thr+eps.
    """
    first_t, past_t, global_t, thr_t, mask_t = prepare_inference_features(
        case, thresholds, scalers, device,
    )

    with torch.no_grad():
        pred_raw_out, ord_logits = model(
            first_t, past_t, global_t, thr_t, key_padding_mask=mask_t,
        )
        pred_bers = model.raw_to_ber(pred_raw_out).cpu().numpy().reshape(-1)
        pred_log = model.raw_to_log10ber(pred_raw_out).cpu().numpy().reshape(-1)
        pred_regions = ordinal_logits_to_region_labels_torch(ord_logits).cpu().numpy().reshape(-1)
        pred_region_probs = torch.sigmoid(ord_logits).cpu().numpy()

    pred_bers = np.clip(pred_bers, EPS, 0.5)
    pred_log = np.log10(np.clip(pred_bers, EPS, 0.5)).astype(np.float32)
    return pred_bers, pred_log, pred_regions, pred_region_probs


def predict_ber_smoothed_3point(
    model,
    case,
    thresholds,
    scalers,
    device,
    eps_frac=SMOOTHING_EPS_FRAC,
    smoothing_space=SMOOTHING_SPACE,
    weights=SMOOTHING_WEIGHTS,
):
    """Three-point test-time smoothing over threshold.

    Predicts at [thr-eps, thr, thr+eps] and averages either in log-BER space or
    raw-BER space. Crucially, this recomputes global features for each shifted
    threshold, rather than only perturbing the already-scaled threshold tensor.
    """
    thresholds = np.asarray(thresholds, dtype=np.float32)
    w_minus, w_mid, w_plus = weights
    weight_sum = float(w_minus + w_mid + w_plus)
    w_minus, w_mid, w_plus = w_minus / weight_sum, w_mid / weight_sum, w_plus / weight_sum

    thr_span = float(np.max(thresholds) - np.min(thresholds))
    eps_thr = float(eps_frac * max(thr_span, EPS))

    thr_minus = np.clip(thresholds - eps_thr, 0.0, None).astype(np.float32)
    thr_mid = thresholds.astype(np.float32)
    thr_plus = (thresholds + eps_thr).astype(np.float32)

    ber_minus, log_minus, _, _ = predict_ber_for_thresholds(
        model, case, thr_minus, scalers, device,
    )
    ber_mid, log_mid, regions_mid, probs_mid = predict_ber_for_thresholds(
        model, case, thr_mid, scalers, device,
    )
    ber_plus, log_plus, _, _ = predict_ber_for_thresholds(
        model, case, thr_plus, scalers, device,
    )

    if smoothing_space.lower() == "raw":
        ber_smooth = (
            w_minus * ber_minus +
            w_mid * ber_mid +
            w_plus * ber_plus
        )
        ber_smooth = np.clip(ber_smooth, EPS, 0.5).astype(np.float32)
        log_smooth = np.log10(ber_smooth).astype(np.float32)
    elif smoothing_space.lower() == "log":
        log_smooth = (
            w_minus * log_minus +
            w_mid * log_mid +
            w_plus * log_plus
        ).astype(np.float32)
        ber_smooth = np.clip(10.0 ** log_smooth, EPS, 0.5).astype(np.float32)
    else:
        raise ValueError("smoothing_space must be either 'log' or 'raw'.")

    return {
        "ber": ber_smooth,
        "log10_ber": log_smooth,
        "regions": regions_mid,
        "region_probs": probs_mid,
        "raw_mid_ber": ber_mid,
        "raw_mid_log10_ber": log_mid,
        "eps_threshold": eps_thr,
        "smoothing_space": smoothing_space,
    }


# =========================
# Generate scenario
# =========================
rng = np.random.default_rng(RANDOM_SEED)
case = generate_physical_case(
    rng,
    min_mem_len=PHYSICS_MIN_MEM_LEN,
    max_mem_len=PHYSICS_MAX_MEM_LEN,
    arrival_coverage=ARRIVAL_COVERAGE,
)

print("Generated physical scenario")
print("radius    =", case["radius"])
print("distance  =", case["distance"])
print("diffusion =", case["diffusion"])
print("Ts        =", case["Ts"])
print("N         =", case["N"])
print("mem_len   =", case["mem_len"])
print("P         =", case["P"])
print("P_scaled  =", case["P_scaled"])
print("variances =", case["variances"])

# =========================
# Threshold sweep — ground truth
# =========================
thr_min = 0.0
thr_max = float(np.sum(case["P_scaled"]))
thresholds = np.linspace(thr_min, thr_max, N_THRESHOLDS)

print("Threshold search interval:", thr_min, "to", thr_max)
print("sum(P_scaled) =", np.sum(case["P_scaled"]))

real_bers = np.array([
    calculate_ber_vectorized(
        mem_len=case["mem_len"],
        threshold=thr,
        P_scaled=case["P_scaled"],
        variances=case["variances"],
    )
    for thr in thresholds
])

real_bers = np.clip(real_bers, EPS, 0.5)
real_regions = raw_to_region_labels_np(real_bers)

# =========================
# Load model + scalers
# =========================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
scalers = joblib.load(SCALER_PATH)

max_past_seq_len = scalers.get("train_max_past_seq_len", scalers.get("max_past_seq_len", case["mem_len"] - 1))
ordinal_thresholds = scalers.get("ordinal_thresholds", ORDINAL_THRESHOLDS)
num_ordinal = len(ordinal_thresholds)
global_dim = scalers.get("global_dim", 7)

model = BERFirstTokenSetTransformerBoundedLogHalfMultiTask(
    max_past_seq_len=max_past_seq_len,
    token_dim=scalers.get("past_token_dim", 4),
    first_token_dim=scalers.get("first_token_dim", 4),
    threshold_dim=1,
    global_dim=global_dim,
    d_model=128,
    num_set_layers=4,
    num_heads=4,
    mlp_ratio=4.0,
    dropout=0.10,
    num_ordinal_thresholds=num_ordinal,
).to(device)

state = torch.load(MODEL_PATH, map_location=device, weights_only=True)
model.load_state_dict(state)
model.eval()

print(f"\nModel loaded: max_past_seq_len={max_past_seq_len}, "
      f"global_dim={global_dim}, num_ordinal={num_ordinal}")
print(f"Scaling strategy: {scalers.get('scaling_strategy', 'unknown')}")
print(f"Threshold feature mode: {scalers.get('threshold_feature_mode', THRESHOLD_FEATURE_MODE)}")
print("Ordinal head: monotonic cumulative logits")

# =========================
# Predict across thresholds
# =========================
raw_bers, raw_log, raw_regions, raw_region_probs = predict_ber_for_thresholds(
    model, case, thresholds, scalers, device,
)

if USE_TEST_TIME_SMOOTHING:
    smooth = predict_ber_smoothed_3point(
        model,
        case,
        thresholds,
        scalers,
        device,
        eps_frac=SMOOTHING_EPS_FRAC,
        smoothing_space=SMOOTHING_SPACE,
        weights=SMOOTHING_WEIGHTS,
    )
    pred_bers = smooth["ber"]
    pred_log = smooth["log10_ber"]
    pred_regions = smooth["regions"]
    pred_region_probs = smooth["region_probs"]
    print(
        f"Test-time smoothing: ON | space={smooth['smoothing_space']} | "
        f"eps_threshold={smooth['eps_threshold']:.6g}"
    )
else:
    pred_bers = raw_bers
    pred_log = raw_log
    pred_regions = raw_regions
    pred_region_probs = raw_region_probs
    print("Test-time smoothing: OFF")

pred_bers = np.clip(pred_bers, EPS, 0.5)
raw_bers = np.clip(raw_bers, EPS, 0.5)

# =========================
# Comparison table
# =========================
results = pd.DataFrame({
    "threshold": thresholds,
    "real_BER": real_bers,
    "estimated_BER": pred_bers,
    "estimated_BER_raw_unsmoothed": raw_bers,
    "smoothing_delta": pred_bers - raw_bers,
    "real_region": real_regions,
    "predicted_region": pred_regions,
    "abs_error": np.abs(pred_bers - real_bers),
    "abs_log10_error": np.abs(
        np.log10(np.clip(pred_bers, EPS, 0.5)) -
        np.log10(np.clip(real_bers, EPS, 0.5))
    ),
    "region_abs_error": np.abs(pred_regions.astype(np.int64) - real_regions.astype(np.int64)),
})

print(results.head(15))

print("\nSummary")
print("Mean abs raw error        :", results["abs_error"].mean())
print("Mean abs log10 error      :", results["abs_log10_error"].mean())
print("Max  abs log10 error      :", results["abs_log10_error"].max())
print("Mean region abs error     :", results["region_abs_error"].mean())
print("Exact region accuracy     :", np.mean(results["real_region"] == results["predicted_region"]))
if USE_TEST_TIME_SMOOTHING:
    print("Mean abs smoothing delta  :", float(np.mean(np.abs(results["smoothing_delta"]))))
    print("Max  abs smoothing delta  :", float(np.max(np.abs(results["smoothing_delta"]))))

best_real_idx = np.argmin(real_bers)
best_est_idx = np.argmin(pred_bers)

print("\nBest threshold from real BER      :", thresholds[best_real_idx])
print("Minimum real BER                  :", real_bers[best_real_idx])
print("Real BER region there             :", REGION_LABELS.get(int(real_regions[best_real_idx]), "?"))

print("Best threshold from estimated BER :", thresholds[best_est_idx])
print("Estimated BER at that threshold   :", pred_bers[best_est_idx])
print("Predicted BER region there        :", REGION_LABELS.get(int(pred_regions[best_est_idx]), "?"))

# =========================
# Plot 1: log-scale BER
# =========================
plt.figure(figsize=(10, 6))
plt.plot(results["threshold"], results["estimated_BER"], label="Estimated BER" + (" smoothed" if USE_TEST_TIME_SMOOTHING else ""))
if USE_TEST_TIME_SMOOTHING:
    plt.plot(results["threshold"], results["estimated_BER_raw_unsmoothed"], label="Estimated BER raw", alpha=0.45)
plt.plot(results["threshold"], results["real_BER"], label="Real BER")
plt.yscale("log")
plt.xlabel("Threshold")
plt.ylabel("BER")
plt.title(f"Threshold vs BER (log scale) — mem_len={case['mem_len']}")
plt.legend()
plt.grid(True, which="both", alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, "01_ber_log_scale.png"), dpi=150)
plt.close()
print(f"Saved: {PLOT_DIR}/01_ber_log_scale.png")

# =========================
# Plot 2: linear-scale BER
# =========================
plt.figure(figsize=(10, 6))
plt.plot(results["threshold"], results["estimated_BER"], label="Estimated BER" + (" smoothed" if USE_TEST_TIME_SMOOTHING else ""))
if USE_TEST_TIME_SMOOTHING:
    plt.plot(results["threshold"], results["estimated_BER_raw_unsmoothed"], label="Estimated BER raw", alpha=0.45)
plt.plot(results["threshold"], results["real_BER"], label="Real BER")
plt.yscale("linear")
plt.xlabel("Threshold")
plt.ylabel("BER")
plt.title(f"Threshold vs BER (linear scale) — mem_len={case['mem_len']}")
plt.legend()
plt.grid(True, which="both", alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, "02_ber_linear_scale.png"), dpi=150)
plt.close()
print(f"Saved: {PLOT_DIR}/02_ber_linear_scale.png")

# =========================
# Plot 3: predicted vs true region
# =========================
plt.figure(figsize=(10, 6))
plt.plot(results["threshold"], results["predicted_region"], label="Predicted region")
plt.plot(results["threshold"], results["real_region"], label="True region")
plt.xlabel("Threshold")
plt.ylabel("BER Region Class")
plt.title(f"Threshold vs BER Region — mem_len={case['mem_len']}")
plt.yticks(list(REGION_LABELS.keys()),
           [REGION_LABELS[k] for k in sorted(REGION_LABELS.keys())],
           fontsize=7)
plt.legend()
plt.grid(True, which="both", alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, "03_region_comparison.png"), dpi=150)
plt.close()
print(f"Saved: {PLOT_DIR}/03_region_comparison.png")

# =========================
# Plot 4: ordinal threshold probabilities
# =========================
plt.figure(figsize=(12, 7))
for i, thr_val in enumerate(ordinal_thresholds):
    plt.plot(thresholds, pred_region_probs[:, i], label=f"P(y >= {thr_val:g})")
plt.xlabel("Threshold")
plt.ylabel("Ordinal Probability")
plt.title(f"Ordinal Head Outputs Across Threshold Sweep — mem_len={case['mem_len']}")
plt.legend(fontsize=7, ncol=2)
plt.grid(True, which="both", alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, "04_ordinal_probabilities.png"), dpi=150)
plt.close()
print(f"Saved: {PLOT_DIR}/04_ordinal_probabilities.png")

# =========================
# Plot 5: absolute error by threshold
# =========================
plt.figure(figsize=(10, 6))
plt.plot(results["threshold"], results["abs_error"], label="Abs raw error", alpha=0.8)
plt.plot(results["threshold"], results["abs_log10_error"], label="Abs log10 error", alpha=0.8)
plt.yscale("log")
plt.xlabel("Threshold")
plt.ylabel("Error")
plt.title(f"Prediction Error Across Threshold Sweep — mem_len={case['mem_len']}")
plt.legend()
plt.grid(True, which="both", alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, "05_error_by_threshold.png"), dpi=150)
plt.close()
print(f"Saved: {PLOT_DIR}/05_error_by_threshold.png")

# =========================
# Plot 6: smoothing delta
# =========================
if USE_TEST_TIME_SMOOTHING:
    plt.figure(figsize=(10, 6))
    plt.plot(results["threshold"], results["smoothing_delta"], label="Smoothed - raw estimate")
    plt.axhline(0.0, linewidth=1.0)
    plt.xlabel("Threshold")
    plt.ylabel("BER delta")
    plt.title(f"Test-Time Smoothing Delta — mem_len={case['mem_len']}")
    plt.legend()
    plt.grid(True, which="both", alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(PLOT_DIR, "06_smoothing_delta.png"), dpi=150)
    plt.close()
    print(f"Saved: {PLOT_DIR}/06_smoothing_delta.png")


Generated physical scenario
radius    = 4.999903475776163
distance  = 13.063595613360427
diffusion = 63.15707394395149
Ts        = 0.5834416162676823
N         = 2026
mem_len   = 14
P         = [0.03545102 0.042582   0.02704795 0.01857409 0.01368091 0.01059455
 0.00850978 0.00702648 0.00592783 0.00508774 0.00442855 0.00390015
 0.00346895 0.00311165]
P_scaled  = [71.82377081 86.27113641 54.79914164 37.63111579 27.71752004 21.46456524
 17.24082032 14.23564842 12.00979138 10.30776824  8.9722469   7.90171201
  7.0280834   6.304198  ]
variances = [69.27754472 82.59753869 53.31693734 36.93215188 27.33831919 21.23715776
 17.09410468 14.13562193 11.93859933 10.25532496  8.93251283  7.87089412
  7.00370336  6.28458156]
Threshold search interval: 0.0 to 383.7075186062634
sum(P_scaled) = 383.7075186062634


/home/birkan/miniconda3/lib/python3.9/site-packages/torch/cuda/__init__.py:829: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")



Model loaded: max_past_seq_len=13, global_dim=7, num_ordinal=13
Scaling strategy: position_independent
Threshold feature mode: normalized_logit
Ordinal head: monotonic cumulative logits
Test-time smoothing: ON | space=log | eps_threshold=1.91854
    threshold  real_BER  estimated_BER  estimated_BER_raw_unsmoothed  \
0    0.000000  0.499999       0.499651                      0.499573   
1    0.768953  0.499937       0.499819                      0.499918   
2    1.537906  0.499935       0.499804                      0.499896   
3    2.306859  0.499930       0.499880                      0.499878   
4    3.075812  0.499923       0.499860                      0.499860   
5    3.844765  0.499912       0.499841                      0.499843   
6    4.613718  0.499895       0.499821                      0.499824   
7    5.382671  0.499873       0.499799                      0.499803   
8    6.151624  0.499845       0.499774                      0.499780   
9    6.920576  0.499811       0.4

In [4]:
"""
Fine-Tuning Script
==================
Continues training from the normalized-threshold + monotonic-ordinal checkpoint on new datasets.

Handles two CSV formats:
  1. Full physics format with var_i columns (existing)
  2. New format without var_i columns — variance is computed from
     binomial physics: var_i = N * tap_i * (1 - tap_i)

The new format also lacks `mem_len` in some cases — derived from the
number of non-NaN tap columns.

Fine-tuning differences vs from-scratch training:
  - Loads model & scalers from the normalized-threshold + monotonic-ordinal run
  - Lower learning rate (1/5 of base)
  - Shorter schedule (60 epochs max)
  - Smaller patience for early stopping
  - Reuses scalers from previous run by default (controlled by REUSE_SCALERS)
  - Saves to a NEW set of files so previous checkpoints stay intact
"""
import os
import re
import copy
import joblib
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torch.multiprocessing as mp
from torch.utils.data import DataLoader, TensorDataset, WeightedRandomSampler
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

mp.set_sharing_strategy("file_system")

# =========================
# Configuration
# =========================
BASE_DIR = "./"
DATA_DIR = "./../ber_data_generation/data/"

# ---- Source: previous training run to resume from ----
PRETRAINED_MODEL_PATH = os.path.join(
    BASE_DIR, "random_extra_multitask_last2_normthr_monoord.pth"
)
PRETRAINED_SCALER_PATH = os.path.join(
    BASE_DIR, "random_extra_multitask_scalers2_normthr_monoord.pkl"
)
# If True, keep using the pretrained scalers (recommended — keeps the model
# and scalers consistent). If False, refit scalers on the fine-tune data.
REUSE_SCALERS = True

# ---- Destination: NEW files for fine-tuned model ----
BEST_MODEL_SAVE_PATH = os.path.join(
    BASE_DIR, "random_extra_multitask_finetune_best_normthr_monoord.pth"
)
LAST_MODEL_SAVE_PATH = os.path.join(
    BASE_DIR, "random_extra_multitask_finetune_last_normthr_monoord.pth"
)
SCALER_SAVE_PATH = os.path.join(
    BASE_DIR, "random_extra_multitask_finetune_scalers_normthr_monoord.pkl"
)

# ---- Fine-tune data (no var_* columns; will be computed) ----
DATA_PATHS = [
    os.path.join(DATA_DIR, "data_physics_total.csv"),
    os.path.join(DATA_DIR, "data_random_total.csv"),
]

NROWS_PER_DATASET = 5_000_000

# ---- Fine-tune hyperparameters (gentler than from-scratch) ----
FINETUNE_LR = 2e-5         # ~1/5 of base 1e-4 — preserve pretrained features
FINETUNE_WEIGHT_DECAY = 1e-5
FINETUNE_MAX_EPOCHS = 60
FINETUNE_MIN_EPOCHS = 25
FINETUNE_PATIENCE = 8
SCHEDULER_PATIENCE = 3
SCHEDULER_FACTOR = 0.5
GRAD_CLIP = 1.0

EPS = 1e-12
LOG10_HALF = float(np.log10(0.5))

# =========================
# New-model threshold / numeric feature options
# =========================
# Must match the normalized-threshold model trained from scratch.
# If REUSE_SCALERS=True, the mode saved in PRETRAINED_SCALER_PATH is used.
THRESHOLD_FEATURE_MODE = "normalized_logit"
THRESHOLD_NORM_CLIP_EPS = 1e-5
HARMONIC_DENOM_EPS = 1e-8
FEATURE_CLIP_ABS = 1e6

ORDINAL_THRESHOLDS = [
    1e-6, 1e-5, 1e-4, 1e-3, 1e-2, 1e-1,
    0.15, 0.20, 0.25, 0.30, 0.35, 0.40, 0.45,
]
NUM_ORDINAL_THRESHOLDS = len(ORDINAL_THRESHOLDS)
NUM_REGION_CLASSES = NUM_ORDINAL_THRESHOLDS + 1


# =========================
# Utilities
# =========================
def get_sorted_seq_cols(columns, prefix):
    pattern = re.compile(rf"^{re.escape(prefix)}_(\d+)$")
    matched = []
    for col in columns:
        m = pattern.match(col)
        if m:
            matched.append((int(m.group(1)), col))
    matched.sort(key=lambda x: x[0])
    return [col for _, col in matched]


def make_strat_bins(y_log, n_bins=10):
    y_flat = y_log.reshape(-1)
    quantiles = np.linspace(0, 1, n_bins + 1)
    edges = np.quantile(y_flat, quantiles)
    edges = np.unique(edges)
    if len(edges) < 3:
        return None
    bins = np.digitize(y_flat, edges[1:-1], right=True)
    counts = np.bincount(bins)
    if np.any(counts < 2):
        return None
    return bins


def has_nonfinite_tensor(x):
    return not torch.isfinite(x).all().item()


def validate_required_columns(df, file_path, allow_missing_vars=True):
    """Validate columns. With allow_missing_vars=True, var_* columns are
    optional and will be computed from taps if absent."""
    if "N" not in df.columns:
        raise ValueError(f"Required column 'N' not found in {file_path}")

    tap_cols = get_sorted_seq_cols(df.columns, "tap")
    var_cols = get_sorted_seq_cols(df.columns, "var")

    if not tap_cols:
        raise ValueError(f"No tap_* columns found in {file_path}")

    if var_cols:
        if len(tap_cols) != len(var_cols):
            raise ValueError(
                f"tap/var length mismatch in {file_path}: "
                f"{len(tap_cols)} tap cols vs {len(var_cols)} var cols"
            )
        has_vars = True
    else:
        if not allow_missing_vars:
            raise ValueError(f"No var_* columns found in {file_path}")
        has_vars = False

    base_required = tap_cols + ["threshold", "BER", "N"]
    if has_vars:
        base_required = base_required + var_cols
    missing = [c for c in base_required if c not in df.columns]
    if missing:
        raise ValueError(f"Missing required columns in {file_path}: {missing}")

    return tap_cols, var_cols, has_vars


def derive_mem_len_from_taps(df, tap_cols):
    """If mem_len column is missing, derive it from the number of leading
    non-null tap columns. Assumes tap_1..tap_K are filled and tap_(K+1)+ are NaN."""
    if "mem_len" in df.columns:
        return df["mem_len"].to_numpy(dtype=np.int64)

    print(f"  mem_len missing; deriving from non-null tap counts.")
    non_null = df[tap_cols].notna().to_numpy()
    # mem_len = number of non-null taps
    derived = non_null.sum(axis=1).astype(np.int64)
    return derived


def compute_variances_from_taps(taps_raw, N_array, valid_mask):
    """Compute binomial variance var_i = N * tap_i * (1 - tap_i) for each
    valid position. Padding positions (where valid_mask==False) get 0.

    Args:
        taps_raw: [N, L] raw tap probabilities (may have NaN/0 in pad positions)
        N_array:  [N] number of molecules
        valid_mask: [N, L] bool, True = real tap

    Returns:
        vars_raw: [N, L] variance values (raw, not multiplied by N already)
                   matches the convention of the var_* columns in original CSVs
    """
    N = N_array.reshape(-1, 1).astype(np.float64)
    taps = np.where(np.isnan(taps_raw), 0.0, taps_raw).astype(np.float64)
    taps = np.clip(taps, 0.0, 1.0)
    vars_full = N * taps * (1.0 - taps)
    # Re-zero invalid positions
    vars_full = vars_full * valid_mask.astype(np.float64)
    return vars_full.astype(np.float32)


def _read_csv_robust(path, nrows):
    """Read a CSV, skipping any rows that have MORE fields than the header
    declares (treated as corrupted). The C engine with on_bad_lines='skip'
    handles this efficiently — pandas drops over-long rows and keeps only
    the well-formed ones.
    """
    try:
        df = pd.read_csv(path, nrows=nrows, on_bad_lines="skip", engine="c")
    except (ValueError, pd.errors.ParserError) as e:
        # Fallback: python engine is more lenient
        print(f"  C engine failed ({e}); using python engine")
        df = pd.read_csv(path, nrows=nrows, on_bad_lines="skip", engine="python")
    return df


def load_and_merge_data(csv_paths, nrows_per_dataset):
    dfs = []
    reference_tap_cols = None
    union_var_cols = None
    has_vars_per_file = []

    for path in csv_paths:
        if not os.path.exists(path):
            raise FileNotFoundError(f"Dataset not found: {path}")

        print(f"Loading up to {nrows_per_dataset:,} rows from: {path}")
        df = _read_csv_robust(path, nrows=nrows_per_dataset)

        tap_cols, var_cols, has_vars = validate_required_columns(
            df, path, allow_missing_vars=True
        )

        if reference_tap_cols is None:
            reference_tap_cols = tap_cols
            union_var_cols = var_cols if has_vars else []
        else:
            # Reconcile tap columns across files: use the UNION sorted by index.
            # If one file has more tap columns than another, missing columns
            # will be filled with NaN (treated as padding downstream).
            if tap_cols != reference_tap_cols:
                ref_set = set(reference_tap_cols)
                this_set = set(tap_cols)
                only_here = sorted(this_set - ref_set,
                                   key=lambda c: int(c.split("_")[1]))
                only_ref = sorted(ref_set - this_set,
                                  key=lambda c: int(c.split("_")[1]))
                if only_here:
                    print(f"  File adds {len(only_here)} tap cols not in "
                          f"reference (e.g., {only_here[:3]}...). Reference will be extended.")
                if only_ref:
                    print(f"  File missing {len(only_ref)} reference tap cols "
                          f"(e.g., {only_ref[:3]}...); will be NaN-filled.")
                # Extend the reference with any new columns
                reference_tap_cols = sorted(
                    ref_set | this_set,
                    key=lambda c: int(c.split("_")[1]),
                )
                # Add NaN columns to existing dfs that lack the new cols
                for prev in dfs:
                    for c in only_here:
                        if c not in prev.columns:
                            prev[c] = np.nan
                # Add NaN columns to current df for missing ref cols
                for c in only_ref:
                    if c not in df.columns:
                        df[c] = np.nan

            if has_vars and not union_var_cols:
                union_var_cols = var_cols

        df["source_dataset"] = os.path.basename(path)
        df["_has_vars"] = has_vars
        has_vars_per_file.append(has_vars)
        dfs.append(df)

    merged = pd.concat(dfs, ignore_index=True)
    print(f"Combined rows before filtering: {len(merged):,}")
    print(f"  Files with var_* columns: {sum(has_vars_per_file)}/{len(has_vars_per_file)}")
    print(f"  Final tap column count: {len(reference_tap_cols)}")

    numeric_cols = list(dict.fromkeys(
        reference_tap_cols + (union_var_cols or []) + ["mem_len", "threshold", "BER", "N"]
    ))
    merged, numeric_bad_by_col = coerce_numeric_columns(
        merged, numeric_cols, context="merged fine-tune data"
    )
    if numeric_bad_by_col:
        print("Malformed numeric fields were found after CSV alignment; invalid required rows "
              "and invalid valid-tap rows will be dropped during preprocessing.")

    return merged, reference_tap_cols, union_var_cols


def y_log_to_raw_np(y_log):
    return np.clip(10 ** y_log, EPS, 0.5)


def raw_to_region_labels_np(y_raw):
    y = np.asarray(y_raw).reshape(-1)
    labels = np.searchsorted(np.asarray(ORDINAL_THRESHOLDS), y, side="right")
    return labels.astype(np.int64)


def region_labels_to_ordinal_targets_np(labels, num_thresholds=NUM_ORDINAL_THRESHOLDS):
    labels = np.asarray(labels).reshape(-1)
    thresholds = np.arange(1, num_thresholds + 1, dtype=np.int64)
    ordinal = (labels[:, None] >= thresholds[None, :]).astype(np.float32)
    return ordinal


def ordinal_logits_to_region_labels_torch(logits):
    probs = torch.sigmoid(logits)
    passed = (probs >= 0.5).sum(dim=1)
    return passed.long()


def compute_region_class_weights(labels_np, num_classes=NUM_REGION_CLASSES, max_weight=8.0):
    counts = np.bincount(labels_np.reshape(-1), minlength=num_classes).astype(np.float64)
    counts = np.maximum(counts, 1.0)
    weights = counts.sum() / (num_classes * counts)
    weights = np.sqrt(weights)
    weights = np.clip(weights, 1.0, max_weight)
    return weights.astype(np.float32)


def compute_ordinal_pos_weights_from_region_labels(
    region_labels_np,
    num_thresholds=NUM_ORDINAL_THRESHOLDS,
    max_weight=20.0,
):
    ordinal_targets = region_labels_to_ordinal_targets_np(region_labels_np, num_thresholds)
    pos_counts = ordinal_targets.sum(axis=0)
    neg_counts = ordinal_targets.shape[0] - pos_counts
    pos_counts = np.maximum(pos_counts, 1.0)
    pos_weight = neg_counts / pos_counts
    pos_weight = np.clip(pos_weight, 1.0, max_weight)
    return pos_weight.astype(np.float32)


# =========================
# Position-Independent Scaling helpers
# =========================
def fit_shared_scalar_scaler(train_data, train_valid_mask):
    entries = train_data[train_valid_mask].reshape(-1)
    if len(entries) == 0:
        entries = train_data.reshape(-1)
    mean = float(entries.mean())
    std = float(max(entries.std(), 1e-12))
    return mean, std


def apply_shared_scale(data, mean, std, valid_mask=None):
    scaled = ((data - mean) / std).astype(np.float32)
    if valid_mask is not None and scaled.ndim == 2 and valid_mask.shape == scaled.shape:
        scaled = scaled * valid_mask.astype(np.float32)
    return scaled


def sanitize_np_feature(x, name, clip_abs=FEATURE_CLIP_ABS):
    """Replace NaN/inf with 0 and clip extreme engineered features."""
    arr = np.asarray(x, dtype=np.float32)
    bad = ~np.isfinite(arr)
    if np.any(bad):
        print(f"Warning: {name} contained {int(bad.sum())} non-finite values; replacing with 0.")
    arr = np.nan_to_num(arr, nan=0.0, posinf=0.0, neginf=0.0)
    arr = np.clip(arr, -clip_abs, clip_abs).astype(np.float32)
    return arr


def stable_harmonic_pair_np(a, b, denom_eps=HARMONIC_DENOM_EPS):
    """Stable 2ab/(a+b), with near-zero denominators mapped to zero."""
    a = np.asarray(a, dtype=np.float32)
    b = np.asarray(b, dtype=np.float32)
    denom = (a + b).astype(np.float32)
    numer = (2.0 * a * b).astype(np.float32)
    out = np.divide(
        numer, denom,
        out=np.zeros_like(numer, dtype=np.float32),
        where=np.abs(denom) > denom_eps,
    ).astype(np.float32)
    return sanitize_np_feature(out, "stable_harmonic_pair")


def compute_threshold_feature_np(threshold_raw, total_scaled_taps, mode=THRESHOLD_FEATURE_MODE):
    """Build model threshold feature.

    normalized_logit: logit(clip(threshold / sum(valid tap_i*N), eps, 1-eps))
    normalized:       threshold / sum(valid tap_i*N)
    log10_raw:        legacy log10(threshold + EPS)
    """
    thr = np.asarray(threshold_raw, dtype=np.float32).reshape(-1, 1)
    total = np.asarray(total_scaled_taps, dtype=np.float32).reshape(-1, 1)
    denom = np.maximum(total, EPS).astype(np.float32)

    if mode == "normalized":
        feat = (thr / denom).astype(np.float32)
    elif mode == "normalized_logit":
        u = (thr / denom).astype(np.float32)
        u = np.clip(u, THRESHOLD_NORM_CLIP_EPS, 1.0 - THRESHOLD_NORM_CLIP_EPS)
        feat = np.log(u / (1.0 - u)).astype(np.float32)
    elif mode == "log10_raw":
        feat = np.log10(thr + EPS).astype(np.float32)
    else:
        raise ValueError(f"Unknown THRESHOLD_FEATURE_MODE: {mode}")

    return sanitize_np_feature(feat, f"threshold_feature_{mode}")


def coerce_numeric_columns(df, numeric_cols, context="dataframe"):
    """Coerce numeric CSV columns and report malformed cells."""
    bad_by_col = {}
    for col in numeric_cols:
        if col not in df.columns:
            continue
        original = df[col]
        converted = pd.to_numeric(original, errors="coerce")
        bad = converted.isna() & original.notna() & (original.astype(str).str.strip() != "")
        n_bad = int(bad.sum())
        if n_bad:
            bad_by_col[col] = n_bad
        df[col] = converted
    if bad_by_col:
        total_bad = sum(bad_by_col.values())
        top = sorted(bad_by_col.items(), key=lambda kv: kv[1], reverse=True)[:8]
        print(f"{context}: coerced {total_bad:,} malformed numeric cells to NaN. Top columns: {top}")
    return df, bad_by_col


# =========================
# Loss
# =========================
class StableMultiObjectiveBERBoundedLogLoss(nn.Module):
    def __init__(self, log_delta=0.35, raw_delta=0.006, rel_delta=0.03,
                 alpha_log=0.45, beta_raw=0.35, gamma_rel=0.20,
                 use_regime_weights=True):
        super().__init__()
        self.log_delta = log_delta
        self.raw_delta = raw_delta
        self.rel_delta = rel_delta
        self.alpha_log = alpha_log
        self.beta_raw = beta_raw
        self.gamma_rel = gamma_rel
        self.use_regime_weights = use_regime_weights

    @staticmethod
    def huber_elementwise(pred, target, delta):
        err = pred - target
        abs_err = err.abs()
        return torch.where(abs_err < delta,
                           0.5 * err * err,
                           delta * (abs_err - 0.5 * delta))

    @staticmethod
    def raw_to_pred_log(pred_raw_unconstrained):
        log10_half = torch.log10(torch.tensor(
            0.5, device=pred_raw_unconstrained.device,
            dtype=pred_raw_unconstrained.dtype))
        return log10_half - F.softplus(pred_raw_unconstrained)

    @staticmethod
    def raw_to_pred_ber(pred_raw_unconstrained):
        pred_log = StableMultiObjectiveBERBoundedLogLoss.raw_to_pred_log(pred_raw_unconstrained)
        return torch.pow(10.0, pred_log).clamp(min=EPS, max=0.5)

    def forward(self, pred_raw_unconstrained, target_log):
        pred_log = self.raw_to_pred_log(pred_raw_unconstrained)
        pred_raw = torch.pow(10.0, pred_log).clamp(min=EPS, max=0.5)
        target_raw = torch.pow(10.0, target_log).clamp(min=EPS, max=0.5)
        target_log_for_loss = torch.log10(target_raw)

        log_loss = self.huber_elementwise(pred_log, target_log_for_loss, self.log_delta)
        raw_loss = self.huber_elementwise(pred_raw, target_raw, self.raw_delta)
        rel_err = (pred_raw - target_raw) / torch.clamp(target_raw, min=1e-6)
        rel_loss = self.huber_elementwise(rel_err, torch.zeros_like(rel_err), self.rel_delta)

        total = (self.alpha_log * log_loss
                 + self.beta_raw * raw_loss
                 + self.gamma_rel * rel_loss)

        if self.use_regime_weights:
            weights = torch.ones_like(target_raw)
            weights = torch.where((target_raw >= 1e-2) & (target_raw < 0.1),
                                  torch.full_like(weights, 1.5), weights)
            weights = torch.where((target_raw >= 0.1) & (target_raw < 0.15),
                                  torch.full_like(weights, 2.5), weights)
            weights = torch.where((target_raw >= 0.15) & (target_raw < 0.2),
                                  torch.full_like(weights, 3.0), weights)
            weights = torch.where((target_raw >= 0.2) & (target_raw < 0.3),
                                  torch.full_like(weights, 4.0), weights)
            weights = torch.where((target_raw >= 0.3) & (target_raw < 0.4),
                                  torch.full_like(weights, 4.5), weights)
            weights = torch.where((target_raw >= 0.4) & (target_raw < 0.45),
                                  torch.full_like(weights, 3.0), weights)
            weights = torch.where(target_raw >= 0.45,
                                  torch.full_like(weights, 2.5), weights)
            total = total * weights

        return total.mean()


class OrdinalBCELoss(nn.Module):
    def __init__(self, pos_weight=None, reduction="mean"):
        super().__init__()
        if pos_weight is not None and not isinstance(pos_weight, torch.Tensor):
            pos_weight = torch.tensor(pos_weight, dtype=torch.float32)
        self.register_buffer("pos_weight", pos_weight if pos_weight is not None else None)
        self.reduction = reduction

    def forward(self, logits, ordinal_targets):
        return F.binary_cross_entropy_with_logits(
            logits, ordinal_targets, pos_weight=self.pos_weight, reduction=self.reduction)


class MultiTaskBERLoss(nn.Module):
    def __init__(self, reg_loss, ord_loss, lambda_ord=0.25):
        super().__init__()
        self.reg_loss = reg_loss
        self.ord_loss = ord_loss
        self.lambda_ord = lambda_ord

    def forward(self, pred_raw_unconstrained, ord_logits, target_log, target_ord):
        reg = self.reg_loss(pred_raw_unconstrained, target_log)
        ordl = self.ord_loss(ord_logits, target_ord)
        total = reg + self.lambda_ord * ordl
        return total, reg.detach(), ordl.detach()


# =========================
# Set Transformer Model (normalized-threshold + monotonic-ordinal architecture)
# =========================
class SetSelfAttentionBlock(nn.Module):
    def __init__(self, d_model, num_heads, mlp_ratio=4.0, dropout=0.1):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model)
        self.attn = nn.MultiheadAttention(
            embed_dim=d_model, num_heads=num_heads,
            dropout=dropout, batch_first=True)
        self.norm2 = nn.LayerNorm(d_model)
        hidden = int(d_model * mlp_ratio)
        self.mlp = nn.Sequential(
            nn.Linear(d_model, hidden), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(hidden, d_model), nn.Dropout(dropout))

    def forward(self, x, key_padding_mask=None):
        y = self.norm1(x)
        attn_out, _ = self.attn(y, y, y, need_weights=False, key_padding_mask=key_padding_mask)
        x = x + attn_out
        x = x + self.mlp(self.norm2(x))
        return x


class ConditionalFeatureModulation(nn.Module):
    def __init__(self, cond_dim, feat_dim, hidden_dim=256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(cond_dim, hidden_dim), nn.GELU(),
            nn.Linear(hidden_dim, 2 * feat_dim))
        self.scale = nn.Parameter(torch.tensor(0.25, dtype=torch.float32))

    def forward(self, x, cond):
        gamma_beta = self.net(cond)
        gamma, beta = torch.chunk(gamma_beta, 2, dim=-1)
        if x.dim() == 3:
            gamma = gamma.unsqueeze(1)
            beta = beta.unsqueeze(1)
        return x * (1.0 + self.scale * gamma) + self.scale * beta


class SetAttentionPooling(nn.Module):
    def __init__(self, d_model, hidden_dim=128):
        super().__init__()
        self.score = nn.Sequential(
            nn.Linear(d_model, hidden_dim), nn.GELU(), nn.Linear(hidden_dim, 1))

    def forward(self, x, key_padding_mask=None):
        logits = self.score(x)
        if key_padding_mask is not None:
            logits = logits.masked_fill(key_padding_mask.unsqueeze(-1), float("-inf"))
        weights = torch.softmax(logits, dim=1)
        weights = torch.nan_to_num(weights, nan=0.0)
        return (weights * x).sum(dim=1)


class MonotonicOrdinalHead(nn.Module):
    """Ordinal head with monotonic probabilities by construction.

    The ordinal targets are P(BER >= threshold_k), which must be non-increasing
    as the threshold index k increases. The head constructs non-increasing logits:
        logit_1 = free
        logit_k = logit_{k-1} - softplus(delta_k)
    Since sigmoid is monotone, probabilities are also non-increasing.
    """
    def __init__(self, head_in, num_thresholds):
        super().__init__()
        self.num_thresholds = num_thresholds
        self.base = nn.Sequential(
            nn.Linear(head_in, 256), nn.GELU(), nn.Dropout(0.20),
            nn.Linear(256, 128), nn.GELU(), nn.Dropout(0.15),
        )
        self.first_logit = nn.Linear(128, 1)
        self.deltas = nn.Linear(128, max(num_thresholds - 1, 1))

    def forward(self, x):
        h = self.base(x)
        first = self.first_logit(h)
        if self.num_thresholds == 1:
            return first
        deltas = -F.softplus(self.deltas(h)[:, : self.num_thresholds - 1])
        return torch.cat([first, deltas], dim=-1).cumsum(dim=-1)


class BERFirstTokenSetTransformerBoundedLogHalfMultiTask(nn.Module):
    def __init__(self, max_past_seq_len, token_dim=4, first_token_dim=4,
                 threshold_dim=1, global_dim=7, d_model=128, num_set_layers=4,
                 num_heads=4, mlp_ratio=4.0, dropout=0.10,
                 num_ordinal_thresholds=NUM_ORDINAL_THRESHOLDS):
        super().__init__()
        self.max_past_seq_len = max_past_seq_len
        self.d_model = d_model
        self.num_ordinal_thresholds = num_ordinal_thresholds

        self.first_proj = nn.Sequential(
            nn.Linear(first_token_dim, d_model), nn.LayerNorm(d_model),
            nn.GELU(), nn.Dropout(dropout))
        self.token_proj = nn.Sequential(
            nn.Linear(token_dim, d_model), nn.LayerNorm(d_model),
            nn.GELU(), nn.Dropout(dropout))
        self.thr_embed = nn.Sequential(
            nn.Linear(threshold_dim, 32), nn.GELU(),
            nn.Linear(32, 32), nn.GELU())
        self.global_embed = nn.Sequential(
            nn.Linear(global_dim, 96), nn.GELU(),
            nn.Linear(96, 96), nn.GELU())

        cond_dim = 32 + 96
        self.first_cond_mod = ConditionalFeatureModulation(cond_dim, d_model, 256)
        self.set_cond_mod = ConditionalFeatureModulation(cond_dim, d_model, 256)

        self.set_blocks = nn.ModuleList([
            SetSelfAttentionBlock(d_model, num_heads, mlp_ratio, dropout)
            for _ in range(num_set_layers)])

        self.first_refine = nn.Sequential(
            nn.LayerNorm(d_model), nn.Linear(d_model, d_model),
            nn.GELU(), nn.Dropout(dropout), nn.Linear(d_model, d_model))
        self.final_set_norm = nn.LayerNorm(d_model)
        self.set_attn_pool = SetAttentionPooling(d_model, 128)

        head_in = d_model + 3 * d_model + cond_dim
        self.reg_head = nn.Sequential(
            nn.Linear(head_in, 256), nn.GELU(), nn.Dropout(0.20),
            nn.Linear(256, 128), nn.GELU(), nn.Dropout(0.15), nn.Linear(128, 1))
        self.ord_head = MonotonicOrdinalHead(head_in, num_ordinal_thresholds)

    @staticmethod
    def raw_to_log10ber(x):
        log10_half = torch.log10(torch.tensor(0.5, device=x.device, dtype=x.dtype))
        return log10_half - F.softplus(x)

    @staticmethod
    def raw_to_ber(x):
        return torch.pow(10.0, BERFirstTokenSetTransformerBoundedLogHalfMultiTask.raw_to_log10ber(x)).clamp(EPS, 0.5)

    def forward(self, first_token, past_tokens, global_feats, threshold, key_padding_mask=None):
        thr = self.thr_embed(threshold)
        g = self.global_embed(global_feats)
        cond = torch.cat([thr, g], dim=-1)

        first_x = self.first_proj(first_token)
        first_x = self.first_cond_mod(first_x, cond)
        first_x = first_x + self.first_refine(first_x)

        set_x = self.token_proj(past_tokens)
        set_x = self.set_cond_mod(set_x, cond)
        for block in self.set_blocks:
            set_x = block(set_x, key_padding_mask=key_padding_mask)
        set_x = self.final_set_norm(set_x)

        if key_padding_mask is not None:
            valid_mask = (~key_padding_mask).unsqueeze(-1).float()
        else:
            valid_mask = torch.ones(set_x.shape[0], set_x.shape[1], 1,
                                    device=set_x.device, dtype=set_x.dtype)

        pooled_attn = self.set_attn_pool(set_x, key_padding_mask=key_padding_mask)
        valid_counts = valid_mask.sum(dim=1).clamp(min=1.0)
        pooled_mean = (set_x * valid_mask).sum(dim=1) / valid_counts

        if key_padding_mask is not None:
            x_for_max = set_x.masked_fill(key_padding_mask.unsqueeze(-1), float("-inf"))
        else:
            x_for_max = set_x
        pooled_max = torch.nan_to_num(x_for_max.amax(dim=1), nan=0.0, posinf=0.0, neginf=0.0)

        fused = torch.cat([first_x, pooled_attn, pooled_mean, pooled_max, cond], dim=-1)
        return self.reg_head(fused), self.ord_head(fused)


# =========================
# Data preparation (handles missing var_* columns)
# =========================
def prepare_data_for_finetune(
    csv_paths, batch_size=256, nrows_per_dataset=2_500_000, num_workers=0,
    pretrained_scalers=None, reuse_scalers=True,
):
    """Same pipeline as v4's prepare_data, plus:
       - var_* columns are computed from taps if missing
       - mem_len column is derived from non-null taps if missing
       - if reuse_scalers=True and pretrained_scalers is provided, those
         scalers are reused (recommended for fine-tuning)
    """
    df, tap_cols, var_cols = load_and_merge_data(csv_paths, nrows_per_dataset)

    # ---- Derive mem_len if missing ----
    if "mem_len" not in df.columns:
        df["mem_len"] = derive_mem_len_from_taps(df, tap_cols)

    # Drop malformed/non-numeric mem_len rows, then remove one-tap cases.
    n_before_mem = len(df)
    df = df[pd.to_numeric(df["mem_len"], errors="coerce").notna()].copy()
    if len(df) < n_before_mem:
        print(f"  Dropped {n_before_mem - len(df):,} rows with invalid mem_len")
    df["mem_len"] = df["mem_len"].astype(np.int64)
    df = df[df["mem_len"] != 1].copy()

    # Do not fill tap NaNs yet: NaN inside the valid mem_len region indicates
    # a malformed/corrupted required tap and is dropped below. NaNs outside the
    # valid region are padding and are zero-filled after the mask is built.
    df["threshold"] = df["threshold"].fillna(0.0)
    df["BER"] = df["BER"].fillna(0.0)
    df["N"] = df["N"].fillna(0.0)

    df = df[(df["BER"] > 0) & (df["N"] > 0)].copy()
    df["BER"] = df["BER"].clip(lower=EPS, upper=0.5)

    # Allow threshold==0 rows? In original training threshold>0 was required.
    # The new random data has rows with threshold=0; filter them out
    # because they don't correspond to a real decision threshold.
    n_before = len(df)
    df = df[df["threshold"] > 0].copy()
    if len(df) < n_before:
        print(f"  Dropped {n_before - len(df):,} rows with threshold=0")

    print(f"Rows after cleaning/filtering: {len(df):,}")

    # ---- Extract raw arrays ----
    X_taps_raw = df[tap_cols].to_numpy(dtype=np.float32)
    num_molecules = df["N"].to_numpy(dtype=np.float32).reshape(-1, 1)
    mem_len = df["mem_len"].to_numpy(dtype=np.int64)

    X_thr_raw = df["threshold"].to_numpy(dtype=np.float32).reshape(-1, 1)
    y_raw = df["BER"].to_numpy(dtype=np.float32).reshape(-1, 1)

    # X_thr is computed after valid-memory masking because normalized threshold
    # needs sum(valid tap_i * N). y_log remains the regression target.
    y_log = np.log10(y_raw + EPS).astype(np.float32)

    L_full = X_taps_raw.shape[1]
    mem_len_clamped = np.minimum(mem_len, L_full)
    valid_full = (np.arange(L_full)[None, :] < mem_len_clamped[:, None])
    valid_past = valid_full[:, 1:]
    valid_past_float = valid_past.astype(np.float32)
    past_lens = np.maximum(mem_len_clamped - 1, 0).astype(np.int64)

    print(f"  Tap column count L_full = {L_full}, "
          f"mem_len range = [{int(mem_len.min())}, {int(mem_len.max())}], "
          f"past_lens range = [{int(past_lens.min())}, {int(past_lens.max())}]")
    if mem_len.max() > L_full:
        print(f"    NOTE: {(mem_len > L_full).sum():,} rows have mem_len > L_full; "
              f"will be clamped to {L_full}")

    bad_valid_tap = (~np.isfinite(X_taps_raw)) & valid_full
    bad_valid_tap_rows = bad_valid_tap.any(axis=1)
    if np.any(bad_valid_tap_rows):
        n_bad = int(bad_valid_tap_rows.sum())
        print(f"  Dropping {n_bad:,} rows with malformed/non-numeric tap values inside valid mem_len region.")
        keep = ~bad_valid_tap_rows
        df = df.iloc[keep].copy()
        X_taps_raw = X_taps_raw[keep]
        num_molecules = num_molecules[keep]
        mem_len = mem_len[keep]
        X_thr_raw = X_thr_raw[keep]
        y_raw = y_raw[keep]
        y_log = y_log[keep]
        mem_len_clamped = mem_len_clamped[keep]
        valid_full = valid_full[keep]
        valid_past = valid_past[keep]
        valid_past_float = valid_past_float[keep]
        past_lens = past_lens[keep]

    # Padding/outside valid memory can safely be zero-filled now.
    X_taps_raw = np.nan_to_num(X_taps_raw, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)
    valid_full_float = valid_full.astype(np.float32)

    # ---- Variances: load from CSV if present, else compute from taps ----
    if var_cols and all(c in df.columns for c in var_cols):
        # File has var columns — use them, fill missing with computed values per row
        X_vars_raw_csv = df[var_cols].to_numpy(dtype=np.float32)

        # Identify rows where _has_vars indicates the file had var columns
        if "_has_vars" in df.columns:
            has_vars_mask = df["_has_vars"].to_numpy(dtype=bool)
        else:
            has_vars_mask = np.ones(len(df), dtype=bool)

        # Compute variance for ALL rows, then overlay CSV values where available
        X_vars_computed = compute_variances_from_taps(X_taps_raw, num_molecules.reshape(-1), valid_full)

        X_vars_raw = X_vars_computed.copy()
        # For rows that DO have var columns in CSV, use those values where non-NaN
        has_vars_idx = np.where(has_vars_mask)[0]
        if len(has_vars_idx) > 0:
            csv_vars = np.where(np.isnan(X_vars_raw_csv[has_vars_idx]),
                                X_vars_computed[has_vars_idx],
                                X_vars_raw_csv[has_vars_idx])
            X_vars_raw[has_vars_idx] = csv_vars

        n_computed = (~has_vars_mask).sum()
        print(f"  Variances: {has_vars_mask.sum():,} loaded from CSV, "
              f"{n_computed:,} computed from N*tap*(1-tap)")
    else:
        # No var columns at all — compute everything
        X_vars_raw = compute_variances_from_taps(
            X_taps_raw, num_molecules.reshape(-1), valid_full)
        print(f"  Variances: all {len(df):,} computed from N*tap*(1-tap)")

    # Sanity: variances should be non-negative and zero on padding
    if np.any(X_vars_raw < 0):
        # Numerical garbage; clamp
        X_vars_raw = np.maximum(X_vars_raw, 0.0)

    # ---- Feature engineering (matches normalized-threshold model) ----
    X_taps_feat = (X_taps_raw * num_molecules).astype(np.float32)
    X_vars_feat = X_vars_raw.astype(np.float32)
    abs_taps_raw = np.abs(X_taps_feat).astype(np.float32)
    snr_raw = np.log10((X_taps_feat ** 2) / (X_vars_raw + EPS) + EPS).astype(np.float32)
    snr_raw = sanitize_np_feature(snr_raw, "snr_raw")

    threshold_mode = (
        pretrained_scalers.get("threshold_feature_mode", THRESHOLD_FEATURE_MODE)
        if (reuse_scalers and pretrained_scalers is not None)
        else THRESHOLD_FEATURE_MODE
    )
    total_scaled_taps_raw = np.sum(
        X_taps_feat * valid_full_float,
        axis=1, keepdims=True,
    ).astype(np.float32)
    X_thr = compute_threshold_feature_np(
        X_thr_raw, total_scaled_taps_raw, mode=threshold_mode
    )

    # ---- Mask-aware sums ----
    first_mean = X_taps_feat[:, 0:1]
    past_means = X_taps_feat[:, 1:]
    first_var = X_vars_feat[:, 0:1]
    past_vars = X_vars_feat[:, 1:]

    past_means_masked = past_means * valid_past_float
    past_vars_masked = past_vars * valid_past_float

    mu0_raw = (0.5 * np.sum(past_means_masked, axis=1, keepdims=True)).astype(np.float32)
    mu1_raw = (first_mean + mu0_raw).astype(np.float32)
    var0_raw = (0.5 * np.sum(past_vars_masked, axis=1, keepdims=True)).astype(np.float32)
    var1_raw = (first_var + var0_raw).astype(np.float32)

    std0_raw = np.sqrt(np.maximum(var0_raw, EPS)).astype(np.float32)
    std1_raw = np.sqrt(np.maximum(var1_raw, EPS)).astype(np.float32)

    z0_raw = ((X_thr_raw - mu0_raw) / (std0_raw + EPS)).astype(np.float32)
    z1_raw = ((mu1_raw - X_thr_raw) / (std1_raw + EPS)).astype(np.float32)

    z0_raw = sanitize_np_feature(z0_raw, "z0_raw")
    z1_raw = sanitize_np_feature(z1_raw, "z1_raw")
    harmonic_side_z_raw = stable_harmonic_pair_np(z0_raw, z1_raw)
    abs_diff_side_z_raw = np.abs(z0_raw - z1_raw).astype(np.float32)
    harmonic_minus_gap_raw = sanitize_np_feature(
        harmonic_side_z_raw - 0.25 * abs_diff_side_z_raw,
        "harmonic_minus_gap_raw",
    )

    # ---- Scenario-level features ----
    signal_raw = first_mean
    isi_raw = np.sum(past_means_masked, axis=1, keepdims=True)
    nsid_raw = ((signal_raw - isi_raw) / (signal_raw + isi_raw + EPS)).astype(np.float32)

    gap_raw = signal_raw
    d0_raw = (gap_raw / (std0_raw + EPS)).astype(np.float32)
    d1_raw = (gap_raw / (std1_raw + EPS)).astype(np.float32)
    harmonic_discrim_raw = (2.0 * d0_raw * d1_raw / (d0_raw + d1_raw + EPS)).astype(np.float32)
    log_harmonic_discrim_raw = np.log10(harmonic_discrim_raw + EPS).astype(np.float32)
    discrim_asymmetry_raw = (np.abs(d0_raw - d1_raw) / (d0_raw + d1_raw + EPS)).astype(np.float32)

    past_taps_sum = np.sum(past_means_masked, axis=1, keepdims=True)
    past_shares = past_means_masked / (past_taps_sum + EPS)
    herfindahl_raw = np.sum(past_shares ** 2, axis=1, keepdims=True).astype(np.float32)

    nsid_raw = sanitize_np_feature(nsid_raw, "nsid_raw")
    log_harmonic_discrim_raw = sanitize_np_feature(log_harmonic_discrim_raw, "log_harmonic_discrim_raw")
    discrim_asymmetry_raw = sanitize_np_feature(discrim_asymmetry_raw, "discrim_asymmetry_raw")
    herfindahl_raw = sanitize_np_feature(herfindahl_raw, "herfindahl_raw")

    # ---- Labels ----
    region_labels = raw_to_region_labels_np(y_raw)
    ordinal_targets = region_labels_to_ordinal_targets_np(region_labels)
    strat_labels = make_strat_bins(y_log, n_bins=10)

    # ---- Train / temp split ----
    split_args = dict(test_size=0.30, random_state=42)
    if strat_labels is not None:
        split_args["stratify"] = strat_labels

    (
        t_taps_feat, temp_taps_feat,
        t_vars_feat, temp_vars_feat,
        t_abs_raw, temp_abs_raw,
        t_snr_raw, temp_snr_raw,
        t_z0_raw, temp_z0_raw,
        t_z1_raw, temp_z1_raw,
        t_hmg_raw, temp_hmg_raw,
        t_nsid_raw, temp_nsid_raw,
        t_log_hd_raw, temp_log_hd_raw,
        t_da_raw, temp_da_raw,
        t_herf_raw, temp_herf_raw,
        t_thr, temp_thr,
        t_y_log, temp_y_log,
        t_region, temp_region,
        t_ord, temp_ord,
        t_past_lens, temp_past_lens,
    ) = train_test_split(
        X_taps_feat, X_vars_feat, abs_taps_raw, snr_raw,
        z0_raw, z1_raw, harmonic_minus_gap_raw,
        nsid_raw, log_harmonic_discrim_raw, discrim_asymmetry_raw, herfindahl_raw,
        X_thr,
        y_log, region_labels, ordinal_targets, past_lens,
        **split_args,
    )

    temp_strat = make_strat_bins(temp_y_log, n_bins=6)
    split_args2 = dict(test_size=0.50, random_state=42)
    if temp_strat is not None:
        split_args2["stratify"] = temp_strat

    (
        v_taps_feat, te_taps_feat,
        v_vars_feat, te_vars_feat,
        v_abs_raw, te_abs_raw,
        v_snr_raw, te_snr_raw,
        v_z0_raw, te_z0_raw,
        v_z1_raw, te_z1_raw,
        v_hmg_raw, te_hmg_raw,
        v_nsid_raw, te_nsid_raw,
        v_log_hd_raw, te_log_hd_raw,
        v_da_raw, te_da_raw,
        v_herf_raw, te_herf_raw,
        v_thr, te_thr,
        v_y_log, te_y_log,
        v_region, te_region,
        v_ord, te_ord,
        v_past_lens, te_past_lens,
    ) = train_test_split(
        temp_taps_feat, temp_vars_feat, temp_abs_raw, temp_snr_raw,
        temp_z0_raw, temp_z1_raw, temp_hmg_raw,
        temp_nsid_raw, temp_log_hd_raw, temp_da_raw, temp_herf_raw,
        temp_thr,
        temp_y_log, temp_region, temp_ord, temp_past_lens,
        **split_args2,
    )

    L_past = L_full - 1
    t_valid_past = (np.arange(L_past)[None, :] < t_past_lens[:, None])

    # =========================================================
    # Scalers: reuse pretrained or fit from scratch
    # =========================================================
    if reuse_scalers and pretrained_scalers is not None:
        print("  Reusing pretrained scalers (recommended for fine-tuning).")

        first_tap_mean = pretrained_scalers["first_tap_mean"]
        first_tap_std = pretrained_scalers["first_tap_std"]
        first_var_mean = pretrained_scalers["first_var_mean"]
        first_var_std = pretrained_scalers["first_var_std"]
        first_abs_mean = pretrained_scalers["first_abs_mean"]
        first_abs_std = pretrained_scalers["first_abs_std"]
        first_snr_mean = pretrained_scalers["first_snr_mean"]
        first_snr_std = pretrained_scalers["first_snr_std"]

        past_tap_mean = pretrained_scalers["past_tap_mean"]
        past_tap_std = pretrained_scalers["past_tap_std"]
        past_var_mean = pretrained_scalers["past_var_mean"]
        past_var_std = pretrained_scalers["past_var_std"]
        past_abs_mean = pretrained_scalers["past_abs_mean"]
        past_abs_std = pretrained_scalers["past_abs_std"]
        past_snr_mean = pretrained_scalers["past_snr_mean"]
        past_snr_std = pretrained_scalers["past_snr_std"]

        z0_scaler = pretrained_scalers["z0_scaler"]
        z1_scaler = pretrained_scalers["z1_scaler"]
        hmg_scaler = pretrained_scalers["harmonic_minus_gap_scaler"]
        nsid_scaler = pretrained_scalers["nsid_scaler"]
        log_hd_scaler = pretrained_scalers["log_hd_scaler"]
        da_scaler = pretrained_scalers["da_scaler"]
        herf_scaler = pretrained_scalers["herf_scaler"]
        thr_scaler = pretrained_scalers["thr_scaler"]
    else:
        print("  Refitting scalers on fine-tune data.")
        first_tap_mean, first_tap_std = float(t_taps_feat[:, 0].mean()), max(float(t_taps_feat[:, 0].std()), 1e-12)
        first_var_mean, first_var_std = float(t_vars_feat[:, 0].mean()), max(float(t_vars_feat[:, 0].std()), 1e-12)
        first_abs_mean, first_abs_std = float(t_abs_raw[:, 0].mean()), max(float(t_abs_raw[:, 0].std()), 1e-12)
        first_snr_mean, first_snr_std = float(t_snr_raw[:, 0].mean()), max(float(t_snr_raw[:, 0].std()), 1e-12)

        past_tap_mean, past_tap_std = fit_shared_scalar_scaler(t_taps_feat[:, 1:], t_valid_past)
        past_var_mean, past_var_std = fit_shared_scalar_scaler(t_vars_feat[:, 1:], t_valid_past)
        past_abs_mean, past_abs_std = fit_shared_scalar_scaler(t_abs_raw[:, 1:], t_valid_past)
        past_snr_mean, past_snr_std = fit_shared_scalar_scaler(t_snr_raw[:, 1:], t_valid_past)

        z0_scaler = StandardScaler().fit(t_z0_raw)
        z1_scaler = StandardScaler().fit(t_z1_raw)
        hmg_scaler = StandardScaler().fit(t_hmg_raw)
        nsid_scaler = StandardScaler().fit(t_nsid_raw)
        log_hd_scaler = StandardScaler().fit(t_log_hd_raw)
        da_scaler = StandardScaler().fit(t_da_raw)
        herf_scaler = StandardScaler().fit(t_herf_raw)
        thr_scaler = StandardScaler().fit(t_thr)

    def build_tokens(taps_feat, vars_feat, abs_raw, snr_raw, p_lens):
        ft_tap = ((taps_feat[:, 0] - first_tap_mean) / first_tap_std).astype(np.float32)
        ft_var = ((vars_feat[:, 0] - first_var_mean) / first_var_std).astype(np.float32)
        ft_abs = ((abs_raw[:, 0] - first_abs_mean) / first_abs_std).astype(np.float32)
        ft_snr = ((snr_raw[:, 0] - first_snr_mean) / first_snr_std).astype(np.float32)
        first_token = np.stack([ft_tap, ft_var, ft_abs, ft_snr], axis=1).astype(np.float32)

        pt_valid = (np.arange(L_past)[None, :] < p_lens[:, None])
        pt_tap = apply_shared_scale(taps_feat[:, 1:], past_tap_mean, past_tap_std, pt_valid)
        pt_var = apply_shared_scale(vars_feat[:, 1:], past_var_mean, past_var_std, pt_valid)
        pt_abs = apply_shared_scale(abs_raw[:, 1:], past_abs_mean, past_abs_std, pt_valid)
        pt_snr = apply_shared_scale(snr_raw[:, 1:], past_snr_mean, past_snr_std, pt_valid)
        past_tokens = np.stack([pt_tap, pt_var, pt_abs, pt_snr], axis=2).astype(np.float32)

        return first_token, past_tokens

    def build_global(z0, z1, hmg, nsid, log_hd, da, herf):
        return np.concatenate([
            z0_scaler.transform(z0),
            z1_scaler.transform(z1),
            hmg_scaler.transform(hmg),
            nsid_scaler.transform(nsid),
            log_hd_scaler.transform(log_hd),
            da_scaler.transform(da),
            herf_scaler.transform(herf),
        ], axis=1).astype(np.float32)

    def build_padding_mask(p_lens, L_max):
        return (np.arange(L_max)[None, :] >= p_lens[:, None])

    t_first, t_past = build_tokens(t_taps_feat, t_vars_feat, t_abs_raw, t_snr_raw, t_past_lens)
    v_first, v_past = build_tokens(v_taps_feat, v_vars_feat, v_abs_raw, v_snr_raw, v_past_lens)
    te_first, te_past = build_tokens(te_taps_feat, te_vars_feat, te_abs_raw, te_snr_raw, te_past_lens)

    t_global = build_global(t_z0_raw, t_z1_raw, t_hmg_raw,
                            t_nsid_raw, t_log_hd_raw, t_da_raw, t_herf_raw)
    v_global = build_global(v_z0_raw, v_z1_raw, v_hmg_raw,
                            v_nsid_raw, v_log_hd_raw, v_da_raw, v_herf_raw)
    te_global = build_global(te_z0_raw, te_z1_raw, te_hmg_raw,
                             te_nsid_raw, te_log_hd_raw, te_da_raw, te_herf_raw)

    t_thr_s = thr_scaler.transform(t_thr).astype(np.float32)
    v_thr_s = thr_scaler.transform(v_thr).astype(np.float32)
    te_thr_s = thr_scaler.transform(te_thr).astype(np.float32)

    L_max_past = L_past
    t_mask = build_padding_mask(t_past_lens, L_max_past)
    v_mask = build_padding_mask(v_past_lens, L_max_past)
    te_mask = build_padding_mask(te_past_lens, L_max_past)

    train_ds = TensorDataset(
        torch.from_numpy(t_first), torch.from_numpy(t_past),
        torch.from_numpy(t_global), torch.from_numpy(t_thr_s),
        torch.from_numpy(t_y_log.astype(np.float32)),
        torch.from_numpy(t_ord.astype(np.float32)),
        torch.from_numpy(t_region.astype(np.int64)),
        torch.from_numpy(t_mask))
    val_ds = TensorDataset(
        torch.from_numpy(v_first), torch.from_numpy(v_past),
        torch.from_numpy(v_global), torch.from_numpy(v_thr_s),
        torch.from_numpy(v_y_log.astype(np.float32)),
        torch.from_numpy(v_ord.astype(np.float32)),
        torch.from_numpy(v_region.astype(np.int64)),
        torch.from_numpy(v_mask))
    test_ds = TensorDataset(
        torch.from_numpy(te_first), torch.from_numpy(te_past),
        torch.from_numpy(te_global), torch.from_numpy(te_thr_s),
        torch.from_numpy(te_y_log.astype(np.float32)),
        torch.from_numpy(te_ord.astype(np.float32)),
        torch.from_numpy(te_region.astype(np.int64)),
        torch.from_numpy(te_mask))

    # ---- Sampler (region + SIR-aware) ----
    region_sample_weights = np.ones_like(t_region, dtype=np.float32)
    region_sample_weights[t_region == 6] = 2.5
    region_sample_weights[t_region == 7] = 3.0
    region_sample_weights[t_region == 8] = 5.0
    region_sample_weights[t_region == 9] = 5.0
    region_sample_weights[t_region == 10] = 5.0
    region_sample_weights[t_region == 11] = 5.0
    region_sample_weights[t_region == 12] = 3.0
    region_sample_weights[t_region == 13] = 2.5

    t_signal = t_taps_feat[:, 0]
    t_valid_past_for_sir = (np.arange(L_past)[None, :] < t_past_lens[:, None]).astype(np.float32)
    t_isi = np.sum(t_taps_feat[:, 1:] * t_valid_past_for_sir, axis=1)
    t_sir = t_signal / (t_isi + EPS)

    sir_sample_weights = np.ones_like(t_sir, dtype=np.float32)
    sir_sample_weights[t_sir < 0.26] = 3.0
    sir_sample_weights[(t_sir >= 0.26) & (t_sir < 0.36)] = 2.0

    combined_sample_weights = region_sample_weights * sir_sample_weights

    sampler = WeightedRandomSampler(
        weights=torch.from_numpy(combined_sample_weights),
        num_samples=len(combined_sample_weights),
        replacement=True)

    pin_mem = torch.cuda.is_available()
    train_loader = DataLoader(train_ds, batch_size=batch_size, sampler=sampler,
                              shuffle=False, pin_memory=pin_mem, num_workers=num_workers)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False,
                            pin_memory=pin_mem, num_workers=num_workers)
    test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False,
                             pin_memory=pin_mem, num_workers=num_workers)

    class_weights = compute_region_class_weights(t_region, NUM_REGION_CLASSES)
    ordinal_pos_weights = compute_ordinal_pos_weights_from_region_labels(
        t_region, num_thresholds=NUM_ORDINAL_THRESHOLDS, max_weight=20.0)

    scalers = {
        "first_tap_mean": first_tap_mean, "first_tap_std": first_tap_std,
        "first_var_mean": first_var_mean, "first_var_std": first_var_std,
        "first_abs_mean": first_abs_mean, "first_abs_std": first_abs_std,
        "first_snr_mean": first_snr_mean, "first_snr_std": first_snr_std,
        "past_tap_mean": past_tap_mean, "past_tap_std": past_tap_std,
        "past_var_mean": past_var_mean, "past_var_std": past_var_std,
        "past_abs_mean": past_abs_mean, "past_abs_std": past_abs_std,
        "past_snr_mean": past_snr_mean, "past_snr_std": past_snr_std,
        "z0_scaler": z0_scaler, "z1_scaler": z1_scaler,
        "harmonic_minus_gap_scaler": hmg_scaler,
        "nsid_scaler": nsid_scaler, "log_hd_scaler": log_hd_scaler,
        "da_scaler": da_scaler, "herf_scaler": herf_scaler,
        "thr_scaler": thr_scaler,
        "threshold_feature_mode": threshold_mode,
        "threshold_norm_clip_eps": THRESHOLD_NORM_CLIP_EPS,
        "threshold_feature_description": (
            "threshold branch input is StandardScaler(threshold / sum(valid tap_i*N)) "
            "or its logit, depending on threshold_feature_mode"
        ),
        "tap_cols": tap_cols, "var_cols": var_cols,
        "first_token_dim": 4, "past_token_dim": 4,
        "global_dim": 7,
        "train_max_past_seq_len": L_past,
        "scaling_strategy": "position_independent",
        "uses_positional_encoding": False,
        "permutation_invariance_post_first": True,
        "variable_length_support": True,
        "target_parameterization": "pred_log10_ber = log10(0.5) - softplus(raw_out)",
        "ordinal_thresholds": ORDINAL_THRESHOLDS,
        "num_region_classes": NUM_REGION_CLASSES,
        "class_weights": class_weights.tolist(),
        "ordinal_pos_weights": ordinal_pos_weights.tolist(),
        "data_paths": csv_paths,
        "nrows_per_dataset": nrows_per_dataset,
        "finetuned_from": PRETRAINED_MODEL_PATH,
        "scalers_reused": reuse_scalers,
        "variances_computed_from_taps": True,
    }

    aux_info = {
        "class_weights": class_weights,
        "ordinal_pos_weights": ordinal_pos_weights,
        "t_region": t_region,
    }

    return train_loader, val_loader, test_loader, scalers, aux_info, L_past


# =========================
# Evaluation (same as v4)
# =========================
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0
    total_reg_loss = 0.0
    total_ord_loss = 0.0
    all_preds_log = []
    all_targets_log = []
    all_pred_regions = []
    all_true_regions = []

    with torch.no_grad():
        for b_first, b_past, b_global, b_thr, b_y_log, b_ord, b_region, b_mask in loader:
            b_first = b_first.to(device, non_blocking=True)
            b_past = b_past.to(device, non_blocking=True)
            b_global = b_global.to(device, non_blocking=True)
            b_thr = b_thr.to(device, non_blocking=True)
            b_y_log = b_y_log.to(device, non_blocking=True)
            b_ord = b_ord.to(device, non_blocking=True)
            b_region = b_region.to(device, non_blocking=True)
            b_mask = b_mask.to(device, non_blocking=True)

            pred_raw_unconstrained, ord_logits = model(
                b_first, b_past, b_global, b_thr, key_padding_mask=b_mask)

            if has_nonfinite_tensor(pred_raw_unconstrained):
                raise RuntimeError("Non-finite prediction during evaluation.")
            if has_nonfinite_tensor(ord_logits):
                raise RuntimeError("Non-finite ordinal logits during evaluation.")

            loss, reg_loss, ord_loss = criterion(
                pred_raw_unconstrained, ord_logits, b_y_log, b_ord)

            if has_nonfinite_tensor(loss):
                raise RuntimeError("Non-finite loss during evaluation.")

            pred_log = model.raw_to_log10ber(pred_raw_unconstrained)
            pred_region = ordinal_logits_to_region_labels_torch(ord_logits)

            total_loss += loss.item()
            total_reg_loss += reg_loss.item()
            total_ord_loss += ord_loss.item()

            all_preds_log.append(pred_log.cpu().numpy())
            all_targets_log.append(b_y_log.cpu().numpy())
            all_pred_regions.append(pred_region.cpu().numpy())
            all_true_regions.append(b_region.cpu().numpy())

    n_batches = max(len(loader), 1)
    avg_loss = total_loss / n_batches
    avg_reg_loss = total_reg_loss / n_batches
    avg_ord_loss = total_ord_loss / n_batches

    preds_log = np.vstack(all_preds_log)
    targets_log = np.vstack(all_targets_log)
    preds_raw = np.clip(10 ** preds_log, EPS, 0.5)
    targets_raw = np.clip(10 ** targets_log, EPS, 0.5)

    rmse_log = float(np.sqrt(np.mean((preds_log - targets_log) ** 2)))
    mae_log = float(np.mean(np.abs(preds_log - targets_log)))
    factor_error = float(10 ** rmse_log)
    rmse_raw = float(np.sqrt(np.mean((preds_raw - targets_raw) ** 2)))
    mae_raw = float(np.mean(np.abs(preds_raw - targets_raw)))
    rel_err = (preds_raw - targets_raw) / np.maximum(targets_raw, 1e-6)
    rmse_rel = float(np.sqrt(np.mean(rel_err ** 2)))
    mae_rel = float(np.mean(np.abs(rel_err)))

    pred_regions = np.concatenate(all_pred_regions).reshape(-1)
    true_regions = np.concatenate(all_true_regions).reshape(-1)
    region_acc = float(np.mean(pred_regions == true_regions))
    region_mae = float(np.mean(np.abs(pred_regions - true_regions)))

    return {
        "loss": avg_loss, "reg_loss": avg_reg_loss, "ord_loss": avg_ord_loss,
        "rmse_log": rmse_log, "mae_log": mae_log, "factor_error": factor_error,
        "rmse_raw": rmse_raw, "mae_raw": mae_raw,
        "rmse_rel": rmse_rel, "mae_rel": mae_rel,
        "region_acc": region_acc, "region_mae": region_mae,
    }


def evaluate_by_target_range(model, loader, device):
    model.eval()
    all_preds_log = []
    all_targets_log = []

    with torch.no_grad():
        for b_first, b_past, b_global, b_thr, b_y_log, _, _, b_mask in loader:
            b_first = b_first.to(device, non_blocking=True)
            b_past = b_past.to(device, non_blocking=True)
            b_global = b_global.to(device, non_blocking=True)
            b_thr = b_thr.to(device, non_blocking=True)
            b_mask = b_mask.to(device, non_blocking=True)

            pred_raw_unconstrained, _ = model(
                b_first, b_past, b_global, b_thr, key_padding_mask=b_mask)
            if has_nonfinite_tensor(pred_raw_unconstrained):
                raise RuntimeError("Non-finite prediction during per-range evaluation.")
            pred_log = model.raw_to_log10ber(pred_raw_unconstrained)
            all_preds_log.append(pred_log.cpu().numpy())
            all_targets_log.append(b_y_log.numpy())

    preds_log = np.vstack(all_preds_log).reshape(-1)
    targets_log = np.vstack(all_targets_log).reshape(-1)
    preds_raw = np.clip(10 ** preds_log, EPS, 0.5)
    targets_raw = np.clip(10 ** targets_log, EPS, 0.5)

    ranges = {
        "low_BER(y<1e-6)": targets_raw < 1e-6,
        "mid_BER(1e-6<=y<1e-3)": (targets_raw >= 1e-6) & (targets_raw < 1e-3),
        "high_BER(1e-3<=y<0.10)": (targets_raw >= 1e-3) & (targets_raw < 0.10),
        "upper_BER(0.10<=y<0.20)": (targets_raw >= 0.10) & (targets_raw < 0.20),
        "target_BER(0.20<=y<0.40)": (targets_raw >= 0.20) & (targets_raw < 0.40),
        "very_high_BER(0.40<=y<=0.50)": targets_raw >= 0.40,
    }

    metrics = {}
    for name, mask in ranges.items():
        if np.any(mask):
            err_log = preds_log[mask] - targets_log[mask]
            err_raw = preds_raw[mask] - targets_raw[mask]
            rel_err = err_raw / np.maximum(targets_raw[mask], 1e-6)
            metrics[name] = {
                "count": int(mask.sum()),
                "rmse_log": float(np.sqrt(np.mean(err_log ** 2))),
                "mae_log": float(np.mean(np.abs(err_log))),
                "factor_error": float(10 ** float(np.sqrt(np.mean(err_log ** 2)))),
                "rmse_raw": float(np.sqrt(np.mean(err_raw ** 2))),
                "mae_raw": float(np.mean(np.abs(err_raw))),
                "rmse_rel": float(np.sqrt(np.mean(rel_err ** 2))),
                "mae_rel": float(np.mean(np.abs(rel_err))),
                "bias_raw": float(np.mean(err_raw)),
                "bias_log": float(np.mean(err_log)),
                "p90_abs_raw": float(np.percentile(np.abs(err_raw), 90)),
                "p95_abs_raw": float(np.percentile(np.abs(err_raw), 95)),
            }
        else:
            metrics[name] = None
    return metrics


# =========================
# Selection scoring (same as v4)
# =========================
SELECTION_REGION_WEIGHTS = {
    "low_BER(y<1e-6)": 0.25,
    "mid_BER(1e-6<=y<1e-3)": 0.50,
    "high_BER(1e-3<=y<0.10)": 1.00,
    "upper_BER(0.10<=y<0.20)": 2.00,
    "target_BER(0.20<=y<0.40)": 3.00,
    "very_high_BER(0.40<=y<=0.50)": 1.50,
}
SELECTION_W_LOG_RMSE = 1.00
SELECTION_W_BIAS = 0.50
SELECTION_W_TAIL = 0.05
SELECTION_TAIL_CAP = 5.0


def compute_selection_score(val_range_metrics, val_metrics, min_count=50):
    log_rmse_sum = 0.0
    bias_sum = 0.0
    tail_sum = 0.0
    total_w = 0.0

    for region_name, w in SELECTION_REGION_WEIGHTS.items():
        m = val_range_metrics.get(region_name)
        if m is None or m["count"] < min_count:
            continue
        log_rmse_sum += w * m["rmse_log"]
        bias_sum += w * abs(m["bias_log"])
        if m["rmse_raw"] > 1e-9:
            tail_ratio = m["p95_abs_raw"] / (m["rmse_raw"] + 1e-9)
            tail_sum += w * min(tail_ratio, SELECTION_TAIL_CAP)
        else:
            tail_sum += w * 1.0
        total_w += w

    if total_w == 0:
        return {
            "composite": float(val_metrics["rmse_log"]),
            "log_rmse_weighted": float(val_metrics["rmse_log"]),
            "bias_weighted": 0.0, "tail_weighted": 0.0,
            "geometric_factor_error": float(val_metrics["factor_error"]),
            "fallback": True,
        }

    log_rmse_weighted = log_rmse_sum / total_w
    bias_weighted = bias_sum / total_w
    tail_weighted = tail_sum / total_w
    composite = (SELECTION_W_LOG_RMSE * log_rmse_weighted
                 + SELECTION_W_BIAS * bias_weighted
                 + SELECTION_W_TAIL * tail_weighted)
    return {
        "composite": float(composite),
        "log_rmse_weighted": float(log_rmse_weighted),
        "bias_weighted": float(bias_weighted),
        "tail_weighted": float(tail_weighted),
        "geometric_factor_error": float(10 ** log_rmse_weighted),
        "fallback": False,
    }


def is_acceptable_checkpoint(val_range_metrics, val_metrics):
    target = val_range_metrics.get("target_BER(0.20<=y<0.40)")
    if target is None or target["count"] < 100:
        return False, "target region too small"
    for region_name, m in val_range_metrics.items():
        if m is None or m["count"] < 50:
            continue
        if abs(m["bias_log"]) > 0.15:
            return False, f"{region_name} has bias_log={m['bias_log']:.3f}"
    if val_metrics["rmse_log"] > 0.30:
        return False, f"overall rmse_log={val_metrics['rmse_log']:.3f} too high"
    return True, "ok"


# =========================
# Fine-tune main
# =========================
def finetune():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Executing on: {device}")
    print(f"\nFINE-TUNING from: {PRETRAINED_MODEL_PATH}")
    print(f"  Pretrained scalers: {PRETRAINED_SCALER_PATH}")
    print(f"  Reuse scalers: {REUSE_SCALERS}")

    if not os.path.exists(PRETRAINED_MODEL_PATH):
        raise FileNotFoundError(f"Pretrained model not found: {PRETRAINED_MODEL_PATH}")
    if not os.path.exists(PRETRAINED_SCALER_PATH):
        raise FileNotFoundError(f"Pretrained scalers not found: {PRETRAINED_SCALER_PATH}")

    pretrained_scalers = joblib.load(PRETRAINED_SCALER_PATH)
    print(f"  Pretrained global_dim: {pretrained_scalers.get('global_dim', '?')}")
    print(f"  Pretrained threshold_feature_mode: {pretrained_scalers.get('threshold_feature_mode', THRESHOLD_FEATURE_MODE)}")

    # ---- Prepare data ----
    train_loader, val_loader, test_loader, scalers, aux_info, max_past_seq_len = (
        prepare_data_for_finetune(
            DATA_PATHS, batch_size=256,
            nrows_per_dataset=NROWS_PER_DATASET, num_workers=0,
            pretrained_scalers=pretrained_scalers, reuse_scalers=REUSE_SCALERS,
        )
    )

    joblib.dump(scalers, SCALER_SAVE_PATH)
    print(f"\nFine-tune scalers saved to: {SCALER_SAVE_PATH}")

    # ---- Build model & load pretrained weights ----
    model = BERFirstTokenSetTransformerBoundedLogHalfMultiTask(
        max_past_seq_len=max_past_seq_len,
        token_dim=4, first_token_dim=4, threshold_dim=1,
        global_dim=7, d_model=128, num_set_layers=4,
        num_heads=4, mlp_ratio=4.0, dropout=0.10,
        num_ordinal_thresholds=NUM_ORDINAL_THRESHOLDS,
    ).to(device)

    pretrained_state = torch.load(PRETRAINED_MODEL_PATH, map_location=device, weights_only=True)
    model.load_state_dict(pretrained_state, strict=True)
    print("Pretrained normalized-threshold + monotonic-ordinal weights loaded with strict=True.")

    # ---- Pre-fine-tune evaluation (sanity check) ----
    print("\n--- Pre-fine-tune evaluation (pretrained model on new data) ---")
    pre_criterion = MultiTaskBERLoss(
        StableMultiObjectiveBERBoundedLogLoss(),
        OrdinalBCELoss(),
        lambda_ord=0.25,
    )
    try:
        pre_val = evaluate(model, val_loader, pre_criterion, device)
        pre_range = evaluate_by_target_range(model, val_loader, device)
        pre_sel = compute_selection_score(pre_range, pre_val)
        print(f"  Pretrained on new val: composite={pre_sel['composite']:.5f} | "
              f"factor~{pre_sel['geometric_factor_error']:.3f}x | "
              f"RMSE(log10)={pre_val['rmse_log']:.4f} | "
              f"region_acc={pre_val['region_acc']:.4f}")
        for rng, m in pre_range.items():
            if m is not None:
                print(f"    {rng}: factor~{m['factor_error']:.3f}x, "
                      f"bias_log={m['bias_log']:+.4f}, count={m['count']}")
    except RuntimeError as e:
        print(f"  Pre-eval failed: {e}")

    # ---- Optimizer & loss for fine-tuning ----
    print(f"\nFine-tune hyperparameters:")
    print(f"  LR={FINETUNE_LR}, weight_decay={FINETUNE_WEIGHT_DECAY}")
    print(f"  max_epochs={FINETUNE_MAX_EPOCHS}, min_epochs={FINETUNE_MIN_EPOCHS}, "
          f"patience={FINETUNE_PATIENCE}")

    optimizer = optim.AdamW(model.parameters(), lr=FINETUNE_LR,
                            weight_decay=FINETUNE_WEIGHT_DECAY)

    reg_loss = StableMultiObjectiveBERBoundedLogLoss(
        log_delta=0.35, raw_delta=0.006, rel_delta=0.03,
        alpha_log=0.45, beta_raw=0.35, gamma_rel=0.20,
        use_regime_weights=True)

    boosted_pos = aux_info["ordinal_pos_weights"].copy()
    for i, thr in enumerate(ORDINAL_THRESHOLDS):
        if 0.20 <= thr <= 0.40:
            boosted_pos[i] *= 2.5
        elif 0.15 <= thr < 0.20:
            boosted_pos[i] *= 1.5
        elif 0.40 < thr <= 0.45:
            boosted_pos[i] *= 1.5

    ord_loss = OrdinalBCELoss(
        pos_weight=torch.tensor(boosted_pos, dtype=torch.float32, device=device),
        reduction="mean")
    criterion = MultiTaskBERLoss(reg_loss=reg_loss, ord_loss=ord_loss, lambda_ord=0.25)

    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="min", patience=SCHEDULER_PATIENCE, factor=SCHEDULER_FACTOR)

    # ---- Multi-component selection state ----
    best_checkpoints = {
        "composite": {"score": float("inf"), "state": None, "epoch": -1},
        "composite_ema": {"score": float("inf"), "state": None, "epoch": -1},
        "log_rmse": {"score": float("inf"), "state": None, "epoch": -1},
        "target_mae": {"score": float("inf"), "state": None, "epoch": -1},
        "low_bias": {"score": float("inf"), "state": None, "epoch": -1},
    }
    EMA_ALPHA = 0.5
    ema_composite = None
    wait = 0

    # Seed best with pre-fine-tune score so fine-tuning must IMPROVE on the
    # pretrained baseline before being marked as best.
    try:
        base_state = copy.deepcopy(model.state_dict())
        pre_acceptable, _ = is_acceptable_checkpoint(pre_range, pre_val)
        if pre_acceptable:
            for ckpt_name, score_value in [
                ("composite", pre_sel["composite"]),
                ("composite_ema", pre_sel["composite"]),
                ("log_rmse", pre_sel["log_rmse_weighted"]),
                ("target_mae", pre_range["target_BER(0.20<=y<0.40)"]["mae_raw"]
                    if pre_range.get("target_BER(0.20<=y<0.40)") else float("inf")),
                ("low_bias", pre_sel["bias_weighted"]),
            ]:
                best_checkpoints[ckpt_name] = {
                    "score": float(score_value),
                    "state": base_state,
                    "epoch": 0,  # 0 = pretrained baseline
                }
            print("\nSeeded best checkpoints with pretrained baseline scores. "
                  "Fine-tune must improve to take over.")
    except Exception as e:
        print(f"  Could not seed pretrained baseline as best: {e}")

    print(f"\nSelection: weighted log-RMSE + bias + tail penalties")
    print(f"  Region weights: {SELECTION_REGION_WEIGHTS}")
    print(f"  Composite weights: log_rmse={SELECTION_W_LOG_RMSE}, "
          f"bias={SELECTION_W_BIAS}, tail={SELECTION_W_TAIL}")
    print(f"  EMA alpha: {EMA_ALPHA}\n")

    training_broke = False

    for epoch in range(FINETUNE_MAX_EPOCHS):
        model.train()

        for batch_idx, (b_first, b_past, b_global, b_thr, b_y_log, b_ord, _, b_mask) in enumerate(train_loader):
            b_first = b_first.to(device, non_blocking=True)
            b_past = b_past.to(device, non_blocking=True)
            b_global = b_global.to(device, non_blocking=True)
            b_thr = b_thr.to(device, non_blocking=True)
            b_y_log = b_y_log.to(device, non_blocking=True)
            b_ord = b_ord.to(device, non_blocking=True)
            b_mask = b_mask.to(device, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)
            pred_raw_unconstrained, ord_logits = model(
                b_first, b_past, b_global, b_thr, key_padding_mask=b_mask)

            if has_nonfinite_tensor(pred_raw_unconstrained):
                print(f"Non-finite prediction at epoch {epoch+1}, batch {batch_idx+1}.")
                training_broke = True; break
            if has_nonfinite_tensor(ord_logits):
                print(f"Non-finite ordinal logits at epoch {epoch+1}, batch {batch_idx+1}.")
                training_broke = True; break

            loss, reg_part, ord_part = criterion(
                pred_raw_unconstrained, ord_logits, b_y_log, b_ord)
            if has_nonfinite_tensor(loss):
                print(f"Non-finite loss at epoch {epoch+1}, batch {batch_idx+1}.")
                training_broke = True; break

            loss.backward()
            grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=GRAD_CLIP)
            if not torch.isfinite(grad_norm):
                print(f"Non-finite gradient norm at epoch {epoch+1}, batch {batch_idx+1}.")
                training_broke = True; break

            optimizer.step()

            bad_param = False
            for nm, param in model.named_parameters():
                if param.requires_grad and param.data is not None and not torch.isfinite(param.data).all():
                    print(f"Non-finite parameter after optimizer step: {nm}")
                    bad_param = True; break
            if bad_param:
                training_broke = True; break

        if training_broke:
            print("Training stopped due to non-finite values.")
            break

        try:
            train_metrics = evaluate(model, train_loader, criterion, device)
            val_metrics = evaluate(model, val_loader, criterion, device)
            val_range_metrics = evaluate_by_target_range(model, val_loader, device)
        except RuntimeError as e:
            print(f"Evaluation failed at epoch {epoch+1}: {e}")
            break

        scheduler.step(val_metrics["loss"])
        current_lr = optimizer.param_groups[0]["lr"]

        sel = compute_selection_score(val_range_metrics, val_metrics)
        composite = sel["composite"]
        if ema_composite is None:
            ema_composite = composite
        else:
            ema_composite = EMA_ALPHA * composite + (1.0 - EMA_ALPHA) * ema_composite

        target_key = "target_BER(0.20<=y<0.40)"
        target_mae = (val_range_metrics[target_key]["mae_raw"]
                      if val_range_metrics[target_key] is not None else float("inf"))

        is_acceptable, reason = is_acceptable_checkpoint(val_range_metrics, val_metrics)

        print(
            f"FT-Epoch {epoch+1:03d} | LR: {current_lr:.2e} | "
            f"Train Loss: {train_metrics['loss']:.4f} | "
            f"Val Loss: {val_metrics['loss']:.4f} | "
            f"Val RMSE(log10): {val_metrics['rmse_log']:.4f} "
            f"(~{val_metrics['factor_error']:.3f}x) | "
            f"Val Region Acc: {val_metrics['region_acc']:.4f}"
        )
        print(
            f"  SELECTION | composite={composite:.5f} | ema={ema_composite:.5f} | "
            f"weighted log-RMSE={sel['log_rmse_weighted']:.5f} "
            f"(~{sel['geometric_factor_error']:.3f}x) | "
            f"|bias_log|={sel['bias_weighted']:.5f} | "
            f"tail={sel['tail_weighted']:.3f} | acceptable={is_acceptable}"
            + ("" if is_acceptable else f" ({reason})")
        )
        for rng_name, rng_stats in val_range_metrics.items():
            if rng_stats is not None:
                print(
                    f"    {rng_name}: n={rng_stats['count']} | "
                    f"factor~{rng_stats['factor_error']:.3f}x | "
                    f"MAE_raw={rng_stats['mae_raw']:.6f} | "
                    f"P90={rng_stats['p90_abs_raw']:.6f} | "
                    f"bias_log={rng_stats['bias_log']:+.4f}"
                )

        improved_any = False
        if is_acceptable:
            current_state_snapshot = None
            checkpoint_candidates = [
                ("composite", composite),
                ("composite_ema", ema_composite),
                ("log_rmse", sel["log_rmse_weighted"]),
                ("target_mae", target_mae),
                ("low_bias", sel["bias_weighted"]),
            ]
            for ckpt_name, score in checkpoint_candidates:
                if score < best_checkpoints[ckpt_name]["score"]:
                    if current_state_snapshot is None:
                        current_state_snapshot = copy.deepcopy(model.state_dict())
                    best_checkpoints[ckpt_name] = {
                        "score": float(score),
                        "state": current_state_snapshot,
                        "epoch": epoch + 1,
                    }
                    improved_any = True
                    print(f"  -> New best [{ckpt_name}] at FT-epoch {epoch+1}: {score:.6f}")

        ema_best = best_checkpoints["composite_ema"]["score"]
        if ema_composite < ema_best + 1e-9 or improved_any:
            wait = 0
        else:
            if epoch + 1 >= FINETUNE_MIN_EPOCHS:
                wait += 1
                if wait >= FINETUNE_PATIENCE:
                    print(f"Early stopping triggered after {wait} non-improving epochs.")
                    break

    # ---- Save last ----
    torch.save(model.state_dict(), LAST_MODEL_SAVE_PATH)
    print(f"\nLast fine-tuned model saved to: {LAST_MODEL_SAVE_PATH}")

    # ---- Save all best checkpoints ----
    print("\n" + "=" * 80)
    print("FINE-TUNE BEST CHECKPOINTS SUMMARY")
    print("=" * 80)
    for ckpt_name, info in best_checkpoints.items():
        if info["state"] is None:
            print(f"  [{ckpt_name:>14s}] never updated")
            continue
        path = BEST_MODEL_SAVE_PATH.replace(".pth", f"_{ckpt_name}.pth")
        torch.save(info["state"], path)
        print(f"  [{ckpt_name:>14s}] epoch={info['epoch']:>3d} | "
              f"score={info['score']:.6f} | saved -> {path}")

    primary_choice = "composite_ema"
    if best_checkpoints[primary_choice]["state"] is None:
        primary_choice = "composite"
    if best_checkpoints[primary_choice]["state"] is None:
        primary_choice = "log_rmse"

    if best_checkpoints[primary_choice]["state"] is not None:
        model.load_state_dict(best_checkpoints[primary_choice]["state"])
        torch.save(model.state_dict(), BEST_MODEL_SAVE_PATH)
        print(f"\nPrimary best ({primary_choice}, "
              f"FT-epoch {best_checkpoints[primary_choice]['epoch']}) "
              f"saved to: {BEST_MODEL_SAVE_PATH}")
    else:
        print("\nWarning: no valid best checkpoint was found.")

    # ---- Test ----
    test_metrics = evaluate(model, test_loader, criterion, device)
    print(
        f"\nTest Loss: {test_metrics['loss']:.4f} | "
        f"Test RMSE(log10): {test_metrics['rmse_log']:.4f} | "
        f"~{test_metrics['factor_error']:.2f}x | "
        f"Test Region Acc: {test_metrics['region_acc']:.4f}"
    )

    range_metrics = evaluate_by_target_range(model, test_loader, device)
    print("\nPer-range test diagnostics:")
    for name, stats in range_metrics.items():
        if stats is None:
            print(f"  {name}: no samples")
        else:
            print(
                f"  {name} | count={stats['count']} | "
                f"factor~{stats['factor_error']:.2f}x | "
                f"RMSE(log10)={stats['rmse_log']:.4f} | "
                f"MAE_raw={stats['mae_raw']:.6f} | "
                f"bias_log={stats['bias_log']:+.4f} | "
                f"P90={stats['p90_abs_raw']:.6f} | "
                f"P95={stats['p95_abs_raw']:.6f}"
            )

    return model


if __name__ == "__main__":
    os.makedirs(BASE_DIR, exist_ok=True)
    missing = [p for p in DATA_PATHS if not os.path.exists(p)]
    if missing:
        print("Critical Error: Missing dataset files:")
        for p in missing:
            print(f"  - {p}")
    else:
        print("Fine-tuning from datasets:")
        for p in DATA_PATHS:
            print(f"  - {p}")
        print(f"Row cap per dataset: {NROWS_PER_DATASET:,}")
        finetune()

Fine-tuning from datasets:
  - ./../ber_data_generation/data/data_physics_total.csv
  - ./../ber_data_generation/data/data_random_total.csv
Row cap per dataset: 5,000,000
Executing on: cuda

FINE-TUNING from: ./random_extra_multitask_last2_normthr_monoord.pth
  Pretrained scalers: ./random_extra_multitask_scalers2_normthr_monoord.pkl
  Reuse scalers: True
  Pretrained global_dim: 7
  Pretrained threshold_feature_mode: normalized_logit
Loading up to 5,000,000 rows from: ./../ber_data_generation/data/data_physics_total.csv
Loading up to 5,000,000 rows from: ./../ber_data_generation/data/data_random_total.csv
Combined rows before filtering: 10,000,000
  Files with var_* columns: 0/2
  Final tap column count: 15
  Dropped 342,676 rows with threshold=0
Rows after cleaning/filtering: 7,990,695
  Tap column count L_full = 15, mem_len range = [2, 15], past_lens range = [1, 14]
  Variances: all 7,990,695 computed from N*tap*(1-tap)
  Reusing pretrained scalers (recommended for fine-tuning).

Fi

FT-Epoch 007 | LR: 2.00e-05 | Train Loss: 0.0009 | Val Loss: 0.0010 | Val RMSE(log10): 0.0462 (~1.112x) | Val Region Acc: 0.9885
  SELECTION | composite=0.13199 | ema=0.12699 | weighted log-RMSE=0.04021 (~1.097x) | |bias_log|=0.00929 | tail=1.743 | acceptable=True
    low_BER(y<1e-6): n=54827 | factor~1.440x | MAE_raw=0.000000 | P90=0.000000 | bias_log=-0.0295
    mid_BER(1e-6<=y<1e-3): n=8285 | factor~2.071x | MAE_raw=0.000065 | P90=0.000222 | bias_log=+0.0164
    high_BER(1e-3<=y<0.10): n=46295 | factor~1.209x | MAE_raw=0.004506 | P90=0.007221 | bias_log=+0.0458
    upper_BER(0.10<=y<0.20): n=82146 | factor~1.030x | MAE_raw=0.002929 | P90=0.005364 | bias_log=+0.0069
    target_BER(0.20<=y<0.40): n=246451 | factor~1.014x | MAE_raw=0.002283 | P90=0.004930 | bias_log=+0.0004
    very_high_BER(0.40<=y<=0.50): n=760600 | factor~1.012x | MAE_raw=0.001402 | P90=0.002622 | bias_log=+0.0002
  -> New best [target_mae] at FT-epoch 7: 0.002283
  -> New best [low_bias] at FT-epoch 7: 0.009291
FT-

FT-Epoch 016 | LR: 1.00e-05 | Train Loss: 0.0008 | Val Loss: 0.0008 | Val RMSE(log10): 0.0322 (~1.077x) | Val Region Acc: 0.9906
  SELECTION | composite=0.11660 | ema=0.11920 | weighted log-RMSE=0.02646 (~1.063x) | |bias_log|=0.01075 | tail=1.695 | acceptable=True
    low_BER(y<1e-6): n=54827 | factor~1.325x | MAE_raw=0.000000 | P90=0.000000 | bias_log=-0.0435
    mid_BER(1e-6<=y<1e-3): n=8285 | factor~1.462x | MAE_raw=0.000048 | P90=0.000152 | bias_log=+0.0644
    high_BER(1e-3<=y<0.10): n=46295 | factor~1.153x | MAE_raw=0.003052 | P90=0.005217 | bias_log=+0.0353
    upper_BER(0.10<=y<0.20): n=82146 | factor~1.027x | MAE_raw=0.002234 | P90=0.004618 | bias_log=+0.0036
    target_BER(0.20<=y<0.40): n=246451 | factor~1.013x | MAE_raw=0.002056 | P90=0.004382 | bias_log=+0.0009
    very_high_BER(0.40<=y<=0.50): n=760600 | factor~1.006x | MAE_raw=0.000789 | P90=0.002205 | bias_log=+0.0003
  -> New best [log_rmse] at FT-epoch 16: 0.026455
FT-Epoch 017 | LR: 1.00e-05 | Train Loss: 0.0008 | Va

FT-Epoch 025 | LR: 1.00e-05 | Train Loss: 0.0007 | Val Loss: 0.0008 | Val RMSE(log10): 0.0289 (~1.069x) | Val Region Acc: 0.9905
  SELECTION | composite=0.12099 | ema=0.11997 | weighted log-RMSE=0.02729 (~1.065x) | |bias_log|=0.01741 | tail=1.700 | acceptable=False (mid_BER(1e-6<=y<1e-3) has bias_log=0.157)
    low_BER(y<1e-6): n=54827 | factor~1.241x | MAE_raw=0.000000 | P90=0.000000 | bias_log=+0.0428
    mid_BER(1e-6<=y<1e-3): n=8285 | factor~1.587x | MAE_raw=0.000081 | P90=0.000265 | bias_log=+0.1573
    high_BER(1e-3<=y<0.10): n=46295 | factor~1.148x | MAE_raw=0.003753 | P90=0.005957 | bias_log=+0.0444
    upper_BER(0.10<=y<0.20): n=82146 | factor~1.027x | MAE_raw=0.002215 | P90=0.004583 | bias_log=+0.0038
    target_BER(0.20<=y<0.40): n=246451 | factor~1.012x | MAE_raw=0.001909 | P90=0.004202 | bias_log=+0.0006
    very_high_BER(0.40<=y<=0.50): n=760600 | factor~1.005x | MAE_raw=0.000608 | P90=0.001804 | bias_log=+0.0003
FT-Epoch 026 | LR: 1.00e-05 | Train Loss: 0.0008 | Val Loss


Test Loss: 0.0008 | Test RMSE(log10): 0.0239 | ~1.06x | Test Region Acc: 0.9905

Per-range test diagnostics:
  low_BER(y<1e-6) | count=54926 | factor~1.19x | RMSE(log10)=0.0754 | MAE_raw=0.000000 | bias_log=+0.0005 | P90=0.000000 | P95=0.000000
  mid_BER(1e-6<=y<1e-3) | count=8300 | factor~1.44x | RMSE(log10)=0.1594 | MAE_raw=0.000057 | bias_log=+0.1352 | P90=0.000165 | P95=0.000213
  high_BER(1e-3<=y<0.10) | count=46361 | factor~1.13x | RMSE(log10)=0.0548 | MAE_raw=0.003501 | bias_log=+0.0407 | P90=0.005929 | P95=0.006991
  upper_BER(0.10<=y<0.20) | count=81865 | factor~1.03x | RMSE(log10)=0.0118 | MAE_raw=0.002133 | bias_log=+0.0021 | P90=0.004637 | P95=0.007202
  target_BER(0.20<=y<0.40) | count=246403 | factor~1.01x | RMSE(log10)=0.0054 | MAE_raw=0.001947 | bias_log=+0.0002 | P90=0.004409 | P95=0.006567
  very_high_BER(0.40<=y<=0.50) | count=760750 | factor~1.01x | RMSE(log10)=0.0024 | MAE_raw=0.000641 | bias_log=+0.0003 | P90=0.001561 | P95=0.003661


In [ ]:
"""
Fine-Tuning Script
==================
Continues training from the normalized-threshold + monotonic-ordinal checkpoint on new datasets.

Handles two CSV formats:
  1. Full physics format with var_i columns (existing)
  2. New format without var_i columns — variance is computed from
     binomial physics: var_i = N * tap_i * (1 - tap_i)

The new format also lacks `mem_len` in some cases — derived from the
number of non-NaN tap columns.

Fine-tuning differences vs from-scratch training:
  - Loads model & scalers from the normalized-threshold + monotonic-ordinal run
  - Lower learning rate (1/5 of base)
  - Shorter schedule (60 epochs max)
  - Smaller patience for early stopping
  - Reuses scalers from previous run by default (controlled by REUSE_SCALERS)
  - Saves to a NEW set of files so previous checkpoints stay intact
"""
import os
import re
import copy
import joblib
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torch.multiprocessing as mp
from torch.utils.data import DataLoader, TensorDataset, WeightedRandomSampler
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

mp.set_sharing_strategy("file_system")

# =========================
# Configuration
# =========================
BASE_DIR = "./"
DATA_DIR = "./../ber_data_generation/data/"

# ---- Source: previous training run to resume from ----
PRETRAINED_MODEL_PATH = os.path.join(
    BASE_DIR, "random_extra_multitask_finetune_best_normthr_monoord.pth"
)
PRETRAINED_SCALER_PATH = os.path.join(
    BASE_DIR, "random_extra_multitask_finetune_scalers_normthr_monoord.pkl"
)
# If True, keep using the pretrained scalers (recommended — keeps the model
# and scalers consistent). If False, refit scalers on the fine-tune data.
REUSE_SCALERS = True

# ---- Destination: NEW files for fine-tuned model ----
BEST_MODEL_SAVE_PATH = os.path.join(
    BASE_DIR, "random_extra_multitask_finetune_best_normthr_monoord.pth"
)
LAST_MODEL_SAVE_PATH = os.path.join(
    BASE_DIR, "random_extra_multitask_finetune_last_normthr_monoord.pth"
)
SCALER_SAVE_PATH = os.path.join(
    BASE_DIR, "random_extra_multitask_finetune_scalers_normthr_monoord.pkl"
)

# ---- Fine-tune data (no var_* columns; will be computed) ----
DATA_PATHS = [
    os.path.join(DATA_DIR, "data_physics_total.csv"),
    os.path.join(DATA_DIR, "data_random_total.csv"),
]

NROWS_PER_DATASET = 10_000_000

# ---- Fine-tune hyperparameters (gentler than from-scratch) ----
FINETUNE_LR = 2e-5         # ~1/5 of base 1e-4 — preserve pretrained features
FINETUNE_WEIGHT_DECAY = 1e-5
FINETUNE_MAX_EPOCHS = 60
FINETUNE_MIN_EPOCHS = 25
FINETUNE_PATIENCE = 8
SCHEDULER_PATIENCE = 3
SCHEDULER_FACTOR = 0.5
GRAD_CLIP = 1.0

EPS = 1e-12
LOG10_HALF = float(np.log10(0.5))

# =========================
# New-model threshold / numeric feature options
# =========================
# Must match the normalized-threshold model trained from scratch.
# If REUSE_SCALERS=True, the mode saved in PRETRAINED_SCALER_PATH is used.
THRESHOLD_FEATURE_MODE = "normalized_logit"
THRESHOLD_NORM_CLIP_EPS = 1e-5
HARMONIC_DENOM_EPS = 1e-8
FEATURE_CLIP_ABS = 1e6

ORDINAL_THRESHOLDS = [
    1e-6, 1e-5, 1e-4, 1e-3, 1e-2, 1e-1,
    0.15, 0.20, 0.25, 0.30, 0.35, 0.40, 0.45,
]
NUM_ORDINAL_THRESHOLDS = len(ORDINAL_THRESHOLDS)
NUM_REGION_CLASSES = NUM_ORDINAL_THRESHOLDS + 1


# =========================
# Utilities
# =========================
def get_sorted_seq_cols(columns, prefix):
    pattern = re.compile(rf"^{re.escape(prefix)}_(\d+)$")
    matched = []
    for col in columns:
        m = pattern.match(col)
        if m:
            matched.append((int(m.group(1)), col))
    matched.sort(key=lambda x: x[0])
    return [col for _, col in matched]


def make_strat_bins(y_log, n_bins=10):
    y_flat = y_log.reshape(-1)
    quantiles = np.linspace(0, 1, n_bins + 1)
    edges = np.quantile(y_flat, quantiles)
    edges = np.unique(edges)
    if len(edges) < 3:
        return None
    bins = np.digitize(y_flat, edges[1:-1], right=True)
    counts = np.bincount(bins)
    if np.any(counts < 2):
        return None
    return bins


def has_nonfinite_tensor(x):
    return not torch.isfinite(x).all().item()


def validate_required_columns(df, file_path, allow_missing_vars=True):
    """Validate columns. With allow_missing_vars=True, var_* columns are
    optional and will be computed from taps if absent."""
    if "N" not in df.columns:
        raise ValueError(f"Required column 'N' not found in {file_path}")

    tap_cols = get_sorted_seq_cols(df.columns, "tap")
    var_cols = get_sorted_seq_cols(df.columns, "var")

    if not tap_cols:
        raise ValueError(f"No tap_* columns found in {file_path}")

    if var_cols:
        if len(tap_cols) != len(var_cols):
            raise ValueError(
                f"tap/var length mismatch in {file_path}: "
                f"{len(tap_cols)} tap cols vs {len(var_cols)} var cols"
            )
        has_vars = True
    else:
        if not allow_missing_vars:
            raise ValueError(f"No var_* columns found in {file_path}")
        has_vars = False

    base_required = tap_cols + ["threshold", "BER", "N"]
    if has_vars:
        base_required = base_required + var_cols
    missing = [c for c in base_required if c not in df.columns]
    if missing:
        raise ValueError(f"Missing required columns in {file_path}: {missing}")

    return tap_cols, var_cols, has_vars


def derive_mem_len_from_taps(df, tap_cols):
    """If mem_len column is missing, derive it from the number of leading
    non-null tap columns. Assumes tap_1..tap_K are filled and tap_(K+1)+ are NaN."""
    if "mem_len" in df.columns:
        return df["mem_len"].to_numpy(dtype=np.int64)

    print(f"  mem_len missing; deriving from non-null tap counts.")
    non_null = df[tap_cols].notna().to_numpy()
    # mem_len = number of non-null taps
    derived = non_null.sum(axis=1).astype(np.int64)
    return derived


def compute_variances_from_taps(taps_raw, N_array, valid_mask):
    """Compute binomial variance var_i = N * tap_i * (1 - tap_i) for each
    valid position. Padding positions (where valid_mask==False) get 0.

    Args:
        taps_raw: [N, L] raw tap probabilities (may have NaN/0 in pad positions)
        N_array:  [N] number of molecules
        valid_mask: [N, L] bool, True = real tap

    Returns:
        vars_raw: [N, L] variance values (raw, not multiplied by N already)
                   matches the convention of the var_* columns in original CSVs
    """
    N = N_array.reshape(-1, 1).astype(np.float64)
    taps = np.where(np.isnan(taps_raw), 0.0, taps_raw).astype(np.float64)
    taps = np.clip(taps, 0.0, 1.0)
    vars_full = N * taps * (1.0 - taps)
    # Re-zero invalid positions
    vars_full = vars_full * valid_mask.astype(np.float64)
    return vars_full.astype(np.float32)


def _read_csv_robust(path, nrows):
    """Read a CSV, skipping any rows that have MORE fields than the header
    declares (treated as corrupted). The C engine with on_bad_lines='skip'
    handles this efficiently — pandas drops over-long rows and keeps only
    the well-formed ones.
    """
    try:
        df = pd.read_csv(path, nrows=nrows, on_bad_lines="skip", engine="c")
    except (ValueError, pd.errors.ParserError) as e:
        # Fallback: python engine is more lenient
        print(f"  C engine failed ({e}); using python engine")
        df = pd.read_csv(path, nrows=nrows, on_bad_lines="skip", engine="python")
    return df


def load_and_merge_data(csv_paths, nrows_per_dataset):
    dfs = []
    reference_tap_cols = None
    union_var_cols = None
    has_vars_per_file = []

    for path in csv_paths:
        if not os.path.exists(path):
            raise FileNotFoundError(f"Dataset not found: {path}")

        print(f"Loading up to {nrows_per_dataset:,} rows from: {path}")
        df = _read_csv_robust(path, nrows=nrows_per_dataset)

        tap_cols, var_cols, has_vars = validate_required_columns(
            df, path, allow_missing_vars=True
        )

        if reference_tap_cols is None:
            reference_tap_cols = tap_cols
            union_var_cols = var_cols if has_vars else []
        else:
            # Reconcile tap columns across files: use the UNION sorted by index.
            # If one file has more tap columns than another, missing columns
            # will be filled with NaN (treated as padding downstream).
            if tap_cols != reference_tap_cols:
                ref_set = set(reference_tap_cols)
                this_set = set(tap_cols)
                only_here = sorted(this_set - ref_set,
                                   key=lambda c: int(c.split("_")[1]))
                only_ref = sorted(ref_set - this_set,
                                  key=lambda c: int(c.split("_")[1]))
                if only_here:
                    print(f"  File adds {len(only_here)} tap cols not in "
                          f"reference (e.g., {only_here[:3]}...). Reference will be extended.")
                if only_ref:
                    print(f"  File missing {len(only_ref)} reference tap cols "
                          f"(e.g., {only_ref[:3]}...); will be NaN-filled.")
                # Extend the reference with any new columns
                reference_tap_cols = sorted(
                    ref_set | this_set,
                    key=lambda c: int(c.split("_")[1]),
                )
                # Add NaN columns to existing dfs that lack the new cols
                for prev in dfs:
                    for c in only_here:
                        if c not in prev.columns:
                            prev[c] = np.nan
                # Add NaN columns to current df for missing ref cols
                for c in only_ref:
                    if c not in df.columns:
                        df[c] = np.nan

            if has_vars and not union_var_cols:
                union_var_cols = var_cols

        df["source_dataset"] = os.path.basename(path)
        df["_has_vars"] = has_vars
        has_vars_per_file.append(has_vars)
        dfs.append(df)

    merged = pd.concat(dfs, ignore_index=True)
    print(f"Combined rows before filtering: {len(merged):,}")
    print(f"  Files with var_* columns: {sum(has_vars_per_file)}/{len(has_vars_per_file)}")
    print(f"  Final tap column count: {len(reference_tap_cols)}")

    numeric_cols = list(dict.fromkeys(
        reference_tap_cols + (union_var_cols or []) + ["mem_len", "threshold", "BER", "N"]
    ))
    merged, numeric_bad_by_col = coerce_numeric_columns(
        merged, numeric_cols, context="merged fine-tune data"
    )
    if numeric_bad_by_col:
        print("Malformed numeric fields were found after CSV alignment; invalid required rows "
              "and invalid valid-tap rows will be dropped during preprocessing.")

    return merged, reference_tap_cols, union_var_cols


def y_log_to_raw_np(y_log):
    return np.clip(10 ** y_log, EPS, 0.5)


def raw_to_region_labels_np(y_raw):
    y = np.asarray(y_raw).reshape(-1)
    labels = np.searchsorted(np.asarray(ORDINAL_THRESHOLDS), y, side="right")
    return labels.astype(np.int64)


def region_labels_to_ordinal_targets_np(labels, num_thresholds=NUM_ORDINAL_THRESHOLDS):
    labels = np.asarray(labels).reshape(-1)
    thresholds = np.arange(1, num_thresholds + 1, dtype=np.int64)
    ordinal = (labels[:, None] >= thresholds[None, :]).astype(np.float32)
    return ordinal


def ordinal_logits_to_region_labels_torch(logits):
    probs = torch.sigmoid(logits)
    passed = (probs >= 0.5).sum(dim=1)
    return passed.long()


def compute_region_class_weights(labels_np, num_classes=NUM_REGION_CLASSES, max_weight=8.0):
    counts = np.bincount(labels_np.reshape(-1), minlength=num_classes).astype(np.float64)
    counts = np.maximum(counts, 1.0)
    weights = counts.sum() / (num_classes * counts)
    weights = np.sqrt(weights)
    weights = np.clip(weights, 1.0, max_weight)
    return weights.astype(np.float32)


def compute_ordinal_pos_weights_from_region_labels(
    region_labels_np,
    num_thresholds=NUM_ORDINAL_THRESHOLDS,
    max_weight=20.0,
):
    ordinal_targets = region_labels_to_ordinal_targets_np(region_labels_np, num_thresholds)
    pos_counts = ordinal_targets.sum(axis=0)
    neg_counts = ordinal_targets.shape[0] - pos_counts
    pos_counts = np.maximum(pos_counts, 1.0)
    pos_weight = neg_counts / pos_counts
    pos_weight = np.clip(pos_weight, 1.0, max_weight)
    return pos_weight.astype(np.float32)


# =========================
# Position-Independent Scaling helpers
# =========================
def fit_shared_scalar_scaler(train_data, train_valid_mask):
    entries = train_data[train_valid_mask].reshape(-1)
    if len(entries) == 0:
        entries = train_data.reshape(-1)
    mean = float(entries.mean())
    std = float(max(entries.std(), 1e-12))
    return mean, std


def apply_shared_scale(data, mean, std, valid_mask=None):
    scaled = ((data - mean) / std).astype(np.float32)
    if valid_mask is not None and scaled.ndim == 2 and valid_mask.shape == scaled.shape:
        scaled = scaled * valid_mask.astype(np.float32)
    return scaled


def sanitize_np_feature(x, name, clip_abs=FEATURE_CLIP_ABS):
    """Replace NaN/inf with 0 and clip extreme engineered features."""
    arr = np.asarray(x, dtype=np.float32)
    bad = ~np.isfinite(arr)
    if np.any(bad):
        print(f"Warning: {name} contained {int(bad.sum())} non-finite values; replacing with 0.")
    arr = np.nan_to_num(arr, nan=0.0, posinf=0.0, neginf=0.0)
    arr = np.clip(arr, -clip_abs, clip_abs).astype(np.float32)
    return arr


def stable_harmonic_pair_np(a, b, denom_eps=HARMONIC_DENOM_EPS):
    """Stable 2ab/(a+b), with near-zero denominators mapped to zero."""
    a = np.asarray(a, dtype=np.float32)
    b = np.asarray(b, dtype=np.float32)
    denom = (a + b).astype(np.float32)
    numer = (2.0 * a * b).astype(np.float32)
    out = np.divide(
        numer, denom,
        out=np.zeros_like(numer, dtype=np.float32),
        where=np.abs(denom) > denom_eps,
    ).astype(np.float32)
    return sanitize_np_feature(out, "stable_harmonic_pair")


def compute_threshold_feature_np(threshold_raw, total_scaled_taps, mode=THRESHOLD_FEATURE_MODE):
    """Build model threshold feature.

    normalized_logit: logit(clip(threshold / sum(valid tap_i*N), eps, 1-eps))
    normalized:       threshold / sum(valid tap_i*N)
    log10_raw:        legacy log10(threshold + EPS)
    """
    thr = np.asarray(threshold_raw, dtype=np.float32).reshape(-1, 1)
    total = np.asarray(total_scaled_taps, dtype=np.float32).reshape(-1, 1)
    denom = np.maximum(total, EPS).astype(np.float32)

    if mode == "normalized":
        feat = (thr / denom).astype(np.float32)
    elif mode == "normalized_logit":
        u = (thr / denom).astype(np.float32)
        u = np.clip(u, THRESHOLD_NORM_CLIP_EPS, 1.0 - THRESHOLD_NORM_CLIP_EPS)
        feat = np.log(u / (1.0 - u)).astype(np.float32)
    elif mode == "log10_raw":
        feat = np.log10(thr + EPS).astype(np.float32)
    else:
        raise ValueError(f"Unknown THRESHOLD_FEATURE_MODE: {mode}")

    return sanitize_np_feature(feat, f"threshold_feature_{mode}")


def coerce_numeric_columns(df, numeric_cols, context="dataframe"):
    """Coerce numeric CSV columns and report malformed cells."""
    bad_by_col = {}
    for col in numeric_cols:
        if col not in df.columns:
            continue
        original = df[col]
        converted = pd.to_numeric(original, errors="coerce")
        bad = converted.isna() & original.notna() & (original.astype(str).str.strip() != "")
        n_bad = int(bad.sum())
        if n_bad:
            bad_by_col[col] = n_bad
        df[col] = converted
    if bad_by_col:
        total_bad = sum(bad_by_col.values())
        top = sorted(bad_by_col.items(), key=lambda kv: kv[1], reverse=True)[:8]
        print(f"{context}: coerced {total_bad:,} malformed numeric cells to NaN. Top columns: {top}")
    return df, bad_by_col


# =========================
# Loss
# =========================
class StableMultiObjectiveBERBoundedLogLoss(nn.Module):
    def __init__(self, log_delta=0.35, raw_delta=0.006, rel_delta=0.03,
                 alpha_log=0.45, beta_raw=0.35, gamma_rel=0.20,
                 use_regime_weights=True):
        super().__init__()
        self.log_delta = log_delta
        self.raw_delta = raw_delta
        self.rel_delta = rel_delta
        self.alpha_log = alpha_log
        self.beta_raw = beta_raw
        self.gamma_rel = gamma_rel
        self.use_regime_weights = use_regime_weights

    @staticmethod
    def huber_elementwise(pred, target, delta):
        err = pred - target
        abs_err = err.abs()
        return torch.where(abs_err < delta,
                           0.5 * err * err,
                           delta * (abs_err - 0.5 * delta))

    @staticmethod
    def raw_to_pred_log(pred_raw_unconstrained):
        log10_half = torch.log10(torch.tensor(
            0.5, device=pred_raw_unconstrained.device,
            dtype=pred_raw_unconstrained.dtype))
        return log10_half - F.softplus(pred_raw_unconstrained)

    @staticmethod
    def raw_to_pred_ber(pred_raw_unconstrained):
        pred_log = StableMultiObjectiveBERBoundedLogLoss.raw_to_pred_log(pred_raw_unconstrained)
        return torch.pow(10.0, pred_log).clamp(min=EPS, max=0.5)

    def forward(self, pred_raw_unconstrained, target_log):
        pred_log = self.raw_to_pred_log(pred_raw_unconstrained)
        pred_raw = torch.pow(10.0, pred_log).clamp(min=EPS, max=0.5)
        target_raw = torch.pow(10.0, target_log).clamp(min=EPS, max=0.5)
        target_log_for_loss = torch.log10(target_raw)

        log_loss = self.huber_elementwise(pred_log, target_log_for_loss, self.log_delta)
        raw_loss = self.huber_elementwise(pred_raw, target_raw, self.raw_delta)
        rel_err = (pred_raw - target_raw) / torch.clamp(target_raw, min=1e-6)
        rel_loss = self.huber_elementwise(rel_err, torch.zeros_like(rel_err), self.rel_delta)

        total = (self.alpha_log * log_loss
                 + self.beta_raw * raw_loss
                 + self.gamma_rel * rel_loss)

        if self.use_regime_weights:
            weights = torch.ones_like(target_raw)
            weights = torch.where((target_raw >= 1e-2) & (target_raw < 0.1),
                                  torch.full_like(weights, 1.5), weights)
            weights = torch.where((target_raw >= 0.1) & (target_raw < 0.15),
                                  torch.full_like(weights, 2.5), weights)
            weights = torch.where((target_raw >= 0.15) & (target_raw < 0.2),
                                  torch.full_like(weights, 3.0), weights)
            weights = torch.where((target_raw >= 0.2) & (target_raw < 0.3),
                                  torch.full_like(weights, 4.0), weights)
            weights = torch.where((target_raw >= 0.3) & (target_raw < 0.4),
                                  torch.full_like(weights, 4.5), weights)
            weights = torch.where((target_raw >= 0.4) & (target_raw < 0.45),
                                  torch.full_like(weights, 3.0), weights)
            weights = torch.where(target_raw >= 0.45,
                                  torch.full_like(weights, 2.5), weights)
            total = total * weights

        return total.mean()


class OrdinalBCELoss(nn.Module):
    def __init__(self, pos_weight=None, reduction="mean"):
        super().__init__()
        if pos_weight is not None and not isinstance(pos_weight, torch.Tensor):
            pos_weight = torch.tensor(pos_weight, dtype=torch.float32)
        self.register_buffer("pos_weight", pos_weight if pos_weight is not None else None)
        self.reduction = reduction

    def forward(self, logits, ordinal_targets):
        return F.binary_cross_entropy_with_logits(
            logits, ordinal_targets, pos_weight=self.pos_weight, reduction=self.reduction)


class MultiTaskBERLoss(nn.Module):
    def __init__(self, reg_loss, ord_loss, lambda_ord=0.25):
        super().__init__()
        self.reg_loss = reg_loss
        self.ord_loss = ord_loss
        self.lambda_ord = lambda_ord

    def forward(self, pred_raw_unconstrained, ord_logits, target_log, target_ord):
        reg = self.reg_loss(pred_raw_unconstrained, target_log)
        ordl = self.ord_loss(ord_logits, target_ord)
        total = reg + self.lambda_ord * ordl
        return total, reg.detach(), ordl.detach()


# =========================
# Set Transformer Model (normalized-threshold + monotonic-ordinal architecture)
# =========================
class SetSelfAttentionBlock(nn.Module):
    def __init__(self, d_model, num_heads, mlp_ratio=4.0, dropout=0.1):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model)
        self.attn = nn.MultiheadAttention(
            embed_dim=d_model, num_heads=num_heads,
            dropout=dropout, batch_first=True)
        self.norm2 = nn.LayerNorm(d_model)
        hidden = int(d_model * mlp_ratio)
        self.mlp = nn.Sequential(
            nn.Linear(d_model, hidden), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(hidden, d_model), nn.Dropout(dropout))

    def forward(self, x, key_padding_mask=None):
        y = self.norm1(x)
        attn_out, _ = self.attn(y, y, y, need_weights=False, key_padding_mask=key_padding_mask)
        x = x + attn_out
        x = x + self.mlp(self.norm2(x))
        return x


class ConditionalFeatureModulation(nn.Module):
    def __init__(self, cond_dim, feat_dim, hidden_dim=256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(cond_dim, hidden_dim), nn.GELU(),
            nn.Linear(hidden_dim, 2 * feat_dim))
        self.scale = nn.Parameter(torch.tensor(0.25, dtype=torch.float32))

    def forward(self, x, cond):
        gamma_beta = self.net(cond)
        gamma, beta = torch.chunk(gamma_beta, 2, dim=-1)
        if x.dim() == 3:
            gamma = gamma.unsqueeze(1)
            beta = beta.unsqueeze(1)
        return x * (1.0 + self.scale * gamma) + self.scale * beta


class SetAttentionPooling(nn.Module):
    def __init__(self, d_model, hidden_dim=128):
        super().__init__()
        self.score = nn.Sequential(
            nn.Linear(d_model, hidden_dim), nn.GELU(), nn.Linear(hidden_dim, 1))

    def forward(self, x, key_padding_mask=None):
        logits = self.score(x)
        if key_padding_mask is not None:
            logits = logits.masked_fill(key_padding_mask.unsqueeze(-1), float("-inf"))
        weights = torch.softmax(logits, dim=1)
        weights = torch.nan_to_num(weights, nan=0.0)
        return (weights * x).sum(dim=1)


class MonotonicOrdinalHead(nn.Module):
    """Ordinal head with monotonic probabilities by construction.

    The ordinal targets are P(BER >= threshold_k), which must be non-increasing
    as the threshold index k increases. The head constructs non-increasing logits:
        logit_1 = free
        logit_k = logit_{k-1} - softplus(delta_k)
    Since sigmoid is monotone, probabilities are also non-increasing.
    """
    def __init__(self, head_in, num_thresholds):
        super().__init__()
        self.num_thresholds = num_thresholds
        self.base = nn.Sequential(
            nn.Linear(head_in, 256), nn.GELU(), nn.Dropout(0.20),
            nn.Linear(256, 128), nn.GELU(), nn.Dropout(0.15),
        )
        self.first_logit = nn.Linear(128, 1)
        self.deltas = nn.Linear(128, max(num_thresholds - 1, 1))

    def forward(self, x):
        h = self.base(x)
        first = self.first_logit(h)
        if self.num_thresholds == 1:
            return first
        deltas = -F.softplus(self.deltas(h)[:, : self.num_thresholds - 1])
        return torch.cat([first, deltas], dim=-1).cumsum(dim=-1)


class BERFirstTokenSetTransformerBoundedLogHalfMultiTask(nn.Module):
    def __init__(self, max_past_seq_len, token_dim=4, first_token_dim=4,
                 threshold_dim=1, global_dim=7, d_model=128, num_set_layers=4,
                 num_heads=4, mlp_ratio=4.0, dropout=0.10,
                 num_ordinal_thresholds=NUM_ORDINAL_THRESHOLDS):
        super().__init__()
        self.max_past_seq_len = max_past_seq_len
        self.d_model = d_model
        self.num_ordinal_thresholds = num_ordinal_thresholds

        self.first_proj = nn.Sequential(
            nn.Linear(first_token_dim, d_model), nn.LayerNorm(d_model),
            nn.GELU(), nn.Dropout(dropout))
        self.token_proj = nn.Sequential(
            nn.Linear(token_dim, d_model), nn.LayerNorm(d_model),
            nn.GELU(), nn.Dropout(dropout))
        self.thr_embed = nn.Sequential(
            nn.Linear(threshold_dim, 32), nn.GELU(),
            nn.Linear(32, 32), nn.GELU())
        self.global_embed = nn.Sequential(
            nn.Linear(global_dim, 96), nn.GELU(),
            nn.Linear(96, 96), nn.GELU())

        cond_dim = 32 + 96
        self.first_cond_mod = ConditionalFeatureModulation(cond_dim, d_model, 256)
        self.set_cond_mod = ConditionalFeatureModulation(cond_dim, d_model, 256)

        self.set_blocks = nn.ModuleList([
            SetSelfAttentionBlock(d_model, num_heads, mlp_ratio, dropout)
            for _ in range(num_set_layers)])

        self.first_refine = nn.Sequential(
            nn.LayerNorm(d_model), nn.Linear(d_model, d_model),
            nn.GELU(), nn.Dropout(dropout), nn.Linear(d_model, d_model))
        self.final_set_norm = nn.LayerNorm(d_model)
        self.set_attn_pool = SetAttentionPooling(d_model, 128)

        head_in = d_model + 3 * d_model + cond_dim
        self.reg_head = nn.Sequential(
            nn.Linear(head_in, 256), nn.GELU(), nn.Dropout(0.20),
            nn.Linear(256, 128), nn.GELU(), nn.Dropout(0.15), nn.Linear(128, 1))
        self.ord_head = MonotonicOrdinalHead(head_in, num_ordinal_thresholds)

    @staticmethod
    def raw_to_log10ber(x):
        log10_half = torch.log10(torch.tensor(0.5, device=x.device, dtype=x.dtype))
        return log10_half - F.softplus(x)

    @staticmethod
    def raw_to_ber(x):
        return torch.pow(10.0, BERFirstTokenSetTransformerBoundedLogHalfMultiTask.raw_to_log10ber(x)).clamp(EPS, 0.5)

    def forward(self, first_token, past_tokens, global_feats, threshold, key_padding_mask=None):
        thr = self.thr_embed(threshold)
        g = self.global_embed(global_feats)
        cond = torch.cat([thr, g], dim=-1)

        first_x = self.first_proj(first_token)
        first_x = self.first_cond_mod(first_x, cond)
        first_x = first_x + self.first_refine(first_x)

        set_x = self.token_proj(past_tokens)
        set_x = self.set_cond_mod(set_x, cond)
        for block in self.set_blocks:
            set_x = block(set_x, key_padding_mask=key_padding_mask)
        set_x = self.final_set_norm(set_x)

        if key_padding_mask is not None:
            valid_mask = (~key_padding_mask).unsqueeze(-1).float()
        else:
            valid_mask = torch.ones(set_x.shape[0], set_x.shape[1], 1,
                                    device=set_x.device, dtype=set_x.dtype)

        pooled_attn = self.set_attn_pool(set_x, key_padding_mask=key_padding_mask)
        valid_counts = valid_mask.sum(dim=1).clamp(min=1.0)
        pooled_mean = (set_x * valid_mask).sum(dim=1) / valid_counts

        if key_padding_mask is not None:
            x_for_max = set_x.masked_fill(key_padding_mask.unsqueeze(-1), float("-inf"))
        else:
            x_for_max = set_x
        pooled_max = torch.nan_to_num(x_for_max.amax(dim=1), nan=0.0, posinf=0.0, neginf=0.0)

        fused = torch.cat([first_x, pooled_attn, pooled_mean, pooled_max, cond], dim=-1)
        return self.reg_head(fused), self.ord_head(fused)


# =========================
# Data preparation (handles missing var_* columns)
# =========================
def prepare_data_for_finetune(
    csv_paths, batch_size=256, nrows_per_dataset=2_500_000, num_workers=0,
    pretrained_scalers=None, reuse_scalers=True,
):
    """Same pipeline as v4's prepare_data, plus:
       - var_* columns are computed from taps if missing
       - mem_len column is derived from non-null taps if missing
       - if reuse_scalers=True and pretrained_scalers is provided, those
         scalers are reused (recommended for fine-tuning)
    """
    df, tap_cols, var_cols = load_and_merge_data(csv_paths, nrows_per_dataset)

    # ---- Derive mem_len if missing ----
    if "mem_len" not in df.columns:
        df["mem_len"] = derive_mem_len_from_taps(df, tap_cols)

    # Drop malformed/non-numeric mem_len rows, then remove one-tap cases.
    n_before_mem = len(df)
    df = df[pd.to_numeric(df["mem_len"], errors="coerce").notna()].copy()
    if len(df) < n_before_mem:
        print(f"  Dropped {n_before_mem - len(df):,} rows with invalid mem_len")
    df["mem_len"] = df["mem_len"].astype(np.int64)
    df = df[df["mem_len"] != 1].copy()

    # Do not fill tap NaNs yet: NaN inside the valid mem_len region indicates
    # a malformed/corrupted required tap and is dropped below. NaNs outside the
    # valid region are padding and are zero-filled after the mask is built.
    df["threshold"] = df["threshold"].fillna(0.0)
    df["BER"] = df["BER"].fillna(0.0)
    df["N"] = df["N"].fillna(0.0)

    df = df[(df["BER"] > 0) & (df["N"] > 0)].copy()
    df["BER"] = df["BER"].clip(lower=EPS, upper=0.5)

    # Allow threshold==0 rows? In original training threshold>0 was required.
    # The new random data has rows with threshold=0; filter them out
    # because they don't correspond to a real decision threshold.
    n_before = len(df)
    df = df[df["threshold"] > 0].copy()
    if len(df) < n_before:
        print(f"  Dropped {n_before - len(df):,} rows with threshold=0")

    print(f"Rows after cleaning/filtering: {len(df):,}")

    # ---- Extract raw arrays ----
    X_taps_raw = df[tap_cols].to_numpy(dtype=np.float32)
    num_molecules = df["N"].to_numpy(dtype=np.float32).reshape(-1, 1)
    mem_len = df["mem_len"].to_numpy(dtype=np.int64)

    X_thr_raw = df["threshold"].to_numpy(dtype=np.float32).reshape(-1, 1)
    y_raw = df["BER"].to_numpy(dtype=np.float32).reshape(-1, 1)

    # X_thr is computed after valid-memory masking because normalized threshold
    # needs sum(valid tap_i * N). y_log remains the regression target.
    y_log = np.log10(y_raw + EPS).astype(np.float32)

    L_full = X_taps_raw.shape[1]
    mem_len_clamped = np.minimum(mem_len, L_full)
    valid_full = (np.arange(L_full)[None, :] < mem_len_clamped[:, None])
    valid_past = valid_full[:, 1:]
    valid_past_float = valid_past.astype(np.float32)
    past_lens = np.maximum(mem_len_clamped - 1, 0).astype(np.int64)

    print(f"  Tap column count L_full = {L_full}, "
          f"mem_len range = [{int(mem_len.min())}, {int(mem_len.max())}], "
          f"past_lens range = [{int(past_lens.min())}, {int(past_lens.max())}]")
    if mem_len.max() > L_full:
        print(f"    NOTE: {(mem_len > L_full).sum():,} rows have mem_len > L_full; "
              f"will be clamped to {L_full}")

    bad_valid_tap = (~np.isfinite(X_taps_raw)) & valid_full
    bad_valid_tap_rows = bad_valid_tap.any(axis=1)
    if np.any(bad_valid_tap_rows):
        n_bad = int(bad_valid_tap_rows.sum())
        print(f"  Dropping {n_bad:,} rows with malformed/non-numeric tap values inside valid mem_len region.")
        keep = ~bad_valid_tap_rows
        df = df.iloc[keep].copy()
        X_taps_raw = X_taps_raw[keep]
        num_molecules = num_molecules[keep]
        mem_len = mem_len[keep]
        X_thr_raw = X_thr_raw[keep]
        y_raw = y_raw[keep]
        y_log = y_log[keep]
        mem_len_clamped = mem_len_clamped[keep]
        valid_full = valid_full[keep]
        valid_past = valid_past[keep]
        valid_past_float = valid_past_float[keep]
        past_lens = past_lens[keep]

    # Padding/outside valid memory can safely be zero-filled now.
    X_taps_raw = np.nan_to_num(X_taps_raw, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)
    valid_full_float = valid_full.astype(np.float32)

    # ---- Variances: load from CSV if present, else compute from taps ----
    if var_cols and all(c in df.columns for c in var_cols):
        # File has var columns — use them, fill missing with computed values per row
        X_vars_raw_csv = df[var_cols].to_numpy(dtype=np.float32)

        # Identify rows where _has_vars indicates the file had var columns
        if "_has_vars" in df.columns:
            has_vars_mask = df["_has_vars"].to_numpy(dtype=bool)
        else:
            has_vars_mask = np.ones(len(df), dtype=bool)

        # Compute variance for ALL rows, then overlay CSV values where available
        X_vars_computed = compute_variances_from_taps(X_taps_raw, num_molecules.reshape(-1), valid_full)

        X_vars_raw = X_vars_computed.copy()
        # For rows that DO have var columns in CSV, use those values where non-NaN
        has_vars_idx = np.where(has_vars_mask)[0]
        if len(has_vars_idx) > 0:
            csv_vars = np.where(np.isnan(X_vars_raw_csv[has_vars_idx]),
                                X_vars_computed[has_vars_idx],
                                X_vars_raw_csv[has_vars_idx])
            X_vars_raw[has_vars_idx] = csv_vars

        n_computed = (~has_vars_mask).sum()
        print(f"  Variances: {has_vars_mask.sum():,} loaded from CSV, "
              f"{n_computed:,} computed from N*tap*(1-tap)")
    else:
        # No var columns at all — compute everything
        X_vars_raw = compute_variances_from_taps(
            X_taps_raw, num_molecules.reshape(-1), valid_full)
        print(f"  Variances: all {len(df):,} computed from N*tap*(1-tap)")

    # Sanity: variances should be non-negative and zero on padding
    if np.any(X_vars_raw < 0):
        # Numerical garbage; clamp
        X_vars_raw = np.maximum(X_vars_raw, 0.0)

    # ---- Feature engineering (matches normalized-threshold model) ----
    X_taps_feat = (X_taps_raw * num_molecules).astype(np.float32)
    X_vars_feat = X_vars_raw.astype(np.float32)
    abs_taps_raw = np.abs(X_taps_feat).astype(np.float32)
    snr_raw = np.log10((X_taps_feat ** 2) / (X_vars_raw + EPS) + EPS).astype(np.float32)
    snr_raw = sanitize_np_feature(snr_raw, "snr_raw")

    threshold_mode = (
        pretrained_scalers.get("threshold_feature_mode", THRESHOLD_FEATURE_MODE)
        if (reuse_scalers and pretrained_scalers is not None)
        else THRESHOLD_FEATURE_MODE
    )
    total_scaled_taps_raw = np.sum(
        X_taps_feat * valid_full_float,
        axis=1, keepdims=True,
    ).astype(np.float32)
    X_thr = compute_threshold_feature_np(
        X_thr_raw, total_scaled_taps_raw, mode=threshold_mode
    )

    # ---- Mask-aware sums ----
    first_mean = X_taps_feat[:, 0:1]
    past_means = X_taps_feat[:, 1:]
    first_var = X_vars_feat[:, 0:1]
    past_vars = X_vars_feat[:, 1:]

    past_means_masked = past_means * valid_past_float
    past_vars_masked = past_vars * valid_past_float

    mu0_raw = (0.5 * np.sum(past_means_masked, axis=1, keepdims=True)).astype(np.float32)
    mu1_raw = (first_mean + mu0_raw).astype(np.float32)
    var0_raw = (0.5 * np.sum(past_vars_masked, axis=1, keepdims=True)).astype(np.float32)
    var1_raw = (first_var + var0_raw).astype(np.float32)

    std0_raw = np.sqrt(np.maximum(var0_raw, EPS)).astype(np.float32)
    std1_raw = np.sqrt(np.maximum(var1_raw, EPS)).astype(np.float32)

    z0_raw = ((X_thr_raw - mu0_raw) / (std0_raw + EPS)).astype(np.float32)
    z1_raw = ((mu1_raw - X_thr_raw) / (std1_raw + EPS)).astype(np.float32)

    z0_raw = sanitize_np_feature(z0_raw, "z0_raw")
    z1_raw = sanitize_np_feature(z1_raw, "z1_raw")
    harmonic_side_z_raw = stable_harmonic_pair_np(z0_raw, z1_raw)
    abs_diff_side_z_raw = np.abs(z0_raw - z1_raw).astype(np.float32)
    harmonic_minus_gap_raw = sanitize_np_feature(
        harmonic_side_z_raw - 0.25 * abs_diff_side_z_raw,
        "harmonic_minus_gap_raw",
    )

    # ---- Scenario-level features ----
    signal_raw = first_mean
    isi_raw = np.sum(past_means_masked, axis=1, keepdims=True)
    nsid_raw = ((signal_raw - isi_raw) / (signal_raw + isi_raw + EPS)).astype(np.float32)

    gap_raw = signal_raw
    d0_raw = (gap_raw / (std0_raw + EPS)).astype(np.float32)
    d1_raw = (gap_raw / (std1_raw + EPS)).astype(np.float32)
    harmonic_discrim_raw = (2.0 * d0_raw * d1_raw / (d0_raw + d1_raw + EPS)).astype(np.float32)
    log_harmonic_discrim_raw = np.log10(harmonic_discrim_raw + EPS).astype(np.float32)
    discrim_asymmetry_raw = (np.abs(d0_raw - d1_raw) / (d0_raw + d1_raw + EPS)).astype(np.float32)

    past_taps_sum = np.sum(past_means_masked, axis=1, keepdims=True)
    past_shares = past_means_masked / (past_taps_sum + EPS)
    herfindahl_raw = np.sum(past_shares ** 2, axis=1, keepdims=True).astype(np.float32)

    nsid_raw = sanitize_np_feature(nsid_raw, "nsid_raw")
    log_harmonic_discrim_raw = sanitize_np_feature(log_harmonic_discrim_raw, "log_harmonic_discrim_raw")
    discrim_asymmetry_raw = sanitize_np_feature(discrim_asymmetry_raw, "discrim_asymmetry_raw")
    herfindahl_raw = sanitize_np_feature(herfindahl_raw, "herfindahl_raw")

    # ---- Labels ----
    region_labels = raw_to_region_labels_np(y_raw)
    ordinal_targets = region_labels_to_ordinal_targets_np(region_labels)
    strat_labels = make_strat_bins(y_log, n_bins=10)

    # ---- Train / temp split ----
    split_args = dict(test_size=0.30, random_state=42)
    if strat_labels is not None:
        split_args["stratify"] = strat_labels

    (
        t_taps_feat, temp_taps_feat,
        t_vars_feat, temp_vars_feat,
        t_abs_raw, temp_abs_raw,
        t_snr_raw, temp_snr_raw,
        t_z0_raw, temp_z0_raw,
        t_z1_raw, temp_z1_raw,
        t_hmg_raw, temp_hmg_raw,
        t_nsid_raw, temp_nsid_raw,
        t_log_hd_raw, temp_log_hd_raw,
        t_da_raw, temp_da_raw,
        t_herf_raw, temp_herf_raw,
        t_thr, temp_thr,
        t_y_log, temp_y_log,
        t_region, temp_region,
        t_ord, temp_ord,
        t_past_lens, temp_past_lens,
    ) = train_test_split(
        X_taps_feat, X_vars_feat, abs_taps_raw, snr_raw,
        z0_raw, z1_raw, harmonic_minus_gap_raw,
        nsid_raw, log_harmonic_discrim_raw, discrim_asymmetry_raw, herfindahl_raw,
        X_thr,
        y_log, region_labels, ordinal_targets, past_lens,
        **split_args,
    )

    temp_strat = make_strat_bins(temp_y_log, n_bins=6)
    split_args2 = dict(test_size=0.50, random_state=42)
    if temp_strat is not None:
        split_args2["stratify"] = temp_strat

    (
        v_taps_feat, te_taps_feat,
        v_vars_feat, te_vars_feat,
        v_abs_raw, te_abs_raw,
        v_snr_raw, te_snr_raw,
        v_z0_raw, te_z0_raw,
        v_z1_raw, te_z1_raw,
        v_hmg_raw, te_hmg_raw,
        v_nsid_raw, te_nsid_raw,
        v_log_hd_raw, te_log_hd_raw,
        v_da_raw, te_da_raw,
        v_herf_raw, te_herf_raw,
        v_thr, te_thr,
        v_y_log, te_y_log,
        v_region, te_region,
        v_ord, te_ord,
        v_past_lens, te_past_lens,
    ) = train_test_split(
        temp_taps_feat, temp_vars_feat, temp_abs_raw, temp_snr_raw,
        temp_z0_raw, temp_z1_raw, temp_hmg_raw,
        temp_nsid_raw, temp_log_hd_raw, temp_da_raw, temp_herf_raw,
        temp_thr,
        temp_y_log, temp_region, temp_ord, temp_past_lens,
        **split_args2,
    )

    L_past = L_full - 1
    t_valid_past = (np.arange(L_past)[None, :] < t_past_lens[:, None])

    # =========================================================
    # Scalers: reuse pretrained or fit from scratch
    # =========================================================
    if reuse_scalers and pretrained_scalers is not None:
        print("  Reusing pretrained scalers (recommended for fine-tuning).")

        first_tap_mean = pretrained_scalers["first_tap_mean"]
        first_tap_std = pretrained_scalers["first_tap_std"]
        first_var_mean = pretrained_scalers["first_var_mean"]
        first_var_std = pretrained_scalers["first_var_std"]
        first_abs_mean = pretrained_scalers["first_abs_mean"]
        first_abs_std = pretrained_scalers["first_abs_std"]
        first_snr_mean = pretrained_scalers["first_snr_mean"]
        first_snr_std = pretrained_scalers["first_snr_std"]

        past_tap_mean = pretrained_scalers["past_tap_mean"]
        past_tap_std = pretrained_scalers["past_tap_std"]
        past_var_mean = pretrained_scalers["past_var_mean"]
        past_var_std = pretrained_scalers["past_var_std"]
        past_abs_mean = pretrained_scalers["past_abs_mean"]
        past_abs_std = pretrained_scalers["past_abs_std"]
        past_snr_mean = pretrained_scalers["past_snr_mean"]
        past_snr_std = pretrained_scalers["past_snr_std"]

        z0_scaler = pretrained_scalers["z0_scaler"]
        z1_scaler = pretrained_scalers["z1_scaler"]
        hmg_scaler = pretrained_scalers["harmonic_minus_gap_scaler"]
        nsid_scaler = pretrained_scalers["nsid_scaler"]
        log_hd_scaler = pretrained_scalers["log_hd_scaler"]
        da_scaler = pretrained_scalers["da_scaler"]
        herf_scaler = pretrained_scalers["herf_scaler"]
        thr_scaler = pretrained_scalers["thr_scaler"]
    else:
        print("  Refitting scalers on fine-tune data.")
        first_tap_mean, first_tap_std = float(t_taps_feat[:, 0].mean()), max(float(t_taps_feat[:, 0].std()), 1e-12)
        first_var_mean, first_var_std = float(t_vars_feat[:, 0].mean()), max(float(t_vars_feat[:, 0].std()), 1e-12)
        first_abs_mean, first_abs_std = float(t_abs_raw[:, 0].mean()), max(float(t_abs_raw[:, 0].std()), 1e-12)
        first_snr_mean, first_snr_std = float(t_snr_raw[:, 0].mean()), max(float(t_snr_raw[:, 0].std()), 1e-12)

        past_tap_mean, past_tap_std = fit_shared_scalar_scaler(t_taps_feat[:, 1:], t_valid_past)
        past_var_mean, past_var_std = fit_shared_scalar_scaler(t_vars_feat[:, 1:], t_valid_past)
        past_abs_mean, past_abs_std = fit_shared_scalar_scaler(t_abs_raw[:, 1:], t_valid_past)
        past_snr_mean, past_snr_std = fit_shared_scalar_scaler(t_snr_raw[:, 1:], t_valid_past)

        z0_scaler = StandardScaler().fit(t_z0_raw)
        z1_scaler = StandardScaler().fit(t_z1_raw)
        hmg_scaler = StandardScaler().fit(t_hmg_raw)
        nsid_scaler = StandardScaler().fit(t_nsid_raw)
        log_hd_scaler = StandardScaler().fit(t_log_hd_raw)
        da_scaler = StandardScaler().fit(t_da_raw)
        herf_scaler = StandardScaler().fit(t_herf_raw)
        thr_scaler = StandardScaler().fit(t_thr)

    def build_tokens(taps_feat, vars_feat, abs_raw, snr_raw, p_lens):
        ft_tap = ((taps_feat[:, 0] - first_tap_mean) / first_tap_std).astype(np.float32)
        ft_var = ((vars_feat[:, 0] - first_var_mean) / first_var_std).astype(np.float32)
        ft_abs = ((abs_raw[:, 0] - first_abs_mean) / first_abs_std).astype(np.float32)
        ft_snr = ((snr_raw[:, 0] - first_snr_mean) / first_snr_std).astype(np.float32)
        first_token = np.stack([ft_tap, ft_var, ft_abs, ft_snr], axis=1).astype(np.float32)

        pt_valid = (np.arange(L_past)[None, :] < p_lens[:, None])
        pt_tap = apply_shared_scale(taps_feat[:, 1:], past_tap_mean, past_tap_std, pt_valid)
        pt_var = apply_shared_scale(vars_feat[:, 1:], past_var_mean, past_var_std, pt_valid)
        pt_abs = apply_shared_scale(abs_raw[:, 1:], past_abs_mean, past_abs_std, pt_valid)
        pt_snr = apply_shared_scale(snr_raw[:, 1:], past_snr_mean, past_snr_std, pt_valid)
        past_tokens = np.stack([pt_tap, pt_var, pt_abs, pt_snr], axis=2).astype(np.float32)

        return first_token, past_tokens

    def build_global(z0, z1, hmg, nsid, log_hd, da, herf):
        return np.concatenate([
            z0_scaler.transform(z0),
            z1_scaler.transform(z1),
            hmg_scaler.transform(hmg),
            nsid_scaler.transform(nsid),
            log_hd_scaler.transform(log_hd),
            da_scaler.transform(da),
            herf_scaler.transform(herf),
        ], axis=1).astype(np.float32)

    def build_padding_mask(p_lens, L_max):
        return (np.arange(L_max)[None, :] >= p_lens[:, None])

    t_first, t_past = build_tokens(t_taps_feat, t_vars_feat, t_abs_raw, t_snr_raw, t_past_lens)
    v_first, v_past = build_tokens(v_taps_feat, v_vars_feat, v_abs_raw, v_snr_raw, v_past_lens)
    te_first, te_past = build_tokens(te_taps_feat, te_vars_feat, te_abs_raw, te_snr_raw, te_past_lens)

    t_global = build_global(t_z0_raw, t_z1_raw, t_hmg_raw,
                            t_nsid_raw, t_log_hd_raw, t_da_raw, t_herf_raw)
    v_global = build_global(v_z0_raw, v_z1_raw, v_hmg_raw,
                            v_nsid_raw, v_log_hd_raw, v_da_raw, v_herf_raw)
    te_global = build_global(te_z0_raw, te_z1_raw, te_hmg_raw,
                             te_nsid_raw, te_log_hd_raw, te_da_raw, te_herf_raw)

    t_thr_s = thr_scaler.transform(t_thr).astype(np.float32)
    v_thr_s = thr_scaler.transform(v_thr).astype(np.float32)
    te_thr_s = thr_scaler.transform(te_thr).astype(np.float32)

    L_max_past = L_past
    t_mask = build_padding_mask(t_past_lens, L_max_past)
    v_mask = build_padding_mask(v_past_lens, L_max_past)
    te_mask = build_padding_mask(te_past_lens, L_max_past)

    train_ds = TensorDataset(
        torch.from_numpy(t_first), torch.from_numpy(t_past),
        torch.from_numpy(t_global), torch.from_numpy(t_thr_s),
        torch.from_numpy(t_y_log.astype(np.float32)),
        torch.from_numpy(t_ord.astype(np.float32)),
        torch.from_numpy(t_region.astype(np.int64)),
        torch.from_numpy(t_mask))
    val_ds = TensorDataset(
        torch.from_numpy(v_first), torch.from_numpy(v_past),
        torch.from_numpy(v_global), torch.from_numpy(v_thr_s),
        torch.from_numpy(v_y_log.astype(np.float32)),
        torch.from_numpy(v_ord.astype(np.float32)),
        torch.from_numpy(v_region.astype(np.int64)),
        torch.from_numpy(v_mask))
    test_ds = TensorDataset(
        torch.from_numpy(te_first), torch.from_numpy(te_past),
        torch.from_numpy(te_global), torch.from_numpy(te_thr_s),
        torch.from_numpy(te_y_log.astype(np.float32)),
        torch.from_numpy(te_ord.astype(np.float32)),
        torch.from_numpy(te_region.astype(np.int64)),
        torch.from_numpy(te_mask))

    # ---- Sampler (region + SIR-aware) ----
    region_sample_weights = np.ones_like(t_region, dtype=np.float32)
    region_sample_weights[t_region == 6] = 2.5
    region_sample_weights[t_region == 7] = 3.0
    region_sample_weights[t_region == 8] = 5.0
    region_sample_weights[t_region == 9] = 5.0
    region_sample_weights[t_region == 10] = 5.0
    region_sample_weights[t_region == 11] = 5.0
    region_sample_weights[t_region == 12] = 3.0
    region_sample_weights[t_region == 13] = 2.5

    t_signal = t_taps_feat[:, 0]
    t_valid_past_for_sir = (np.arange(L_past)[None, :] < t_past_lens[:, None]).astype(np.float32)
    t_isi = np.sum(t_taps_feat[:, 1:] * t_valid_past_for_sir, axis=1)
    t_sir = t_signal / (t_isi + EPS)

    sir_sample_weights = np.ones_like(t_sir, dtype=np.float32)
    sir_sample_weights[t_sir < 0.26] = 3.0
    sir_sample_weights[(t_sir >= 0.26) & (t_sir < 0.36)] = 2.0

    combined_sample_weights = region_sample_weights * sir_sample_weights

    sampler = WeightedRandomSampler(
        weights=torch.from_numpy(combined_sample_weights),
        num_samples=len(combined_sample_weights),
        replacement=True)

    pin_mem = torch.cuda.is_available()
    train_loader = DataLoader(train_ds, batch_size=batch_size, sampler=sampler,
                              shuffle=False, pin_memory=pin_mem, num_workers=num_workers)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False,
                            pin_memory=pin_mem, num_workers=num_workers)
    test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False,
                             pin_memory=pin_mem, num_workers=num_workers)

    class_weights = compute_region_class_weights(t_region, NUM_REGION_CLASSES)
    ordinal_pos_weights = compute_ordinal_pos_weights_from_region_labels(
        t_region, num_thresholds=NUM_ORDINAL_THRESHOLDS, max_weight=20.0)

    scalers = {
        "first_tap_mean": first_tap_mean, "first_tap_std": first_tap_std,
        "first_var_mean": first_var_mean, "first_var_std": first_var_std,
        "first_abs_mean": first_abs_mean, "first_abs_std": first_abs_std,
        "first_snr_mean": first_snr_mean, "first_snr_std": first_snr_std,
        "past_tap_mean": past_tap_mean, "past_tap_std": past_tap_std,
        "past_var_mean": past_var_mean, "past_var_std": past_var_std,
        "past_abs_mean": past_abs_mean, "past_abs_std": past_abs_std,
        "past_snr_mean": past_snr_mean, "past_snr_std": past_snr_std,
        "z0_scaler": z0_scaler, "z1_scaler": z1_scaler,
        "harmonic_minus_gap_scaler": hmg_scaler,
        "nsid_scaler": nsid_scaler, "log_hd_scaler": log_hd_scaler,
        "da_scaler": da_scaler, "herf_scaler": herf_scaler,
        "thr_scaler": thr_scaler,
        "threshold_feature_mode": threshold_mode,
        "threshold_norm_clip_eps": THRESHOLD_NORM_CLIP_EPS,
        "threshold_feature_description": (
            "threshold branch input is StandardScaler(threshold / sum(valid tap_i*N)) "
            "or its logit, depending on threshold_feature_mode"
        ),
        "tap_cols": tap_cols, "var_cols": var_cols,
        "first_token_dim": 4, "past_token_dim": 4,
        "global_dim": 7,
        "train_max_past_seq_len": L_past,
        "scaling_strategy": "position_independent",
        "uses_positional_encoding": False,
        "permutation_invariance_post_first": True,
        "variable_length_support": True,
        "target_parameterization": "pred_log10_ber = log10(0.5) - softplus(raw_out)",
        "ordinal_thresholds": ORDINAL_THRESHOLDS,
        "num_region_classes": NUM_REGION_CLASSES,
        "class_weights": class_weights.tolist(),
        "ordinal_pos_weights": ordinal_pos_weights.tolist(),
        "data_paths": csv_paths,
        "nrows_per_dataset": nrows_per_dataset,
        "finetuned_from": PRETRAINED_MODEL_PATH,
        "scalers_reused": reuse_scalers,
        "variances_computed_from_taps": True,
    }

    aux_info = {
        "class_weights": class_weights,
        "ordinal_pos_weights": ordinal_pos_weights,
        "t_region": t_region,
    }

    return train_loader, val_loader, test_loader, scalers, aux_info, L_past


# =========================
# Evaluation (same as v4)
# =========================
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0
    total_reg_loss = 0.0
    total_ord_loss = 0.0
    all_preds_log = []
    all_targets_log = []
    all_pred_regions = []
    all_true_regions = []

    with torch.no_grad():
        for b_first, b_past, b_global, b_thr, b_y_log, b_ord, b_region, b_mask in loader:
            b_first = b_first.to(device, non_blocking=True)
            b_past = b_past.to(device, non_blocking=True)
            b_global = b_global.to(device, non_blocking=True)
            b_thr = b_thr.to(device, non_blocking=True)
            b_y_log = b_y_log.to(device, non_blocking=True)
            b_ord = b_ord.to(device, non_blocking=True)
            b_region = b_region.to(device, non_blocking=True)
            b_mask = b_mask.to(device, non_blocking=True)

            pred_raw_unconstrained, ord_logits = model(
                b_first, b_past, b_global, b_thr, key_padding_mask=b_mask)

            if has_nonfinite_tensor(pred_raw_unconstrained):
                raise RuntimeError("Non-finite prediction during evaluation.")
            if has_nonfinite_tensor(ord_logits):
                raise RuntimeError("Non-finite ordinal logits during evaluation.")

            loss, reg_loss, ord_loss = criterion(
                pred_raw_unconstrained, ord_logits, b_y_log, b_ord)

            if has_nonfinite_tensor(loss):
                raise RuntimeError("Non-finite loss during evaluation.")

            pred_log = model.raw_to_log10ber(pred_raw_unconstrained)
            pred_region = ordinal_logits_to_region_labels_torch(ord_logits)

            total_loss += loss.item()
            total_reg_loss += reg_loss.item()
            total_ord_loss += ord_loss.item()

            all_preds_log.append(pred_log.cpu().numpy())
            all_targets_log.append(b_y_log.cpu().numpy())
            all_pred_regions.append(pred_region.cpu().numpy())
            all_true_regions.append(b_region.cpu().numpy())

    n_batches = max(len(loader), 1)
    avg_loss = total_loss / n_batches
    avg_reg_loss = total_reg_loss / n_batches
    avg_ord_loss = total_ord_loss / n_batches

    preds_log = np.vstack(all_preds_log)
    targets_log = np.vstack(all_targets_log)
    preds_raw = np.clip(10 ** preds_log, EPS, 0.5)
    targets_raw = np.clip(10 ** targets_log, EPS, 0.5)

    rmse_log = float(np.sqrt(np.mean((preds_log - targets_log) ** 2)))
    mae_log = float(np.mean(np.abs(preds_log - targets_log)))
    factor_error = float(10 ** rmse_log)
    rmse_raw = float(np.sqrt(np.mean((preds_raw - targets_raw) ** 2)))
    mae_raw = float(np.mean(np.abs(preds_raw - targets_raw)))
    rel_err = (preds_raw - targets_raw) / np.maximum(targets_raw, 1e-6)
    rmse_rel = float(np.sqrt(np.mean(rel_err ** 2)))
    mae_rel = float(np.mean(np.abs(rel_err)))

    pred_regions = np.concatenate(all_pred_regions).reshape(-1)
    true_regions = np.concatenate(all_true_regions).reshape(-1)
    region_acc = float(np.mean(pred_regions == true_regions))
    region_mae = float(np.mean(np.abs(pred_regions - true_regions)))

    return {
        "loss": avg_loss, "reg_loss": avg_reg_loss, "ord_loss": avg_ord_loss,
        "rmse_log": rmse_log, "mae_log": mae_log, "factor_error": factor_error,
        "rmse_raw": rmse_raw, "mae_raw": mae_raw,
        "rmse_rel": rmse_rel, "mae_rel": mae_rel,
        "region_acc": region_acc, "region_mae": region_mae,
    }


def evaluate_by_target_range(model, loader, device):
    model.eval()
    all_preds_log = []
    all_targets_log = []

    with torch.no_grad():
        for b_first, b_past, b_global, b_thr, b_y_log, _, _, b_mask in loader:
            b_first = b_first.to(device, non_blocking=True)
            b_past = b_past.to(device, non_blocking=True)
            b_global = b_global.to(device, non_blocking=True)
            b_thr = b_thr.to(device, non_blocking=True)
            b_mask = b_mask.to(device, non_blocking=True)

            pred_raw_unconstrained, _ = model(
                b_first, b_past, b_global, b_thr, key_padding_mask=b_mask)
            if has_nonfinite_tensor(pred_raw_unconstrained):
                raise RuntimeError("Non-finite prediction during per-range evaluation.")
            pred_log = model.raw_to_log10ber(pred_raw_unconstrained)
            all_preds_log.append(pred_log.cpu().numpy())
            all_targets_log.append(b_y_log.numpy())

    preds_log = np.vstack(all_preds_log).reshape(-1)
    targets_log = np.vstack(all_targets_log).reshape(-1)
    preds_raw = np.clip(10 ** preds_log, EPS, 0.5)
    targets_raw = np.clip(10 ** targets_log, EPS, 0.5)

    ranges = {
        "low_BER(y<1e-6)": targets_raw < 1e-6,
        "mid_BER(1e-6<=y<1e-3)": (targets_raw >= 1e-6) & (targets_raw < 1e-3),
        "high_BER(1e-3<=y<0.10)": (targets_raw >= 1e-3) & (targets_raw < 0.10),
        "upper_BER(0.10<=y<0.20)": (targets_raw >= 0.10) & (targets_raw < 0.20),
        "target_BER(0.20<=y<0.40)": (targets_raw >= 0.20) & (targets_raw < 0.40),
        "very_high_BER(0.40<=y<=0.50)": targets_raw >= 0.40,
    }

    metrics = {}
    for name, mask in ranges.items():
        if np.any(mask):
            err_log = preds_log[mask] - targets_log[mask]
            err_raw = preds_raw[mask] - targets_raw[mask]
            rel_err = err_raw / np.maximum(targets_raw[mask], 1e-6)
            metrics[name] = {
                "count": int(mask.sum()),
                "rmse_log": float(np.sqrt(np.mean(err_log ** 2))),
                "mae_log": float(np.mean(np.abs(err_log))),
                "factor_error": float(10 ** float(np.sqrt(np.mean(err_log ** 2)))),
                "rmse_raw": float(np.sqrt(np.mean(err_raw ** 2))),
                "mae_raw": float(np.mean(np.abs(err_raw))),
                "rmse_rel": float(np.sqrt(np.mean(rel_err ** 2))),
                "mae_rel": float(np.mean(np.abs(rel_err))),
                "bias_raw": float(np.mean(err_raw)),
                "bias_log": float(np.mean(err_log)),
                "p90_abs_raw": float(np.percentile(np.abs(err_raw), 90)),
                "p95_abs_raw": float(np.percentile(np.abs(err_raw), 95)),
            }
        else:
            metrics[name] = None
    return metrics


# =========================
# Selection scoring (same as v4)
# =========================
SELECTION_REGION_WEIGHTS = {
    "low_BER(y<1e-6)": 0.25,
    "mid_BER(1e-6<=y<1e-3)": 0.50,
    "high_BER(1e-3<=y<0.10)": 1.00,
    "upper_BER(0.10<=y<0.20)": 2.00,
    "target_BER(0.20<=y<0.40)": 3.00,
    "very_high_BER(0.40<=y<=0.50)": 1.50,
}
SELECTION_W_LOG_RMSE = 1.00
SELECTION_W_BIAS = 0.50
SELECTION_W_TAIL = 0.05
SELECTION_TAIL_CAP = 5.0


def compute_selection_score(val_range_metrics, val_metrics, min_count=50):
    log_rmse_sum = 0.0
    bias_sum = 0.0
    tail_sum = 0.0
    total_w = 0.0

    for region_name, w in SELECTION_REGION_WEIGHTS.items():
        m = val_range_metrics.get(region_name)
        if m is None or m["count"] < min_count:
            continue
        log_rmse_sum += w * m["rmse_log"]
        bias_sum += w * abs(m["bias_log"])
        if m["rmse_raw"] > 1e-9:
            tail_ratio = m["p95_abs_raw"] / (m["rmse_raw"] + 1e-9)
            tail_sum += w * min(tail_ratio, SELECTION_TAIL_CAP)
        else:
            tail_sum += w * 1.0
        total_w += w

    if total_w == 0:
        return {
            "composite": float(val_metrics["rmse_log"]),
            "log_rmse_weighted": float(val_metrics["rmse_log"]),
            "bias_weighted": 0.0, "tail_weighted": 0.0,
            "geometric_factor_error": float(val_metrics["factor_error"]),
            "fallback": True,
        }

    log_rmse_weighted = log_rmse_sum / total_w
    bias_weighted = bias_sum / total_w
    tail_weighted = tail_sum / total_w
    composite = (SELECTION_W_LOG_RMSE * log_rmse_weighted
                 + SELECTION_W_BIAS * bias_weighted
                 + SELECTION_W_TAIL * tail_weighted)
    return {
        "composite": float(composite),
        "log_rmse_weighted": float(log_rmse_weighted),
        "bias_weighted": float(bias_weighted),
        "tail_weighted": float(tail_weighted),
        "geometric_factor_error": float(10 ** log_rmse_weighted),
        "fallback": False,
    }


def is_acceptable_checkpoint(val_range_metrics, val_metrics):
    target = val_range_metrics.get("target_BER(0.20<=y<0.40)")
    if target is None or target["count"] < 100:
        return False, "target region too small"
    for region_name, m in val_range_metrics.items():
        if m is None or m["count"] < 50:
            continue
        if abs(m["bias_log"]) > 0.15:
            return False, f"{region_name} has bias_log={m['bias_log']:.3f}"
    if val_metrics["rmse_log"] > 0.30:
        return False, f"overall rmse_log={val_metrics['rmse_log']:.3f} too high"
    return True, "ok"


# =========================
# Fine-tune main
# =========================
def finetune():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Executing on: {device}")
    print(f"\nFINE-TUNING from: {PRETRAINED_MODEL_PATH}")
    print(f"  Pretrained scalers: {PRETRAINED_SCALER_PATH}")
    print(f"  Reuse scalers: {REUSE_SCALERS}")

    if not os.path.exists(PRETRAINED_MODEL_PATH):
        raise FileNotFoundError(f"Pretrained model not found: {PRETRAINED_MODEL_PATH}")
    if not os.path.exists(PRETRAINED_SCALER_PATH):
        raise FileNotFoundError(f"Pretrained scalers not found: {PRETRAINED_SCALER_PATH}")

    pretrained_scalers = joblib.load(PRETRAINED_SCALER_PATH)
    print(f"  Pretrained global_dim: {pretrained_scalers.get('global_dim', '?')}")
    print(f"  Pretrained threshold_feature_mode: {pretrained_scalers.get('threshold_feature_mode', THRESHOLD_FEATURE_MODE)}")

    # ---- Prepare data ----
    train_loader, val_loader, test_loader, scalers, aux_info, max_past_seq_len = (
        prepare_data_for_finetune(
            DATA_PATHS, batch_size=256,
            nrows_per_dataset=NROWS_PER_DATASET, num_workers=0,
            pretrained_scalers=pretrained_scalers, reuse_scalers=REUSE_SCALERS,
        )
    )

    joblib.dump(scalers, SCALER_SAVE_PATH)
    print(f"\nFine-tune scalers saved to: {SCALER_SAVE_PATH}")

    # ---- Build model & load pretrained weights ----
    model = BERFirstTokenSetTransformerBoundedLogHalfMultiTask(
        max_past_seq_len=max_past_seq_len,
        token_dim=4, first_token_dim=4, threshold_dim=1,
        global_dim=7, d_model=128, num_set_layers=4,
        num_heads=4, mlp_ratio=4.0, dropout=0.10,
        num_ordinal_thresholds=NUM_ORDINAL_THRESHOLDS,
    ).to(device)

    pretrained_state = torch.load(PRETRAINED_MODEL_PATH, map_location=device, weights_only=True)
    model.load_state_dict(pretrained_state, strict=True)
    print("Pretrained normalized-threshold + monotonic-ordinal weights loaded with strict=True.")

    # ---- Pre-fine-tune evaluation (sanity check) ----
    print("\n--- Pre-fine-tune evaluation (pretrained model on new data) ---")
    pre_criterion = MultiTaskBERLoss(
        StableMultiObjectiveBERBoundedLogLoss(),
        OrdinalBCELoss(),
        lambda_ord=0.25,
    )
    try:
        pre_val = evaluate(model, val_loader, pre_criterion, device)
        pre_range = evaluate_by_target_range(model, val_loader, device)
        pre_sel = compute_selection_score(pre_range, pre_val)
        print(f"  Pretrained on new val: composite={pre_sel['composite']:.5f} | "
              f"factor~{pre_sel['geometric_factor_error']:.3f}x | "
              f"RMSE(log10)={pre_val['rmse_log']:.4f} | "
              f"region_acc={pre_val['region_acc']:.4f}")
        for rng, m in pre_range.items():
            if m is not None:
                print(f"    {rng}: factor~{m['factor_error']:.3f}x, "
                      f"bias_log={m['bias_log']:+.4f}, count={m['count']}")
    except RuntimeError as e:
        print(f"  Pre-eval failed: {e}")

    # ---- Optimizer & loss for fine-tuning ----
    print(f"\nFine-tune hyperparameters:")
    print(f"  LR={FINETUNE_LR}, weight_decay={FINETUNE_WEIGHT_DECAY}")
    print(f"  max_epochs={FINETUNE_MAX_EPOCHS}, min_epochs={FINETUNE_MIN_EPOCHS}, "
          f"patience={FINETUNE_PATIENCE}")

    optimizer = optim.AdamW(model.parameters(), lr=FINETUNE_LR,
                            weight_decay=FINETUNE_WEIGHT_DECAY)

    reg_loss = StableMultiObjectiveBERBoundedLogLoss(
        log_delta=0.35, raw_delta=0.006, rel_delta=0.03,
        alpha_log=0.45, beta_raw=0.35, gamma_rel=0.20,
        use_regime_weights=True)

    boosted_pos = aux_info["ordinal_pos_weights"].copy()
    for i, thr in enumerate(ORDINAL_THRESHOLDS):
        if 0.20 <= thr <= 0.40:
            boosted_pos[i] *= 2.5
        elif 0.15 <= thr < 0.20:
            boosted_pos[i] *= 1.5
        elif 0.40 < thr <= 0.45:
            boosted_pos[i] *= 1.5

    ord_loss = OrdinalBCELoss(
        pos_weight=torch.tensor(boosted_pos, dtype=torch.float32, device=device),
        reduction="mean")
    criterion = MultiTaskBERLoss(reg_loss=reg_loss, ord_loss=ord_loss, lambda_ord=0.25)

    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="min", patience=SCHEDULER_PATIENCE, factor=SCHEDULER_FACTOR)

    # ---- Multi-component selection state ----
    best_checkpoints = {
        "composite": {"score": float("inf"), "state": None, "epoch": -1},
        "composite_ema": {"score": float("inf"), "state": None, "epoch": -1},
        "log_rmse": {"score": float("inf"), "state": None, "epoch": -1},
        "target_mae": {"score": float("inf"), "state": None, "epoch": -1},
        "low_bias": {"score": float("inf"), "state": None, "epoch": -1},
    }
    EMA_ALPHA = 0.5
    ema_composite = None
    wait = 0

    # Seed best with pre-fine-tune score so fine-tuning must IMPROVE on the
    # pretrained baseline before being marked as best.
    try:
        base_state = copy.deepcopy(model.state_dict())
        pre_acceptable, _ = is_acceptable_checkpoint(pre_range, pre_val)
        if pre_acceptable:
            for ckpt_name, score_value in [
                ("composite", pre_sel["composite"]),
                ("composite_ema", pre_sel["composite"]),
                ("log_rmse", pre_sel["log_rmse_weighted"]),
                ("target_mae", pre_range["target_BER(0.20<=y<0.40)"]["mae_raw"]
                    if pre_range.get("target_BER(0.20<=y<0.40)") else float("inf")),
                ("low_bias", pre_sel["bias_weighted"]),
            ]:
                best_checkpoints[ckpt_name] = {
                    "score": float(score_value),
                    "state": base_state,
                    "epoch": 0,  # 0 = pretrained baseline
                }
            print("\nSeeded best checkpoints with pretrained baseline scores. "
                  "Fine-tune must improve to take over.")
    except Exception as e:
        print(f"  Could not seed pretrained baseline as best: {e}")

    print(f"\nSelection: weighted log-RMSE + bias + tail penalties")
    print(f"  Region weights: {SELECTION_REGION_WEIGHTS}")
    print(f"  Composite weights: log_rmse={SELECTION_W_LOG_RMSE}, "
          f"bias={SELECTION_W_BIAS}, tail={SELECTION_W_TAIL}")
    print(f"  EMA alpha: {EMA_ALPHA}\n")

    training_broke = False

    for epoch in range(FINETUNE_MAX_EPOCHS):
        model.train()

        for batch_idx, (b_first, b_past, b_global, b_thr, b_y_log, b_ord, _, b_mask) in enumerate(train_loader):
            b_first = b_first.to(device, non_blocking=True)
            b_past = b_past.to(device, non_blocking=True)
            b_global = b_global.to(device, non_blocking=True)
            b_thr = b_thr.to(device, non_blocking=True)
            b_y_log = b_y_log.to(device, non_blocking=True)
            b_ord = b_ord.to(device, non_blocking=True)
            b_mask = b_mask.to(device, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)
            pred_raw_unconstrained, ord_logits = model(
                b_first, b_past, b_global, b_thr, key_padding_mask=b_mask)

            if has_nonfinite_tensor(pred_raw_unconstrained):
                print(f"Non-finite prediction at epoch {epoch+1}, batch {batch_idx+1}.")
                training_broke = True; break
            if has_nonfinite_tensor(ord_logits):
                print(f"Non-finite ordinal logits at epoch {epoch+1}, batch {batch_idx+1}.")
                training_broke = True; break

            loss, reg_part, ord_part = criterion(
                pred_raw_unconstrained, ord_logits, b_y_log, b_ord)
            if has_nonfinite_tensor(loss):
                print(f"Non-finite loss at epoch {epoch+1}, batch {batch_idx+1}.")
                training_broke = True; break

            loss.backward()
            grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=GRAD_CLIP)
            if not torch.isfinite(grad_norm):
                print(f"Non-finite gradient norm at epoch {epoch+1}, batch {batch_idx+1}.")
                training_broke = True; break

            optimizer.step()

            bad_param = False
            for nm, param in model.named_parameters():
                if param.requires_grad and param.data is not None and not torch.isfinite(param.data).all():
                    print(f"Non-finite parameter after optimizer step: {nm}")
                    bad_param = True; break
            if bad_param:
                training_broke = True; break

        if training_broke:
            print("Training stopped due to non-finite values.")
            break

        try:
            train_metrics = evaluate(model, train_loader, criterion, device)
            val_metrics = evaluate(model, val_loader, criterion, device)
            val_range_metrics = evaluate_by_target_range(model, val_loader, device)
        except RuntimeError as e:
            print(f"Evaluation failed at epoch {epoch+1}: {e}")
            break

        scheduler.step(val_metrics["loss"])
        current_lr = optimizer.param_groups[0]["lr"]

        sel = compute_selection_score(val_range_metrics, val_metrics)
        composite = sel["composite"]
        if ema_composite is None:
            ema_composite = composite
        else:
            ema_composite = EMA_ALPHA * composite + (1.0 - EMA_ALPHA) * ema_composite

        target_key = "target_BER(0.20<=y<0.40)"
        target_mae = (val_range_metrics[target_key]["mae_raw"]
                      if val_range_metrics[target_key] is not None else float("inf"))

        is_acceptable, reason = is_acceptable_checkpoint(val_range_metrics, val_metrics)

        print(
            f"FT-Epoch {epoch+1:03d} | LR: {current_lr:.2e} | "
            f"Train Loss: {train_metrics['loss']:.4f} | "
            f"Val Loss: {val_metrics['loss']:.4f} | "
            f"Val RMSE(log10): {val_metrics['rmse_log']:.4f} "
            f"(~{val_metrics['factor_error']:.3f}x) | "
            f"Val Region Acc: {val_metrics['region_acc']:.4f}"
        )
        print(
            f"  SELECTION | composite={composite:.5f} | ema={ema_composite:.5f} | "
            f"weighted log-RMSE={sel['log_rmse_weighted']:.5f} "
            f"(~{sel['geometric_factor_error']:.3f}x) | "
            f"|bias_log|={sel['bias_weighted']:.5f} | "
            f"tail={sel['tail_weighted']:.3f} | acceptable={is_acceptable}"
            + ("" if is_acceptable else f" ({reason})")
        )
        for rng_name, rng_stats in val_range_metrics.items():
            if rng_stats is not None:
                print(
                    f"    {rng_name}: n={rng_stats['count']} | "
                    f"factor~{rng_stats['factor_error']:.3f}x | "
                    f"MAE_raw={rng_stats['mae_raw']:.6f} | "
                    f"P90={rng_stats['p90_abs_raw']:.6f} | "
                    f"bias_log={rng_stats['bias_log']:+.4f}"
                )

        improved_any = False
        if is_acceptable:
            current_state_snapshot = None
            checkpoint_candidates = [
                ("composite", composite),
                ("composite_ema", ema_composite),
                ("log_rmse", sel["log_rmse_weighted"]),
                ("target_mae", target_mae),
                ("low_bias", sel["bias_weighted"]),
            ]
            for ckpt_name, score in checkpoint_candidates:
                if score < best_checkpoints[ckpt_name]["score"]:
                    if current_state_snapshot is None:
                        current_state_snapshot = copy.deepcopy(model.state_dict())
                    best_checkpoints[ckpt_name] = {
                        "score": float(score),
                        "state": current_state_snapshot,
                        "epoch": epoch + 1,
                    }
                    improved_any = True
                    print(f"  -> New best [{ckpt_name}] at FT-epoch {epoch+1}: {score:.6f}")

        ema_best = best_checkpoints["composite_ema"]["score"]
        if ema_composite < ema_best + 1e-9 or improved_any:
            wait = 0
        else:
            if epoch + 1 >= FINETUNE_MIN_EPOCHS:
                wait += 1
                if wait >= FINETUNE_PATIENCE:
                    print(f"Early stopping triggered after {wait} non-improving epochs.")
                    break

    # ---- Save last ----
    torch.save(model.state_dict(), LAST_MODEL_SAVE_PATH)
    print(f"\nLast fine-tuned model saved to: {LAST_MODEL_SAVE_PATH}")

    # ---- Save all best checkpoints ----
    print("\n" + "=" * 80)
    print("FINE-TUNE BEST CHECKPOINTS SUMMARY")
    print("=" * 80)
    for ckpt_name, info in best_checkpoints.items():
        if info["state"] is None:
            print(f"  [{ckpt_name:>14s}] never updated")
            continue
        path = BEST_MODEL_SAVE_PATH.replace(".pth", f"_{ckpt_name}.pth")
        torch.save(info["state"], path)
        print(f"  [{ckpt_name:>14s}] epoch={info['epoch']:>3d} | "
              f"score={info['score']:.6f} | saved -> {path}")

    primary_choice = "composite_ema"
    if best_checkpoints[primary_choice]["state"] is None:
        primary_choice = "composite"
    if best_checkpoints[primary_choice]["state"] is None:
        primary_choice = "log_rmse"

    if best_checkpoints[primary_choice]["state"] is not None:
        model.load_state_dict(best_checkpoints[primary_choice]["state"])
        torch.save(model.state_dict(), BEST_MODEL_SAVE_PATH)
        print(f"\nPrimary best ({primary_choice}, "
              f"FT-epoch {best_checkpoints[primary_choice]['epoch']}) "
              f"saved to: {BEST_MODEL_SAVE_PATH}")
    else:
        print("\nWarning: no valid best checkpoint was found.")

    # ---- Test ----
    test_metrics = evaluate(model, test_loader, criterion, device)
    print(
        f"\nTest Loss: {test_metrics['loss']:.4f} | "
        f"Test RMSE(log10): {test_metrics['rmse_log']:.4f} | "
        f"~{test_metrics['factor_error']:.2f}x | "
        f"Test Region Acc: {test_metrics['region_acc']:.4f}"
    )

    range_metrics = evaluate_by_target_range(model, test_loader, device)
    print("\nPer-range test diagnostics:")
    for name, stats in range_metrics.items():
        if stats is None:
            print(f"  {name}: no samples")
        else:
            print(
                f"  {name} | count={stats['count']} | "
                f"factor~{stats['factor_error']:.2f}x | "
                f"RMSE(log10)={stats['rmse_log']:.4f} | "
                f"MAE_raw={stats['mae_raw']:.6f} | "
                f"bias_log={stats['bias_log']:+.4f} | "
                f"P90={stats['p90_abs_raw']:.6f} | "
                f"P95={stats['p95_abs_raw']:.6f}"
            )

    return model


if __name__ == "__main__":
    os.makedirs(BASE_DIR, exist_ok=True)
    missing = [p for p in DATA_PATHS if not os.path.exists(p)]
    if missing:
        print("Critical Error: Missing dataset files:")
        for p in missing:
            print(f"  - {p}")
    else:
        print("Fine-tuning from datasets:")
        for p in DATA_PATHS:
            print(f"  - {p}")
        print(f"Row cap per dataset: {NROWS_PER_DATASET:,}")
        finetune()

Fine-tuning from datasets:
  - ./../ber_data_generation/data/data_physics_total.csv
  - ./../ber_data_generation/data/data_random_total.csv
Row cap per dataset: 10,000,000
Executing on: cuda

FINE-TUNING from: ./random_extra_multitask_finetune_best_normthr_monoord.pth
  Pretrained scalers: ./random_extra_multitask_finetune_scalers_normthr_monoord.pkl
  Reuse scalers: True
  Pretrained global_dim: 7
  Pretrained threshold_feature_mode: normalized_logit
Loading up to 10,000,000 rows from: ./../ber_data_generation/data/data_physics_total.csv
Loading up to 10,000,000 rows from: ./../ber_data_generation/data/data_random_total.csv
Combined rows before filtering: 20,000,000
  Files with var_* columns: 0/2
  Final tap column count: 15


In [2]:
import os
import joblib
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy.special import erfc
from itertools import product as iproduct

EPS = 1e-12
PLOT_DIR = "./plots_mixed_newdata-experiments/6"
os.makedirs(PLOT_DIR, exist_ok=True)

# =========================
# Paths
# =========================
MODEL_PATH = "random_extra_multitask_finetune_best_normthr_monoord.pth"
SCALER_PATH = "random_extra_multitask_finetune_scalers_normthr_monoord.pkl"

# =========================
# Config
# =========================
PHYSICS_MAX_MEM_LEN = 20
PHYSICS_MIN_MEM_LEN = 15
ARRIVAL_COVERAGE = 0.70
N_THRESHOLDS = 500
RANDOM_SEED = 59

# =========================
# Test-time smoothing config
# =========================
# The smoother recomputes ALL threshold-dependent features for thr-eps, thr, thr+eps.
# This is important because threshold enters both thr_t and global features such as z0/z1/HMG.
USE_TEST_TIME_SMOOTHING = False
SMOOTHING_EPS_FRAC = 0.005     # eps = SMOOTHING_EPS_FRAC * threshold sweep span
SMOOTHING_SPACE = "log"         # "log" is usually best globally; "raw" can look better at high BER
SMOOTHING_WEIGHTS = (0.25, 0.50, 0.25)

# This must match training. If the scaler file contains "threshold_feature_mode",
# that saved value is used; this default is only a fallback.
THRESHOLD_FEATURE_MODE = "normalized_logit"
THRESHOLD_NORM_CLIP_EPS = 1e-5


ORDINAL_THRESHOLDS = [
    1e-6, 1e-5, 1e-4, 1e-3, 1e-2, 1e-1,
    0.15, 0.20, 0.25, 0.30, 0.35, 0.40, 0.45,
]
NUM_ORDINAL_THRESHOLDS = len(ORDINAL_THRESHOLDS)
NUM_REGION_CLASSES = NUM_ORDINAL_THRESHOLDS + 1

REGION_LABELS = {
    0: "y < 1e-6",
    1: "1e-6 <= y < 1e-5",
    2: "1e-5 <= y < 1e-4",
    3: "1e-4 <= y < 1e-3",
    4: "1e-3 <= y < 1e-2",
    5: "1e-2 <= y < 1e-1",
    6: "0.10 <= y < 0.15",
    7: "0.15 <= y < 0.20",
    8: "0.20 <= y < 0.25",
    9: "0.25 <= y < 0.30",
    10: "0.30 <= y < 0.35",
    11: "0.35 <= y < 0.40",
    12: "0.40 <= y < 0.45",
    13: "0.45 <= y <= 0.50",
}


# =========================
# Physics helpers
# =========================
def Fhit_function(radius, distance, diffusionCoef, t):
    if t <= 0:
        return 0.0
    return (radius / (distance + radius)) * erfc(distance / np.sqrt(4 * diffusionCoef * t))


def calculate_hitting_probabilities(mem_len, radius, distance, diffusionCoef, Ts):
    P = np.zeros(mem_len)
    for i in range(mem_len):
        t_end = (i + 1) * Ts
        t_start = i * Ts
        P[i] = Fhit_function(radius, distance, diffusionCoef, t_end) - Fhit_function(
            radius, distance, diffusionCoef, t_start
        )
    return P


def calculate_ber_vectorized(mem_len, threshold, P_scaled, variances):
    P_arr = np.asarray(P_scaled, dtype=float)[:mem_len]
    vars_arr = np.asarray(variances, dtype=float)[:mem_len]

    seqs = np.array(list(iproduct([0, 1], repeat=mem_len)), dtype=np.float64)[:, ::-1]
    c_bit = seqs[:, 0]

    mu = (seqs * P_arr).sum(axis=1)
    var_total = (seqs * vars_arr).sum(axis=1)
    std = np.sqrt(np.maximum(var_total, 0.0))

    pe = np.empty_like(mu)
    zero_std = (std == 0)
    if np.any(zero_std):
        pe[zero_std & (c_bit == 1)] = np.where(
            mu[zero_std & (c_bit == 1)] < threshold, 1.0, 0.0)
        pe[zero_std & (c_bit == 0)] = np.where(
            mu[zero_std & (c_bit == 0)] >= threshold, 1.0, 0.0)
    nz = ~zero_std
    if np.any(nz):
        pe[nz & (c_bit == 1)] = 0.5 * erfc(
            (mu[nz & (c_bit == 1)] - threshold) / (std[nz & (c_bit == 1)] * np.sqrt(2)))
        pe[nz & (c_bit == 0)] = 0.5 * erfc(
            (threshold - mu[nz & (c_bit == 0)]) / (std[nz & (c_bit == 0)] * np.sqrt(2)))
    return float(np.mean(pe))


def raw_to_region_labels_np(y_raw):
    y = np.asarray(y_raw).reshape(-1)
    return np.searchsorted(np.asarray(ORDINAL_THRESHOLDS), y, side="right").astype(np.int64)


def ordinal_logits_to_region_labels_torch(logits):
    probs = torch.sigmoid(logits)
    passed = (probs >= 0.5).sum(dim=1)
    return passed.long()


# =========================
# Shared scaling helper
# =========================
def apply_shared_scale(data, mean, std, valid_mask=None):
    scaled = ((data - mean) / std).astype(np.float32)
    if valid_mask is not None and scaled.ndim == 2 and valid_mask.shape == scaled.shape:
        scaled = scaled * valid_mask.astype(np.float32)
    return scaled


def safe_harmonic_z(z0, z1, denom_eps=1e-9):
    """Finite-safe version of 2/(1/z0 + 1/z1) = 2*z0*z1/(z0+z1).

    The direct reciprocal form can create +/-inf when z0 or z1 is near zero.
    This helper uses np.divide(..., where=...) and sanitizes non-finite outputs.
    """
    z0 = np.asarray(z0, dtype=np.float32)
    z1 = np.asarray(z1, dtype=np.float32)
    num = (2.0 * z0 * z1).astype(np.float32)
    den = (z0 + z1).astype(np.float32)
    out = np.divide(
        num,
        den,
        out=np.zeros_like(num, dtype=np.float32),
        where=np.abs(den) > denom_eps,
    ).astype(np.float32)
    return np.nan_to_num(out, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)


def compute_threshold_feature_np(threshold_raw, total_scaled_taps, mode=THRESHOLD_FEATURE_MODE):
    thr = np.asarray(threshold_raw, dtype=np.float32).reshape(-1, 1)
    total = np.asarray(total_scaled_taps, dtype=np.float32).reshape(-1, 1)
    denom = np.maximum(total, EPS).astype(np.float32)

    if mode == "normalized":
        feat = (thr / denom).astype(np.float32)
    elif mode == "normalized_logit":
        u = (thr / denom).astype(np.float32)
        u = np.clip(u, THRESHOLD_NORM_CLIP_EPS, 1.0 - THRESHOLD_NORM_CLIP_EPS)
        feat = np.log(u / (1.0 - u)).astype(np.float32)
    elif mode == "log10_raw":
        feat = np.log10(thr + EPS).astype(np.float32)
    else:
        raise ValueError(f"Unknown threshold feature mode: {mode}")

    feat = np.nan_to_num(feat, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)
    return feat


# =========================
# Generate one physical scenario
# =========================
def generate_physical_case(rng, min_mem_len, max_mem_len, arrival_coverage, max_tries=5000):
    for _ in range(max_tries):
        radius = rng.uniform(3.0, 5.0)
        distance = rng.uniform(10.0, 15.0)
        diff = rng.uniform(50.0, 75.0)
        Ts = rng.uniform(0.2, 1.2)
        N = int(10 ** rng.uniform(3.0, 6.0))

        f_inf = radius / (radius + distance)
        target = arrival_coverage * f_inf

        cumsum, k = 0.0, 0
        while k < max_mem_len:
            pk = Fhit_function(radius, distance, diff, (k + 1) * Ts) - Fhit_function(
                radius, distance, diff, k * Ts)
            cumsum += pk
            k += 1
            if cumsum >= target:
                break

        if k < min_mem_len:
            continue

        P_ext = calculate_hitting_probabilities(k + 1, radius, distance, diff, Ts)
        P_main = P_ext[:k]
        P_extra = float(P_ext[k]) if k < len(P_ext) else 0.0

        P_scaled = P_main * N
        variances = N * P_main * (1.0 - P_main)

        return {
            "radius": radius, "distance": distance, "diffusion": diff,
            "Ts": Ts, "N": N, "mem_len": k, "P": P_main,
            "P_scaled": P_scaled, "variances": variances,
            "P_mem_len_extra": P_extra,
            "P_mem_len_extra_var": P_extra * (1.0 - P_extra),
        }

    raise RuntimeError(
        f"Could not generate a physical case with mem_len >= {min_mem_len} "
        f"after {max_tries} tries."
    )


# =========================
# Model (matches v3 training: global_dim=7, global_embed=96, cond_dim=128)
# =========================
class ConditionalFeatureModulation(nn.Module):
    def __init__(self, cond_dim, feat_dim, hidden_dim=256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(cond_dim, hidden_dim), nn.GELU(),
            nn.Linear(hidden_dim, 2 * feat_dim),
        )
        self.scale = nn.Parameter(torch.tensor(0.25, dtype=torch.float32))

    def forward(self, x, cond):
        gamma_beta = self.net(cond)
        gamma, beta = torch.chunk(gamma_beta, 2, dim=-1)
        if x.dim() == 3:
            gamma = gamma.unsqueeze(1)
            beta = beta.unsqueeze(1)
        return x * (1.0 + self.scale * gamma) + self.scale * beta


class SetSelfAttentionBlock(nn.Module):
    def __init__(self, d_model, num_heads, mlp_ratio=4.0, dropout=0.10):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model)
        self.attn = nn.MultiheadAttention(
            embed_dim=d_model, num_heads=num_heads,
            dropout=dropout, batch_first=True)
        self.norm2 = nn.LayerNorm(d_model)
        hidden = int(d_model * mlp_ratio)
        self.mlp = nn.Sequential(
            nn.Linear(d_model, hidden), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(hidden, d_model), nn.Dropout(dropout))

    def forward(self, x, key_padding_mask=None):
        y = self.norm1(x)
        attn_out, _ = self.attn(y, y, y, need_weights=False, key_padding_mask=key_padding_mask)
        x = x + attn_out
        x = x + self.mlp(self.norm2(x))
        return x


class SetAttentionPooling(nn.Module):
    def __init__(self, d_model, hidden_dim=128):
        super().__init__()
        self.score = nn.Sequential(
            nn.Linear(d_model, hidden_dim), nn.GELU(), nn.Linear(hidden_dim, 1))

    def forward(self, x, key_padding_mask=None):
        logits = self.score(x)
        if key_padding_mask is not None:
            logits = logits.masked_fill(key_padding_mask.unsqueeze(-1), float("-inf"))
        weights = torch.softmax(logits, dim=1)
        weights = torch.nan_to_num(weights, nan=0.0)
        return (weights * x).sum(dim=1)


class MonotonicOrdinalHead(nn.Module):
    def __init__(self, head_in, num_thresholds):
        super().__init__()
        self.num_thresholds = num_thresholds
        self.base = nn.Sequential(
            nn.Linear(head_in, 256), nn.GELU(), nn.Dropout(0.20),
            nn.Linear(256, 128), nn.GELU(), nn.Dropout(0.15),
        )
        self.first_logit = nn.Linear(128, 1)
        self.deltas = nn.Linear(128, max(num_thresholds - 1, 1))

    def forward(self, x):
        h = self.base(x)
        first = self.first_logit(h)
        if self.num_thresholds == 1:
            return first
        deltas = -F.softplus(self.deltas(h)[:, : self.num_thresholds - 1])
        return torch.cat([first, deltas], dim=-1).cumsum(dim=-1)


class BERFirstTokenSetTransformerBoundedLogHalfMultiTask(nn.Module):
    def __init__(self, max_past_seq_len, token_dim=4, first_token_dim=4,
                 threshold_dim=1, global_dim=7, d_model=128, num_set_layers=4,
                 num_heads=4, mlp_ratio=4.0, dropout=0.10,
                 num_ordinal_thresholds=NUM_ORDINAL_THRESHOLDS):
        super().__init__()
        self.max_past_seq_len = max_past_seq_len
        self.d_model = d_model
        self.num_ordinal_thresholds = num_ordinal_thresholds

        self.first_proj = nn.Sequential(
            nn.Linear(first_token_dim, d_model), nn.LayerNorm(d_model),
            nn.GELU(), nn.Dropout(dropout))
        self.token_proj = nn.Sequential(
            nn.Linear(token_dim, d_model), nn.LayerNorm(d_model),
            nn.GELU(), nn.Dropout(dropout))
        self.thr_embed = nn.Sequential(
            nn.Linear(threshold_dim, 32), nn.GELU(),
            nn.Linear(32, 32), nn.GELU())
        self.global_embed = nn.Sequential(
            nn.Linear(global_dim, 96), nn.GELU(),
            nn.Linear(96, 96), nn.GELU())

        cond_dim = 32 + 96
        self.first_cond_mod = ConditionalFeatureModulation(cond_dim, d_model, 256)
        self.set_cond_mod = ConditionalFeatureModulation(cond_dim, d_model, 256)

        self.set_blocks = nn.ModuleList([
            SetSelfAttentionBlock(d_model, num_heads, mlp_ratio, dropout)
            for _ in range(num_set_layers)])

        self.first_refine = nn.Sequential(
            nn.LayerNorm(d_model), nn.Linear(d_model, d_model),
            nn.GELU(), nn.Dropout(dropout), nn.Linear(d_model, d_model))
        self.final_set_norm = nn.LayerNorm(d_model)
        self.set_attn_pool = SetAttentionPooling(d_model, 128)

        set_summary_dim = 3 * d_model
        head_in = d_model + set_summary_dim + cond_dim

        self.reg_head = nn.Sequential(
            nn.Linear(head_in, 256), nn.GELU(), nn.Dropout(0.20),
            nn.Linear(256, 128), nn.GELU(), nn.Dropout(0.15),
            nn.Linear(128, 1))
        self.ord_head = MonotonicOrdinalHead(head_in, num_ordinal_thresholds)

    @staticmethod
    def raw_to_log10ber(pred_raw_unconstrained):
        log10_half = torch.log10(torch.tensor(
            0.5, device=pred_raw_unconstrained.device,
            dtype=pred_raw_unconstrained.dtype))
        return log10_half - F.softplus(pred_raw_unconstrained)

    @staticmethod
    def raw_to_ber(pred_raw_unconstrained):
        pred_log = BERFirstTokenSetTransformerBoundedLogHalfMultiTask.raw_to_log10ber(
            pred_raw_unconstrained)
        return torch.pow(10.0, pred_log).clamp(min=EPS, max=0.5)

    def forward(self, first_token, past_tokens, global_feats, threshold, key_padding_mask=None):
        thr = self.thr_embed(threshold)
        g = self.global_embed(global_feats)
        cond = torch.cat([thr, g], dim=-1)

        first_x = self.first_proj(first_token)
        first_x = self.first_cond_mod(first_x, cond)
        first_x = first_x + self.first_refine(first_x)

        set_x = self.token_proj(past_tokens)
        set_x = self.set_cond_mod(set_x, cond)
        for block in self.set_blocks:
            set_x = block(set_x, key_padding_mask=key_padding_mask)
        set_x = self.final_set_norm(set_x)

        if key_padding_mask is not None:
            valid_mask = (~key_padding_mask).unsqueeze(-1).float()
        else:
            valid_mask = torch.ones(
                set_x.shape[0], set_x.shape[1], 1,
                device=set_x.device, dtype=set_x.dtype)

        pooled_attn = self.set_attn_pool(set_x, key_padding_mask=key_padding_mask)
        valid_counts = valid_mask.sum(dim=1).clamp(min=1.0)
        pooled_mean = (set_x * valid_mask).sum(dim=1) / valid_counts

        if key_padding_mask is not None:
            x_for_max = set_x.masked_fill(key_padding_mask.unsqueeze(-1), float("-inf"))
        else:
            x_for_max = set_x
        pooled_max = x_for_max.amax(dim=1)
        pooled_max = torch.nan_to_num(pooled_max, nan=0.0, posinf=0.0, neginf=0.0)

        set_summary = torch.cat([pooled_attn, pooled_mean, pooled_max], dim=-1)
        fused = torch.cat([first_x, set_summary, cond], dim=-1)

        pred_raw_unconstrained = self.reg_head(fused)
        ord_logits = self.ord_head(fused)
        return pred_raw_unconstrained, ord_logits


# =========================
# Build inference inputs (7 global features, position-independent scaling)
# =========================
def prepare_inference_features(case, thresholds, scalers, device):
    """Build model inputs for a threshold sweep over a single physical case.

    Returns:
        first_t, past_t, global_t, thr_t, mask_t — all torch tensors on device
        global_t has shape [B, 7]: [z0, z1, hmg, nsid, log_hd, da, herf]
    """
    B = len(thresholds)
    mem_len = case["mem_len"]
    N = float(case["N"])

    P_raw = np.asarray(case["P"], dtype=np.float32)
    var_raw = np.asarray(case["variances"], dtype=np.float32)
    thr_raw = np.asarray(thresholds, dtype=np.float32).reshape(-1, 1)

    # Feature engineering
    taps_feat = (P_raw * N).astype(np.float32)
    vars_feat = var_raw.astype(np.float32)
    abs_feat = np.abs(taps_feat).astype(np.float32)
    snr_feat = np.log10((taps_feat ** 2) / (var_raw + EPS) + EPS).astype(np.float32)

    # Broadcast to [B, mem_len]
    taps_2d = np.broadcast_to(taps_feat[None, :], (B, mem_len)).copy()
    vars_2d = np.broadcast_to(vars_feat[None, :], (B, mem_len)).copy()
    abs_2d = np.broadcast_to(abs_feat[None, :], (B, mem_len)).copy()
    snr_2d = np.broadcast_to(snr_feat[None, :], (B, mem_len)).copy()

    L_past = mem_len - 1
    valid_past = np.ones((B, L_past), dtype=bool)

    # --- Threshold-dependent global features ---
    first_mean = taps_2d[:, 0:1]
    past_means = taps_2d[:, 1:]
    first_var = vars_2d[:, 0:1]
    past_vars = vars_2d[:, 1:]

    mu0 = (0.5 * past_means.sum(axis=1, keepdims=True)).astype(np.float32)
    mu1 = (first_mean + mu0).astype(np.float32)
    var0 = (0.5 * past_vars.sum(axis=1, keepdims=True)).astype(np.float32)
    var1 = (first_var + var0).astype(np.float32)
    std0 = np.sqrt(np.maximum(var0, EPS)).astype(np.float32)
    std1 = np.sqrt(np.maximum(var1, EPS)).astype(np.float32)

    z0 = ((thr_raw - mu0) / (std0 + EPS)).astype(np.float32)
    z1 = ((mu1 - thr_raw) / (std1 + EPS)).astype(np.float32)
    harmonic = safe_harmonic_z(z0, z1)
    abs_diff = np.abs(z0 - z1).astype(np.float32)
    hmg = (harmonic - 0.25 * abs_diff).astype(np.float32)
    hmg = np.nan_to_num(hmg, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)

    # --- Scenario-level features (threshold-independent) ---
    signal = first_mean  # [B, 1]
    isi = past_means.sum(axis=1, keepdims=True)
    nsid = ((signal - isi) / (signal + isi + EPS)).astype(np.float32)

    gap = signal
    d0_inf = (gap / (std0 + EPS)).astype(np.float32)
    d1_inf = (gap / (std1 + EPS)).astype(np.float32)
    hd = (2.0 * d0_inf * d1_inf / (d0_inf + d1_inf + EPS)).astype(np.float32)
    log_hd = np.log10(hd + EPS).astype(np.float32)
    da = (np.abs(d0_inf - d1_inf) / (d0_inf + d1_inf + EPS)).astype(np.float32)

    past_taps_sum = past_means.sum(axis=1, keepdims=True)
    past_shares = past_means / (past_taps_sum + EPS)
    herf = (past_shares ** 2).sum(axis=1, keepdims=True).astype(np.float32)

    # --- Scale first token ---
    ft_tap = ((taps_2d[:, 0] - scalers["first_tap_mean"]) / scalers["first_tap_std"]).astype(np.float32)
    ft_var = ((vars_2d[:, 0] - scalers["first_var_mean"]) / scalers["first_var_std"]).astype(np.float32)
    ft_abs = ((abs_2d[:, 0] - scalers["first_abs_mean"]) / scalers["first_abs_std"]).astype(np.float32)
    ft_snr = ((snr_2d[:, 0] - scalers["first_snr_mean"]) / scalers["first_snr_std"]).astype(np.float32)
    first_token = np.stack([ft_tap, ft_var, ft_abs, ft_snr], axis=1).astype(np.float32)

    # --- Scale past tokens ---
    pt_tap = apply_shared_scale(taps_2d[:, 1:], scalers["past_tap_mean"], scalers["past_tap_std"], valid_past)
    pt_var = apply_shared_scale(vars_2d[:, 1:], scalers["past_var_mean"], scalers["past_var_std"], valid_past)
    pt_abs = apply_shared_scale(abs_2d[:, 1:], scalers["past_abs_mean"], scalers["past_abs_std"], valid_past)
    pt_snr = apply_shared_scale(snr_2d[:, 1:], scalers["past_snr_mean"], scalers["past_snr_std"], valid_past)
    past_tokens = np.stack([pt_tap, pt_var, pt_abs, pt_snr], axis=2).astype(np.float32)

    # --- Scale globals (7 features) ---
    z0_s = scalers["z0_scaler"].transform(z0).astype(np.float32)
    z1_s = scalers["z1_scaler"].transform(z1).astype(np.float32)
    hmg_s = scalers["harmonic_minus_gap_scaler"].transform(hmg).astype(np.float32)
    nsid_s = scalers["nsid_scaler"].transform(nsid).astype(np.float32)
    log_hd_s = scalers["log_hd_scaler"].transform(log_hd).astype(np.float32)
    da_s = scalers["da_scaler"].transform(da).astype(np.float32)
    herf_s = scalers["herf_scaler"].transform(herf).astype(np.float32)
    global_feats = np.concatenate(
        [z0_s, z1_s, hmg_s, nsid_s, log_hd_s, da_s, herf_s], axis=1
    ).astype(np.float32)

    # --- Scale threshold using scenario-normalized feature from training ---
    total_scaled_taps = np.sum(taps_2d, axis=1, keepdims=True).astype(np.float32)
    threshold_mode = scalers.get("threshold_feature_mode", THRESHOLD_FEATURE_MODE)
    thr_feature = compute_threshold_feature_np(thr_raw, total_scaled_taps, mode=threshold_mode)
    thr_s = scalers["thr_scaler"].transform(thr_feature).astype(np.float32)

    # --- Padding mask (no padding needed here) ---
    pad_mask = np.zeros((B, L_past), dtype=bool)

    first_t = torch.from_numpy(first_token).to(device)
    past_t = torch.from_numpy(past_tokens).to(device)
    global_t = torch.from_numpy(global_feats).to(device)
    thr_t = torch.from_numpy(thr_s).to(device)
    mask_t = torch.from_numpy(pad_mask).to(device)

    return first_t, past_t, global_t, thr_t, mask_t


# =========================
# Prediction helpers
# =========================
def predict_ber_for_thresholds(model, case, thresholds, scalers, device):
    """Predict BER for a threshold vector.

    This recomputes threshold-dependent engineered features, so it is safe to call
    for shifted threshold vectors such as thr-eps and thr+eps.
    """
    first_t, past_t, global_t, thr_t, mask_t = prepare_inference_features(
        case, thresholds, scalers, device,
    )

    with torch.no_grad():
        pred_raw_out, ord_logits = model(
            first_t, past_t, global_t, thr_t, key_padding_mask=mask_t,
        )
        pred_bers = model.raw_to_ber(pred_raw_out).cpu().numpy().reshape(-1)
        pred_log = model.raw_to_log10ber(pred_raw_out).cpu().numpy().reshape(-1)
        pred_regions = ordinal_logits_to_region_labels_torch(ord_logits).cpu().numpy().reshape(-1)
        pred_region_probs = torch.sigmoid(ord_logits).cpu().numpy()

    pred_bers = np.clip(pred_bers, EPS, 0.5)
    pred_log = np.log10(np.clip(pred_bers, EPS, 0.5)).astype(np.float32)
    return pred_bers, pred_log, pred_regions, pred_region_probs


def predict_ber_smoothed_3point(
    model,
    case,
    thresholds,
    scalers,
    device,
    eps_frac=SMOOTHING_EPS_FRAC,
    smoothing_space=SMOOTHING_SPACE,
    weights=SMOOTHING_WEIGHTS,
):
    """Three-point test-time smoothing over threshold.

    Predicts at [thr-eps, thr, thr+eps] and averages either in log-BER space or
    raw-BER space. Crucially, this recomputes global features for each shifted
    threshold, rather than only perturbing the already-scaled threshold tensor.
    """
    thresholds = np.asarray(thresholds, dtype=np.float32)
    w_minus, w_mid, w_plus = weights
    weight_sum = float(w_minus + w_mid + w_plus)
    w_minus, w_mid, w_plus = w_minus / weight_sum, w_mid / weight_sum, w_plus / weight_sum

    thr_span = float(np.max(thresholds) - np.min(thresholds))
    eps_thr = float(eps_frac * max(thr_span, EPS))

    thr_minus = np.clip(thresholds - eps_thr, 0.0, None).astype(np.float32)
    thr_mid = thresholds.astype(np.float32)
    thr_plus = (thresholds + eps_thr).astype(np.float32)

    ber_minus, log_minus, _, _ = predict_ber_for_thresholds(
        model, case, thr_minus, scalers, device,
    )
    ber_mid, log_mid, regions_mid, probs_mid = predict_ber_for_thresholds(
        model, case, thr_mid, scalers, device,
    )
    ber_plus, log_plus, _, _ = predict_ber_for_thresholds(
        model, case, thr_plus, scalers, device,
    )

    if smoothing_space.lower() == "raw":
        ber_smooth = (
            w_minus * ber_minus +
            w_mid * ber_mid +
            w_plus * ber_plus
        )
        ber_smooth = np.clip(ber_smooth, EPS, 0.5).astype(np.float32)
        log_smooth = np.log10(ber_smooth).astype(np.float32)
    elif smoothing_space.lower() == "log":
        log_smooth = (
            w_minus * log_minus +
            w_mid * log_mid +
            w_plus * log_plus
        ).astype(np.float32)
        ber_smooth = np.clip(10.0 ** log_smooth, EPS, 0.5).astype(np.float32)
    else:
        raise ValueError("smoothing_space must be either 'log' or 'raw'.")

    return {
        "ber": ber_smooth,
        "log10_ber": log_smooth,
        "regions": regions_mid,
        "region_probs": probs_mid,
        "raw_mid_ber": ber_mid,
        "raw_mid_log10_ber": log_mid,
        "eps_threshold": eps_thr,
        "smoothing_space": smoothing_space,
    }


# =========================
# Generate scenario
# =========================
rng = np.random.default_rng(RANDOM_SEED)
case = generate_physical_case(
    rng,
    min_mem_len=PHYSICS_MIN_MEM_LEN,
    max_mem_len=PHYSICS_MAX_MEM_LEN,
    arrival_coverage=ARRIVAL_COVERAGE,
)

print("Generated physical scenario")
print("radius    =", case["radius"])
print("distance  =", case["distance"])
print("diffusion =", case["diffusion"])
print("Ts        =", case["Ts"])
print("N         =", case["N"])
print("mem_len   =", case["mem_len"])
print("P         =", case["P"])
print("P_scaled  =", case["P_scaled"])
print("variances =", case["variances"])

# =========================
# Threshold sweep — ground truth
# =========================
thr_min = 0.0
thr_max = float(np.sum(case["P_scaled"]))
thresholds = np.linspace(thr_min, thr_max, N_THRESHOLDS)

print("Threshold search interval:", thr_min, "to", thr_max)
print("sum(P_scaled) =", np.sum(case["P_scaled"]))

real_bers = np.array([
    calculate_ber_vectorized(
        mem_len=case["mem_len"],
        threshold=thr,
        P_scaled=case["P_scaled"],
        variances=case["variances"],
    )
    for thr in thresholds
])

real_bers = np.clip(real_bers, EPS, 0.5)
real_regions = raw_to_region_labels_np(real_bers)

# =========================
# Load model + scalers
# =========================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
scalers = joblib.load(SCALER_PATH)

max_past_seq_len = scalers.get("train_max_past_seq_len", scalers.get("max_past_seq_len", case["mem_len"] - 1))
ordinal_thresholds = scalers.get("ordinal_thresholds", ORDINAL_THRESHOLDS)
num_ordinal = len(ordinal_thresholds)
global_dim = scalers.get("global_dim", 7)

model = BERFirstTokenSetTransformerBoundedLogHalfMultiTask(
    max_past_seq_len=max_past_seq_len,
    token_dim=scalers.get("past_token_dim", 4),
    first_token_dim=scalers.get("first_token_dim", 4),
    threshold_dim=1,
    global_dim=global_dim,
    d_model=128,
    num_set_layers=4,
    num_heads=4,
    mlp_ratio=4.0,
    dropout=0.10,
    num_ordinal_thresholds=num_ordinal,
).to(device)

state = torch.load(MODEL_PATH, map_location=device, weights_only=True)
model.load_state_dict(state)
model.eval()

print(f"\nModel loaded: max_past_seq_len={max_past_seq_len}, "
      f"global_dim={global_dim}, num_ordinal={num_ordinal}")
print(f"Scaling strategy: {scalers.get('scaling_strategy', 'unknown')}")
print(f"Threshold feature mode: {scalers.get('threshold_feature_mode', THRESHOLD_FEATURE_MODE)}")
print("Ordinal head: monotonic cumulative logits")

# =========================
# Predict across thresholds
# =========================
raw_bers, raw_log, raw_regions, raw_region_probs = predict_ber_for_thresholds(
    model, case, thresholds, scalers, device,
)

if USE_TEST_TIME_SMOOTHING:
    smooth = predict_ber_smoothed_3point(
        model,
        case,
        thresholds,
        scalers,
        device,
        eps_frac=SMOOTHING_EPS_FRAC,
        smoothing_space=SMOOTHING_SPACE,
        weights=SMOOTHING_WEIGHTS,
    )
    pred_bers = smooth["ber"]
    pred_log = smooth["log10_ber"]
    pred_regions = smooth["regions"]
    pred_region_probs = smooth["region_probs"]
    print(
        f"Test-time smoothing: ON | space={smooth['smoothing_space']} | "
        f"eps_threshold={smooth['eps_threshold']:.6g}"
    )
else:
    pred_bers = raw_bers
    pred_log = raw_log
    pred_regions = raw_regions
    pred_region_probs = raw_region_probs
    print("Test-time smoothing: OFF")

pred_bers = np.clip(pred_bers, EPS, 0.5)
raw_bers = np.clip(raw_bers, EPS, 0.5)

# =========================
# Comparison table
# =========================
results = pd.DataFrame({
    "threshold": thresholds,
    "real_BER": real_bers,
    "estimated_BER": pred_bers,
    "estimated_BER_raw_unsmoothed": raw_bers,
    "smoothing_delta": pred_bers - raw_bers,
    "real_region": real_regions,
    "predicted_region": pred_regions,
    "abs_error": np.abs(pred_bers - real_bers),
    "abs_log10_error": np.abs(
        np.log10(np.clip(pred_bers, EPS, 0.5)) -
        np.log10(np.clip(real_bers, EPS, 0.5))
    ),
    "region_abs_error": np.abs(pred_regions.astype(np.int64) - real_regions.astype(np.int64)),
})

print(results.head(15))

print("\nSummary")
print("Mean abs raw error        :", results["abs_error"].mean())
print("Mean abs log10 error      :", results["abs_log10_error"].mean())
print("Max  abs log10 error      :", results["abs_log10_error"].max())
print("Mean region abs error     :", results["region_abs_error"].mean())
print("Exact region accuracy     :", np.mean(results["real_region"] == results["predicted_region"]))
if USE_TEST_TIME_SMOOTHING:
    print("Mean abs smoothing delta  :", float(np.mean(np.abs(results["smoothing_delta"]))))
    print("Max  abs smoothing delta  :", float(np.max(np.abs(results["smoothing_delta"]))))

best_real_idx = np.argmin(real_bers)
best_est_idx = np.argmin(pred_bers)

print("\nBest threshold from real BER      :", thresholds[best_real_idx])
print("Minimum real BER                  :", real_bers[best_real_idx])
print("Real BER region there             :", REGION_LABELS.get(int(real_regions[best_real_idx]), "?"))

print("Best threshold from estimated BER :", thresholds[best_est_idx])
print("Estimated BER at that threshold   :", pred_bers[best_est_idx])
print("Predicted BER region there        :", REGION_LABELS.get(int(pred_regions[best_est_idx]), "?"))

# =========================
# Plot 1: log-scale BER
# =========================
plt.figure(figsize=(10, 6))
plt.plot(results["threshold"], results["estimated_BER"], label="Estimated BER" + (" smoothed" if USE_TEST_TIME_SMOOTHING else ""))
if USE_TEST_TIME_SMOOTHING:
    plt.plot(results["threshold"], results["estimated_BER_raw_unsmoothed"], label="Estimated BER raw", alpha=0.45)
plt.plot(results["threshold"], results["real_BER"], label="Real BER")
plt.yscale("log")
plt.xlabel("Threshold")
plt.ylabel("BER")
plt.title(f"Threshold vs BER (log scale) — mem_len={case['mem_len']}")
plt.legend()
plt.grid(True, which="both", alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, "01_ber_log_scale.png"), dpi=150)
plt.close()
print(f"Saved: {PLOT_DIR}/01_ber_log_scale.png")

# =========================
# Plot 2: linear-scale BER
# =========================
plt.figure(figsize=(10, 6))
plt.plot(results["threshold"], results["estimated_BER"], label="Estimated BER" + (" smoothed" if USE_TEST_TIME_SMOOTHING else ""))
if USE_TEST_TIME_SMOOTHING:
    plt.plot(results["threshold"], results["estimated_BER_raw_unsmoothed"], label="Estimated BER raw", alpha=0.45)
plt.plot(results["threshold"], results["real_BER"], label="Real BER")
plt.yscale("linear")
plt.xlabel("Threshold")
plt.ylabel("BER")
plt.title(f"Threshold vs BER (linear scale) — mem_len={case['mem_len']}")
plt.legend()
plt.grid(True, which="both", alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, "02_ber_linear_scale.png"), dpi=150)
plt.close()
print(f"Saved: {PLOT_DIR}/02_ber_linear_scale.png")

# =========================
# Plot 3: predicted vs true region
# =========================
plt.figure(figsize=(10, 6))
plt.plot(results["threshold"], results["predicted_region"], label="Predicted region")
plt.plot(results["threshold"], results["real_region"], label="True region")
plt.xlabel("Threshold")
plt.ylabel("BER Region Class")
plt.title(f"Threshold vs BER Region — mem_len={case['mem_len']}")
plt.yticks(list(REGION_LABELS.keys()),
           [REGION_LABELS[k] for k in sorted(REGION_LABELS.keys())],
           fontsize=7)
plt.legend()
plt.grid(True, which="both", alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, "03_region_comparison.png"), dpi=150)
plt.close()
print(f"Saved: {PLOT_DIR}/03_region_comparison.png")

# =========================
# Plot 4: ordinal threshold probabilities
# =========================
plt.figure(figsize=(12, 7))
for i, thr_val in enumerate(ordinal_thresholds):
    plt.plot(thresholds, pred_region_probs[:, i], label=f"P(y >= {thr_val:g})")
plt.xlabel("Threshold")
plt.ylabel("Ordinal Probability")
plt.title(f"Ordinal Head Outputs Across Threshold Sweep — mem_len={case['mem_len']}")
plt.legend(fontsize=7, ncol=2)
plt.grid(True, which="both", alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, "04_ordinal_probabilities.png"), dpi=150)
plt.close()
print(f"Saved: {PLOT_DIR}/04_ordinal_probabilities.png")

# =========================
# Plot 5: absolute error by threshold
# =========================
plt.figure(figsize=(10, 6))
plt.plot(results["threshold"], results["abs_error"], label="Abs raw error", alpha=0.8)
plt.plot(results["threshold"], results["abs_log10_error"], label="Abs log10 error", alpha=0.8)
plt.yscale("log")
plt.xlabel("Threshold")
plt.ylabel("Error")
plt.title(f"Prediction Error Across Threshold Sweep — mem_len={case['mem_len']}")
plt.legend()
plt.grid(True, which="both", alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, "05_error_by_threshold.png"), dpi=150)
plt.close()
print(f"Saved: {PLOT_DIR}/05_error_by_threshold.png")

# =========================
# Plot 6: smoothing delta
# =========================
if USE_TEST_TIME_SMOOTHING:
    plt.figure(figsize=(10, 6))
    plt.plot(results["threshold"], results["smoothing_delta"], label="Smoothed - raw estimate")
    plt.axhline(0.0, linewidth=1.0)
    plt.xlabel("Threshold")
    plt.ylabel("BER delta")
    plt.title(f"Test-Time Smoothing Delta — mem_len={case['mem_len']}")
    plt.legend()
    plt.grid(True, which="both", alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(PLOT_DIR, "06_smoothing_delta.png"), dpi=150)
    plt.close()
    print(f"Saved: {PLOT_DIR}/06_smoothing_delta.png")


Generated physical scenario
radius    = 4.175494317332729
distance  = 14.130330840994176
diffusion = 52.923594506181246
Ts        = 0.577429608675686
N         = 3620
mem_len   = 20
P         = [0.0161251  0.0297751  0.02177731 0.01583882 0.01203602 0.0095056
 0.00773896 0.00645339 0.00548549 0.0047361  0.00414228 0.00366251
 0.00326841 0.00294007 0.00266313 0.00242701 0.00222377 0.00204735
 0.00189306 0.0017572 ]
P_scaled  = [ 58.37284513 107.78585809  78.83385227  57.33652944  43.5703904
  34.4102711   28.0150473   23.36127993  19.85746581  17.14466926
  14.99506554  13.25827868  11.83165655  10.64306554   9.64053005
   8.78577127   8.05005117   7.41142503   6.85288047   6.36104915]
variances = [ 57.43157744 104.57652351  77.11706324  56.42838646  43.04597633
  34.08318084  27.79823988  23.21052043  19.74853792  17.06347045
  14.93295173  13.20972013  11.7929858   10.61177415   9.61485607
   8.76444813   8.0321497    7.39625122   6.83990755   6.34987154]
Threshold search interval: 0.

In [5]:
import os
import gc
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq


def clean_csv_to_parquet(
    csv_path,
    parquet_path,
    chunksize=500_000,
    compression="zstd",
):
    """
    Stream-clean CSV -> Parquet.

    Handles:
      - rows with too many fields: skipped by pandas on_bad_lines='skip'
      - malformed numeric cells: coerced to NaN
      - invalid BER / N / threshold rows: removed
      - missing mem_len: derived from non-null tap_* columns
    """

    print(f"\nCleaning/converting:\n  {csv_path}\n  -> {parquet_path}")

    writer = None
    total_in = 0
    total_out = 0
    chunk_idx = 0

    reader = pd.read_csv(
        csv_path,
        chunksize=chunksize,
        on_bad_lines="skip",
        engine="c",
        low_memory=False,
    )

    try:
        for chunk in reader:
            chunk_idx += 1
            total_in += len(chunk)

            # Normalize column names
            chunk.columns = [str(c).strip() for c in chunk.columns]

            tap_cols = sorted(
                [c for c in chunk.columns if c.startswith("tap_")],
                key=lambda x: int(x.split("_")[1])
            )
            var_cols = sorted(
                [c for c in chunk.columns if c.startswith("var_")],
                key=lambda x: int(x.split("_")[1])
            )

            required = ["N", "threshold", "BER"]
            missing = [c for c in required if c not in chunk.columns]
            if missing:
                raise ValueError(f"{csv_path}: missing required columns {missing}")

            if not tap_cols:
                raise ValueError(f"{csv_path}: no tap_* columns found")

            # Derive mem_len if absent
            if "mem_len" not in chunk.columns:
                chunk["mem_len"] = chunk[tap_cols].notna().sum(axis=1)

            numeric_cols = list(dict.fromkeys(
                tap_cols + var_cols + ["mem_len", "N", "threshold", "BER"]
            ))

            # Coerce malformed numeric cells to NaN
            for col in numeric_cols:
                if col in chunk.columns:
                    chunk[col] = pd.to_numeric(chunk[col], errors="coerce")

            # Basic row-level cleaning
            before = len(chunk)

            chunk = chunk[chunk["mem_len"].notna()]
            chunk["mem_len"] = chunk["mem_len"].astype("int64")

            chunk = chunk[
                (chunk["mem_len"] != 1)
                & (chunk["BER"] > 0)
                & (chunk["N"] > 0)
                & (chunk["threshold"] > 0)
            ].copy()

            chunk["BER"] = chunk["BER"].clip(lower=1e-12, upper=0.5)

            # Drop rows where valid tap positions contain NaN/non-numeric values.
            # Padding taps after mem_len are allowed to be NaN.
            if len(chunk) > 0:
                tap_values = chunk[tap_cols].to_numpy()
                mem_len = chunk["mem_len"].to_numpy()
                L = len(tap_cols)

                # Clamp mem_len to available tap columns
                mem_len_clamped = mem_len.clip(max=L)

                valid_mask = (
                    pd.DataFrame(
                        {
                            c: range(len(chunk))
                            for c in []
                        }
                    )
                )

                import numpy as np

                valid = np.arange(L)[None, :] < mem_len_clamped[:, None]
                bad_valid_tap = pd.isna(tap_values) & valid
                keep = ~bad_valid_tap.any(axis=1)

                chunk = chunk.iloc[keep].copy()

            # Fill padding NaNs only after valid-region check
            chunk[tap_cols] = chunk[tap_cols].fillna(0.0)

            if len(var_cols) > 0:
                chunk[var_cols] = chunk[var_cols].fillna(0.0)

            # Convert to Arrow table and append to Parquet
            if len(chunk) > 0:
                table = pa.Table.from_pandas(chunk, preserve_index=False)

                if writer is None:
                    writer = pq.ParquetWriter(
                        parquet_path,
                        table.schema,
                        compression=compression,
                        use_dictionary=False,
                    )

                writer.write_table(table, row_group_size=200_000)

            total_out += len(chunk)

            print(
                f"  chunk {chunk_idx:04d}: "
                f"read={before:,}, kept={len(chunk):,}, "
                f"total_kept={total_out:,}"
            )

            del chunk
            gc.collect()

    finally:
        if writer is not None:
            writer.close()

    print(f"Done: read approximately {total_in:,} rows, wrote {total_out:,} cleaned rows")

In [6]:
clean_csv_to_parquet(
    "./../ber_data_generation/data/data_physics_total.csv",
    "./../ber_data_generation/data/data_physics_total.cleaned.parquet",
)

clean_csv_to_parquet(
    "./../ber_data_generation/data/data_random_total.csv",
    "./../ber_data_generation/data/data_random_total.cleaned.parquet",
)


Cleaning/converting:
  ./../ber_data_generation/data/data_physics_total.csv
  -> ./../ber_data_generation/data/data_physics_total.cleaned.parquet
  chunk 0001: read=500,000, kept=321,933, total_kept=321,933
  chunk 0002: read=500,000, kept=317,381, total_kept=639,314
  chunk 0003: read=500,000, kept=318,609, total_kept=957,923
  chunk 0004: read=500,000, kept=321,793, total_kept=1,279,716
  chunk 0005: read=500,000, kept=317,815, total_kept=1,597,531
  chunk 0006: read=500,000, kept=319,537, total_kept=1,917,068
  chunk 0007: read=500,000, kept=322,816, total_kept=2,239,884
  chunk 0008: read=500,000, kept=319,632, total_kept=2,559,516
  chunk 0009: read=500,000, kept=317,622, total_kept=2,877,138
  chunk 0010: read=500,000, kept=316,992, total_kept=3,194,130
  chunk 0011: read=500,000, kept=319,633, total_kept=3,513,763
  chunk 0012: read=500,000, kept=316,902, total_kept=3,830,665
  chunk 0013: read=500,000, kept=314,754, total_kept=4,145,419
  chunk 0014: read=500,000, kept=318,904

  chunk 0041: read=500,000, kept=479,699, total_kept=19,666,178
  chunk 0042: read=500,000, kept=479,677, total_kept=20,145,855
  chunk 0043: read=500,000, kept=479,685, total_kept=20,625,540
  chunk 0044: read=500,000, kept=479,639, total_kept=21,105,179
  chunk 0045: read=500,000, kept=479,687, total_kept=21,584,866
  chunk 0046: read=500,000, kept=479,649, total_kept=22,064,515
  chunk 0047: read=500,000, kept=479,720, total_kept=22,544,235
  chunk 0048: read=500,000, kept=479,639, total_kept=23,023,874
  chunk 0049: read=500,000, kept=479,700, total_kept=23,503,574
  chunk 0050: read=500,000, kept=479,664, total_kept=23,983,238
  chunk 0051: read=500,000, kept=479,634, total_kept=24,462,872
  chunk 0052: read=500,000, kept=479,660, total_kept=24,942,532
  chunk 0053: read=500,000, kept=479,632, total_kept=25,422,164
  chunk 0054: read=500,000, kept=479,695, total_kept=25,901,859
  chunk 0055: read=500,000, kept=479,685, total_kept=26,381,544
  chunk 0056: read=500,000, kept=479,627

In [7]:
"""
Fine-Tuning Script: Parquet Streaming Version
============================================
Continues training from the normalized-threshold + monotonic-ordinal checkpoint.

What changed vs the eager CSV version:
  - CSVs are cleaned and converted to Parquet once.
  - Training/validation/test data are streamed from Parquet row groups.
  - No full pandas concat.
  - No sklearn train_test_split on full arrays.
  - No full TensorDataset in RAM.
  - No WeightedRandomSampler, because that requires all labels/weights in memory.
  - Scalers are reused from the pretrained run.

Recommended flow:
  1. Set RUN_PARQUET_CONVERSION = True once and run this file.
  2. Set RUN_PARQUET_CONVERSION = False for later training runs.

Dependencies:
  pip install pandas pyarrow joblib torch numpy
"""

import os
import re
import gc
import copy
import joblib
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torch.multiprocessing as mp
from torch.utils.data import DataLoader, IterableDataset, get_worker_info

mp.set_sharing_strategy("file_system")


# =========================
# Configuration
# =========================
BASE_DIR = "./"
DATA_DIR = "./../ber_data_generation/data/"

# ---- Source: previous training run to resume from ----
PRETRAINED_MODEL_PATH = os.path.join(
    BASE_DIR, "random_extra_multitask_finetune_best_normthr_monoord.pth"
)
PRETRAINED_SCALER_PATH = os.path.join(
    BASE_DIR, "random_extra_multitask_finetune_scalers_normthr_monoord.pkl"
)

# This streaming version is designed for reused scalers.
REUSE_SCALERS = True

# ---- Destination: fine-tuned model ----
BEST_MODEL_SAVE_PATH = os.path.join(
    BASE_DIR, "random_extra_multitask_finetune_best_normthr_monoord2.pth"
)
LAST_MODEL_SAVE_PATH = os.path.join(
    BASE_DIR, "random_extra_multitask_finetune_last_normthr_monoord2.pth"
)
SCALER_SAVE_PATH = os.path.join(
    BASE_DIR, "random_extra_multitask_finetune_scalers_normthr_monoord2.pkl"
)

# ---- Raw CSVs ----
CSV_DATA_PATHS = [
    os.path.join(DATA_DIR, "data_physics_total.csv"),
    os.path.join(DATA_DIR, "data_random_total.csv"),
]

# ---- Cleaned Parquet files ----
PARQUET_DATA_PATHS = [
    os.path.join(DATA_DIR, "data_physics_total.cleaned.parquet"),
    os.path.join(DATA_DIR, "data_random_total.cleaned.parquet"),
]

# Training reads Parquet.
DATA_PATHS = PARQUET_DATA_PATHS

# Set True only once to create Parquet files. Then set False for normal training.
RUN_PARQUET_CONVERSION = False
OVERWRITE_PARQUET = False

# Cap per source file. None = use all rows. 10_000_000 matches your old config.
NROWS_PER_DATASET = 10_000_000
# NROWS_PER_DATASET = None

# Conversion and streaming batch sizes.
PARQUET_CSV_CHUNKSIZE = 500_000
PARQUET_BATCH_ROWS = 8192
DATALOADER_NUM_WORKERS = 0

# ---- Fine-tune hyperparameters ----
FINETUNE_LR = 2e-5
FINETUNE_WEIGHT_DECAY = 1e-5
FINETUNE_MAX_EPOCHS = 60
FINETUNE_MIN_EPOCHS = 25
FINETUNE_PATIENCE = 8
SCHEDULER_PATIENCE = 3
SCHEDULER_FACTOR = 0.5
GRAD_CLIP = 1.0

EPS = 1e-12
LOG10_HALF = float(np.log10(0.5))

# =========================
# New-model threshold / numeric feature options
# =========================
THRESHOLD_FEATURE_MODE = "normalized_logit"
THRESHOLD_NORM_CLIP_EPS = 1e-5
HARMONIC_DENOM_EPS = 1e-8
FEATURE_CLIP_ABS = 1e6

ORDINAL_THRESHOLDS = [
    1e-6, 1e-5, 1e-4, 1e-3, 1e-2, 1e-1,
    0.15, 0.20, 0.25, 0.30, 0.35, 0.40, 0.45,
]
NUM_ORDINAL_THRESHOLDS = len(ORDINAL_THRESHOLDS)
NUM_REGION_CLASSES = NUM_ORDINAL_THRESHOLDS + 1


# =========================
# Utilities
# =========================
def get_sorted_seq_cols(columns, prefix):
    pattern = re.compile(rf"^{re.escape(prefix)}_(\d+)$")
    matched = []
    for col in columns:
        m = pattern.match(str(col))
        if m:
            matched.append((int(m.group(1)), str(col)))
    matched.sort(key=lambda x: x[0])
    return [col for _, col in matched]


def has_nonfinite_tensor(x):
    return not torch.isfinite(x).all().item()


def y_log_to_raw_np(y_log):
    return np.clip(10 ** y_log, EPS, 0.5)


def raw_to_region_labels_np(y_raw):
    y = np.asarray(y_raw).reshape(-1)
    labels = np.searchsorted(np.asarray(ORDINAL_THRESHOLDS), y, side="right")
    return labels.astype(np.int64)


def region_labels_to_ordinal_targets_np(labels, num_thresholds=NUM_ORDINAL_THRESHOLDS):
    labels = np.asarray(labels).reshape(-1)
    thresholds = np.arange(1, num_thresholds + 1, dtype=np.int64)
    ordinal = (labels[:, None] >= thresholds[None, :]).astype(np.float32)
    return ordinal


def ordinal_logits_to_region_labels_torch(logits):
    probs = torch.sigmoid(logits)
    passed = (probs >= 0.5).sum(dim=1)
    return passed.long()


def compute_region_class_weights(labels_np, num_classes=NUM_REGION_CLASSES, max_weight=8.0):
    counts = np.bincount(labels_np.reshape(-1), minlength=num_classes).astype(np.float64)
    counts = np.maximum(counts, 1.0)
    weights = counts.sum() / (num_classes * counts)
    weights = np.sqrt(weights)
    weights = np.clip(weights, 1.0, max_weight)
    return weights.astype(np.float32)


def compute_ordinal_pos_weights_from_region_labels(
    region_labels_np,
    num_thresholds=NUM_ORDINAL_THRESHOLDS,
    max_weight=20.0,
):
    ordinal_targets = region_labels_to_ordinal_targets_np(region_labels_np, num_thresholds)
    pos_counts = ordinal_targets.sum(axis=0)
    neg_counts = ordinal_targets.shape[0] - pos_counts
    pos_counts = np.maximum(pos_counts, 1.0)
    pos_weight = neg_counts / pos_counts
    pos_weight = np.clip(pos_weight, 1.0, max_weight)
    return pos_weight.astype(np.float32)


def apply_shared_scale(data, mean, std, valid_mask=None):
    scaled = ((data - mean) / std).astype(np.float32)
    if valid_mask is not None and scaled.ndim == 2 and valid_mask.shape == scaled.shape:
        scaled = scaled * valid_mask.astype(np.float32)
    return scaled


def sanitize_np_feature(x, name, clip_abs=FEATURE_CLIP_ABS):
    arr = np.asarray(x, dtype=np.float32)
    bad = ~np.isfinite(arr)
    if np.any(bad):
        print(f"Warning: {name} contained {int(bad.sum())} non-finite values; replacing with 0.")
    arr = np.nan_to_num(arr, nan=0.0, posinf=0.0, neginf=0.0)
    arr = np.clip(arr, -clip_abs, clip_abs).astype(np.float32)
    return arr


def stable_harmonic_pair_np(a, b, denom_eps=HARMONIC_DENOM_EPS):
    a = np.asarray(a, dtype=np.float32)
    b = np.asarray(b, dtype=np.float32)
    denom = (a + b).astype(np.float32)
    numer = (2.0 * a * b).astype(np.float32)
    out = np.divide(
        numer,
        denom,
        out=np.zeros_like(numer, dtype=np.float32),
        where=np.abs(denom) > denom_eps,
    ).astype(np.float32)
    return sanitize_np_feature(out, "stable_harmonic_pair")


def compute_threshold_feature_np(threshold_raw, total_scaled_taps, mode=THRESHOLD_FEATURE_MODE):
    thr = np.asarray(threshold_raw, dtype=np.float32).reshape(-1, 1)
    total = np.asarray(total_scaled_taps, dtype=np.float32).reshape(-1, 1)
    denom = np.maximum(total, EPS).astype(np.float32)

    if mode == "normalized":
        feat = (thr / denom).astype(np.float32)
    elif mode == "normalized_logit":
        u = (thr / denom).astype(np.float32)
        u = np.clip(u, THRESHOLD_NORM_CLIP_EPS, 1.0 - THRESHOLD_NORM_CLIP_EPS)
        feat = np.log(u / (1.0 - u)).astype(np.float32)
    elif mode == "log10_raw":
        feat = np.log10(thr + EPS).astype(np.float32)
    else:
        raise ValueError(f"Unknown THRESHOLD_FEATURE_MODE: {mode}")

    return sanitize_np_feature(feat, f"threshold_feature_{mode}")


def compute_variances_from_taps(taps_raw, N_array, valid_mask):
    N = N_array.reshape(-1, 1).astype(np.float64)
    taps = np.where(np.isnan(taps_raw), 0.0, taps_raw).astype(np.float64)
    taps = np.clip(taps, 0.0, 1.0)
    vars_full = N * taps * (1.0 - taps)
    vars_full = vars_full * valid_mask.astype(np.float64)
    return vars_full.astype(np.float32)


def safe_torch_load_state_dict(path, map_location):
    try:
        return torch.load(path, map_location=map_location, weights_only=True)
    except TypeError:
        return torch.load(path, map_location=map_location)


# =========================
# CSV -> Parquet conversion
# =========================
def clean_csv_to_parquet(
    csv_path,
    parquet_path,
    chunksize=PARQUET_CSV_CHUNKSIZE,
    compression="zstd",
    overwrite=False,
):
    """
    Stream-clean CSV -> Parquet without loading the full CSV.

    Handles:
      - over-long corrupted rows: skipped by pandas on_bad_lines='skip'
      - malformed numeric cells: coerced to NaN
      - invalid BER/N/threshold rows: removed
      - missing mem_len: derived from non-null tap_* columns
      - malformed valid tap positions: dropped
      - padding tap NaNs: filled with 0 after valid-region check
    """
    print(f"\nCleaning/converting:\n  {csv_path}\n  -> {parquet_path}")

    if not os.path.exists(csv_path):
        raise FileNotFoundError(f"CSV not found: {csv_path}")

    os.makedirs(os.path.dirname(parquet_path), exist_ok=True)

    if os.path.exists(parquet_path):
        if overwrite:
            print(f"  Removing existing parquet: {parquet_path}")
            os.remove(parquet_path)
        else:
            print(f"  Parquet already exists; skipping conversion: {parquet_path}")
            return

    writer = None
    total_read = 0
    total_written = 0
    total_dropped_bad_taps = 0
    total_dropped_filters = 0

    reader = pd.read_csv(
        csv_path,
        chunksize=chunksize,
        on_bad_lines="skip",
        engine="c",
        low_memory=False,
    )

    try:
        for chunk_idx, chunk in enumerate(reader, start=1):
            chunk.columns = [str(c).strip() for c in chunk.columns]
            before = len(chunk)
            total_read += before

            tap_cols = get_sorted_seq_cols(chunk.columns, "tap")
            var_cols = get_sorted_seq_cols(chunk.columns, "var")

            if not tap_cols:
                raise ValueError(f"No tap_* columns found in {csv_path}")

            for required in ["N", "threshold", "BER"]:
                if required not in chunk.columns:
                    raise ValueError(f"Missing required column {required!r} in {csv_path}")

            # Derive mem_len before padding NaNs are filled.
            if "mem_len" not in chunk.columns:
                chunk["mem_len"] = chunk[tap_cols].notna().sum(axis=1)

            numeric_cols = list(dict.fromkeys(
                tap_cols + var_cols + ["mem_len", "N", "threshold", "BER"]
            ))

            for col in numeric_cols:
                if col in chunk.columns:
                    chunk[col] = pd.to_numeric(chunk[col], errors="coerce")

            chunk = chunk[chunk["mem_len"].notna()].copy()
            if len(chunk) == 0:
                print(f"  chunk {chunk_idx:04d}: read={before:,}, wrote=0")
                continue

            chunk["mem_len"] = chunk["mem_len"].astype(np.int64)
            chunk["threshold"] = chunk["threshold"].fillna(0.0)
            chunk["BER"] = chunk["BER"].fillna(0.0)
            chunk["N"] = chunk["N"].fillna(0.0)

            before_filters = len(chunk)
            chunk = chunk[
                (chunk["mem_len"] > 1)
                & (chunk["BER"] > 0)
                & (chunk["N"] > 0)
                & (chunk["threshold"] > 0)
            ].copy()
            total_dropped_filters += before_filters - len(chunk)

            if len(chunk) == 0:
                print(f"  chunk {chunk_idx:04d}: read={before:,}, wrote=0")
                continue

            chunk["BER"] = chunk["BER"].clip(lower=EPS, upper=0.5)

            # Drop rows where valid tap positions contain NaN/non-numeric values.
            tap_values = chunk[tap_cols].to_numpy(dtype=np.float32)
            mem_len = chunk["mem_len"].to_numpy(dtype=np.int64)
            L = len(tap_cols)

            mem_len_clamped = np.minimum(mem_len, L)
            valid = np.arange(L)[None, :] < mem_len_clamped[:, None]

            bad_valid_tap = (~np.isfinite(tap_values)) & valid
            keep = ~bad_valid_tap.any(axis=1)

            dropped_bad_taps = int((~keep).sum())
            total_dropped_bad_taps += dropped_bad_taps

            if dropped_bad_taps:
                print(f"    dropped malformed valid-tap rows: {dropped_bad_taps:,}")

            chunk = chunk.iloc[keep].copy()

            if len(chunk) == 0:
                print(f"  chunk {chunk_idx:04d}: read={before:,}, wrote=0")
                continue

            # Padding/outside-valid-memory NaNs are safe to zero-fill now.
            chunk[tap_cols] = chunk[tap_cols].fillna(0.0)
            if var_cols:
                chunk[var_cols] = chunk[var_cols].fillna(0.0)

            # Stable dtypes.
            for col in tap_cols + var_cols + ["N", "threshold", "BER"]:
                if col in chunk.columns:
                    chunk[col] = chunk[col].astype(np.float32)
            chunk["mem_len"] = chunk["mem_len"].astype(np.int64)

            chunk["source_dataset"] = os.path.basename(csv_path)
            chunk["_has_vars"] = bool(len(var_cols) > 0)

            table = pa.Table.from_pandas(chunk, preserve_index=False)

            if writer is None:
                writer = pq.ParquetWriter(
                    parquet_path,
                    table.schema,
                    compression=compression,
                    use_dictionary=False,
                )

            writer.write_table(table, row_group_size=200_000)
            total_written += len(chunk)

            print(
                f"  chunk {chunk_idx:04d}: "
                f"read={before:,}, wrote={len(chunk):,}, total_written={total_written:,}"
            )

            del chunk, table
            gc.collect()

    finally:
        if writer is not None:
            writer.close()

    print(
        f"Done: read={total_read:,}, wrote={total_written:,}, "
        f"dropped_filters≈{total_dropped_filters:,}, "
        f"dropped_bad_valid_taps={total_dropped_bad_taps:,}"
    )


def build_clean_parquet_files(overwrite=OVERWRITE_PARQUET):
    for csv_path, parquet_path in zip(CSV_DATA_PATHS, PARQUET_DATA_PATHS):
        clean_csv_to_parquet(csv_path, parquet_path, overwrite=overwrite)


# =========================
# Parquet streaming helpers
# =========================
def _parquet_schema_names(path):
    return set(pq.ParquetFile(path).schema_arrow.names)


def infer_parquet_sequence_columns(parquet_paths, pretrained_scalers=None):
    all_cols = set()
    for path in parquet_paths:
        if not os.path.exists(path):
            raise FileNotFoundError(f"Parquet dataset not found: {path}")
        all_cols |= _parquet_schema_names(path)

    tap_cols = get_sorted_seq_cols(all_cols, "tap")

    if pretrained_scalers is not None and "tap_cols" in pretrained_scalers:
        tap_cols = sorted(
            set(tap_cols) | set(pretrained_scalers["tap_cols"]),
            key=lambda c: int(c.split("_")[1]),
        )

    if not tap_cols:
        raise ValueError("No tap_* columns found in Parquet files.")

    tap_indices = [int(c.split("_")[1]) for c in tap_cols]
    var_cols = [f"var_{i}" for i in tap_indices]

    return tap_cols, var_cols


def stable_hash_percent(row_ids, seed=42):
    """Deterministic split hash in [0, 10000)."""
    x = row_ids.astype(np.uint64) + np.uint64(seed)
    x ^= x >> np.uint64(33)
    x *= np.uint64(0xff51afd7ed558ccd)
    x ^= x >> np.uint64(33)
    x *= np.uint64(0xc4ceb9fe1a85ec53)
    x ^= x >> np.uint64(33)
    return (x % np.uint64(10000)).astype(np.int32)


def split_keep_mask(row_ids, split, seed=42):
    pct = stable_hash_percent(row_ids, seed=seed)
    if split == "train":
        return pct < 7000
    if split == "val":
        return (pct >= 7000) & (pct < 8500)
    if split == "test":
        return pct >= 8500
    raise ValueError(f"Unknown split: {split}")


def build_parquet_row_group_index(parquet_paths, max_rows_per_file=None):
    items = []
    global_start = 0

    for path in parquet_paths:
        pf = pq.ParquetFile(path)
        file_seen = 0

        for rg_idx in range(pf.num_row_groups):
            rg_rows = pf.metadata.row_group(rg_idx).num_rows

            if max_rows_per_file is None:
                take_rows = rg_rows
            else:
                remaining = max_rows_per_file - file_seen
                if remaining <= 0:
                    break
                take_rows = min(rg_rows, remaining)

            if take_rows > 0:
                items.append({
                    "path": path,
                    "row_group": rg_idx,
                    "global_start": global_start,
                    "take_rows": take_rows,
                    "rg_rows": rg_rows,
                })
                global_start += take_rows

            file_seen += rg_rows

    return items, global_start


class ParquetBERIterableDataset(IterableDataset):
    """
    Streams Parquet row groups, cleans each batch, computes features, applies
    pretrained scalers, and yields already-batched tensors.

    DataLoader must use batch_size=None.
    """

    def __init__(
        self,
        parquet_paths,
        scalers,
        tap_cols,
        var_cols,
        split,
        batch_rows=8192,
        max_rows_per_file=None,
        split_seed=42,
    ):
        super().__init__()
        self.parquet_paths = list(parquet_paths)
        self.scalers = scalers
        self.tap_cols = list(tap_cols)
        self.var_cols = list(var_cols)
        self.split = split
        self.batch_rows = int(batch_rows)
        self.max_rows_per_file = max_rows_per_file
        self.split_seed = int(split_seed)

        self.row_groups, self.total_indexed_rows = build_parquet_row_group_index(
            self.parquet_paths,
            max_rows_per_file=max_rows_per_file,
        )

        self.schema_names_by_path = {
            path: _parquet_schema_names(path)
            for path in self.parquet_paths
        }

    def __iter__(self):
        worker = get_worker_info()
        if worker is None:
            worker_id = 0
            num_workers = 1
        else:
            worker_id = worker.id
            num_workers = worker.num_workers

        for rg_global_idx, item in enumerate(self.row_groups):
            if rg_global_idx % num_workers != worker_id:
                continue

            path = item["path"]
            rg_idx = item["row_group"]
            global_start = item["global_start"]
            take_rows = item["take_rows"]

            schema_names = self.schema_names_by_path[path]
            wanted_cols = list(dict.fromkeys(
                self.tap_cols
                + self.var_cols
                + ["mem_len", "N", "threshold", "BER", "source_dataset", "_has_vars"]
            ))
            read_cols = [c for c in wanted_cols if c in schema_names]

            pf = pq.ParquetFile(path)
            batch_offset = 0

            for record_batch in pf.iter_batches(
                batch_size=self.batch_rows,
                row_groups=[rg_idx],
                columns=read_cols,
            ):
                df = record_batch.to_pandas()
                n = len(df)

                local_in_group = np.arange(batch_offset, batch_offset + n)
                allowed = local_in_group < take_rows

                if not np.any(allowed):
                    break

                if not np.all(allowed):
                    df = df.iloc[allowed].copy()
                    local_in_group = local_in_group[allowed]
                    n = len(df)

                row_ids = global_start + local_in_group.astype(np.int64)
                batch_offset += len(record_batch)

                split_keep = split_keep_mask(row_ids, self.split, seed=self.split_seed)
                if not np.any(split_keep):
                    continue

                df = df.iloc[split_keep].copy()
                tensors = self._prepare_tensor_batch(df)
                if tensors is not None:
                    yield tensors

    def _prepare_tensor_batch(self, df):
        for col in self.tap_cols:
            if col not in df.columns:
                df[col] = np.nan

        for required in ["N", "threshold", "BER"]:
            if required not in df.columns:
                raise ValueError(f"Missing required column {required!r} in Parquet batch.")

        if "mem_len" not in df.columns:
            df["mem_len"] = df[self.tap_cols].notna().sum(axis=1)

        numeric_cols = list(dict.fromkeys(
            self.tap_cols + self.var_cols + ["mem_len", "N", "threshold", "BER"]
        ))
        for col in numeric_cols:
            if col in df.columns:
                df[col] = pd.to_numeric(df[col], errors="coerce")

        df = df[df["mem_len"].notna()].copy()
        if len(df) == 0:
            return None

        df["mem_len"] = df["mem_len"].astype(np.int64)
        df["threshold"] = df["threshold"].fillna(0.0)
        df["BER"] = df["BER"].fillna(0.0)
        df["N"] = df["N"].fillna(0.0)

        df = df[
            (df["mem_len"] > 1)
            & (df["BER"] > 0)
            & (df["N"] > 0)
            & (df["threshold"] > 0)
        ].copy()
        if len(df) == 0:
            return None

        df["BER"] = df["BER"].clip(lower=EPS, upper=0.5)

        X_taps_raw = df[self.tap_cols].to_numpy(dtype=np.float32)
        num_molecules = df["N"].to_numpy(dtype=np.float32).reshape(-1, 1)
        mem_len = df["mem_len"].to_numpy(dtype=np.int64)
        X_thr_raw = df["threshold"].to_numpy(dtype=np.float32).reshape(-1, 1)
        y_raw = df["BER"].to_numpy(dtype=np.float32).reshape(-1, 1)
        y_raw = np.clip(y_raw, EPS, 0.5).astype(np.float32)
        y_log = np.log10(y_raw + EPS).astype(np.float32)

        L_full = X_taps_raw.shape[1]
        L_past = L_full - 1

        mem_len_clamped = np.minimum(mem_len, L_full)
        valid_full = np.arange(L_full)[None, :] < mem_len_clamped[:, None]
        valid_past = valid_full[:, 1:]
        valid_past_float = valid_past.astype(np.float32)
        past_lens = np.maximum(mem_len_clamped - 1, 0).astype(np.int64)

        bad_valid_tap = (~np.isfinite(X_taps_raw)) & valid_full
        keep = ~bad_valid_tap.any(axis=1)
        if not np.any(keep):
            return None

        if not np.all(keep):
            X_taps_raw = X_taps_raw[keep]
            num_molecules = num_molecules[keep]
            X_thr_raw = X_thr_raw[keep]
            y_raw = y_raw[keep]
            y_log = y_log[keep]
            valid_full = valid_full[keep]
            valid_past = valid_past[keep]
            valid_past_float = valid_past_float[keep]
            past_lens = past_lens[keep]
            df = df.iloc[keep].copy()

        X_taps_raw = np.nan_to_num(
            X_taps_raw,
            nan=0.0,
            posinf=0.0,
            neginf=0.0,
        ).astype(np.float32)

        valid_full_float = valid_full.astype(np.float32)

        X_vars_computed = compute_variances_from_taps(
            X_taps_raw,
            num_molecules.reshape(-1),
            valid_full,
        )

        X_vars_csv = np.full_like(X_vars_computed, np.nan, dtype=np.float32)
        for j, var_col in enumerate(self.var_cols):
            if var_col in df.columns and j < X_vars_csv.shape[1]:
                X_vars_csv[:, j] = pd.to_numeric(
                    df[var_col],
                    errors="coerce",
                ).to_numpy(dtype=np.float32)

        X_vars_raw = np.where(
            np.isfinite(X_vars_csv),
            X_vars_csv,
            X_vars_computed,
        ).astype(np.float32)

        X_vars_raw = np.maximum(X_vars_raw, 0.0)
        X_vars_raw = X_vars_raw * valid_full_float

        # Feature engineering: matches original normalized-threshold model.
        X_taps_feat = (X_taps_raw * num_molecules).astype(np.float32)
        X_vars_feat = X_vars_raw.astype(np.float32)
        abs_taps_raw = np.abs(X_taps_feat).astype(np.float32)

        snr_raw = np.log10(
            (X_taps_feat ** 2) / (X_vars_raw + EPS) + EPS
        ).astype(np.float32)
        snr_raw = sanitize_np_feature(snr_raw, "snr_raw")

        threshold_mode = self.scalers.get(
            "threshold_feature_mode",
            THRESHOLD_FEATURE_MODE,
        )

        total_scaled_taps_raw = np.sum(
            X_taps_feat * valid_full_float,
            axis=1,
            keepdims=True,
        ).astype(np.float32)

        X_thr = compute_threshold_feature_np(
            X_thr_raw,
            total_scaled_taps_raw,
            mode=threshold_mode,
        )

        first_mean = X_taps_feat[:, 0:1]
        past_means = X_taps_feat[:, 1:]
        first_var = X_vars_feat[:, 0:1]
        past_vars = X_vars_feat[:, 1:]

        past_means_masked = past_means * valid_past_float
        past_vars_masked = past_vars * valid_past_float

        mu0_raw = (0.5 * np.sum(past_means_masked, axis=1, keepdims=True)).astype(np.float32)
        mu1_raw = (first_mean + mu0_raw).astype(np.float32)
        var0_raw = (0.5 * np.sum(past_vars_masked, axis=1, keepdims=True)).astype(np.float32)
        var1_raw = (first_var + var0_raw).astype(np.float32)

        std0_raw = np.sqrt(np.maximum(var0_raw, EPS)).astype(np.float32)
        std1_raw = np.sqrt(np.maximum(var1_raw, EPS)).astype(np.float32)

        z0_raw = ((X_thr_raw - mu0_raw) / (std0_raw + EPS)).astype(np.float32)
        z1_raw = ((mu1_raw - X_thr_raw) / (std1_raw + EPS)).astype(np.float32)

        z0_raw = sanitize_np_feature(z0_raw, "z0_raw")
        z1_raw = sanitize_np_feature(z1_raw, "z1_raw")

        harmonic_side_z_raw = stable_harmonic_pair_np(z0_raw, z1_raw)
        abs_diff_side_z_raw = np.abs(z0_raw - z1_raw).astype(np.float32)
        harmonic_minus_gap_raw = sanitize_np_feature(
            harmonic_side_z_raw - 0.25 * abs_diff_side_z_raw,
            "harmonic_minus_gap_raw",
        )

        signal_raw = first_mean
        isi_raw = np.sum(past_means_masked, axis=1, keepdims=True)
        nsid_raw = ((signal_raw - isi_raw) / (signal_raw + isi_raw + EPS)).astype(np.float32)

        gap_raw = signal_raw
        d0_raw = (gap_raw / (std0_raw + EPS)).astype(np.float32)
        d1_raw = (gap_raw / (std1_raw + EPS)).astype(np.float32)
        harmonic_discrim_raw = (
            2.0 * d0_raw * d1_raw / (d0_raw + d1_raw + EPS)
        ).astype(np.float32)
        log_harmonic_discrim_raw = np.log10(
            harmonic_discrim_raw + EPS
        ).astype(np.float32)
        discrim_asymmetry_raw = (
            np.abs(d0_raw - d1_raw) / (d0_raw + d1_raw + EPS)
        ).astype(np.float32)

        past_taps_sum = np.sum(past_means_masked, axis=1, keepdims=True)
        past_shares = past_means_masked / (past_taps_sum + EPS)
        herfindahl_raw = np.sum(past_shares ** 2, axis=1, keepdims=True).astype(np.float32)

        nsid_raw = sanitize_np_feature(nsid_raw, "nsid_raw")
        log_harmonic_discrim_raw = sanitize_np_feature(
            log_harmonic_discrim_raw,
            "log_harmonic_discrim_raw",
        )
        discrim_asymmetry_raw = sanitize_np_feature(
            discrim_asymmetry_raw,
            "discrim_asymmetry_raw",
        )
        herfindahl_raw = sanitize_np_feature(herfindahl_raw, "herfindahl_raw")

        region_labels = raw_to_region_labels_np(y_raw)
        ordinal_targets = region_labels_to_ordinal_targets_np(region_labels)

        first_token, past_tokens = self._build_tokens(
            X_taps_feat,
            X_vars_feat,
            abs_taps_raw,
            snr_raw,
            past_lens,
            L_past,
        )

        global_feats = self._build_global(
            z0_raw,
            z1_raw,
            harmonic_minus_gap_raw,
            nsid_raw,
            log_harmonic_discrim_raw,
            discrim_asymmetry_raw,
            herfindahl_raw,
        )

        thr_s = self.scalers["thr_scaler"].transform(X_thr).astype(np.float32)
        padding_mask = np.arange(L_past)[None, :] >= past_lens[:, None]

        return (
            torch.from_numpy(first_token.astype(np.float32)),
            torch.from_numpy(past_tokens.astype(np.float32)),
            torch.from_numpy(global_feats.astype(np.float32)),
            torch.from_numpy(thr_s.astype(np.float32)),
            torch.from_numpy(y_log.astype(np.float32)),
            torch.from_numpy(ordinal_targets.astype(np.float32)),
            torch.from_numpy(region_labels.astype(np.int64)),
            torch.from_numpy(padding_mask),
        )

    def _build_tokens(self, taps_feat, vars_feat, abs_raw, snr_raw, past_lens, L_past):
        s = self.scalers

        ft_tap = ((taps_feat[:, 0] - s["first_tap_mean"]) / s["first_tap_std"]).astype(np.float32)
        ft_var = ((vars_feat[:, 0] - s["first_var_mean"]) / s["first_var_std"]).astype(np.float32)
        ft_abs = ((abs_raw[:, 0] - s["first_abs_mean"]) / s["first_abs_std"]).astype(np.float32)
        ft_snr = ((snr_raw[:, 0] - s["first_snr_mean"]) / s["first_snr_std"]).astype(np.float32)

        first_token = np.stack(
            [ft_tap, ft_var, ft_abs, ft_snr],
            axis=1,
        ).astype(np.float32)

        pt_valid = np.arange(L_past)[None, :] < past_lens[:, None]

        pt_tap = apply_shared_scale(
            taps_feat[:, 1:],
            s["past_tap_mean"],
            s["past_tap_std"],
            pt_valid,
        )
        pt_var = apply_shared_scale(
            vars_feat[:, 1:],
            s["past_var_mean"],
            s["past_var_std"],
            pt_valid,
        )
        pt_abs = apply_shared_scale(
            abs_raw[:, 1:],
            s["past_abs_mean"],
            s["past_abs_std"],
            pt_valid,
        )
        pt_snr = apply_shared_scale(
            snr_raw[:, 1:],
            s["past_snr_mean"],
            s["past_snr_std"],
            pt_valid,
        )

        past_tokens = np.stack(
            [pt_tap, pt_var, pt_abs, pt_snr],
            axis=2,
        ).astype(np.float32)

        return first_token, past_tokens

    def _build_global(self, z0, z1, hmg, nsid, log_hd, da, herf):
        s = self.scalers
        return np.concatenate([
            s["z0_scaler"].transform(z0),
            s["z1_scaler"].transform(z1),
            s["harmonic_minus_gap_scaler"].transform(hmg),
            s["nsid_scaler"].transform(nsid),
            s["log_hd_scaler"].transform(log_hd),
            s["da_scaler"].transform(da),
            s["herf_scaler"].transform(herf),
        ], axis=1).astype(np.float32)


def prepare_data_for_finetune(
    parquet_paths,
    batch_size=256,
    nrows_per_dataset=None,
    num_workers=0,
    pretrained_scalers=None,
    reuse_scalers=True,
):
    """
    Memory-safe Parquet loader.

    Returns:
      train_loader, val_loader, test_loader, scalers, aux_info, L_past
    """
    del batch_size

    if not reuse_scalers:
        raise NotImplementedError(
            "Streaming scaler refit is intentionally disabled. Use REUSE_SCALERS=True."
        )

    if pretrained_scalers is None:
        raise ValueError("pretrained_scalers must be provided for streaming Parquet loading.")

    for path in parquet_paths:
        if not os.path.exists(path):
            raise FileNotFoundError(
                f"Parquet file not found: {path}\n"
                f"Set RUN_PARQUET_CONVERSION=True or run build_clean_parquet_files() first."
            )

    tap_cols, var_cols = infer_parquet_sequence_columns(
        parquet_paths,
        pretrained_scalers=pretrained_scalers,
    )

    L_full = len(tap_cols)
    L_past = L_full - 1

    print("\nStreaming Parquet data:")
    for p in parquet_paths:
        print(f"  {p}")
    print(f"  tap columns: {L_full}")
    print(f"  past length: {L_past}")
    print(f"  rows/file cap: {nrows_per_dataset}")

    scalers = dict(pretrained_scalers)
    scalers.update({
        "tap_cols": tap_cols,
        "var_cols": var_cols,
        "train_max_past_seq_len": L_past,
        "data_paths": parquet_paths,
        "nrows_per_dataset": nrows_per_dataset,
        "finetuned_from": PRETRAINED_MODEL_PATH,
        "scalers_reused": True,
        "streaming_parquet": True,
        "variances_computed_from_taps": True,
    })

    train_ds = ParquetBERIterableDataset(
        parquet_paths=parquet_paths,
        scalers=scalers,
        tap_cols=tap_cols,
        var_cols=var_cols,
        split="train",
        batch_rows=PARQUET_BATCH_ROWS,
        max_rows_per_file=nrows_per_dataset,
        split_seed=42,
    )
    val_ds = ParquetBERIterableDataset(
        parquet_paths=parquet_paths,
        scalers=scalers,
        tap_cols=tap_cols,
        var_cols=var_cols,
        split="val",
        batch_rows=PARQUET_BATCH_ROWS,
        max_rows_per_file=nrows_per_dataset,
        split_seed=42,
    )
    test_ds = ParquetBERIterableDataset(
        parquet_paths=parquet_paths,
        scalers=scalers,
        tap_cols=tap_cols,
        var_cols=var_cols,
        split="test",
        batch_rows=PARQUET_BATCH_ROWS,
        max_rows_per_file=nrows_per_dataset,
        split_seed=42,
    )

    pin_mem = torch.cuda.is_available()

    train_loader = DataLoader(
        train_ds,
        batch_size=None,
        num_workers=num_workers,
        pin_memory=pin_mem,
    )
    val_loader = DataLoader(
        val_ds,
        batch_size=None,
        num_workers=num_workers,
        pin_memory=pin_mem,
    )
    test_loader = DataLoader(
        test_ds,
        batch_size=None,
        num_workers=num_workers,
        pin_memory=pin_mem,
    )

    ordinal_pos_weights = np.asarray(
        pretrained_scalers.get(
            "ordinal_pos_weights",
            np.ones(NUM_ORDINAL_THRESHOLDS, dtype=np.float32),
        ),
        dtype=np.float32,
    )
    if ordinal_pos_weights.shape[0] != NUM_ORDINAL_THRESHOLDS:
        print("  Warning: pretrained ordinal_pos_weights length mismatch; using ones.")
        ordinal_pos_weights = np.ones(NUM_ORDINAL_THRESHOLDS, dtype=np.float32)

    class_weights = np.asarray(
        pretrained_scalers.get(
            "class_weights",
            np.ones(NUM_REGION_CLASSES, dtype=np.float32),
        ),
        dtype=np.float32,
    )
    if class_weights.shape[0] != NUM_REGION_CLASSES:
        class_weights = np.ones(NUM_REGION_CLASSES, dtype=np.float32)

    aux_info = {
        "class_weights": class_weights,
        "ordinal_pos_weights": ordinal_pos_weights,
    }

    return train_loader, val_loader, test_loader, scalers, aux_info, L_past


# =========================
# Loss
# =========================
class StableMultiObjectiveBERBoundedLogLoss(nn.Module):
    def __init__(self, log_delta=0.35, raw_delta=0.006, rel_delta=0.03,
                 alpha_log=0.45, beta_raw=0.35, gamma_rel=0.20,
                 use_regime_weights=True):
        super().__init__()
        self.log_delta = log_delta
        self.raw_delta = raw_delta
        self.rel_delta = rel_delta
        self.alpha_log = alpha_log
        self.beta_raw = beta_raw
        self.gamma_rel = gamma_rel
        self.use_regime_weights = use_regime_weights

    @staticmethod
    def huber_elementwise(pred, target, delta):
        err = pred - target
        abs_err = err.abs()
        return torch.where(abs_err < delta,
                           0.5 * err * err,
                           delta * (abs_err - 0.5 * delta))

    @staticmethod
    def raw_to_pred_log(pred_raw_unconstrained):
        log10_half = torch.log10(torch.tensor(
            0.5, device=pred_raw_unconstrained.device,
            dtype=pred_raw_unconstrained.dtype))
        return log10_half - F.softplus(pred_raw_unconstrained)

    @staticmethod
    def raw_to_pred_ber(pred_raw_unconstrained):
        pred_log = StableMultiObjectiveBERBoundedLogLoss.raw_to_pred_log(pred_raw_unconstrained)
        return torch.pow(10.0, pred_log).clamp(min=EPS, max=0.5)

    def forward(self, pred_raw_unconstrained, target_log):
        pred_log = self.raw_to_pred_log(pred_raw_unconstrained)
        pred_raw = torch.pow(10.0, pred_log).clamp(min=EPS, max=0.5)
        target_raw = torch.pow(10.0, target_log).clamp(min=EPS, max=0.5)
        target_log_for_loss = torch.log10(target_raw)

        log_loss = self.huber_elementwise(pred_log, target_log_for_loss, self.log_delta)
        raw_loss = self.huber_elementwise(pred_raw, target_raw, self.raw_delta)
        rel_err = (pred_raw - target_raw) / torch.clamp(target_raw, min=1e-6)
        rel_loss = self.huber_elementwise(rel_err, torch.zeros_like(rel_err), self.rel_delta)

        total = (self.alpha_log * log_loss
                 + self.beta_raw * raw_loss
                 + self.gamma_rel * rel_loss)

        if self.use_regime_weights:
            weights = torch.ones_like(target_raw)
            weights = torch.where((target_raw >= 1e-2) & (target_raw < 0.1),
                                  torch.full_like(weights, 1.5), weights)
            weights = torch.where((target_raw >= 0.1) & (target_raw < 0.15),
                                  torch.full_like(weights, 2.5), weights)
            weights = torch.where((target_raw >= 0.15) & (target_raw < 0.2),
                                  torch.full_like(weights, 3.0), weights)
            weights = torch.where((target_raw >= 0.2) & (target_raw < 0.3),
                                  torch.full_like(weights, 4.0), weights)
            weights = torch.where((target_raw >= 0.3) & (target_raw < 0.4),
                                  torch.full_like(weights, 4.5), weights)
            weights = torch.where((target_raw >= 0.4) & (target_raw < 0.45),
                                  torch.full_like(weights, 3.0), weights)
            weights = torch.where(target_raw >= 0.45,
                                  torch.full_like(weights, 2.5), weights)
            total = total * weights

        return total.mean()


class OrdinalBCELoss(nn.Module):
    def __init__(self, pos_weight=None, reduction="mean"):
        super().__init__()
        if pos_weight is not None and not isinstance(pos_weight, torch.Tensor):
            pos_weight = torch.tensor(pos_weight, dtype=torch.float32)
        self.register_buffer("pos_weight", pos_weight if pos_weight is not None else None)
        self.reduction = reduction

    def forward(self, logits, ordinal_targets):
        return F.binary_cross_entropy_with_logits(
            logits,
            ordinal_targets,
            pos_weight=self.pos_weight,
            reduction=self.reduction,
        )


class MultiTaskBERLoss(nn.Module):
    def __init__(self, reg_loss, ord_loss, lambda_ord=0.25):
        super().__init__()
        self.reg_loss = reg_loss
        self.ord_loss = ord_loss
        self.lambda_ord = lambda_ord

    def forward(self, pred_raw_unconstrained, ord_logits, target_log, target_ord):
        reg = self.reg_loss(pred_raw_unconstrained, target_log)
        ordl = self.ord_loss(ord_logits, target_ord)
        total = reg + self.lambda_ord * ordl
        return total, reg.detach(), ordl.detach()


# =========================
# Set Transformer Model
# =========================
class SetSelfAttentionBlock(nn.Module):
    def __init__(self, d_model, num_heads, mlp_ratio=4.0, dropout=0.1):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model)
        self.attn = nn.MultiheadAttention(
            embed_dim=d_model,
            num_heads=num_heads,
            dropout=dropout,
            batch_first=True,
        )
        self.norm2 = nn.LayerNorm(d_model)
        hidden = int(d_model * mlp_ratio)
        self.mlp = nn.Sequential(
            nn.Linear(d_model, hidden), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(hidden, d_model), nn.Dropout(dropout),
        )

    def forward(self, x, key_padding_mask=None):
        y = self.norm1(x)
        attn_out, _ = self.attn(
            y,
            y,
            y,
            need_weights=False,
            key_padding_mask=key_padding_mask,
        )
        x = x + attn_out
        x = x + self.mlp(self.norm2(x))
        return x


class ConditionalFeatureModulation(nn.Module):
    def __init__(self, cond_dim, feat_dim, hidden_dim=256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(cond_dim, hidden_dim), nn.GELU(),
            nn.Linear(hidden_dim, 2 * feat_dim),
        )
        self.scale = nn.Parameter(torch.tensor(0.25, dtype=torch.float32))

    def forward(self, x, cond):
        gamma_beta = self.net(cond)
        gamma, beta = torch.chunk(gamma_beta, 2, dim=-1)
        if x.dim() == 3:
            gamma = gamma.unsqueeze(1)
            beta = beta.unsqueeze(1)
        return x * (1.0 + self.scale * gamma) + self.scale * beta


class SetAttentionPooling(nn.Module):
    def __init__(self, d_model, hidden_dim=128):
        super().__init__()
        self.score = nn.Sequential(
            nn.Linear(d_model, hidden_dim), nn.GELU(), nn.Linear(hidden_dim, 1),
        )

    def forward(self, x, key_padding_mask=None):
        logits = self.score(x)
        if key_padding_mask is not None:
            logits = logits.masked_fill(key_padding_mask.unsqueeze(-1), float("-inf"))
        weights = torch.softmax(logits, dim=1)
        weights = torch.nan_to_num(weights, nan=0.0)
        return (weights * x).sum(dim=1)


class MonotonicOrdinalHead(nn.Module):
    def __init__(self, head_in, num_thresholds):
        super().__init__()
        self.num_thresholds = num_thresholds
        self.base = nn.Sequential(
            nn.Linear(head_in, 256), nn.GELU(), nn.Dropout(0.20),
            nn.Linear(256, 128), nn.GELU(), nn.Dropout(0.15),
        )
        self.first_logit = nn.Linear(128, 1)
        self.deltas = nn.Linear(128, max(num_thresholds - 1, 1))

    def forward(self, x):
        h = self.base(x)
        first = self.first_logit(h)
        if self.num_thresholds == 1:
            return first
        deltas = -F.softplus(self.deltas(h)[:, : self.num_thresholds - 1])
        return torch.cat([first, deltas], dim=-1).cumsum(dim=-1)


class BERFirstTokenSetTransformerBoundedLogHalfMultiTask(nn.Module):
    def __init__(self, max_past_seq_len, token_dim=4, first_token_dim=4,
                 threshold_dim=1, global_dim=7, d_model=128, num_set_layers=4,
                 num_heads=4, mlp_ratio=4.0, dropout=0.10,
                 num_ordinal_thresholds=NUM_ORDINAL_THRESHOLDS):
        super().__init__()
        self.max_past_seq_len = max_past_seq_len
        self.d_model = d_model
        self.num_ordinal_thresholds = num_ordinal_thresholds

        self.first_proj = nn.Sequential(
            nn.Linear(first_token_dim, d_model), nn.LayerNorm(d_model),
            nn.GELU(), nn.Dropout(dropout),
        )
        self.token_proj = nn.Sequential(
            nn.Linear(token_dim, d_model), nn.LayerNorm(d_model),
            nn.GELU(), nn.Dropout(dropout),
        )
        self.thr_embed = nn.Sequential(
            nn.Linear(threshold_dim, 32), nn.GELU(),
            nn.Linear(32, 32), nn.GELU(),
        )
        self.global_embed = nn.Sequential(
            nn.Linear(global_dim, 96), nn.GELU(),
            nn.Linear(96, 96), nn.GELU(),
        )

        cond_dim = 32 + 96
        self.first_cond_mod = ConditionalFeatureModulation(cond_dim, d_model, 256)
        self.set_cond_mod = ConditionalFeatureModulation(cond_dim, d_model, 256)

        self.set_blocks = nn.ModuleList([
            SetSelfAttentionBlock(d_model, num_heads, mlp_ratio, dropout)
            for _ in range(num_set_layers)
        ])

        self.first_refine = nn.Sequential(
            nn.LayerNorm(d_model), nn.Linear(d_model, d_model),
            nn.GELU(), nn.Dropout(dropout), nn.Linear(d_model, d_model),
        )
        self.final_set_norm = nn.LayerNorm(d_model)
        self.set_attn_pool = SetAttentionPooling(d_model, 128)

        head_in = d_model + 3 * d_model + cond_dim
        self.reg_head = nn.Sequential(
            nn.Linear(head_in, 256), nn.GELU(), nn.Dropout(0.20),
            nn.Linear(256, 128), nn.GELU(), nn.Dropout(0.15), nn.Linear(128, 1),
        )
        self.ord_head = MonotonicOrdinalHead(head_in, num_ordinal_thresholds)

    @staticmethod
    def raw_to_log10ber(x):
        log10_half = torch.log10(torch.tensor(0.5, device=x.device, dtype=x.dtype))
        return log10_half - F.softplus(x)

    @staticmethod
    def raw_to_ber(x):
        return torch.pow(
            10.0,
            BERFirstTokenSetTransformerBoundedLogHalfMultiTask.raw_to_log10ber(x),
        ).clamp(EPS, 0.5)

    def forward(self, first_token, past_tokens, global_feats, threshold, key_padding_mask=None):
        thr = self.thr_embed(threshold)
        g = self.global_embed(global_feats)
        cond = torch.cat([thr, g], dim=-1)

        first_x = self.first_proj(first_token)
        first_x = self.first_cond_mod(first_x, cond)
        first_x = first_x + self.first_refine(first_x)

        set_x = self.token_proj(past_tokens)
        set_x = self.set_cond_mod(set_x, cond)
        for block in self.set_blocks:
            set_x = block(set_x, key_padding_mask=key_padding_mask)
        set_x = self.final_set_norm(set_x)

        if key_padding_mask is not None:
            valid_mask = (~key_padding_mask).unsqueeze(-1).float()
        else:
            valid_mask = torch.ones(
                set_x.shape[0],
                set_x.shape[1],
                1,
                device=set_x.device,
                dtype=set_x.dtype,
            )

        pooled_attn = self.set_attn_pool(set_x, key_padding_mask=key_padding_mask)
        valid_counts = valid_mask.sum(dim=1).clamp(min=1.0)
        pooled_mean = (set_x * valid_mask).sum(dim=1) / valid_counts

        if key_padding_mask is not None:
            x_for_max = set_x.masked_fill(key_padding_mask.unsqueeze(-1), float("-inf"))
        else:
            x_for_max = set_x
        pooled_max = torch.nan_to_num(
            x_for_max.amax(dim=1),
            nan=0.0,
            posinf=0.0,
            neginf=0.0,
        )

        fused = torch.cat([first_x, pooled_attn, pooled_mean, pooled_max, cond], dim=-1)
        return self.reg_head(fused), self.ord_head(fused)


# =========================
# Evaluation
# =========================
def evaluate(model, loader, criterion, device):
    model.eval()

    total_loss = 0.0
    total_reg_loss = 0.0
    total_ord_loss = 0.0
    n_batches = 0

    all_preds_log = []
    all_targets_log = []
    all_pred_regions = []
    all_true_regions = []

    with torch.no_grad():
        for b_first, b_past, b_global, b_thr, b_y_log, b_ord, b_region, b_mask in loader:
            b_first = b_first.to(device, non_blocking=True)
            b_past = b_past.to(device, non_blocking=True)
            b_global = b_global.to(device, non_blocking=True)
            b_thr = b_thr.to(device, non_blocking=True)
            b_y_log = b_y_log.to(device, non_blocking=True)
            b_ord = b_ord.to(device, non_blocking=True)
            b_region = b_region.to(device, non_blocking=True)
            b_mask = b_mask.to(device, non_blocking=True)

            pred_raw_unconstrained, ord_logits = model(
                b_first,
                b_past,
                b_global,
                b_thr,
                key_padding_mask=b_mask,
            )

            if has_nonfinite_tensor(pred_raw_unconstrained):
                raise RuntimeError("Non-finite prediction during evaluation.")
            if has_nonfinite_tensor(ord_logits):
                raise RuntimeError("Non-finite ordinal logits during evaluation.")

            loss, reg_loss, ord_loss = criterion(
                pred_raw_unconstrained,
                ord_logits,
                b_y_log,
                b_ord,
            )

            if has_nonfinite_tensor(loss):
                raise RuntimeError("Non-finite loss during evaluation.")

            pred_log = model.raw_to_log10ber(pred_raw_unconstrained)
            pred_region = ordinal_logits_to_region_labels_torch(ord_logits)

            total_loss += loss.item()
            total_reg_loss += reg_loss.item()
            total_ord_loss += ord_loss.item()
            n_batches += 1

            all_preds_log.append(pred_log.cpu().numpy())
            all_targets_log.append(b_y_log.cpu().numpy())
            all_pred_regions.append(pred_region.cpu().numpy())
            all_true_regions.append(b_region.cpu().numpy())

    if n_batches == 0:
        raise RuntimeError("Evaluation loader produced zero batches.")

    preds_log = np.vstack(all_preds_log)
    targets_log = np.vstack(all_targets_log)

    preds_raw = np.clip(10 ** preds_log, EPS, 0.5)
    targets_raw = np.clip(10 ** targets_log, EPS, 0.5)

    rmse_log = float(np.sqrt(np.mean((preds_log - targets_log) ** 2)))
    mae_log = float(np.mean(np.abs(preds_log - targets_log)))
    factor_error = float(10 ** rmse_log)

    rmse_raw = float(np.sqrt(np.mean((preds_raw - targets_raw) ** 2)))
    mae_raw = float(np.mean(np.abs(preds_raw - targets_raw)))

    rel_err = (preds_raw - targets_raw) / np.maximum(targets_raw, 1e-6)
    rmse_rel = float(np.sqrt(np.mean(rel_err ** 2)))
    mae_rel = float(np.mean(np.abs(rel_err)))

    pred_regions = np.concatenate(all_pred_regions).reshape(-1)
    true_regions = np.concatenate(all_true_regions).reshape(-1)

    region_acc = float(np.mean(pred_regions == true_regions))
    region_mae = float(np.mean(np.abs(pred_regions - true_regions)))

    return {
        "loss": total_loss / n_batches,
        "reg_loss": total_reg_loss / n_batches,
        "ord_loss": total_ord_loss / n_batches,
        "rmse_log": rmse_log,
        "mae_log": mae_log,
        "factor_error": factor_error,
        "rmse_raw": rmse_raw,
        "mae_raw": mae_raw,
        "rmse_rel": rmse_rel,
        "mae_rel": mae_rel,
        "region_acc": region_acc,
        "region_mae": region_mae,
    }


def evaluate_by_target_range(model, loader, device):
    model.eval()
    all_preds_log = []
    all_targets_log = []

    with torch.no_grad():
        for b_first, b_past, b_global, b_thr, b_y_log, _, _, b_mask in loader:
            b_first = b_first.to(device, non_blocking=True)
            b_past = b_past.to(device, non_blocking=True)
            b_global = b_global.to(device, non_blocking=True)
            b_thr = b_thr.to(device, non_blocking=True)
            b_mask = b_mask.to(device, non_blocking=True)

            pred_raw_unconstrained, _ = model(
                b_first,
                b_past,
                b_global,
                b_thr,
                key_padding_mask=b_mask,
            )
            if has_nonfinite_tensor(pred_raw_unconstrained):
                raise RuntimeError("Non-finite prediction during per-range evaluation.")
            pred_log = model.raw_to_log10ber(pred_raw_unconstrained)
            all_preds_log.append(pred_log.cpu().numpy())
            all_targets_log.append(b_y_log.numpy())

    if not all_preds_log:
        raise RuntimeError("Range evaluation loader produced zero batches.")

    preds_log = np.vstack(all_preds_log).reshape(-1)
    targets_log = np.vstack(all_targets_log).reshape(-1)
    preds_raw = np.clip(10 ** preds_log, EPS, 0.5)
    targets_raw = np.clip(10 ** targets_log, EPS, 0.5)

    ranges = {
        "low_BER(y<1e-6)": targets_raw < 1e-6,
        "mid_BER(1e-6<=y<1e-3)": (targets_raw >= 1e-6) & (targets_raw < 1e-3),
        "high_BER(1e-3<=y<0.10)": (targets_raw >= 1e-3) & (targets_raw < 0.10),
        "upper_BER(0.10<=y<0.20)": (targets_raw >= 0.10) & (targets_raw < 0.20),
        "target_BER(0.20<=y<0.40)": (targets_raw >= 0.20) & (targets_raw < 0.40),
        "very_high_BER(0.40<=y<=0.50)": targets_raw >= 0.40,
    }

    metrics = {}
    for name, mask in ranges.items():
        if np.any(mask):
            err_log = preds_log[mask] - targets_log[mask]
            err_raw = preds_raw[mask] - targets_raw[mask]
            rel_err = err_raw / np.maximum(targets_raw[mask], 1e-6)
            metrics[name] = {
                "count": int(mask.sum()),
                "rmse_log": float(np.sqrt(np.mean(err_log ** 2))),
                "mae_log": float(np.mean(np.abs(err_log))),
                "factor_error": float(10 ** float(np.sqrt(np.mean(err_log ** 2)))),
                "rmse_raw": float(np.sqrt(np.mean(err_raw ** 2))),
                "mae_raw": float(np.mean(np.abs(err_raw))),
                "rmse_rel": float(np.sqrt(np.mean(rel_err ** 2))),
                "mae_rel": float(np.mean(np.abs(rel_err))),
                "bias_raw": float(np.mean(err_raw)),
                "bias_log": float(np.mean(err_log)),
                "p90_abs_raw": float(np.percentile(np.abs(err_raw), 90)),
                "p95_abs_raw": float(np.percentile(np.abs(err_raw), 95)),
            }
        else:
            metrics[name] = None
    return metrics


# =========================
# Selection scoring
# =========================
SELECTION_REGION_WEIGHTS = {
    "low_BER(y<1e-6)": 0.25,
    "mid_BER(1e-6<=y<1e-3)": 0.50,
    "high_BER(1e-3<=y<0.10)": 1.00,
    "upper_BER(0.10<=y<0.20)": 2.00,
    "target_BER(0.20<=y<0.40)": 3.00,
    "very_high_BER(0.40<=y<=0.50)": 1.50,
}
SELECTION_W_LOG_RMSE = 1.00
SELECTION_W_BIAS = 0.50
SELECTION_W_TAIL = 0.05
SELECTION_TAIL_CAP = 5.0


def compute_selection_score(val_range_metrics, val_metrics, min_count=50):
    log_rmse_sum = 0.0
    bias_sum = 0.0
    tail_sum = 0.0
    total_w = 0.0

    for region_name, w in SELECTION_REGION_WEIGHTS.items():
        m = val_range_metrics.get(region_name)
        if m is None or m["count"] < min_count:
            continue
        log_rmse_sum += w * m["rmse_log"]
        bias_sum += w * abs(m["bias_log"])
        if m["rmse_raw"] > 1e-9:
            tail_ratio = m["p95_abs_raw"] / (m["rmse_raw"] + 1e-9)
            tail_sum += w * min(tail_ratio, SELECTION_TAIL_CAP)
        else:
            tail_sum += w * 1.0
        total_w += w

    if total_w == 0:
        return {
            "composite": float(val_metrics["rmse_log"]),
            "log_rmse_weighted": float(val_metrics["rmse_log"]),
            "bias_weighted": 0.0,
            "tail_weighted": 0.0,
            "geometric_factor_error": float(val_metrics["factor_error"]),
            "fallback": True,
        }

    log_rmse_weighted = log_rmse_sum / total_w
    bias_weighted = bias_sum / total_w
    tail_weighted = tail_sum / total_w
    composite = (SELECTION_W_LOG_RMSE * log_rmse_weighted
                 + SELECTION_W_BIAS * bias_weighted
                 + SELECTION_W_TAIL * tail_weighted)
    return {
        "composite": float(composite),
        "log_rmse_weighted": float(log_rmse_weighted),
        "bias_weighted": float(bias_weighted),
        "tail_weighted": float(tail_weighted),
        "geometric_factor_error": float(10 ** log_rmse_weighted),
        "fallback": False,
    }


def is_acceptable_checkpoint(val_range_metrics, val_metrics):
    target = val_range_metrics.get("target_BER(0.20<=y<0.40)")
    if target is None or target["count"] < 100:
        return False, "target region too small"
    for region_name, m in val_range_metrics.items():
        if m is None or m["count"] < 50:
            continue
        if abs(m["bias_log"]) > 0.15:
            return False, f"{region_name} has bias_log={m['bias_log']:.3f}"
    if val_metrics["rmse_log"] > 0.30:
        return False, f"overall rmse_log={val_metrics['rmse_log']:.3f} too high"
    return True, "ok"


# =========================
# Fine-tune main
# =========================
def finetune():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Executing on: {device}")
    print(f"\nFINE-TUNING from: {PRETRAINED_MODEL_PATH}")
    print(f"  Pretrained scalers: {PRETRAINED_SCALER_PATH}")
    print(f"  Reuse scalers: {REUSE_SCALERS}")

    if not os.path.exists(PRETRAINED_MODEL_PATH):
        raise FileNotFoundError(f"Pretrained model not found: {PRETRAINED_MODEL_PATH}")
    if not os.path.exists(PRETRAINED_SCALER_PATH):
        raise FileNotFoundError(f"Pretrained scalers not found: {PRETRAINED_SCALER_PATH}")

    pretrained_scalers = joblib.load(PRETRAINED_SCALER_PATH)
    print(f"  Pretrained global_dim: {pretrained_scalers.get('global_dim', '?')}")
    print(f"  Pretrained threshold_feature_mode: {pretrained_scalers.get('threshold_feature_mode', THRESHOLD_FEATURE_MODE)}")

    train_loader, val_loader, test_loader, scalers, aux_info, max_past_seq_len = (
        prepare_data_for_finetune(
            DATA_PATHS,
            batch_size=256,
            nrows_per_dataset=NROWS_PER_DATASET,
            num_workers=DATALOADER_NUM_WORKERS,
            pretrained_scalers=pretrained_scalers,
            reuse_scalers=REUSE_SCALERS,
        )
    )

    joblib.dump(scalers, SCALER_SAVE_PATH)
    print(f"\nFine-tune scalers saved to: {SCALER_SAVE_PATH}")

    model = BERFirstTokenSetTransformerBoundedLogHalfMultiTask(
        max_past_seq_len=max_past_seq_len,
        token_dim=4,
        first_token_dim=4,
        threshold_dim=1,
        global_dim=7,
        d_model=128,
        num_set_layers=4,
        num_heads=4,
        mlp_ratio=4.0,
        dropout=0.10,
        num_ordinal_thresholds=NUM_ORDINAL_THRESHOLDS,
    ).to(device)

    pretrained_state = safe_torch_load_state_dict(PRETRAINED_MODEL_PATH, map_location=device)
    model.load_state_dict(pretrained_state, strict=True)
    print("Pretrained normalized-threshold + monotonic-ordinal weights loaded with strict=True.")

    print("\n--- Pre-fine-tune evaluation (pretrained model on new data) ---")
    pre_criterion = MultiTaskBERLoss(
        StableMultiObjectiveBERBoundedLogLoss(),
        OrdinalBCELoss(),
        lambda_ord=0.25,
    )

    try:
        pre_val = evaluate(model, val_loader, pre_criterion, device)
        pre_range = evaluate_by_target_range(model, val_loader, device)
        pre_sel = compute_selection_score(pre_range, pre_val)
        print(
            f"  Pretrained on new val: composite={pre_sel['composite']:.5f} | "
            f"factor~{pre_sel['geometric_factor_error']:.3f}x | "
            f"RMSE(log10)={pre_val['rmse_log']:.4f} | "
            f"region_acc={pre_val['region_acc']:.4f}"
        )
        for rng, m in pre_range.items():
            if m is not None:
                print(
                    f"    {rng}: factor~{m['factor_error']:.3f}x, "
                    f"bias_log={m['bias_log']:+.4f}, count={m['count']}"
                )
            else:
                print(f"    {rng}: no samples")
    except Exception as e:
        print(f"  Pre-fine-tune evaluation failed/skipped: {e}")
        pre_val = {"rmse_log": float("inf"), "factor_error": float("inf")}
        pre_range = {}
        pre_sel = {
            "composite": float("inf"),
            "log_rmse_weighted": float("inf"),
            "bias_weighted": float("inf"),
        }

    print("\nFine-tune hyperparameters:")
    print(f"  LR={FINETUNE_LR}, weight_decay={FINETUNE_WEIGHT_DECAY}")
    print(
        f"  max_epochs={FINETUNE_MAX_EPOCHS}, min_epochs={FINETUNE_MIN_EPOCHS}, "
        f"patience={FINETUNE_PATIENCE}"
    )

    optimizer = optim.AdamW(
        model.parameters(),
        lr=FINETUNE_LR,
        weight_decay=FINETUNE_WEIGHT_DECAY,
    )

    reg_loss = StableMultiObjectiveBERBoundedLogLoss(
        log_delta=0.35,
        raw_delta=0.006,
        rel_delta=0.03,
        alpha_log=0.45,
        beta_raw=0.35,
        gamma_rel=0.20,
        use_regime_weights=True,
    )

    boosted_pos = aux_info["ordinal_pos_weights"].copy()
    for i, thr in enumerate(ORDINAL_THRESHOLDS):
        if 0.20 <= thr <= 0.40:
            boosted_pos[i] *= 2.5
        elif 0.15 <= thr < 0.20:
            boosted_pos[i] *= 1.5
        elif 0.40 < thr <= 0.45:
            boosted_pos[i] *= 1.5

    ord_loss = OrdinalBCELoss(
        pos_weight=torch.tensor(boosted_pos, dtype=torch.float32, device=device),
        reduction="mean",
    )
    criterion = MultiTaskBERLoss(reg_loss=reg_loss, ord_loss=ord_loss, lambda_ord=0.25)

    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="min",
        patience=SCHEDULER_PATIENCE,
        factor=SCHEDULER_FACTOR,
    )

    best_checkpoints = {
        "composite": {"score": float("inf"), "state": None, "epoch": -1},
        "composite_ema": {"score": float("inf"), "state": None, "epoch": -1},
        "log_rmse": {"score": float("inf"), "state": None, "epoch": -1},
        "target_mae": {"score": float("inf"), "state": None, "epoch": -1},
        "low_bias": {"score": float("inf"), "state": None, "epoch": -1},
    }
    EMA_ALPHA = 0.5
    ema_composite = None
    wait = 0

    try:
        base_state = copy.deepcopy(model.state_dict())
        pre_acceptable, _ = is_acceptable_checkpoint(pre_range, pre_val)
        if pre_acceptable:
            for ckpt_name, score_value in [
                ("composite", pre_sel["composite"]),
                ("composite_ema", pre_sel["composite"]),
                ("log_rmse", pre_sel["log_rmse_weighted"]),
                (
                    "target_mae",
                    pre_range["target_BER(0.20<=y<0.40)"]["mae_raw"]
                    if pre_range.get("target_BER(0.20<=y<0.40)") else float("inf"),
                ),
                ("low_bias", pre_sel["bias_weighted"]),
            ]:
                best_checkpoints[ckpt_name] = {
                    "score": float(score_value),
                    "state": base_state,
                    "epoch": 0,
                }
            print("\nSeeded best checkpoints with pretrained baseline scores.")
    except Exception as e:
        print(f"  Could not seed pretrained baseline as best: {e}")

    print("\nSelection: weighted log-RMSE + bias + tail penalties")
    print(f"  Region weights: {SELECTION_REGION_WEIGHTS}")
    print(
        f"  Composite weights: log_rmse={SELECTION_W_LOG_RMSE}, "
        f"bias={SELECTION_W_BIAS}, tail={SELECTION_W_TAIL}"
    )
    print(f"  EMA alpha: {EMA_ALPHA}\n")

    training_broke = False

    for epoch in range(FINETUNE_MAX_EPOCHS):
        model.train()
        running_loss = 0.0
        running_reg = 0.0
        running_ord = 0.0
        num_train_batches = 0

        for batch_idx, (b_first, b_past, b_global, b_thr, b_y_log, b_ord, _, b_mask) in enumerate(train_loader):
            b_first = b_first.to(device, non_blocking=True)
            b_past = b_past.to(device, non_blocking=True)
            b_global = b_global.to(device, non_blocking=True)
            b_thr = b_thr.to(device, non_blocking=True)
            b_y_log = b_y_log.to(device, non_blocking=True)
            b_ord = b_ord.to(device, non_blocking=True)
            b_mask = b_mask.to(device, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)
            pred_raw_unconstrained, ord_logits = model(
                b_first,
                b_past,
                b_global,
                b_thr,
                key_padding_mask=b_mask,
            )

            if has_nonfinite_tensor(pred_raw_unconstrained):
                print(f"Non-finite prediction at epoch {epoch + 1}, batch {batch_idx + 1}.")
                training_broke = True
                break
            if has_nonfinite_tensor(ord_logits):
                print(f"Non-finite ordinal logits at epoch {epoch + 1}, batch {batch_idx + 1}.")
                training_broke = True
                break

            loss, reg_part, ord_part = criterion(
                pred_raw_unconstrained,
                ord_logits,
                b_y_log,
                b_ord,
            )
            if has_nonfinite_tensor(loss):
                print(f"Non-finite loss at epoch {epoch + 1}, batch {batch_idx + 1}.")
                training_broke = True
                break

            loss.backward()
            grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=GRAD_CLIP)
            if not torch.isfinite(grad_norm):
                print(f"Non-finite gradient norm at epoch {epoch + 1}, batch {batch_idx + 1}.")
                training_broke = True
                break

            optimizer.step()

            running_loss += float(loss.item())
            running_reg += float(reg_part.item())
            running_ord += float(ord_part.item())
            num_train_batches += 1

        if training_broke:
            break

        if num_train_batches == 0:
            print(f"No training batches were produced at epoch {epoch + 1}. Stopping.")
            break

        train_loss = running_loss / num_train_batches
        train_reg = running_reg / num_train_batches
        train_ord = running_ord / num_train_batches

        val_metrics = evaluate(model, val_loader, criterion, device)
        val_range_metrics = evaluate_by_target_range(model, val_loader, device)
        val_selection = compute_selection_score(val_range_metrics, val_metrics)
        acceptable, accept_reason = is_acceptable_checkpoint(val_range_metrics, val_metrics)

        composite = val_selection["composite"]
        if ema_composite is None:
            ema_composite = composite
        else:
            ema_composite = EMA_ALPHA * composite + (1.0 - EMA_ALPHA) * ema_composite

        scheduler.step(composite)
        current_lr = optimizer.param_groups[0]["lr"]

        print(
            f"Epoch {epoch + 1:03d}/{FINETUNE_MAX_EPOCHS} | "
            f"train={train_loss:.5f} reg={train_reg:.5f} ord={train_ord:.5f} | "
            f"val={val_metrics['loss']:.5f} rmse_log={val_metrics['rmse_log']:.4f} "
            f"factor~{val_metrics['factor_error']:.3f}x | "
            f"region_acc={val_metrics['region_acc']:.4f} | "
            f"composite={composite:.5f} ema={ema_composite:.5f} | "
            f"lr={current_lr:.2e} | acceptable={acceptable} ({accept_reason})"
        )

        target_stats = val_range_metrics.get("target_BER(0.20<=y<0.40)")
        if target_stats is not None:
            print(
                f"  target_BER: count={target_stats['count']} | "
                f"factor~{target_stats['factor_error']:.3f}x | "
                f"MAE_raw={target_stats['mae_raw']:.6f} | "
                f"bias_log={target_stats['bias_log']:+.4f} | "
                f"P95={target_stats['p95_abs_raw']:.6f}"
            )

        improved = False

        if acceptable and composite < best_checkpoints["composite"]["score"]:
            best_checkpoints["composite"] = {
                "score": float(composite),
                "state": copy.deepcopy(model.state_dict()),
                "epoch": epoch + 1,
            }
            print(f"  New best composite: {composite:.5f}")
            improved = True

        if acceptable and ema_composite < best_checkpoints["composite_ema"]["score"]:
            best_checkpoints["composite_ema"] = {
                "score": float(ema_composite),
                "state": copy.deepcopy(model.state_dict()),
                "epoch": epoch + 1,
            }
            print(f"  New best composite EMA: {ema_composite:.5f}")
            improved = True

        log_rmse_score = val_selection["log_rmse_weighted"]
        if acceptable and log_rmse_score < best_checkpoints["log_rmse"]["score"]:
            best_checkpoints["log_rmse"] = {
                "score": float(log_rmse_score),
                "state": copy.deepcopy(model.state_dict()),
                "epoch": epoch + 1,
            }
            print(f"  New best weighted log-RMSE: {log_rmse_score:.5f}")
            improved = True

        if acceptable and target_stats is not None:
            target_mae = target_stats["mae_raw"]
            if target_mae < best_checkpoints["target_mae"]["score"]:
                best_checkpoints["target_mae"] = {
                    "score": float(target_mae),
                    "state": copy.deepcopy(model.state_dict()),
                    "epoch": epoch + 1,
                }
                print(f"  New best target MAE raw: {target_mae:.6f}")
                improved = True

        bias_score = val_selection["bias_weighted"]
        if acceptable and bias_score < best_checkpoints["low_bias"]["score"]:
            best_checkpoints["low_bias"] = {
                "score": float(bias_score),
                "state": copy.deepcopy(model.state_dict()),
                "epoch": epoch + 1,
            }
            print(f"  New best low-bias score: {bias_score:.5f}")
            improved = True

        torch.save(model.state_dict(), LAST_MODEL_SAVE_PATH)

        if improved:
            wait = 0
        else:
            wait += 1

        if (epoch + 1) >= FINETUNE_MIN_EPOCHS and wait >= FINETUNE_PATIENCE:
            print(
                f"Early stopping at epoch {epoch + 1}: "
                f"no acceptable improvement for {wait} epochs."
            )
            break

    if training_broke:
        print("Training stopped because of non-finite values.")

    primary_choice = "composite_ema"
    if best_checkpoints[primary_choice]["state"] is None:
        primary_choice = "composite"
    if best_checkpoints[primary_choice]["state"] is None:
        primary_choice = "log_rmse"

    if best_checkpoints[primary_choice]["state"] is not None:
        model.load_state_dict(best_checkpoints[primary_choice]["state"])
        torch.save(model.state_dict(), BEST_MODEL_SAVE_PATH)
        print(
            f"\nPrimary best ({primary_choice}, "
            f"FT-epoch {best_checkpoints[primary_choice]['epoch']}) "
            f"saved to: {BEST_MODEL_SAVE_PATH}"
        )
    else:
        print("\nWarning: no valid best checkpoint was found. Saving last model as best fallback.")
        torch.save(model.state_dict(), BEST_MODEL_SAVE_PATH)

    test_metrics = evaluate(model, test_loader, criterion, device)
    print(
        f"\nTest Loss: {test_metrics['loss']:.4f} | "
        f"Test RMSE(log10): {test_metrics['rmse_log']:.4f} | "
        f"~{test_metrics['factor_error']:.2f}x | "
        f"Test Region Acc: {test_metrics['region_acc']:.4f}"
    )

    range_metrics = evaluate_by_target_range(model, test_loader, device)
    print("\nPer-range test diagnostics:")
    for name, stats in range_metrics.items():
        if stats is None:
            print(f"  {name}: no samples")
        else:
            print(
                f"  {name} | count={stats['count']} | "
                f"factor~{stats['factor_error']:.2f}x | "
                f"RMSE(log10)={stats['rmse_log']:.4f} | "
                f"MAE_raw={stats['mae_raw']:.6f} | "
                f"bias_log={stats['bias_log']:+.4f} | "
                f"P90={stats['p90_abs_raw']:.6f} | "
                f"P95={stats['p95_abs_raw']:.6f}"
            )

    return model


if __name__ == "__main__":
    os.makedirs(BASE_DIR, exist_ok=True)

    if RUN_PARQUET_CONVERSION:
        print("Running CSV -> cleaned Parquet conversion.")
        build_clean_parquet_files(overwrite=OVERWRITE_PARQUET)

    missing_parquet = [p for p in DATA_PATHS if not os.path.exists(p)]
    if missing_parquet:
        print("Critical Error: Missing Parquet dataset files:")
        for p in missing_parquet:
            print(f"  - {p}")
        print("Set RUN_PARQUET_CONVERSION=True, or run build_clean_parquet_files() first.")
    else:
        print("Fine-tuning from Parquet datasets:")
        for p in DATA_PATHS:
            print(f"  - {p}")
        print(f"Row cap per dataset: {NROWS_PER_DATASET}")
        finetune()

Fine-tuning from Parquet datasets:
  - ./../ber_data_generation/data/data_physics_total.cleaned.parquet
  - ./../ber_data_generation/data/data_random_total.cleaned.parquet
Row cap per dataset: 10000000
Executing on: cuda

FINE-TUNING from: ./random_extra_multitask_finetune_best_normthr_monoord.pth
  Pretrained scalers: ./random_extra_multitask_finetune_scalers_normthr_monoord.pkl
  Reuse scalers: True
  Pretrained global_dim: 7
  Pretrained threshold_feature_mode: normalized_logit

Streaming Parquet data:
  ./../ber_data_generation/data/data_physics_total.cleaned.parquet
  ./../ber_data_generation/data/data_random_total.cleaned.parquet
  tap columns: 15
  past length: 14
  rows/file cap: 10000000

Fine-tune scalers saved to: ./random_extra_multitask_finetune_scalers_normthr_monoord2.pkl
Pretrained normalized-threshold + monotonic-ordinal weights loaded with strict=True.

--- Pre-fine-tune evaluation (pretrained model on new data) ---
  Pretrained on new val: composite=0.11223 | factor~

Epoch 021/60 | train=0.00255 reg=0.00207 ord=0.00191 | val=0.00065 rmse_log=0.0198 factor~1.047x | region_acc=0.9917 | composite=0.11045 ema=0.11045 | lr=5.00e-06 | acceptable=True (ok)
  target_BER: count=592624 | factor~1.012x | MAE_raw=0.001953 | bias_log=+0.0006 | P95=0.006378
Epoch 022/60 | train=0.00255 reg=0.00207 ord=0.00192 | val=0.00066 rmse_log=0.0207 factor~1.049x | region_acc=0.9916 | composite=0.10942 ema=0.10994 | lr=5.00e-06 | acceptable=True (ok)
  target_BER: count=592624 | factor~1.012x | MAE_raw=0.001998 | bias_log=+0.0004 | P95=0.006522
  New best composite EMA: 0.10994
Epoch 023/60 | train=0.00254 reg=0.00207 ord=0.00190 | val=0.00065 rmse_log=0.0208 factor~1.049x | region_acc=0.9918 | composite=0.11115 ema=0.11054 | lr=2.50e-06 | acceptable=True (ok)
  target_BER: count=592624 | factor~1.012x | MAE_raw=0.002007 | bias_log=+0.0003 | P95=0.006451
Epoch 024/60 | train=0.00254 reg=0.00207 ord=0.00190 | val=0.00063 rmse_log=0.0203 factor~1.048x | region_acc=0.9922 | c

In [10]:
"""
Fine-Tuning Script: Parquet Streaming Version
============================================
Continues training from the normalized-threshold + monotonic-ordinal checkpoint.

What changed vs the eager CSV version:
  - CSVs are cleaned and converted to Parquet once.
  - Training/validation/test data are streamed from Parquet row groups.
  - No full pandas concat.
  - No sklearn train_test_split on full arrays.
  - No full TensorDataset in RAM.
  - No WeightedRandomSampler, because that requires all labels/weights in memory.
  - Scalers are reused from the pretrained run.

Recommended flow:
  1. Set RUN_PARQUET_CONVERSION = True once and run this file.
  2. Set RUN_PARQUET_CONVERSION = False for later training runs.

Dependencies:
  pip install pandas pyarrow joblib torch numpy
"""

import os
import re
import gc
import copy
import joblib
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torch.multiprocessing as mp
from torch.utils.data import DataLoader, IterableDataset, get_worker_info

mp.set_sharing_strategy("file_system")


# =========================
# Configuration
# =========================
BASE_DIR = "./"
DATA_DIR = "./../ber_data_generation/data/"

# ---- Source: previous training run to resume from ----
PRETRAINED_MODEL_PATH = os.path.join(
    BASE_DIR, "random_extra_multitask_finetune_best_normthr_monoord2.pth"
)
PRETRAINED_SCALER_PATH = os.path.join(
    BASE_DIR, "random_extra_multitask_finetune_scalers_normthr_monoord2.pkl"
)

# This streaming version is designed for reused scalers.
REUSE_SCALERS = True

# ---- Destination: fine-tuned model ----
BEST_MODEL_SAVE_PATH = os.path.join(
    BASE_DIR, "random_extra_multitask_finetune_best_normthr_monoord3.pth"
)
LAST_MODEL_SAVE_PATH = os.path.join(
    BASE_DIR, "random_extra_multitask_finetune_last_normthr_monoord3.pth"
)
SCALER_SAVE_PATH = os.path.join(
    BASE_DIR, "random_extra_multitask_finetune_scalers_normthr_monoord3.pkl"
)

# ---- Raw CSVs ----
CSV_DATA_PATHS = [
    os.path.join(DATA_DIR, "data_physics_total.csv"),
    os.path.join(DATA_DIR, "data_random_total.csv"),
]

# ---- Cleaned Parquet files ----
PARQUET_DATA_PATHS = [
    os.path.join(DATA_DIR, "data_physics_total.cleaned.parquet"),
    os.path.join(DATA_DIR, "data_random_total.cleaned.parquet"),
]

# Training reads Parquet.
DATA_PATHS = PARQUET_DATA_PATHS

# Set True only once to create Parquet files. Then set False for normal training.
RUN_PARQUET_CONVERSION = False
OVERWRITE_PARQUET = False

# Cap per source file. None = use all rows. 10_000_000 matches your old config.
NROWS_PER_DATASET = 10_000_000
# NROWS_PER_DATASET = None

# Conversion and streaming batch sizes.
PARQUET_CSV_CHUNKSIZE = 500_000
PARQUET_BATCH_ROWS = 8192
DATALOADER_NUM_WORKERS = 0

# ---- Fine-tune hyperparameters ----
FINETUNE_LR = 2e-5
FINETUNE_WEIGHT_DECAY = 1e-5
FINETUNE_MAX_EPOCHS = 60
FINETUNE_MIN_EPOCHS = 25
FINETUNE_PATIENCE = 8
SCHEDULER_PATIENCE = 3
SCHEDULER_FACTOR = 0.5
GRAD_CLIP = 1.0

EPS = 1e-12
LOG10_HALF = float(np.log10(0.5))

# =========================
# New-model threshold / numeric feature options
# =========================
THRESHOLD_FEATURE_MODE = "normalized_logit"
THRESHOLD_NORM_CLIP_EPS = 1e-5
HARMONIC_DENOM_EPS = 1e-8
FEATURE_CLIP_ABS = 1e6

ORDINAL_THRESHOLDS = [
    1e-6, 1e-5, 1e-4, 1e-3, 1e-2, 1e-1,
    0.15, 0.20, 0.25, 0.30, 0.35, 0.40, 0.45,
]
NUM_ORDINAL_THRESHOLDS = len(ORDINAL_THRESHOLDS)
NUM_REGION_CLASSES = NUM_ORDINAL_THRESHOLDS + 1


# =========================
# Utilities
# =========================
def get_sorted_seq_cols(columns, prefix):
    pattern = re.compile(rf"^{re.escape(prefix)}_(\d+)$")
    matched = []
    for col in columns:
        m = pattern.match(str(col))
        if m:
            matched.append((int(m.group(1)), str(col)))
    matched.sort(key=lambda x: x[0])
    return [col for _, col in matched]


def has_nonfinite_tensor(x):
    return not torch.isfinite(x).all().item()


def y_log_to_raw_np(y_log):
    return np.clip(10 ** y_log, EPS, 0.5)


def raw_to_region_labels_np(y_raw):
    y = np.asarray(y_raw).reshape(-1)
    labels = np.searchsorted(np.asarray(ORDINAL_THRESHOLDS), y, side="right")
    return labels.astype(np.int64)


def region_labels_to_ordinal_targets_np(labels, num_thresholds=NUM_ORDINAL_THRESHOLDS):
    labels = np.asarray(labels).reshape(-1)
    thresholds = np.arange(1, num_thresholds + 1, dtype=np.int64)
    ordinal = (labels[:, None] >= thresholds[None, :]).astype(np.float32)
    return ordinal


def ordinal_logits_to_region_labels_torch(logits):
    probs = torch.sigmoid(logits)
    passed = (probs >= 0.5).sum(dim=1)
    return passed.long()


def compute_region_class_weights(labels_np, num_classes=NUM_REGION_CLASSES, max_weight=8.0):
    counts = np.bincount(labels_np.reshape(-1), minlength=num_classes).astype(np.float64)
    counts = np.maximum(counts, 1.0)
    weights = counts.sum() / (num_classes * counts)
    weights = np.sqrt(weights)
    weights = np.clip(weights, 1.0, max_weight)
    return weights.astype(np.float32)


def compute_ordinal_pos_weights_from_region_labels(
    region_labels_np,
    num_thresholds=NUM_ORDINAL_THRESHOLDS,
    max_weight=20.0,
):
    ordinal_targets = region_labels_to_ordinal_targets_np(region_labels_np, num_thresholds)
    pos_counts = ordinal_targets.sum(axis=0)
    neg_counts = ordinal_targets.shape[0] - pos_counts
    pos_counts = np.maximum(pos_counts, 1.0)
    pos_weight = neg_counts / pos_counts
    pos_weight = np.clip(pos_weight, 1.0, max_weight)
    return pos_weight.astype(np.float32)


def apply_shared_scale(data, mean, std, valid_mask=None):
    scaled = ((data - mean) / std).astype(np.float32)
    if valid_mask is not None and scaled.ndim == 2 and valid_mask.shape == scaled.shape:
        scaled = scaled * valid_mask.astype(np.float32)
    return scaled


def sanitize_np_feature(x, name, clip_abs=FEATURE_CLIP_ABS):
    arr = np.asarray(x, dtype=np.float32)
    bad = ~np.isfinite(arr)
    if np.any(bad):
        print(f"Warning: {name} contained {int(bad.sum())} non-finite values; replacing with 0.")
    arr = np.nan_to_num(arr, nan=0.0, posinf=0.0, neginf=0.0)
    arr = np.clip(arr, -clip_abs, clip_abs).astype(np.float32)
    return arr


def stable_harmonic_pair_np(a, b, denom_eps=HARMONIC_DENOM_EPS):
    a = np.asarray(a, dtype=np.float32)
    b = np.asarray(b, dtype=np.float32)
    denom = (a + b).astype(np.float32)
    numer = (2.0 * a * b).astype(np.float32)
    out = np.divide(
        numer,
        denom,
        out=np.zeros_like(numer, dtype=np.float32),
        where=np.abs(denom) > denom_eps,
    ).astype(np.float32)
    return sanitize_np_feature(out, "stable_harmonic_pair")


def compute_threshold_feature_np(threshold_raw, total_scaled_taps, mode=THRESHOLD_FEATURE_MODE):
    thr = np.asarray(threshold_raw, dtype=np.float32).reshape(-1, 1)
    total = np.asarray(total_scaled_taps, dtype=np.float32).reshape(-1, 1)
    denom = np.maximum(total, EPS).astype(np.float32)

    if mode == "normalized":
        feat = (thr / denom).astype(np.float32)
    elif mode == "normalized_logit":
        u = (thr / denom).astype(np.float32)
        u = np.clip(u, THRESHOLD_NORM_CLIP_EPS, 1.0 - THRESHOLD_NORM_CLIP_EPS)
        feat = np.log(u / (1.0 - u)).astype(np.float32)
    elif mode == "log10_raw":
        feat = np.log10(thr + EPS).astype(np.float32)
    else:
        raise ValueError(f"Unknown THRESHOLD_FEATURE_MODE: {mode}")

    return sanitize_np_feature(feat, f"threshold_feature_{mode}")


def compute_variances_from_taps(taps_raw, N_array, valid_mask):
    N = N_array.reshape(-1, 1).astype(np.float64)
    taps = np.where(np.isnan(taps_raw), 0.0, taps_raw).astype(np.float64)
    taps = np.clip(taps, 0.0, 1.0)
    vars_full = N * taps * (1.0 - taps)
    vars_full = vars_full * valid_mask.astype(np.float64)
    return vars_full.astype(np.float32)


def safe_torch_load_state_dict(path, map_location):
    try:
        return torch.load(path, map_location=map_location, weights_only=True)
    except TypeError:
        return torch.load(path, map_location=map_location)


# =========================
# CSV -> Parquet conversion
# =========================
def clean_csv_to_parquet(
    csv_path,
    parquet_path,
    chunksize=PARQUET_CSV_CHUNKSIZE,
    compression="zstd",
    overwrite=False,
):
    """
    Stream-clean CSV -> Parquet without loading the full CSV.

    Handles:
      - over-long corrupted rows: skipped by pandas on_bad_lines='skip'
      - malformed numeric cells: coerced to NaN
      - invalid BER/N/threshold rows: removed
      - missing mem_len: derived from non-null tap_* columns
      - malformed valid tap positions: dropped
      - padding tap NaNs: filled with 0 after valid-region check
    """
    print(f"\nCleaning/converting:\n  {csv_path}\n  -> {parquet_path}")

    if not os.path.exists(csv_path):
        raise FileNotFoundError(f"CSV not found: {csv_path}")

    os.makedirs(os.path.dirname(parquet_path), exist_ok=True)

    if os.path.exists(parquet_path):
        if overwrite:
            print(f"  Removing existing parquet: {parquet_path}")
            os.remove(parquet_path)
        else:
            print(f"  Parquet already exists; skipping conversion: {parquet_path}")
            return

    writer = None
    total_read = 0
    total_written = 0
    total_dropped_bad_taps = 0
    total_dropped_filters = 0

    reader = pd.read_csv(
        csv_path,
        chunksize=chunksize,
        on_bad_lines="skip",
        engine="c",
        low_memory=False,
    )

    try:
        for chunk_idx, chunk in enumerate(reader, start=1):
            chunk.columns = [str(c).strip() for c in chunk.columns]
            before = len(chunk)
            total_read += before

            tap_cols = get_sorted_seq_cols(chunk.columns, "tap")
            var_cols = get_sorted_seq_cols(chunk.columns, "var")

            if not tap_cols:
                raise ValueError(f"No tap_* columns found in {csv_path}")

            for required in ["N", "threshold", "BER"]:
                if required not in chunk.columns:
                    raise ValueError(f"Missing required column {required!r} in {csv_path}")

            # Derive mem_len before padding NaNs are filled.
            if "mem_len" not in chunk.columns:
                chunk["mem_len"] = chunk[tap_cols].notna().sum(axis=1)

            numeric_cols = list(dict.fromkeys(
                tap_cols + var_cols + ["mem_len", "N", "threshold", "BER"]
            ))

            for col in numeric_cols:
                if col in chunk.columns:
                    chunk[col] = pd.to_numeric(chunk[col], errors="coerce")

            chunk = chunk[chunk["mem_len"].notna()].copy()
            if len(chunk) == 0:
                print(f"  chunk {chunk_idx:04d}: read={before:,}, wrote=0")
                continue

            chunk["mem_len"] = chunk["mem_len"].astype(np.int64)
            chunk["threshold"] = chunk["threshold"].fillna(0.0)
            chunk["BER"] = chunk["BER"].fillna(0.0)
            chunk["N"] = chunk["N"].fillna(0.0)

            before_filters = len(chunk)
            chunk = chunk[
                (chunk["mem_len"] > 1)
                & (chunk["BER"] > 0)
                & (chunk["N"] > 0)
                & (chunk["threshold"] > 0)
            ].copy()
            total_dropped_filters += before_filters - len(chunk)

            if len(chunk) == 0:
                print(f"  chunk {chunk_idx:04d}: read={before:,}, wrote=0")
                continue

            chunk["BER"] = chunk["BER"].clip(lower=EPS, upper=0.5)

            # Drop rows where valid tap positions contain NaN/non-numeric values.
            tap_values = chunk[tap_cols].to_numpy(dtype=np.float32)
            mem_len = chunk["mem_len"].to_numpy(dtype=np.int64)
            L = len(tap_cols)

            mem_len_clamped = np.minimum(mem_len, L)
            valid = np.arange(L)[None, :] < mem_len_clamped[:, None]

            bad_valid_tap = (~np.isfinite(tap_values)) & valid
            keep = ~bad_valid_tap.any(axis=1)

            dropped_bad_taps = int((~keep).sum())
            total_dropped_bad_taps += dropped_bad_taps

            if dropped_bad_taps:
                print(f"    dropped malformed valid-tap rows: {dropped_bad_taps:,}")

            chunk = chunk.iloc[keep].copy()

            if len(chunk) == 0:
                print(f"  chunk {chunk_idx:04d}: read={before:,}, wrote=0")
                continue

            # Padding/outside-valid-memory NaNs are safe to zero-fill now.
            chunk[tap_cols] = chunk[tap_cols].fillna(0.0)
            if var_cols:
                chunk[var_cols] = chunk[var_cols].fillna(0.0)

            # Stable dtypes.
            for col in tap_cols + var_cols + ["N", "threshold", "BER"]:
                if col in chunk.columns:
                    chunk[col] = chunk[col].astype(np.float32)
            chunk["mem_len"] = chunk["mem_len"].astype(np.int64)

            chunk["source_dataset"] = os.path.basename(csv_path)
            chunk["_has_vars"] = bool(len(var_cols) > 0)

            table = pa.Table.from_pandas(chunk, preserve_index=False)

            if writer is None:
                writer = pq.ParquetWriter(
                    parquet_path,
                    table.schema,
                    compression=compression,
                    use_dictionary=False,
                )

            writer.write_table(table, row_group_size=200_000)
            total_written += len(chunk)

            print(
                f"  chunk {chunk_idx:04d}: "
                f"read={before:,}, wrote={len(chunk):,}, total_written={total_written:,}"
            )

            del chunk, table
            gc.collect()

    finally:
        if writer is not None:
            writer.close()

    print(
        f"Done: read={total_read:,}, wrote={total_written:,}, "
        f"dropped_filters≈{total_dropped_filters:,}, "
        f"dropped_bad_valid_taps={total_dropped_bad_taps:,}"
    )


def build_clean_parquet_files(overwrite=OVERWRITE_PARQUET):
    for csv_path, parquet_path in zip(CSV_DATA_PATHS, PARQUET_DATA_PATHS):
        clean_csv_to_parquet(csv_path, parquet_path, overwrite=overwrite)


# =========================
# Parquet streaming helpers
# =========================
def _parquet_schema_names(path):
    return set(pq.ParquetFile(path).schema_arrow.names)


def infer_parquet_sequence_columns(parquet_paths, pretrained_scalers=None):
    all_cols = set()
    for path in parquet_paths:
        if not os.path.exists(path):
            raise FileNotFoundError(f"Parquet dataset not found: {path}")
        all_cols |= _parquet_schema_names(path)

    tap_cols = get_sorted_seq_cols(all_cols, "tap")

    if pretrained_scalers is not None and "tap_cols" in pretrained_scalers:
        tap_cols = sorted(
            set(tap_cols) | set(pretrained_scalers["tap_cols"]),
            key=lambda c: int(c.split("_")[1]),
        )

    if not tap_cols:
        raise ValueError("No tap_* columns found in Parquet files.")

    tap_indices = [int(c.split("_")[1]) for c in tap_cols]
    var_cols = [f"var_{i}" for i in tap_indices]

    return tap_cols, var_cols


def stable_hash_percent(row_ids, seed=42):
    """Deterministic split hash in [0, 10000)."""
    x = row_ids.astype(np.uint64) + np.uint64(seed)
    x ^= x >> np.uint64(33)
    x *= np.uint64(0xff51afd7ed558ccd)
    x ^= x >> np.uint64(33)
    x *= np.uint64(0xc4ceb9fe1a85ec53)
    x ^= x >> np.uint64(33)
    return (x % np.uint64(10000)).astype(np.int32)


def split_keep_mask(row_ids, split, seed=42):
    pct = stable_hash_percent(row_ids, seed=seed)
    if split == "train":
        return pct < 7000
    if split == "val":
        return (pct >= 7000) & (pct < 8500)
    if split == "test":
        return pct >= 8500
    raise ValueError(f"Unknown split: {split}")


def build_parquet_row_group_index(parquet_paths, max_rows_per_file=None):
    items = []
    global_start = 0

    for path in parquet_paths:
        pf = pq.ParquetFile(path)
        file_seen = 0

        for rg_idx in range(pf.num_row_groups):
            rg_rows = pf.metadata.row_group(rg_idx).num_rows

            if max_rows_per_file is None:
                take_rows = rg_rows
            else:
                remaining = max_rows_per_file - file_seen
                if remaining <= 0:
                    break
                take_rows = min(rg_rows, remaining)

            if take_rows > 0:
                items.append({
                    "path": path,
                    "row_group": rg_idx,
                    "global_start": global_start,
                    "take_rows": take_rows,
                    "rg_rows": rg_rows,
                })
                global_start += take_rows

            file_seen += rg_rows

    return items, global_start


class ParquetBERIterableDataset(IterableDataset):
    """
    Streams Parquet row groups, cleans each batch, computes features, applies
    pretrained scalers, and yields already-batched tensors.

    DataLoader must use batch_size=None.
    """

    def __init__(
        self,
        parquet_paths,
        scalers,
        tap_cols,
        var_cols,
        split,
        batch_rows=8192,
        max_rows_per_file=None,
        split_seed=42,
    ):
        super().__init__()
        self.parquet_paths = list(parquet_paths)
        self.scalers = scalers
        self.tap_cols = list(tap_cols)
        self.var_cols = list(var_cols)
        self.split = split
        self.batch_rows = int(batch_rows)
        self.max_rows_per_file = max_rows_per_file
        self.split_seed = int(split_seed)

        self.row_groups, self.total_indexed_rows = build_parquet_row_group_index(
            self.parquet_paths,
            max_rows_per_file=max_rows_per_file,
        )

        self.schema_names_by_path = {
            path: _parquet_schema_names(path)
            for path in self.parquet_paths
        }

    def __iter__(self):
        worker = get_worker_info()
        if worker is None:
            worker_id = 0
            num_workers = 1
        else:
            worker_id = worker.id
            num_workers = worker.num_workers

        for rg_global_idx, item in enumerate(self.row_groups):
            if rg_global_idx % num_workers != worker_id:
                continue

            path = item["path"]
            rg_idx = item["row_group"]
            global_start = item["global_start"]
            take_rows = item["take_rows"]

            schema_names = self.schema_names_by_path[path]
            wanted_cols = list(dict.fromkeys(
                self.tap_cols
                + self.var_cols
                + ["mem_len", "N", "threshold", "BER", "source_dataset", "_has_vars"]
            ))
            read_cols = [c for c in wanted_cols if c in schema_names]

            pf = pq.ParquetFile(path)
            batch_offset = 0

            for record_batch in pf.iter_batches(
                batch_size=self.batch_rows,
                row_groups=[rg_idx],
                columns=read_cols,
            ):
                df = record_batch.to_pandas()
                n = len(df)

                local_in_group = np.arange(batch_offset, batch_offset + n)
                allowed = local_in_group < take_rows

                if not np.any(allowed):
                    break

                if not np.all(allowed):
                    df = df.iloc[allowed].copy()
                    local_in_group = local_in_group[allowed]
                    n = len(df)

                row_ids = global_start + local_in_group.astype(np.int64)
                batch_offset += len(record_batch)

                split_keep = split_keep_mask(row_ids, self.split, seed=self.split_seed)
                if not np.any(split_keep):
                    continue

                df = df.iloc[split_keep].copy()
                tensors = self._prepare_tensor_batch(df)
                if tensors is not None:
                    yield tensors

    def _prepare_tensor_batch(self, df):
        for col in self.tap_cols:
            if col not in df.columns:
                df[col] = np.nan

        for required in ["N", "threshold", "BER"]:
            if required not in df.columns:
                raise ValueError(f"Missing required column {required!r} in Parquet batch.")

        if "mem_len" not in df.columns:
            df["mem_len"] = df[self.tap_cols].notna().sum(axis=1)

        numeric_cols = list(dict.fromkeys(
            self.tap_cols + self.var_cols + ["mem_len", "N", "threshold", "BER"]
        ))
        for col in numeric_cols:
            if col in df.columns:
                df[col] = pd.to_numeric(df[col], errors="coerce")

        df = df[df["mem_len"].notna()].copy()
        if len(df) == 0:
            return None

        df["mem_len"] = df["mem_len"].astype(np.int64)
        df["threshold"] = df["threshold"].fillna(0.0)
        df["BER"] = df["BER"].fillna(0.0)
        df["N"] = df["N"].fillna(0.0)

        df = df[
            (df["mem_len"] > 1)
            & (df["BER"] > 0)
            & (df["N"] > 0)
            & (df["threshold"] > 0)
        ].copy()
        if len(df) == 0:
            return None

        df["BER"] = df["BER"].clip(lower=EPS, upper=0.5)

        X_taps_raw = df[self.tap_cols].to_numpy(dtype=np.float32)
        num_molecules = df["N"].to_numpy(dtype=np.float32).reshape(-1, 1)
        mem_len = df["mem_len"].to_numpy(dtype=np.int64)
        X_thr_raw = df["threshold"].to_numpy(dtype=np.float32).reshape(-1, 1)
        y_raw = df["BER"].to_numpy(dtype=np.float32).reshape(-1, 1)
        y_raw = np.clip(y_raw, EPS, 0.5).astype(np.float32)
        y_log = np.log10(y_raw + EPS).astype(np.float32)

        L_full = X_taps_raw.shape[1]
        L_past = L_full - 1

        mem_len_clamped = np.minimum(mem_len, L_full)
        valid_full = np.arange(L_full)[None, :] < mem_len_clamped[:, None]
        valid_past = valid_full[:, 1:]
        valid_past_float = valid_past.astype(np.float32)
        past_lens = np.maximum(mem_len_clamped - 1, 0).astype(np.int64)

        bad_valid_tap = (~np.isfinite(X_taps_raw)) & valid_full
        keep = ~bad_valid_tap.any(axis=1)
        if not np.any(keep):
            return None

        if not np.all(keep):
            X_taps_raw = X_taps_raw[keep]
            num_molecules = num_molecules[keep]
            X_thr_raw = X_thr_raw[keep]
            y_raw = y_raw[keep]
            y_log = y_log[keep]
            valid_full = valid_full[keep]
            valid_past = valid_past[keep]
            valid_past_float = valid_past_float[keep]
            past_lens = past_lens[keep]
            df = df.iloc[keep].copy()

        X_taps_raw = np.nan_to_num(
            X_taps_raw,
            nan=0.0,
            posinf=0.0,
            neginf=0.0,
        ).astype(np.float32)

        valid_full_float = valid_full.astype(np.float32)

        X_vars_computed = compute_variances_from_taps(
            X_taps_raw,
            num_molecules.reshape(-1),
            valid_full,
        )

        X_vars_csv = np.full_like(X_vars_computed, np.nan, dtype=np.float32)
        for j, var_col in enumerate(self.var_cols):
            if var_col in df.columns and j < X_vars_csv.shape[1]:
                X_vars_csv[:, j] = pd.to_numeric(
                    df[var_col],
                    errors="coerce",
                ).to_numpy(dtype=np.float32)

        X_vars_raw = np.where(
            np.isfinite(X_vars_csv),
            X_vars_csv,
            X_vars_computed,
        ).astype(np.float32)

        X_vars_raw = np.maximum(X_vars_raw, 0.0)
        X_vars_raw = X_vars_raw * valid_full_float

        # Feature engineering: matches original normalized-threshold model.
        X_taps_feat = (X_taps_raw * num_molecules).astype(np.float32)
        X_vars_feat = X_vars_raw.astype(np.float32)
        abs_taps_raw = np.abs(X_taps_feat).astype(np.float32)

        snr_raw = np.log10(
            (X_taps_feat ** 2) / (X_vars_raw + EPS) + EPS
        ).astype(np.float32)
        snr_raw = sanitize_np_feature(snr_raw, "snr_raw")

        threshold_mode = self.scalers.get(
            "threshold_feature_mode",
            THRESHOLD_FEATURE_MODE,
        )

        total_scaled_taps_raw = np.sum(
            X_taps_feat * valid_full_float,
            axis=1,
            keepdims=True,
        ).astype(np.float32)

        X_thr = compute_threshold_feature_np(
            X_thr_raw,
            total_scaled_taps_raw,
            mode=threshold_mode,
        )

        first_mean = X_taps_feat[:, 0:1]
        past_means = X_taps_feat[:, 1:]
        first_var = X_vars_feat[:, 0:1]
        past_vars = X_vars_feat[:, 1:]

        past_means_masked = past_means * valid_past_float
        past_vars_masked = past_vars * valid_past_float

        mu0_raw = (0.5 * np.sum(past_means_masked, axis=1, keepdims=True)).astype(np.float32)
        mu1_raw = (first_mean + mu0_raw).astype(np.float32)
        var0_raw = (0.5 * np.sum(past_vars_masked, axis=1, keepdims=True)).astype(np.float32)
        var1_raw = (first_var + var0_raw).astype(np.float32)

        std0_raw = np.sqrt(np.maximum(var0_raw, EPS)).astype(np.float32)
        std1_raw = np.sqrt(np.maximum(var1_raw, EPS)).astype(np.float32)

        z0_raw = ((X_thr_raw - mu0_raw) / (std0_raw + EPS)).astype(np.float32)
        z1_raw = ((mu1_raw - X_thr_raw) / (std1_raw + EPS)).astype(np.float32)

        z0_raw = sanitize_np_feature(z0_raw, "z0_raw")
        z1_raw = sanitize_np_feature(z1_raw, "z1_raw")

        harmonic_side_z_raw = stable_harmonic_pair_np(z0_raw, z1_raw)
        abs_diff_side_z_raw = np.abs(z0_raw - z1_raw).astype(np.float32)
        harmonic_minus_gap_raw = sanitize_np_feature(
            harmonic_side_z_raw - 0.25 * abs_diff_side_z_raw,
            "harmonic_minus_gap_raw",
        )

        signal_raw = first_mean
        isi_raw = np.sum(past_means_masked, axis=1, keepdims=True)
        nsid_raw = ((signal_raw - isi_raw) / (signal_raw + isi_raw + EPS)).astype(np.float32)

        gap_raw = signal_raw
        d0_raw = (gap_raw / (std0_raw + EPS)).astype(np.float32)
        d1_raw = (gap_raw / (std1_raw + EPS)).astype(np.float32)
        harmonic_discrim_raw = (
            2.0 * d0_raw * d1_raw / (d0_raw + d1_raw + EPS)
        ).astype(np.float32)
        log_harmonic_discrim_raw = np.log10(
            harmonic_discrim_raw + EPS
        ).astype(np.float32)
        discrim_asymmetry_raw = (
            np.abs(d0_raw - d1_raw) / (d0_raw + d1_raw + EPS)
        ).astype(np.float32)

        past_taps_sum = np.sum(past_means_masked, axis=1, keepdims=True)
        past_shares = past_means_masked / (past_taps_sum + EPS)
        herfindahl_raw = np.sum(past_shares ** 2, axis=1, keepdims=True).astype(np.float32)

        nsid_raw = sanitize_np_feature(nsid_raw, "nsid_raw")
        log_harmonic_discrim_raw = sanitize_np_feature(
            log_harmonic_discrim_raw,
            "log_harmonic_discrim_raw",
        )
        discrim_asymmetry_raw = sanitize_np_feature(
            discrim_asymmetry_raw,
            "discrim_asymmetry_raw",
        )
        herfindahl_raw = sanitize_np_feature(herfindahl_raw, "herfindahl_raw")

        region_labels = raw_to_region_labels_np(y_raw)
        ordinal_targets = region_labels_to_ordinal_targets_np(region_labels)

        first_token, past_tokens = self._build_tokens(
            X_taps_feat,
            X_vars_feat,
            abs_taps_raw,
            snr_raw,
            past_lens,
            L_past,
        )

        global_feats = self._build_global(
            z0_raw,
            z1_raw,
            harmonic_minus_gap_raw,
            nsid_raw,
            log_harmonic_discrim_raw,
            discrim_asymmetry_raw,
            herfindahl_raw,
        )

        thr_s = self.scalers["thr_scaler"].transform(X_thr).astype(np.float32)
        padding_mask = np.arange(L_past)[None, :] >= past_lens[:, None]

        return (
            torch.from_numpy(first_token.astype(np.float32)),
            torch.from_numpy(past_tokens.astype(np.float32)),
            torch.from_numpy(global_feats.astype(np.float32)),
            torch.from_numpy(thr_s.astype(np.float32)),
            torch.from_numpy(y_log.astype(np.float32)),
            torch.from_numpy(ordinal_targets.astype(np.float32)),
            torch.from_numpy(region_labels.astype(np.int64)),
            torch.from_numpy(padding_mask),
        )

    def _build_tokens(self, taps_feat, vars_feat, abs_raw, snr_raw, past_lens, L_past):
        s = self.scalers

        ft_tap = ((taps_feat[:, 0] - s["first_tap_mean"]) / s["first_tap_std"]).astype(np.float32)
        ft_var = ((vars_feat[:, 0] - s["first_var_mean"]) / s["first_var_std"]).astype(np.float32)
        ft_abs = ((abs_raw[:, 0] - s["first_abs_mean"]) / s["first_abs_std"]).astype(np.float32)
        ft_snr = ((snr_raw[:, 0] - s["first_snr_mean"]) / s["first_snr_std"]).astype(np.float32)

        first_token = np.stack(
            [ft_tap, ft_var, ft_abs, ft_snr],
            axis=1,
        ).astype(np.float32)

        pt_valid = np.arange(L_past)[None, :] < past_lens[:, None]

        pt_tap = apply_shared_scale(
            taps_feat[:, 1:],
            s["past_tap_mean"],
            s["past_tap_std"],
            pt_valid,
        )
        pt_var = apply_shared_scale(
            vars_feat[:, 1:],
            s["past_var_mean"],
            s["past_var_std"],
            pt_valid,
        )
        pt_abs = apply_shared_scale(
            abs_raw[:, 1:],
            s["past_abs_mean"],
            s["past_abs_std"],
            pt_valid,
        )
        pt_snr = apply_shared_scale(
            snr_raw[:, 1:],
            s["past_snr_mean"],
            s["past_snr_std"],
            pt_valid,
        )

        past_tokens = np.stack(
            [pt_tap, pt_var, pt_abs, pt_snr],
            axis=2,
        ).astype(np.float32)

        return first_token, past_tokens

    def _build_global(self, z0, z1, hmg, nsid, log_hd, da, herf):
        s = self.scalers
        return np.concatenate([
            s["z0_scaler"].transform(z0),
            s["z1_scaler"].transform(z1),
            s["harmonic_minus_gap_scaler"].transform(hmg),
            s["nsid_scaler"].transform(nsid),
            s["log_hd_scaler"].transform(log_hd),
            s["da_scaler"].transform(da),
            s["herf_scaler"].transform(herf),
        ], axis=1).astype(np.float32)


def prepare_data_for_finetune(
    parquet_paths,
    batch_size=256,
    nrows_per_dataset=None,
    num_workers=0,
    pretrained_scalers=None,
    reuse_scalers=True,
):
    """
    Memory-safe Parquet loader.

    Returns:
      train_loader, val_loader, test_loader, scalers, aux_info, L_past
    """
    del batch_size

    if not reuse_scalers:
        raise NotImplementedError(
            "Streaming scaler refit is intentionally disabled. Use REUSE_SCALERS=True."
        )

    if pretrained_scalers is None:
        raise ValueError("pretrained_scalers must be provided for streaming Parquet loading.")

    for path in parquet_paths:
        if not os.path.exists(path):
            raise FileNotFoundError(
                f"Parquet file not found: {path}\n"
                f"Set RUN_PARQUET_CONVERSION=True or run build_clean_parquet_files() first."
            )

    tap_cols, var_cols = infer_parquet_sequence_columns(
        parquet_paths,
        pretrained_scalers=pretrained_scalers,
    )

    L_full = len(tap_cols)
    L_past = L_full - 1

    print("\nStreaming Parquet data:")
    for p in parquet_paths:
        print(f"  {p}")
    print(f"  tap columns: {L_full}")
    print(f"  past length: {L_past}")
    print(f"  rows/file cap: {nrows_per_dataset}")

    scalers = dict(pretrained_scalers)
    scalers.update({
        "tap_cols": tap_cols,
        "var_cols": var_cols,
        "train_max_past_seq_len": L_past,
        "data_paths": parquet_paths,
        "nrows_per_dataset": nrows_per_dataset,
        "finetuned_from": PRETRAINED_MODEL_PATH,
        "scalers_reused": True,
        "streaming_parquet": True,
        "variances_computed_from_taps": True,
    })

    train_ds = ParquetBERIterableDataset(
        parquet_paths=parquet_paths,
        scalers=scalers,
        tap_cols=tap_cols,
        var_cols=var_cols,
        split="train",
        batch_rows=PARQUET_BATCH_ROWS,
        max_rows_per_file=nrows_per_dataset,
        split_seed=42,
    )
    val_ds = ParquetBERIterableDataset(
        parquet_paths=parquet_paths,
        scalers=scalers,
        tap_cols=tap_cols,
        var_cols=var_cols,
        split="val",
        batch_rows=PARQUET_BATCH_ROWS,
        max_rows_per_file=nrows_per_dataset,
        split_seed=42,
    )
    test_ds = ParquetBERIterableDataset(
        parquet_paths=parquet_paths,
        scalers=scalers,
        tap_cols=tap_cols,
        var_cols=var_cols,
        split="test",
        batch_rows=PARQUET_BATCH_ROWS,
        max_rows_per_file=nrows_per_dataset,
        split_seed=42,
    )

    pin_mem = torch.cuda.is_available()

    train_loader = DataLoader(
        train_ds,
        batch_size=None,
        num_workers=num_workers,
        pin_memory=pin_mem,
    )
    val_loader = DataLoader(
        val_ds,
        batch_size=None,
        num_workers=num_workers,
        pin_memory=pin_mem,
    )
    test_loader = DataLoader(
        test_ds,
        batch_size=None,
        num_workers=num_workers,
        pin_memory=pin_mem,
    )

    ordinal_pos_weights = np.asarray(
        pretrained_scalers.get(
            "ordinal_pos_weights",
            np.ones(NUM_ORDINAL_THRESHOLDS, dtype=np.float32),
        ),
        dtype=np.float32,
    )
    if ordinal_pos_weights.shape[0] != NUM_ORDINAL_THRESHOLDS:
        print("  Warning: pretrained ordinal_pos_weights length mismatch; using ones.")
        ordinal_pos_weights = np.ones(NUM_ORDINAL_THRESHOLDS, dtype=np.float32)

    class_weights = np.asarray(
        pretrained_scalers.get(
            "class_weights",
            np.ones(NUM_REGION_CLASSES, dtype=np.float32),
        ),
        dtype=np.float32,
    )
    if class_weights.shape[0] != NUM_REGION_CLASSES:
        class_weights = np.ones(NUM_REGION_CLASSES, dtype=np.float32)

    aux_info = {
        "class_weights": class_weights,
        "ordinal_pos_weights": ordinal_pos_weights,
    }

    return train_loader, val_loader, test_loader, scalers, aux_info, L_past


# =========================
# Loss
# =========================
class StableMultiObjectiveBERBoundedLogLoss(nn.Module):
    def __init__(self, log_delta=0.35, raw_delta=0.006, rel_delta=0.03,
                 alpha_log=0.45, beta_raw=0.35, gamma_rel=0.20,
                 use_regime_weights=True):
        super().__init__()
        self.log_delta = log_delta
        self.raw_delta = raw_delta
        self.rel_delta = rel_delta
        self.alpha_log = alpha_log
        self.beta_raw = beta_raw
        self.gamma_rel = gamma_rel
        self.use_regime_weights = use_regime_weights

    @staticmethod
    def huber_elementwise(pred, target, delta):
        err = pred - target
        abs_err = err.abs()
        return torch.where(abs_err < delta,
                           0.5 * err * err,
                           delta * (abs_err - 0.5 * delta))

    @staticmethod
    def raw_to_pred_log(pred_raw_unconstrained):
        log10_half = torch.log10(torch.tensor(
            0.5, device=pred_raw_unconstrained.device,
            dtype=pred_raw_unconstrained.dtype))
        return log10_half - F.softplus(pred_raw_unconstrained)

    @staticmethod
    def raw_to_pred_ber(pred_raw_unconstrained):
        pred_log = StableMultiObjectiveBERBoundedLogLoss.raw_to_pred_log(pred_raw_unconstrained)
        return torch.pow(10.0, pred_log).clamp(min=EPS, max=0.5)

    def forward(self, pred_raw_unconstrained, target_log):
        pred_log = self.raw_to_pred_log(pred_raw_unconstrained)
        pred_raw = torch.pow(10.0, pred_log).clamp(min=EPS, max=0.5)
        target_raw = torch.pow(10.0, target_log).clamp(min=EPS, max=0.5)
        target_log_for_loss = torch.log10(target_raw)

        log_loss = self.huber_elementwise(pred_log, target_log_for_loss, self.log_delta)
        raw_loss = self.huber_elementwise(pred_raw, target_raw, self.raw_delta)
        rel_err = (pred_raw - target_raw) / torch.clamp(target_raw, min=1e-6)
        rel_loss = self.huber_elementwise(rel_err, torch.zeros_like(rel_err), self.rel_delta)

        total = (self.alpha_log * log_loss
                 + self.beta_raw * raw_loss
                 + self.gamma_rel * rel_loss)

        if self.use_regime_weights:
            weights = torch.ones_like(target_raw)
            weights = torch.where((target_raw >= 1e-2) & (target_raw < 0.1),
                                  torch.full_like(weights, 1.5), weights)
            weights = torch.where((target_raw >= 0.1) & (target_raw < 0.15),
                                  torch.full_like(weights, 2.5), weights)
            weights = torch.where((target_raw >= 0.15) & (target_raw < 0.2),
                                  torch.full_like(weights, 3.0), weights)
            weights = torch.where((target_raw >= 0.2) & (target_raw < 0.3),
                                  torch.full_like(weights, 4.0), weights)
            weights = torch.where((target_raw >= 0.3) & (target_raw < 0.4),
                                  torch.full_like(weights, 4.5), weights)
            weights = torch.where((target_raw >= 0.4) & (target_raw < 0.45),
                                  torch.full_like(weights, 3.0), weights)
            weights = torch.where(target_raw >= 0.45,
                                  torch.full_like(weights, 2.5), weights)
            total = total * weights

        return total.mean()


class OrdinalBCELoss(nn.Module):
    def __init__(self, pos_weight=None, reduction="mean"):
        super().__init__()
        if pos_weight is not None and not isinstance(pos_weight, torch.Tensor):
            pos_weight = torch.tensor(pos_weight, dtype=torch.float32)
        self.register_buffer("pos_weight", pos_weight if pos_weight is not None else None)
        self.reduction = reduction

    def forward(self, logits, ordinal_targets):
        return F.binary_cross_entropy_with_logits(
            logits,
            ordinal_targets,
            pos_weight=self.pos_weight,
            reduction=self.reduction,
        )


class MultiTaskBERLoss(nn.Module):
    def __init__(self, reg_loss, ord_loss, lambda_ord=0.25):
        super().__init__()
        self.reg_loss = reg_loss
        self.ord_loss = ord_loss
        self.lambda_ord = lambda_ord

    def forward(self, pred_raw_unconstrained, ord_logits, target_log, target_ord):
        reg = self.reg_loss(pred_raw_unconstrained, target_log)
        ordl = self.ord_loss(ord_logits, target_ord)
        total = reg + self.lambda_ord * ordl
        return total, reg.detach(), ordl.detach()


# =========================
# Set Transformer Model
# =========================
class SetSelfAttentionBlock(nn.Module):
    def __init__(self, d_model, num_heads, mlp_ratio=4.0, dropout=0.1):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model)
        self.attn = nn.MultiheadAttention(
            embed_dim=d_model,
            num_heads=num_heads,
            dropout=dropout,
            batch_first=True,
        )
        self.norm2 = nn.LayerNorm(d_model)
        hidden = int(d_model * mlp_ratio)
        self.mlp = nn.Sequential(
            nn.Linear(d_model, hidden), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(hidden, d_model), nn.Dropout(dropout),
        )

    def forward(self, x, key_padding_mask=None):
        y = self.norm1(x)
        attn_out, _ = self.attn(
            y,
            y,
            y,
            need_weights=False,
            key_padding_mask=key_padding_mask,
        )
        x = x + attn_out
        x = x + self.mlp(self.norm2(x))
        return x


class ConditionalFeatureModulation(nn.Module):
    def __init__(self, cond_dim, feat_dim, hidden_dim=256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(cond_dim, hidden_dim), nn.GELU(),
            nn.Linear(hidden_dim, 2 * feat_dim),
        )
        self.scale = nn.Parameter(torch.tensor(0.25, dtype=torch.float32))

    def forward(self, x, cond):
        gamma_beta = self.net(cond)
        gamma, beta = torch.chunk(gamma_beta, 2, dim=-1)
        if x.dim() == 3:
            gamma = gamma.unsqueeze(1)
            beta = beta.unsqueeze(1)
        return x * (1.0 + self.scale * gamma) + self.scale * beta


class SetAttentionPooling(nn.Module):
    def __init__(self, d_model, hidden_dim=128):
        super().__init__()
        self.score = nn.Sequential(
            nn.Linear(d_model, hidden_dim), nn.GELU(), nn.Linear(hidden_dim, 1),
        )

    def forward(self, x, key_padding_mask=None):
        logits = self.score(x)
        if key_padding_mask is not None:
            logits = logits.masked_fill(key_padding_mask.unsqueeze(-1), float("-inf"))
        weights = torch.softmax(logits, dim=1)
        weights = torch.nan_to_num(weights, nan=0.0)
        return (weights * x).sum(dim=1)


class MonotonicOrdinalHead(nn.Module):
    def __init__(self, head_in, num_thresholds):
        super().__init__()
        self.num_thresholds = num_thresholds
        self.base = nn.Sequential(
            nn.Linear(head_in, 256), nn.GELU(), nn.Dropout(0.20),
            nn.Linear(256, 128), nn.GELU(), nn.Dropout(0.15),
        )
        self.first_logit = nn.Linear(128, 1)
        self.deltas = nn.Linear(128, max(num_thresholds - 1, 1))

    def forward(self, x):
        h = self.base(x)
        first = self.first_logit(h)
        if self.num_thresholds == 1:
            return first
        deltas = -F.softplus(self.deltas(h)[:, : self.num_thresholds - 1])
        return torch.cat([first, deltas], dim=-1).cumsum(dim=-1)


class BERFirstTokenSetTransformerBoundedLogHalfMultiTask(nn.Module):
    def __init__(self, max_past_seq_len, token_dim=4, first_token_dim=4,
                 threshold_dim=1, global_dim=7, d_model=128, num_set_layers=4,
                 num_heads=4, mlp_ratio=4.0, dropout=0.10,
                 num_ordinal_thresholds=NUM_ORDINAL_THRESHOLDS):
        super().__init__()
        self.max_past_seq_len = max_past_seq_len
        self.d_model = d_model
        self.num_ordinal_thresholds = num_ordinal_thresholds

        self.first_proj = nn.Sequential(
            nn.Linear(first_token_dim, d_model), nn.LayerNorm(d_model),
            nn.GELU(), nn.Dropout(dropout),
        )
        self.token_proj = nn.Sequential(
            nn.Linear(token_dim, d_model), nn.LayerNorm(d_model),
            nn.GELU(), nn.Dropout(dropout),
        )
        self.thr_embed = nn.Sequential(
            nn.Linear(threshold_dim, 32), nn.GELU(),
            nn.Linear(32, 32), nn.GELU(),
        )
        self.global_embed = nn.Sequential(
            nn.Linear(global_dim, 96), nn.GELU(),
            nn.Linear(96, 96), nn.GELU(),
        )

        cond_dim = 32 + 96
        self.first_cond_mod = ConditionalFeatureModulation(cond_dim, d_model, 256)
        self.set_cond_mod = ConditionalFeatureModulation(cond_dim, d_model, 256)

        self.set_blocks = nn.ModuleList([
            SetSelfAttentionBlock(d_model, num_heads, mlp_ratio, dropout)
            for _ in range(num_set_layers)
        ])

        self.first_refine = nn.Sequential(
            nn.LayerNorm(d_model), nn.Linear(d_model, d_model),
            nn.GELU(), nn.Dropout(dropout), nn.Linear(d_model, d_model),
        )
        self.final_set_norm = nn.LayerNorm(d_model)
        self.set_attn_pool = SetAttentionPooling(d_model, 128)

        head_in = d_model + 3 * d_model + cond_dim
        self.reg_head = nn.Sequential(
            nn.Linear(head_in, 256), nn.GELU(), nn.Dropout(0.20),
            nn.Linear(256, 128), nn.GELU(), nn.Dropout(0.15), nn.Linear(128, 1),
        )
        self.ord_head = MonotonicOrdinalHead(head_in, num_ordinal_thresholds)

    @staticmethod
    def raw_to_log10ber(x):
        log10_half = torch.log10(torch.tensor(0.5, device=x.device, dtype=x.dtype))
        return log10_half - F.softplus(x)

    @staticmethod
    def raw_to_ber(x):
        return torch.pow(
            10.0,
            BERFirstTokenSetTransformerBoundedLogHalfMultiTask.raw_to_log10ber(x),
        ).clamp(EPS, 0.5)

    def forward(self, first_token, past_tokens, global_feats, threshold, key_padding_mask=None):
        thr = self.thr_embed(threshold)
        g = self.global_embed(global_feats)
        cond = torch.cat([thr, g], dim=-1)

        first_x = self.first_proj(first_token)
        first_x = self.first_cond_mod(first_x, cond)
        first_x = first_x + self.first_refine(first_x)

        set_x = self.token_proj(past_tokens)
        set_x = self.set_cond_mod(set_x, cond)
        for block in self.set_blocks:
            set_x = block(set_x, key_padding_mask=key_padding_mask)
        set_x = self.final_set_norm(set_x)

        if key_padding_mask is not None:
            valid_mask = (~key_padding_mask).unsqueeze(-1).float()
        else:
            valid_mask = torch.ones(
                set_x.shape[0],
                set_x.shape[1],
                1,
                device=set_x.device,
                dtype=set_x.dtype,
            )

        pooled_attn = self.set_attn_pool(set_x, key_padding_mask=key_padding_mask)
        valid_counts = valid_mask.sum(dim=1).clamp(min=1.0)
        pooled_mean = (set_x * valid_mask).sum(dim=1) / valid_counts

        if key_padding_mask is not None:
            x_for_max = set_x.masked_fill(key_padding_mask.unsqueeze(-1), float("-inf"))
        else:
            x_for_max = set_x
        pooled_max = torch.nan_to_num(
            x_for_max.amax(dim=1),
            nan=0.0,
            posinf=0.0,
            neginf=0.0,
        )

        fused = torch.cat([first_x, pooled_attn, pooled_mean, pooled_max, cond], dim=-1)
        return self.reg_head(fused), self.ord_head(fused)


# =========================
# Evaluation
# =========================
def evaluate(model, loader, criterion, device):
    model.eval()

    total_loss = 0.0
    total_reg_loss = 0.0
    total_ord_loss = 0.0
    n_batches = 0

    all_preds_log = []
    all_targets_log = []
    all_pred_regions = []
    all_true_regions = []

    with torch.no_grad():
        for b_first, b_past, b_global, b_thr, b_y_log, b_ord, b_region, b_mask in loader:
            b_first = b_first.to(device, non_blocking=True)
            b_past = b_past.to(device, non_blocking=True)
            b_global = b_global.to(device, non_blocking=True)
            b_thr = b_thr.to(device, non_blocking=True)
            b_y_log = b_y_log.to(device, non_blocking=True)
            b_ord = b_ord.to(device, non_blocking=True)
            b_region = b_region.to(device, non_blocking=True)
            b_mask = b_mask.to(device, non_blocking=True)

            pred_raw_unconstrained, ord_logits = model(
                b_first,
                b_past,
                b_global,
                b_thr,
                key_padding_mask=b_mask,
            )

            if has_nonfinite_tensor(pred_raw_unconstrained):
                raise RuntimeError("Non-finite prediction during evaluation.")
            if has_nonfinite_tensor(ord_logits):
                raise RuntimeError("Non-finite ordinal logits during evaluation.")

            loss, reg_loss, ord_loss = criterion(
                pred_raw_unconstrained,
                ord_logits,
                b_y_log,
                b_ord,
            )

            if has_nonfinite_tensor(loss):
                raise RuntimeError("Non-finite loss during evaluation.")

            pred_log = model.raw_to_log10ber(pred_raw_unconstrained)
            pred_region = ordinal_logits_to_region_labels_torch(ord_logits)

            total_loss += loss.item()
            total_reg_loss += reg_loss.item()
            total_ord_loss += ord_loss.item()
            n_batches += 1

            all_preds_log.append(pred_log.cpu().numpy())
            all_targets_log.append(b_y_log.cpu().numpy())
            all_pred_regions.append(pred_region.cpu().numpy())
            all_true_regions.append(b_region.cpu().numpy())

    if n_batches == 0:
        raise RuntimeError("Evaluation loader produced zero batches.")

    preds_log = np.vstack(all_preds_log)
    targets_log = np.vstack(all_targets_log)

    preds_raw = np.clip(10 ** preds_log, EPS, 0.5)
    targets_raw = np.clip(10 ** targets_log, EPS, 0.5)

    rmse_log = float(np.sqrt(np.mean((preds_log - targets_log) ** 2)))
    mae_log = float(np.mean(np.abs(preds_log - targets_log)))
    factor_error = float(10 ** rmse_log)

    rmse_raw = float(np.sqrt(np.mean((preds_raw - targets_raw) ** 2)))
    mae_raw = float(np.mean(np.abs(preds_raw - targets_raw)))

    rel_err = (preds_raw - targets_raw) / np.maximum(targets_raw, 1e-6)
    rmse_rel = float(np.sqrt(np.mean(rel_err ** 2)))
    mae_rel = float(np.mean(np.abs(rel_err)))

    pred_regions = np.concatenate(all_pred_regions).reshape(-1)
    true_regions = np.concatenate(all_true_regions).reshape(-1)

    region_acc = float(np.mean(pred_regions == true_regions))
    region_mae = float(np.mean(np.abs(pred_regions - true_regions)))

    return {
        "loss": total_loss / n_batches,
        "reg_loss": total_reg_loss / n_batches,
        "ord_loss": total_ord_loss / n_batches,
        "rmse_log": rmse_log,
        "mae_log": mae_log,
        "factor_error": factor_error,
        "rmse_raw": rmse_raw,
        "mae_raw": mae_raw,
        "rmse_rel": rmse_rel,
        "mae_rel": mae_rel,
        "region_acc": region_acc,
        "region_mae": region_mae,
    }


def evaluate_by_target_range(model, loader, device):
    model.eval()
    all_preds_log = []
    all_targets_log = []

    with torch.no_grad():
        for b_first, b_past, b_global, b_thr, b_y_log, _, _, b_mask in loader:
            b_first = b_first.to(device, non_blocking=True)
            b_past = b_past.to(device, non_blocking=True)
            b_global = b_global.to(device, non_blocking=True)
            b_thr = b_thr.to(device, non_blocking=True)
            b_mask = b_mask.to(device, non_blocking=True)

            pred_raw_unconstrained, _ = model(
                b_first,
                b_past,
                b_global,
                b_thr,
                key_padding_mask=b_mask,
            )
            if has_nonfinite_tensor(pred_raw_unconstrained):
                raise RuntimeError("Non-finite prediction during per-range evaluation.")
            pred_log = model.raw_to_log10ber(pred_raw_unconstrained)
            all_preds_log.append(pred_log.cpu().numpy())
            all_targets_log.append(b_y_log.numpy())

    if not all_preds_log:
        raise RuntimeError("Range evaluation loader produced zero batches.")

    preds_log = np.vstack(all_preds_log).reshape(-1)
    targets_log = np.vstack(all_targets_log).reshape(-1)
    preds_raw = np.clip(10 ** preds_log, EPS, 0.5)
    targets_raw = np.clip(10 ** targets_log, EPS, 0.5)

    ranges = {
        "low_BER(y<1e-6)": targets_raw < 1e-6,
        "mid_BER(1e-6<=y<1e-3)": (targets_raw >= 1e-6) & (targets_raw < 1e-3),
        "high_BER(1e-3<=y<0.10)": (targets_raw >= 1e-3) & (targets_raw < 0.10),
        "upper_BER(0.10<=y<0.20)": (targets_raw >= 0.10) & (targets_raw < 0.20),
        "target_BER(0.20<=y<0.40)": (targets_raw >= 0.20) & (targets_raw < 0.40),
        "very_high_BER(0.40<=y<=0.50)": targets_raw >= 0.40,
    }

    metrics = {}
    for name, mask in ranges.items():
        if np.any(mask):
            err_log = preds_log[mask] - targets_log[mask]
            err_raw = preds_raw[mask] - targets_raw[mask]
            rel_err = err_raw / np.maximum(targets_raw[mask], 1e-6)
            metrics[name] = {
                "count": int(mask.sum()),
                "rmse_log": float(np.sqrt(np.mean(err_log ** 2))),
                "mae_log": float(np.mean(np.abs(err_log))),
                "factor_error": float(10 ** float(np.sqrt(np.mean(err_log ** 2)))),
                "rmse_raw": float(np.sqrt(np.mean(err_raw ** 2))),
                "mae_raw": float(np.mean(np.abs(err_raw))),
                "rmse_rel": float(np.sqrt(np.mean(rel_err ** 2))),
                "mae_rel": float(np.mean(np.abs(rel_err))),
                "bias_raw": float(np.mean(err_raw)),
                "bias_log": float(np.mean(err_log)),
                "p90_abs_raw": float(np.percentile(np.abs(err_raw), 90)),
                "p95_abs_raw": float(np.percentile(np.abs(err_raw), 95)),
            }
        else:
            metrics[name] = None
    return metrics


# =========================
# Selection scoring
# =========================
SELECTION_REGION_WEIGHTS = {
    "low_BER(y<1e-6)": 1,
    "mid_BER(1e-6<=y<1e-3)": 1,
    "high_BER(1e-3<=y<0.10)": 1,
    "upper_BER(0.10<=y<0.20)": 2.00,
    "target_BER(0.20<=y<0.40)": 2.00,
    "very_high_BER(0.40<=y<=0.50)": 2.0,
}
SELECTION_W_LOG_RMSE = 1.00
SELECTION_W_BIAS = 0.50
SELECTION_W_TAIL = 0.1
SELECTION_TAIL_CAP = 5.0


def compute_selection_score(val_range_metrics, val_metrics, min_count=50):
    log_rmse_sum = 0.0
    bias_sum = 0.0
    tail_sum = 0.0
    total_w = 0.0

    for region_name, w in SELECTION_REGION_WEIGHTS.items():
        m = val_range_metrics.get(region_name)
        if m is None or m["count"] < min_count:
            continue
        log_rmse_sum += w * m["rmse_log"]
        bias_sum += w * abs(m["bias_log"])
        if m["rmse_raw"] > 1e-9:
            tail_ratio = m["p95_abs_raw"] / (m["rmse_raw"] + 1e-9)
            tail_sum += w * min(tail_ratio, SELECTION_TAIL_CAP)
        else:
            tail_sum += w * 1.0
        total_w += w

    if total_w == 0:
        return {
            "composite": float(val_metrics["rmse_log"]),
            "log_rmse_weighted": float(val_metrics["rmse_log"]),
            "bias_weighted": 0.0,
            "tail_weighted": 0.0,
            "geometric_factor_error": float(val_metrics["factor_error"]),
            "fallback": True,
        }

    log_rmse_weighted = log_rmse_sum / total_w
    bias_weighted = bias_sum / total_w
    tail_weighted = tail_sum / total_w
    composite = (SELECTION_W_LOG_RMSE * log_rmse_weighted
                 + SELECTION_W_BIAS * bias_weighted
                 + SELECTION_W_TAIL * tail_weighted)
    return {
        "composite": float(composite),
        "log_rmse_weighted": float(log_rmse_weighted),
        "bias_weighted": float(bias_weighted),
        "tail_weighted": float(tail_weighted),
        "geometric_factor_error": float(10 ** log_rmse_weighted),
        "fallback": False,
    }


def is_acceptable_checkpoint(val_range_metrics, val_metrics):
    target = val_range_metrics.get("target_BER(0.20<=y<0.40)")
    if target is None or target["count"] < 100:
        return False, "target region too small"
    for region_name, m in val_range_metrics.items():
        if m is None or m["count"] < 50:
            continue
        if abs(m["bias_log"]) > 0.15:
            return False, f"{region_name} has bias_log={m['bias_log']:.3f}"
    if val_metrics["rmse_log"] > 0.30:
        return False, f"overall rmse_log={val_metrics['rmse_log']:.3f} too high"
    return True, "ok"


# =========================
# Fine-tune main
# =========================
def finetune():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Executing on: {device}")
    print(f"\nFINE-TUNING from: {PRETRAINED_MODEL_PATH}")
    print(f"  Pretrained scalers: {PRETRAINED_SCALER_PATH}")
    print(f"  Reuse scalers: {REUSE_SCALERS}")

    if not os.path.exists(PRETRAINED_MODEL_PATH):
        raise FileNotFoundError(f"Pretrained model not found: {PRETRAINED_MODEL_PATH}")
    if not os.path.exists(PRETRAINED_SCALER_PATH):
        raise FileNotFoundError(f"Pretrained scalers not found: {PRETRAINED_SCALER_PATH}")

    pretrained_scalers = joblib.load(PRETRAINED_SCALER_PATH)
    print(f"  Pretrained global_dim: {pretrained_scalers.get('global_dim', '?')}")
    print(f"  Pretrained threshold_feature_mode: {pretrained_scalers.get('threshold_feature_mode', THRESHOLD_FEATURE_MODE)}")

    train_loader, val_loader, test_loader, scalers, aux_info, max_past_seq_len = (
        prepare_data_for_finetune(
            DATA_PATHS,
            batch_size=256,
            nrows_per_dataset=NROWS_PER_DATASET,
            num_workers=DATALOADER_NUM_WORKERS,
            pretrained_scalers=pretrained_scalers,
            reuse_scalers=REUSE_SCALERS,
        )
    )

    joblib.dump(scalers, SCALER_SAVE_PATH)
    print(f"\nFine-tune scalers saved to: {SCALER_SAVE_PATH}")

    model = BERFirstTokenSetTransformerBoundedLogHalfMultiTask(
        max_past_seq_len=max_past_seq_len,
        token_dim=4,
        first_token_dim=4,
        threshold_dim=1,
        global_dim=7,
        d_model=128,
        num_set_layers=4,
        num_heads=4,
        mlp_ratio=4.0,
        dropout=0.10,
        num_ordinal_thresholds=NUM_ORDINAL_THRESHOLDS,
    ).to(device)

    pretrained_state = safe_torch_load_state_dict(PRETRAINED_MODEL_PATH, map_location=device)
    model.load_state_dict(pretrained_state, strict=True)
    print("Pretrained normalized-threshold + monotonic-ordinal weights loaded with strict=True.")

    print("\n--- Pre-fine-tune evaluation (pretrained model on new data) ---")
    pre_criterion = MultiTaskBERLoss(
        StableMultiObjectiveBERBoundedLogLoss(),
        OrdinalBCELoss(),
        lambda_ord=0.25,
    )

    try:
        pre_val = evaluate(model, val_loader, pre_criterion, device)
        pre_range = evaluate_by_target_range(model, val_loader, device)
        pre_sel = compute_selection_score(pre_range, pre_val)
        print(
            f"  Pretrained on new val: composite={pre_sel['composite']:.5f} | "
            f"factor~{pre_sel['geometric_factor_error']:.3f}x | "
            f"RMSE(log10)={pre_val['rmse_log']:.4f} | "
            f"region_acc={pre_val['region_acc']:.4f}"
        )
        for rng, m in pre_range.items():
            if m is not None:
                print(
                    f"    {rng}: factor~{m['factor_error']:.3f}x, "
                    f"bias_log={m['bias_log']:+.4f}, count={m['count']}"
                )
            else:
                print(f"    {rng}: no samples")
    except Exception as e:
        print(f"  Pre-fine-tune evaluation failed/skipped: {e}")
        pre_val = {"rmse_log": float("inf"), "factor_error": float("inf")}
        pre_range = {}
        pre_sel = {
            "composite": float("inf"),
            "log_rmse_weighted": float("inf"),
            "bias_weighted": float("inf"),
        }

    print("\nFine-tune hyperparameters:")
    print(f"  LR={FINETUNE_LR}, weight_decay={FINETUNE_WEIGHT_DECAY}")
    print(
        f"  max_epochs={FINETUNE_MAX_EPOCHS}, min_epochs={FINETUNE_MIN_EPOCHS}, "
        f"patience={FINETUNE_PATIENCE}"
    )

    optimizer = optim.AdamW(
        model.parameters(),
        lr=FINETUNE_LR,
        weight_decay=FINETUNE_WEIGHT_DECAY,
    )

    reg_loss = StableMultiObjectiveBERBoundedLogLoss(
        log_delta=0.35,
        raw_delta=0.006,
        rel_delta=0.03,
        alpha_log=0.45,
        beta_raw=0.35,
        gamma_rel=0.20,
        use_regime_weights=True,
    )

    boosted_pos = aux_info["ordinal_pos_weights"].copy()
    for i, thr in enumerate(ORDINAL_THRESHOLDS):
        if 0.20 <= thr <= 0.40:
            boosted_pos[i] *= 2.5
        elif 0.15 <= thr < 0.20:
            boosted_pos[i] *= 1.5
        elif 0.40 < thr <= 0.45:
            boosted_pos[i] *= 1.5

    ord_loss = OrdinalBCELoss(
        pos_weight=torch.tensor(boosted_pos, dtype=torch.float32, device=device),
        reduction="mean",
    )
    criterion = MultiTaskBERLoss(reg_loss=reg_loss, ord_loss=ord_loss, lambda_ord=0.25)

    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="min",
        patience=SCHEDULER_PATIENCE,
        factor=SCHEDULER_FACTOR,
    )

    best_checkpoints = {
        "composite": {"score": float("inf"), "state": None, "epoch": -1},
        "composite_ema": {"score": float("inf"), "state": None, "epoch": -1},
        "log_rmse": {"score": float("inf"), "state": None, "epoch": -1},
        "target_mae": {"score": float("inf"), "state": None, "epoch": -1},
        "low_bias": {"score": float("inf"), "state": None, "epoch": -1},
    }
    EMA_ALPHA = 0.5
    ema_composite = None
    wait = 0

    try:
        base_state = copy.deepcopy(model.state_dict())
        pre_acceptable, _ = is_acceptable_checkpoint(pre_range, pre_val)
        if pre_acceptable:
            for ckpt_name, score_value in [
                ("composite", pre_sel["composite"]),
                ("composite_ema", pre_sel["composite"]),
                ("log_rmse", pre_sel["log_rmse_weighted"]),
                (
                    "target_mae",
                    pre_range["target_BER(0.20<=y<0.40)"]["mae_raw"]
                    if pre_range.get("target_BER(0.20<=y<0.40)") else float("inf"),
                ),
                ("low_bias", pre_sel["bias_weighted"]),
            ]:
                best_checkpoints[ckpt_name] = {
                    "score": float(score_value),
                    "state": base_state,
                    "epoch": 0,
                }
            print("\nSeeded best checkpoints with pretrained baseline scores.")
    except Exception as e:
        print(f"  Could not seed pretrained baseline as best: {e}")

    print("\nSelection: weighted log-RMSE + bias + tail penalties")
    print(f"  Region weights: {SELECTION_REGION_WEIGHTS}")
    print(
        f"  Composite weights: log_rmse={SELECTION_W_LOG_RMSE}, "
        f"bias={SELECTION_W_BIAS}, tail={SELECTION_W_TAIL}"
    )
    print(f"  EMA alpha: {EMA_ALPHA}\n")

    training_broke = False

    for epoch in range(FINETUNE_MAX_EPOCHS):
        model.train()
        running_loss = 0.0
        running_reg = 0.0
        running_ord = 0.0
        num_train_batches = 0

        for batch_idx, (b_first, b_past, b_global, b_thr, b_y_log, b_ord, _, b_mask) in enumerate(train_loader):
            b_first = b_first.to(device, non_blocking=True)
            b_past = b_past.to(device, non_blocking=True)
            b_global = b_global.to(device, non_blocking=True)
            b_thr = b_thr.to(device, non_blocking=True)
            b_y_log = b_y_log.to(device, non_blocking=True)
            b_ord = b_ord.to(device, non_blocking=True)
            b_mask = b_mask.to(device, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)
            pred_raw_unconstrained, ord_logits = model(
                b_first,
                b_past,
                b_global,
                b_thr,
                key_padding_mask=b_mask,
            )

            if has_nonfinite_tensor(pred_raw_unconstrained):
                print(f"Non-finite prediction at epoch {epoch + 1}, batch {batch_idx + 1}.")
                training_broke = True
                break
            if has_nonfinite_tensor(ord_logits):
                print(f"Non-finite ordinal logits at epoch {epoch + 1}, batch {batch_idx + 1}.")
                training_broke = True
                break

            loss, reg_part, ord_part = criterion(
                pred_raw_unconstrained,
                ord_logits,
                b_y_log,
                b_ord,
            )
            if has_nonfinite_tensor(loss):
                print(f"Non-finite loss at epoch {epoch + 1}, batch {batch_idx + 1}.")
                training_broke = True
                break

            loss.backward()
            grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=GRAD_CLIP)
            if not torch.isfinite(grad_norm):
                print(f"Non-finite gradient norm at epoch {epoch + 1}, batch {batch_idx + 1}.")
                training_broke = True
                break

            optimizer.step()

            running_loss += float(loss.item())
            running_reg += float(reg_part.item())
            running_ord += float(ord_part.item())
            num_train_batches += 1

        if training_broke:
            break

        if num_train_batches == 0:
            print(f"No training batches were produced at epoch {epoch + 1}. Stopping.")
            break

        train_loss = running_loss / num_train_batches
        train_reg = running_reg / num_train_batches
        train_ord = running_ord / num_train_batches

        val_metrics = evaluate(model, val_loader, criterion, device)
        val_range_metrics = evaluate_by_target_range(model, val_loader, device)
        val_selection = compute_selection_score(val_range_metrics, val_metrics)
        acceptable, accept_reason = is_acceptable_checkpoint(val_range_metrics, val_metrics)

        composite = val_selection["composite"]
        if ema_composite is None:
            ema_composite = composite
        else:
            ema_composite = EMA_ALPHA * composite + (1.0 - EMA_ALPHA) * ema_composite

        scheduler.step(composite)
        current_lr = optimizer.param_groups[0]["lr"]

        print(
            f"Epoch {epoch + 1:03d}/{FINETUNE_MAX_EPOCHS} | "
            f"train={train_loss:.5f} reg={train_reg:.5f} ord={train_ord:.5f} | "
            f"val={val_metrics['loss']:.5f} rmse_log={val_metrics['rmse_log']:.4f} "
            f"factor~{val_metrics['factor_error']:.3f}x | "
            f"region_acc={val_metrics['region_acc']:.4f} | "
            f"composite={composite:.5f} ema={ema_composite:.5f} | "
            f"lr={current_lr:.2e} | acceptable={acceptable} ({accept_reason})"
        )

        target_stats = val_range_metrics.get("target_BER(0.20<=y<0.40)")
        if target_stats is not None:
            print(
                f"  target_BER: count={target_stats['count']} | "
                f"factor~{target_stats['factor_error']:.3f}x | "
                f"MAE_raw={target_stats['mae_raw']:.6f} | "
                f"bias_log={target_stats['bias_log']:+.4f} | "
                f"P95={target_stats['p95_abs_raw']:.6f}"
            )

        improved = False

        if acceptable and composite < best_checkpoints["composite"]["score"]:
            best_checkpoints["composite"] = {
                "score": float(composite),
                "state": copy.deepcopy(model.state_dict()),
                "epoch": epoch + 1,
            }
            print(f"  New best composite: {composite:.5f}")
            improved = True

        if acceptable and ema_composite < best_checkpoints["composite_ema"]["score"]:
            best_checkpoints["composite_ema"] = {
                "score": float(ema_composite),
                "state": copy.deepcopy(model.state_dict()),
                "epoch": epoch + 1,
            }
            print(f"  New best composite EMA: {ema_composite:.5f}")
            improved = True

        log_rmse_score = val_selection["log_rmse_weighted"]
        if acceptable and log_rmse_score < best_checkpoints["log_rmse"]["score"]:
            best_checkpoints["log_rmse"] = {
                "score": float(log_rmse_score),
                "state": copy.deepcopy(model.state_dict()),
                "epoch": epoch + 1,
            }
            print(f"  New best weighted log-RMSE: {log_rmse_score:.5f}")
            improved = True

        if acceptable and target_stats is not None:
            target_mae = target_stats["mae_raw"]
            if target_mae < best_checkpoints["target_mae"]["score"]:
                best_checkpoints["target_mae"] = {
                    "score": float(target_mae),
                    "state": copy.deepcopy(model.state_dict()),
                    "epoch": epoch + 1,
                }
                print(f"  New best target MAE raw: {target_mae:.6f}")
                improved = True

        bias_score = val_selection["bias_weighted"]
        if acceptable and bias_score < best_checkpoints["low_bias"]["score"]:
            best_checkpoints["low_bias"] = {
                "score": float(bias_score),
                "state": copy.deepcopy(model.state_dict()),
                "epoch": epoch + 1,
            }
            print(f"  New best low-bias score: {bias_score:.5f}")
            improved = True

        torch.save(model.state_dict(), LAST_MODEL_SAVE_PATH)

        if improved:
            wait = 0
        else:
            wait += 1

        if (epoch + 1) >= FINETUNE_MIN_EPOCHS and wait >= FINETUNE_PATIENCE:
            print(
                f"Early stopping at epoch {epoch + 1}: "
                f"no acceptable improvement for {wait} epochs."
            )
            break

    if training_broke:
        print("Training stopped because of non-finite values.")

    primary_choice = "composite_ema"
    if best_checkpoints[primary_choice]["state"] is None:
        primary_choice = "composite"
    if best_checkpoints[primary_choice]["state"] is None:
        primary_choice = "log_rmse"

    if best_checkpoints[primary_choice]["state"] is not None:
        model.load_state_dict(best_checkpoints[primary_choice]["state"])
        torch.save(model.state_dict(), BEST_MODEL_SAVE_PATH)
        print(
            f"\nPrimary best ({primary_choice}, "
            f"FT-epoch {best_checkpoints[primary_choice]['epoch']}) "
            f"saved to: {BEST_MODEL_SAVE_PATH}"
        )
    else:
        print("\nWarning: no valid best checkpoint was found. Saving last model as best fallback.")
        torch.save(model.state_dict(), BEST_MODEL_SAVE_PATH)

    test_metrics = evaluate(model, test_loader, criterion, device)
    print(
        f"\nTest Loss: {test_metrics['loss']:.4f} | "
        f"Test RMSE(log10): {test_metrics['rmse_log']:.4f} | "
        f"~{test_metrics['factor_error']:.2f}x | "
        f"Test Region Acc: {test_metrics['region_acc']:.4f}"
    )

    range_metrics = evaluate_by_target_range(model, test_loader, device)
    print("\nPer-range test diagnostics:")
    for name, stats in range_metrics.items():
        if stats is None:
            print(f"  {name}: no samples")
        else:
            print(
                f"  {name} | count={stats['count']} | "
                f"factor~{stats['factor_error']:.2f}x | "
                f"RMSE(log10)={stats['rmse_log']:.4f} | "
                f"MAE_raw={stats['mae_raw']:.6f} | "
                f"bias_log={stats['bias_log']:+.4f} | "
                f"P90={stats['p90_abs_raw']:.6f} | "
                f"P95={stats['p95_abs_raw']:.6f}"
            )

    return model


if __name__ == "__main__":
    os.makedirs(BASE_DIR, exist_ok=True)

    if RUN_PARQUET_CONVERSION:
        print("Running CSV -> cleaned Parquet conversion.")
        build_clean_parquet_files(overwrite=OVERWRITE_PARQUET)

    missing_parquet = [p for p in DATA_PATHS if not os.path.exists(p)]
    if missing_parquet:
        print("Critical Error: Missing Parquet dataset files:")
        for p in missing_parquet:
            print(f"  - {p}")
        print("Set RUN_PARQUET_CONVERSION=True, or run build_clean_parquet_files() first.")
    else:
        print("Fine-tuning from Parquet datasets:")
        for p in DATA_PATHS:
            print(f"  - {p}")
        print(f"Row cap per dataset: {NROWS_PER_DATASET}")
        finetune()

Fine-tuning from Parquet datasets:
  - ./../ber_data_generation/data/data_physics_total.cleaned.parquet
  - ./../ber_data_generation/data/data_random_total.cleaned.parquet
Row cap per dataset: 10000000
Executing on: cuda

FINE-TUNING from: ./random_extra_multitask_finetune_best_normthr_monoord2.pth
  Pretrained scalers: ./random_extra_multitask_finetune_scalers_normthr_monoord2.pkl
  Reuse scalers: True
  Pretrained global_dim: 7
  Pretrained threshold_feature_mode: normalized_logit

Streaming Parquet data:
  ./../ber_data_generation/data/data_physics_total.cleaned.parquet
  ./../ber_data_generation/data/data_random_total.cleaned.parquet
  tap columns: 15
  past length: 14
  rows/file cap: 10000000

Fine-tune scalers saved to: ./random_extra_multitask_finetune_scalers_normthr_monoord3.pkl
Pretrained normalized-threshold + monotonic-ordinal weights loaded with strict=True.

--- Pre-fine-tune evaluation (pretrained model on new data) ---
  Pretrained on new val: composite=0.19518 | facto

Epoch 022/60 | train=0.00248 reg=0.00203 ord=0.00183 | val=0.00061 rmse_log=0.0193 factor~1.046x | region_acc=0.9924 | composite=0.19481 ema=0.19516 | lr=1.25e-06 | acceptable=True (ok)
  target_BER: count=592624 | factor~1.012x | MAE_raw=0.001949 | bias_log=+0.0003 | P95=0.006410
  New best target MAE raw: 0.001949
Epoch 023/60 | train=0.00248 reg=0.00203 ord=0.00182 | val=0.00062 rmse_log=0.0194 factor~1.046x | region_acc=0.9922 | composite=0.19513 ema=0.19514 | lr=6.25e-07 | acceptable=True (ok)
  target_BER: count=592624 | factor~1.012x | MAE_raw=0.001999 | bias_log=+0.0007 | P95=0.006531
Epoch 024/60 | train=0.00248 reg=0.00203 ord=0.00183 | val=0.00060 rmse_log=0.0189 factor~1.044x | region_acc=0.9924 | composite=0.19521 ema=0.19518 | lr=6.25e-07 | acceptable=True (ok)
  target_BER: count=592624 | factor~1.012x | MAE_raw=0.001963 | bias_log=+0.0005 | P95=0.006441
Epoch 025/60 | train=0.00248 reg=0.00203 ord=0.00182 | val=0.00060 rmse_log=0.0189 factor~1.044x | region_acc=0.9924 |

Epoch 050/60 | train=0.00248 reg=0.00203 ord=0.00182 | val=0.00060 rmse_log=0.0194 factor~1.046x | region_acc=0.9925 | composite=0.19877 ema=0.19877 | lr=1.95e-08 | acceptable=True (ok)
  target_BER: count=592624 | factor~1.012x | MAE_raw=0.001919 | bias_log=+0.0006 | P95=0.006382
Epoch 051/60 | train=0.00248 reg=0.00202 ord=0.00182 | val=0.00060 rmse_log=0.0194 factor~1.046x | region_acc=0.9925 | composite=0.19892 ema=0.19885 | lr=1.95e-08 | acceptable=True (ok)
  target_BER: count=592624 | factor~1.012x | MAE_raw=0.001923 | bias_log=+0.0006 | P95=0.006388
Epoch 052/60 | train=0.00248 reg=0.00203 ord=0.00182 | val=0.00060 rmse_log=0.0193 factor~1.046x | region_acc=0.9925 | composite=0.19869 ema=0.19877 | lr=1.95e-08 | acceptable=True (ok)
  target_BER: count=592624 | factor~1.012x | MAE_raw=0.001922 | bias_log=+0.0006 | P95=0.006387
Epoch 053/60 | train=0.00248 reg=0.00203 ord=0.00182 | val=0.00060 rmse_log=0.0195 factor~1.046x | region_acc=0.9924 | composite=0.19889 ema=0.19883 | lr=

In [9]:
import os
import joblib
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy.special import erfc
from itertools import product as iproduct

EPS = 1e-12
PLOT_DIR = "./plots_mixed_newdata-experiments2/8"
os.makedirs(PLOT_DIR, exist_ok=True)

# =========================
# Paths
# =========================
MODEL_PATH = "random_extra_multitask_finetune_best_normthr_monoord2.pth"
SCALER_PATH = "random_extra_multitask_finetune_scalers_normthr_monoord2.pkl"

# =========================
# Config
# =========================
PHYSICS_MAX_MEM_LEN = 20
PHYSICS_MIN_MEM_LEN = 16
ARRIVAL_COVERAGE = 0.7
N_THRESHOLDS = 500
RANDOM_SEED = 60

# =========================
# Test-time smoothing config
# =========================
# The smoother recomputes ALL threshold-dependent features for thr-eps, thr, thr+eps.
# This is important because threshold enters both thr_t and global features such as z0/z1/HMG.
USE_TEST_TIME_SMOOTHING = False
SMOOTHING_EPS_FRAC = 0.005     # eps = SMOOTHING_EPS_FRAC * threshold sweep span
SMOOTHING_SPACE = "log"         # "log" is usually best globally; "raw" can look better at high BER
SMOOTHING_WEIGHTS = (0.25, 0.50, 0.25)

# This must match training. If the scaler file contains "threshold_feature_mode",
# that saved value is used; this default is only a fallback.
THRESHOLD_FEATURE_MODE = "normalized_logit"
THRESHOLD_NORM_CLIP_EPS = 1e-5


ORDINAL_THRESHOLDS = [
    1e-6, 1e-5, 1e-4, 1e-3, 1e-2, 1e-1,
    0.15, 0.20, 0.25, 0.30, 0.35, 0.40, 0.45,
]
NUM_ORDINAL_THRESHOLDS = len(ORDINAL_THRESHOLDS)
NUM_REGION_CLASSES = NUM_ORDINAL_THRESHOLDS + 1

REGION_LABELS = {
    0: "y < 1e-6",
    1: "1e-6 <= y < 1e-5",
    2: "1e-5 <= y < 1e-4",
    3: "1e-4 <= y < 1e-3",
    4: "1e-3 <= y < 1e-2",
    5: "1e-2 <= y < 1e-1",
    6: "0.10 <= y < 0.15",
    7: "0.15 <= y < 0.20",
    8: "0.20 <= y < 0.25",
    9: "0.25 <= y < 0.30",
    10: "0.30 <= y < 0.35",
    11: "0.35 <= y < 0.40",
    12: "0.40 <= y < 0.45",
    13: "0.45 <= y <= 0.50",
}


# =========================
# Physics helpers
# =========================
def Fhit_function(radius, distance, diffusionCoef, t):
    if t <= 0:
        return 0.0
    return (radius / (distance + radius)) * erfc(distance / np.sqrt(4 * diffusionCoef * t))


def calculate_hitting_probabilities(mem_len, radius, distance, diffusionCoef, Ts):
    P = np.zeros(mem_len)
    for i in range(mem_len):
        t_end = (i + 1) * Ts
        t_start = i * Ts
        P[i] = Fhit_function(radius, distance, diffusionCoef, t_end) - Fhit_function(
            radius, distance, diffusionCoef, t_start
        )
    return P


def calculate_ber_vectorized(mem_len, threshold, P_scaled, variances):
    P_arr = np.asarray(P_scaled, dtype=float)[:mem_len]
    vars_arr = np.asarray(variances, dtype=float)[:mem_len]

    seqs = np.array(list(iproduct([0, 1], repeat=mem_len)), dtype=np.float64)[:, ::-1]
    c_bit = seqs[:, 0]

    mu = (seqs * P_arr).sum(axis=1)
    var_total = (seqs * vars_arr).sum(axis=1)
    std = np.sqrt(np.maximum(var_total, 0.0))

    pe = np.empty_like(mu)
    zero_std = (std == 0)
    if np.any(zero_std):
        pe[zero_std & (c_bit == 1)] = np.where(
            mu[zero_std & (c_bit == 1)] < threshold, 1.0, 0.0)
        pe[zero_std & (c_bit == 0)] = np.where(
            mu[zero_std & (c_bit == 0)] >= threshold, 1.0, 0.0)
    nz = ~zero_std
    if np.any(nz):
        pe[nz & (c_bit == 1)] = 0.5 * erfc(
            (mu[nz & (c_bit == 1)] - threshold) / (std[nz & (c_bit == 1)] * np.sqrt(2)))
        pe[nz & (c_bit == 0)] = 0.5 * erfc(
            (threshold - mu[nz & (c_bit == 0)]) / (std[nz & (c_bit == 0)] * np.sqrt(2)))
    return float(np.mean(pe))


def raw_to_region_labels_np(y_raw):
    y = np.asarray(y_raw).reshape(-1)
    return np.searchsorted(np.asarray(ORDINAL_THRESHOLDS), y, side="right").astype(np.int64)


def ordinal_logits_to_region_labels_torch(logits):
    probs = torch.sigmoid(logits)
    passed = (probs >= 0.5).sum(dim=1)
    return passed.long()


# =========================
# Shared scaling helper
# =========================
def apply_shared_scale(data, mean, std, valid_mask=None):
    scaled = ((data - mean) / std).astype(np.float32)
    if valid_mask is not None and scaled.ndim == 2 and valid_mask.shape == scaled.shape:
        scaled = scaled * valid_mask.astype(np.float32)
    return scaled


def safe_harmonic_z(z0, z1, denom_eps=1e-9):
    """Finite-safe version of 2/(1/z0 + 1/z1) = 2*z0*z1/(z0+z1).

    The direct reciprocal form can create +/-inf when z0 or z1 is near zero.
    This helper uses np.divide(..., where=...) and sanitizes non-finite outputs.
    """
    z0 = np.asarray(z0, dtype=np.float32)
    z1 = np.asarray(z1, dtype=np.float32)
    num = (2.0 * z0 * z1).astype(np.float32)
    den = (z0 + z1).astype(np.float32)
    out = np.divide(
        num,
        den,
        out=np.zeros_like(num, dtype=np.float32),
        where=np.abs(den) > denom_eps,
    ).astype(np.float32)
    return np.nan_to_num(out, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)


def compute_threshold_feature_np(threshold_raw, total_scaled_taps, mode=THRESHOLD_FEATURE_MODE):
    thr = np.asarray(threshold_raw, dtype=np.float32).reshape(-1, 1)
    total = np.asarray(total_scaled_taps, dtype=np.float32).reshape(-1, 1)
    denom = np.maximum(total, EPS).astype(np.float32)

    if mode == "normalized":
        feat = (thr / denom).astype(np.float32)
    elif mode == "normalized_logit":
        u = (thr / denom).astype(np.float32)
        u = np.clip(u, THRESHOLD_NORM_CLIP_EPS, 1.0 - THRESHOLD_NORM_CLIP_EPS)
        feat = np.log(u / (1.0 - u)).astype(np.float32)
    elif mode == "log10_raw":
        feat = np.log10(thr + EPS).astype(np.float32)
    else:
        raise ValueError(f"Unknown threshold feature mode: {mode}")

    feat = np.nan_to_num(feat, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)
    return feat


# =========================
# Generate one physical scenario
# =========================
def generate_physical_case(rng, min_mem_len, max_mem_len, arrival_coverage, max_tries=5000):
    for _ in range(max_tries):
        radius = rng.uniform(3.0, 5.0)
        distance = rng.uniform(10.0, 15.0)
        diff = rng.uniform(50.0, 75.0)
        Ts = rng.uniform(0.3,0.4)
        N = int(10 ** rng.uniform(3.0, 6.0))

        f_inf = radius / (radius + distance)
        target = arrival_coverage * f_inf

        cumsum, k = 0.0, 0
        while k < max_mem_len:
            pk = Fhit_function(radius, distance, diff, (k + 1) * Ts) - Fhit_function(
                radius, distance, diff, k * Ts)
            cumsum += pk
            k += 1
            if cumsum >= target:
                break

        if k < min_mem_len:
            continue

        P_ext = calculate_hitting_probabilities(k + 1, radius, distance, diff, Ts)
        P_main = P_ext[:k]
        P_extra = float(P_ext[k]) if k < len(P_ext) else 0.0

        P_scaled = P_main * N
        variances = N * P_main * (1.0 - P_main)

        return {
            "radius": radius, "distance": distance, "diffusion": diff,
            "Ts": Ts, "N": N, "mem_len": k, "P": P_main,
            "P_scaled": P_scaled, "variances": variances,
            "P_mem_len_extra": P_extra,
            "P_mem_len_extra_var": P_extra * (1.0 - P_extra),
        }

    raise RuntimeError(
        f"Could not generate a physical case with mem_len >= {min_mem_len} "
        f"after {max_tries} tries."
    )


# =========================
# Model (matches v3 training: global_dim=7, global_embed=96, cond_dim=128)
# =========================
class ConditionalFeatureModulation(nn.Module):
    def __init__(self, cond_dim, feat_dim, hidden_dim=256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(cond_dim, hidden_dim), nn.GELU(),
            nn.Linear(hidden_dim, 2 * feat_dim),
        )
        self.scale = nn.Parameter(torch.tensor(0.25, dtype=torch.float32))

    def forward(self, x, cond):
        gamma_beta = self.net(cond)
        gamma, beta = torch.chunk(gamma_beta, 2, dim=-1)
        if x.dim() == 3:
            gamma = gamma.unsqueeze(1)
            beta = beta.unsqueeze(1)
        return x * (1.0 + self.scale * gamma) + self.scale * beta


class SetSelfAttentionBlock(nn.Module):
    def __init__(self, d_model, num_heads, mlp_ratio=4.0, dropout=0.10):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model)
        self.attn = nn.MultiheadAttention(
            embed_dim=d_model, num_heads=num_heads,
            dropout=dropout, batch_first=True)
        self.norm2 = nn.LayerNorm(d_model)
        hidden = int(d_model * mlp_ratio)
        self.mlp = nn.Sequential(
            nn.Linear(d_model, hidden), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(hidden, d_model), nn.Dropout(dropout))

    def forward(self, x, key_padding_mask=None):
        y = self.norm1(x)
        attn_out, _ = self.attn(y, y, y, need_weights=False, key_padding_mask=key_padding_mask)
        x = x + attn_out
        x = x + self.mlp(self.norm2(x))
        return x


class SetAttentionPooling(nn.Module):
    def __init__(self, d_model, hidden_dim=128):
        super().__init__()
        self.score = nn.Sequential(
            nn.Linear(d_model, hidden_dim), nn.GELU(), nn.Linear(hidden_dim, 1))

    def forward(self, x, key_padding_mask=None):
        logits = self.score(x)
        if key_padding_mask is not None:
            logits = logits.masked_fill(key_padding_mask.unsqueeze(-1), float("-inf"))
        weights = torch.softmax(logits, dim=1)
        weights = torch.nan_to_num(weights, nan=0.0)
        return (weights * x).sum(dim=1)


class MonotonicOrdinalHead(nn.Module):
    def __init__(self, head_in, num_thresholds):
        super().__init__()
        self.num_thresholds = num_thresholds
        self.base = nn.Sequential(
            nn.Linear(head_in, 256), nn.GELU(), nn.Dropout(0.20),
            nn.Linear(256, 128), nn.GELU(), nn.Dropout(0.15),
        )
        self.first_logit = nn.Linear(128, 1)
        self.deltas = nn.Linear(128, max(num_thresholds - 1, 1))

    def forward(self, x):
        h = self.base(x)
        first = self.first_logit(h)
        if self.num_thresholds == 1:
            return first
        deltas = -F.softplus(self.deltas(h)[:, : self.num_thresholds - 1])
        return torch.cat([first, deltas], dim=-1).cumsum(dim=-1)


class BERFirstTokenSetTransformerBoundedLogHalfMultiTask(nn.Module):
    def __init__(self, max_past_seq_len, token_dim=4, first_token_dim=4,
                 threshold_dim=1, global_dim=7, d_model=128, num_set_layers=4,
                 num_heads=4, mlp_ratio=4.0, dropout=0.10,
                 num_ordinal_thresholds=NUM_ORDINAL_THRESHOLDS):
        super().__init__()
        self.max_past_seq_len = max_past_seq_len
        self.d_model = d_model
        self.num_ordinal_thresholds = num_ordinal_thresholds

        self.first_proj = nn.Sequential(
            nn.Linear(first_token_dim, d_model), nn.LayerNorm(d_model),
            nn.GELU(), nn.Dropout(dropout))
        self.token_proj = nn.Sequential(
            nn.Linear(token_dim, d_model), nn.LayerNorm(d_model),
            nn.GELU(), nn.Dropout(dropout))
        self.thr_embed = nn.Sequential(
            nn.Linear(threshold_dim, 32), nn.GELU(),
            nn.Linear(32, 32), nn.GELU())
        self.global_embed = nn.Sequential(
            nn.Linear(global_dim, 96), nn.GELU(),
            nn.Linear(96, 96), nn.GELU())

        cond_dim = 32 + 96
        self.first_cond_mod = ConditionalFeatureModulation(cond_dim, d_model, 256)
        self.set_cond_mod = ConditionalFeatureModulation(cond_dim, d_model, 256)

        self.set_blocks = nn.ModuleList([
            SetSelfAttentionBlock(d_model, num_heads, mlp_ratio, dropout)
            for _ in range(num_set_layers)])

        self.first_refine = nn.Sequential(
            nn.LayerNorm(d_model), nn.Linear(d_model, d_model),
            nn.GELU(), nn.Dropout(dropout), nn.Linear(d_model, d_model))
        self.final_set_norm = nn.LayerNorm(d_model)
        self.set_attn_pool = SetAttentionPooling(d_model, 128)

        set_summary_dim = 3 * d_model
        head_in = d_model + set_summary_dim + cond_dim

        self.reg_head = nn.Sequential(
            nn.Linear(head_in, 256), nn.GELU(), nn.Dropout(0.20),
            nn.Linear(256, 128), nn.GELU(), nn.Dropout(0.15),
            nn.Linear(128, 1))
        self.ord_head = MonotonicOrdinalHead(head_in, num_ordinal_thresholds)

    @staticmethod
    def raw_to_log10ber(pred_raw_unconstrained):
        log10_half = torch.log10(torch.tensor(
            0.5, device=pred_raw_unconstrained.device,
            dtype=pred_raw_unconstrained.dtype))
        return log10_half - F.softplus(pred_raw_unconstrained)

    @staticmethod
    def raw_to_ber(pred_raw_unconstrained):
        pred_log = BERFirstTokenSetTransformerBoundedLogHalfMultiTask.raw_to_log10ber(
            pred_raw_unconstrained)
        return torch.pow(10.0, pred_log).clamp(min=EPS, max=0.5)

    def forward(self, first_token, past_tokens, global_feats, threshold, key_padding_mask=None):
        thr = self.thr_embed(threshold)
        g = self.global_embed(global_feats)
        cond = torch.cat([thr, g], dim=-1)

        first_x = self.first_proj(first_token)
        first_x = self.first_cond_mod(first_x, cond)
        first_x = first_x + self.first_refine(first_x)

        set_x = self.token_proj(past_tokens)
        set_x = self.set_cond_mod(set_x, cond)
        for block in self.set_blocks:
            set_x = block(set_x, key_padding_mask=key_padding_mask)
        set_x = self.final_set_norm(set_x)

        if key_padding_mask is not None:
            valid_mask = (~key_padding_mask).unsqueeze(-1).float()
        else:
            valid_mask = torch.ones(
                set_x.shape[0], set_x.shape[1], 1,
                device=set_x.device, dtype=set_x.dtype)

        pooled_attn = self.set_attn_pool(set_x, key_padding_mask=key_padding_mask)
        valid_counts = valid_mask.sum(dim=1).clamp(min=1.0)
        pooled_mean = (set_x * valid_mask).sum(dim=1) / valid_counts

        if key_padding_mask is not None:
            x_for_max = set_x.masked_fill(key_padding_mask.unsqueeze(-1), float("-inf"))
        else:
            x_for_max = set_x
        pooled_max = x_for_max.amax(dim=1)
        pooled_max = torch.nan_to_num(pooled_max, nan=0.0, posinf=0.0, neginf=0.0)

        set_summary = torch.cat([pooled_attn, pooled_mean, pooled_max], dim=-1)
        fused = torch.cat([first_x, set_summary, cond], dim=-1)

        pred_raw_unconstrained = self.reg_head(fused)
        ord_logits = self.ord_head(fused)
        return pred_raw_unconstrained, ord_logits


# =========================
# Build inference inputs (7 global features, position-independent scaling)
# =========================
def prepare_inference_features(case, thresholds, scalers, device):
    """Build model inputs for a threshold sweep over a single physical case.

    Returns:
        first_t, past_t, global_t, thr_t, mask_t — all torch tensors on device
        global_t has shape [B, 7]: [z0, z1, hmg, nsid, log_hd, da, herf]
    """
    B = len(thresholds)
    mem_len = case["mem_len"]
    N = float(case["N"])

    P_raw = np.asarray(case["P"], dtype=np.float32)
    var_raw = np.asarray(case["variances"], dtype=np.float32)
    thr_raw = np.asarray(thresholds, dtype=np.float32).reshape(-1, 1)

    # Feature engineering
    taps_feat = (P_raw * N).astype(np.float32)
    vars_feat = var_raw.astype(np.float32)
    abs_feat = np.abs(taps_feat).astype(np.float32)
    snr_feat = np.log10((taps_feat ** 2) / (var_raw + EPS) + EPS).astype(np.float32)

    # Broadcast to [B, mem_len]
    taps_2d = np.broadcast_to(taps_feat[None, :], (B, mem_len)).copy()
    vars_2d = np.broadcast_to(vars_feat[None, :], (B, mem_len)).copy()
    abs_2d = np.broadcast_to(abs_feat[None, :], (B, mem_len)).copy()
    snr_2d = np.broadcast_to(snr_feat[None, :], (B, mem_len)).copy()

    L_past = mem_len - 1
    valid_past = np.ones((B, L_past), dtype=bool)

    # --- Threshold-dependent global features ---
    first_mean = taps_2d[:, 0:1]
    past_means = taps_2d[:, 1:]
    first_var = vars_2d[:, 0:1]
    past_vars = vars_2d[:, 1:]

    mu0 = (0.5 * past_means.sum(axis=1, keepdims=True)).astype(np.float32)
    mu1 = (first_mean + mu0).astype(np.float32)
    var0 = (0.5 * past_vars.sum(axis=1, keepdims=True)).astype(np.float32)
    var1 = (first_var + var0).astype(np.float32)
    std0 = np.sqrt(np.maximum(var0, EPS)).astype(np.float32)
    std1 = np.sqrt(np.maximum(var1, EPS)).astype(np.float32)

    z0 = ((thr_raw - mu0) / (std0 + EPS)).astype(np.float32)
    z1 = ((mu1 - thr_raw) / (std1 + EPS)).astype(np.float32)
    harmonic = safe_harmonic_z(z0, z1)
    abs_diff = np.abs(z0 - z1).astype(np.float32)
    hmg = (harmonic - 0.25 * abs_diff).astype(np.float32)
    hmg = np.nan_to_num(hmg, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)

    # --- Scenario-level features (threshold-independent) ---
    signal = first_mean  # [B, 1]
    isi = past_means.sum(axis=1, keepdims=True)
    nsid = ((signal - isi) / (signal + isi + EPS)).astype(np.float32)

    gap = signal
    d0_inf = (gap / (std0 + EPS)).astype(np.float32)
    d1_inf = (gap / (std1 + EPS)).astype(np.float32)
    hd = (2.0 * d0_inf * d1_inf / (d0_inf + d1_inf + EPS)).astype(np.float32)
    log_hd = np.log10(hd + EPS).astype(np.float32)
    da = (np.abs(d0_inf - d1_inf) / (d0_inf + d1_inf + EPS)).astype(np.float32)

    past_taps_sum = past_means.sum(axis=1, keepdims=True)
    past_shares = past_means / (past_taps_sum + EPS)
    herf = (past_shares ** 2).sum(axis=1, keepdims=True).astype(np.float32)

    # --- Scale first token ---
    ft_tap = ((taps_2d[:, 0] - scalers["first_tap_mean"]) / scalers["first_tap_std"]).astype(np.float32)
    ft_var = ((vars_2d[:, 0] - scalers["first_var_mean"]) / scalers["first_var_std"]).astype(np.float32)
    ft_abs = ((abs_2d[:, 0] - scalers["first_abs_mean"]) / scalers["first_abs_std"]).astype(np.float32)
    ft_snr = ((snr_2d[:, 0] - scalers["first_snr_mean"]) / scalers["first_snr_std"]).astype(np.float32)
    first_token = np.stack([ft_tap, ft_var, ft_abs, ft_snr], axis=1).astype(np.float32)

    # --- Scale past tokens ---
    pt_tap = apply_shared_scale(taps_2d[:, 1:], scalers["past_tap_mean"], scalers["past_tap_std"], valid_past)
    pt_var = apply_shared_scale(vars_2d[:, 1:], scalers["past_var_mean"], scalers["past_var_std"], valid_past)
    pt_abs = apply_shared_scale(abs_2d[:, 1:], scalers["past_abs_mean"], scalers["past_abs_std"], valid_past)
    pt_snr = apply_shared_scale(snr_2d[:, 1:], scalers["past_snr_mean"], scalers["past_snr_std"], valid_past)
    past_tokens = np.stack([pt_tap, pt_var, pt_abs, pt_snr], axis=2).astype(np.float32)

    # --- Scale globals (7 features) ---
    z0_s = scalers["z0_scaler"].transform(z0).astype(np.float32)
    z1_s = scalers["z1_scaler"].transform(z1).astype(np.float32)
    hmg_s = scalers["harmonic_minus_gap_scaler"].transform(hmg).astype(np.float32)
    nsid_s = scalers["nsid_scaler"].transform(nsid).astype(np.float32)
    log_hd_s = scalers["log_hd_scaler"].transform(log_hd).astype(np.float32)
    da_s = scalers["da_scaler"].transform(da).astype(np.float32)
    herf_s = scalers["herf_scaler"].transform(herf).astype(np.float32)
    global_feats = np.concatenate(
        [z0_s, z1_s, hmg_s, nsid_s, log_hd_s, da_s, herf_s], axis=1
    ).astype(np.float32)

    # --- Scale threshold using scenario-normalized feature from training ---
    total_scaled_taps = np.sum(taps_2d, axis=1, keepdims=True).astype(np.float32)
    threshold_mode = scalers.get("threshold_feature_mode", THRESHOLD_FEATURE_MODE)
    thr_feature = compute_threshold_feature_np(thr_raw, total_scaled_taps, mode=threshold_mode)
    thr_s = scalers["thr_scaler"].transform(thr_feature).astype(np.float32)

    # --- Padding mask (no padding needed here) ---
    pad_mask = np.zeros((B, L_past), dtype=bool)

    first_t = torch.from_numpy(first_token).to(device)
    past_t = torch.from_numpy(past_tokens).to(device)
    global_t = torch.from_numpy(global_feats).to(device)
    thr_t = torch.from_numpy(thr_s).to(device)
    mask_t = torch.from_numpy(pad_mask).to(device)

    return first_t, past_t, global_t, thr_t, mask_t


# =========================
# Prediction helpers
# =========================
def predict_ber_for_thresholds(model, case, thresholds, scalers, device):
    """Predict BER for a threshold vector.

    This recomputes threshold-dependent engineered features, so it is safe to call
    for shifted threshold vectors such as thr-eps and thr+eps.
    """
    first_t, past_t, global_t, thr_t, mask_t = prepare_inference_features(
        case, thresholds, scalers, device,
    )

    with torch.no_grad():
        pred_raw_out, ord_logits = model(
            first_t, past_t, global_t, thr_t, key_padding_mask=mask_t,
        )
        pred_bers = model.raw_to_ber(pred_raw_out).cpu().numpy().reshape(-1)
        pred_log = model.raw_to_log10ber(pred_raw_out).cpu().numpy().reshape(-1)
        pred_regions = ordinal_logits_to_region_labels_torch(ord_logits).cpu().numpy().reshape(-1)
        pred_region_probs = torch.sigmoid(ord_logits).cpu().numpy()

    pred_bers = np.clip(pred_bers, EPS, 0.5)
    pred_log = np.log10(np.clip(pred_bers, EPS, 0.5)).astype(np.float32)
    return pred_bers, pred_log, pred_regions, pred_region_probs


def predict_ber_smoothed_3point(
    model,
    case,
    thresholds,
    scalers,
    device,
    eps_frac=SMOOTHING_EPS_FRAC,
    smoothing_space=SMOOTHING_SPACE,
    weights=SMOOTHING_WEIGHTS,
):
    """Three-point test-time smoothing over threshold.

    Predicts at [thr-eps, thr, thr+eps] and averages either in log-BER space or
    raw-BER space. Crucially, this recomputes global features for each shifted
    threshold, rather than only perturbing the already-scaled threshold tensor.
    """
    thresholds = np.asarray(thresholds, dtype=np.float32)
    w_minus, w_mid, w_plus = weights
    weight_sum = float(w_minus + w_mid + w_plus)
    w_minus, w_mid, w_plus = w_minus / weight_sum, w_mid / weight_sum, w_plus / weight_sum

    thr_span = float(np.max(thresholds) - np.min(thresholds))
    eps_thr = float(eps_frac * max(thr_span, EPS))

    thr_minus = np.clip(thresholds - eps_thr, 0.0, None).astype(np.float32)
    thr_mid = thresholds.astype(np.float32)
    thr_plus = (thresholds + eps_thr).astype(np.float32)

    ber_minus, log_minus, _, _ = predict_ber_for_thresholds(
        model, case, thr_minus, scalers, device,
    )
    ber_mid, log_mid, regions_mid, probs_mid = predict_ber_for_thresholds(
        model, case, thr_mid, scalers, device,
    )
    ber_plus, log_plus, _, _ = predict_ber_for_thresholds(
        model, case, thr_plus, scalers, device,
    )

    if smoothing_space.lower() == "raw":
        ber_smooth = (
            w_minus * ber_minus +
            w_mid * ber_mid +
            w_plus * ber_plus
        )
        ber_smooth = np.clip(ber_smooth, EPS, 0.5).astype(np.float32)
        log_smooth = np.log10(ber_smooth).astype(np.float32)
    elif smoothing_space.lower() == "log":
        log_smooth = (
            w_minus * log_minus +
            w_mid * log_mid +
            w_plus * log_plus
        ).astype(np.float32)
        ber_smooth = np.clip(10.0 ** log_smooth, EPS, 0.5).astype(np.float32)
    else:
        raise ValueError("smoothing_space must be either 'log' or 'raw'.")

    return {
        "ber": ber_smooth,
        "log10_ber": log_smooth,
        "regions": regions_mid,
        "region_probs": probs_mid,
        "raw_mid_ber": ber_mid,
        "raw_mid_log10_ber": log_mid,
        "eps_threshold": eps_thr,
        "smoothing_space": smoothing_space,
    }


# =========================
# Generate scenario
# =========================
rng = np.random.default_rng(RANDOM_SEED)
case = generate_physical_case(
    rng,
    min_mem_len=PHYSICS_MIN_MEM_LEN,
    max_mem_len=PHYSICS_MAX_MEM_LEN,
    arrival_coverage=ARRIVAL_COVERAGE,
)

print("Generated physical scenario")
print("radius    =", case["radius"])
print("distance  =", case["distance"])
print("diffusion =", case["diffusion"])
print("Ts        =", case["Ts"])
print("N         =", case["N"])
print("mem_len   =", case["mem_len"])
print("P         =", case["P"])
print("P_scaled  =", case["P_scaled"])
print("variances =", case["variances"])

# =========================
# Threshold sweep — ground truth
# =========================
thr_min = 0.0
thr_max = float(np.sum(case["P_scaled"]))
thresholds = np.linspace(thr_min, thr_max, N_THRESHOLDS)

print("Threshold search interval:", thr_min, "to", thr_max)
print("sum(P_scaled) =", np.sum(case["P_scaled"]))

real_bers = np.array([
    calculate_ber_vectorized(
        mem_len=case["mem_len"],
        threshold=thr,
        P_scaled=case["P_scaled"],
        variances=case["variances"],
    )
    for thr in thresholds
])

real_bers = np.clip(real_bers, EPS, 0.5)
real_regions = raw_to_region_labels_np(real_bers)

# =========================
# Load model + scalers
# =========================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
scalers = joblib.load(SCALER_PATH)

max_past_seq_len = scalers.get("train_max_past_seq_len", scalers.get("max_past_seq_len", case["mem_len"] - 1))
ordinal_thresholds = scalers.get("ordinal_thresholds", ORDINAL_THRESHOLDS)
num_ordinal = len(ordinal_thresholds)
global_dim = scalers.get("global_dim", 7)

model = BERFirstTokenSetTransformerBoundedLogHalfMultiTask(
    max_past_seq_len=max_past_seq_len,
    token_dim=scalers.get("past_token_dim", 4),
    first_token_dim=scalers.get("first_token_dim", 4),
    threshold_dim=1,
    global_dim=global_dim,
    d_model=128,
    num_set_layers=4,
    num_heads=4,
    mlp_ratio=4.0,
    dropout=0.10,
    num_ordinal_thresholds=num_ordinal,
).to(device)

state = torch.load(MODEL_PATH, map_location=device, weights_only=True)
model.load_state_dict(state)
model.eval()

print(f"\nModel loaded: max_past_seq_len={max_past_seq_len}, "
      f"global_dim={global_dim}, num_ordinal={num_ordinal}")
print(f"Scaling strategy: {scalers.get('scaling_strategy', 'unknown')}")
print(f"Threshold feature mode: {scalers.get('threshold_feature_mode', THRESHOLD_FEATURE_MODE)}")
print("Ordinal head: monotonic cumulative logits")

# =========================
# Predict across thresholds
# =========================
raw_bers, raw_log, raw_regions, raw_region_probs = predict_ber_for_thresholds(
    model, case, thresholds, scalers, device,
)

if USE_TEST_TIME_SMOOTHING:
    smooth = predict_ber_smoothed_3point(
        model,
        case,
        thresholds,
        scalers,
        device,
        eps_frac=SMOOTHING_EPS_FRAC,
        smoothing_space=SMOOTHING_SPACE,
        weights=SMOOTHING_WEIGHTS,
    )
    pred_bers = smooth["ber"]
    pred_log = smooth["log10_ber"]
    pred_regions = smooth["regions"]
    pred_region_probs = smooth["region_probs"]
    print(
        f"Test-time smoothing: ON | space={smooth['smoothing_space']} | "
        f"eps_threshold={smooth['eps_threshold']:.6g}"
    )
else:
    pred_bers = raw_bers
    pred_log = raw_log
    pred_regions = raw_regions
    pred_region_probs = raw_region_probs
    print("Test-time smoothing: OFF")

pred_bers = np.clip(pred_bers, EPS, 0.5)
raw_bers = np.clip(raw_bers, EPS, 0.5)

# =========================
# Comparison table
# =========================
results = pd.DataFrame({
    "threshold": thresholds,
    "real_BER": real_bers,
    "estimated_BER": pred_bers,
    "estimated_BER_raw_unsmoothed": raw_bers,
    "smoothing_delta": pred_bers - raw_bers,
    "real_region": real_regions,
    "predicted_region": pred_regions,
    "abs_error": np.abs(pred_bers - real_bers),
    "abs_log10_error": np.abs(
        np.log10(np.clip(pred_bers, EPS, 0.5)) -
        np.log10(np.clip(real_bers, EPS, 0.5))
    ),
    "region_abs_error": np.abs(pred_regions.astype(np.int64) - real_regions.astype(np.int64)),
})

print(results.head(15))

print("\nSummary")
print("Mean abs raw error        :", results["abs_error"].mean())
print("Mean abs log10 error      :", results["abs_log10_error"].mean())
print("Max  abs log10 error      :", results["abs_log10_error"].max())
print("Mean region abs error     :", results["region_abs_error"].mean())
print("Exact region accuracy     :", np.mean(results["real_region"] == results["predicted_region"]))
if USE_TEST_TIME_SMOOTHING:
    print("Mean abs smoothing delta  :", float(np.mean(np.abs(results["smoothing_delta"]))))
    print("Max  abs smoothing delta  :", float(np.max(np.abs(results["smoothing_delta"]))))

best_real_idx = np.argmin(real_bers)
best_est_idx = np.argmin(pred_bers)

print("\nBest threshold from real BER      :", thresholds[best_real_idx])
print("Minimum real BER                  :", real_bers[best_real_idx])
print("Real BER region there             :", REGION_LABELS.get(int(real_regions[best_real_idx]), "?"))

print("Best threshold from estimated BER :", thresholds[best_est_idx])
print("Estimated BER at that threshold   :", pred_bers[best_est_idx])
print("Predicted BER region there        :", REGION_LABELS.get(int(pred_regions[best_est_idx]), "?"))

# =========================
# Plot 1: log-scale BER
# =========================
plt.figure(figsize=(10, 6))
plt.plot(results["threshold"], results["estimated_BER"], label="Estimated BER" + (" smoothed" if USE_TEST_TIME_SMOOTHING else ""))
if USE_TEST_TIME_SMOOTHING:
    plt.plot(results["threshold"], results["estimated_BER_raw_unsmoothed"], label="Estimated BER raw", alpha=0.45)
plt.plot(results["threshold"], results["real_BER"], label="Real BER")
plt.yscale("log")
plt.xlabel("Threshold")
plt.ylabel("BER")
plt.title(f"Threshold vs BER (log scale) — mem_len={case['mem_len']}")
plt.legend()
plt.grid(True, which="both", alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, "01_ber_log_scale.png"), dpi=150)
plt.close()
print(f"Saved: {PLOT_DIR}/01_ber_log_scale.png")

# =========================
# Plot 2: linear-scale BER
# =========================
plt.figure(figsize=(10, 6))
plt.plot(results["threshold"], results["estimated_BER"], label="Estimated BER" + (" smoothed" if USE_TEST_TIME_SMOOTHING else ""))
if USE_TEST_TIME_SMOOTHING:
    plt.plot(results["threshold"], results["estimated_BER_raw_unsmoothed"], label="Estimated BER raw", alpha=0.45)
plt.plot(results["threshold"], results["real_BER"], label="Real BER")
plt.yscale("linear")
plt.xlabel("Threshold")
plt.ylabel("BER")
plt.title(f"Threshold vs BER (linear scale) — mem_len={case['mem_len']}")
plt.legend()
plt.grid(True, which="both", alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, "02_ber_linear_scale.png"), dpi=150)
plt.close()
print(f"Saved: {PLOT_DIR}/02_ber_linear_scale.png")

# =========================
# Plot 3: predicted vs true region
# =========================
plt.figure(figsize=(10, 6))
plt.plot(results["threshold"], results["predicted_region"], label="Predicted region")
plt.plot(results["threshold"], results["real_region"], label="True region")
plt.xlabel("Threshold")
plt.ylabel("BER Region Class")
plt.title(f"Threshold vs BER Region — mem_len={case['mem_len']}")
plt.yticks(list(REGION_LABELS.keys()),
           [REGION_LABELS[k] for k in sorted(REGION_LABELS.keys())],
           fontsize=7)
plt.legend()
plt.grid(True, which="both", alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, "03_region_comparison.png"), dpi=150)
plt.close()
print(f"Saved: {PLOT_DIR}/03_region_comparison.png")

# =========================
# Plot 4: ordinal threshold probabilities
# =========================
plt.figure(figsize=(12, 7))
for i, thr_val in enumerate(ordinal_thresholds):
    plt.plot(thresholds, pred_region_probs[:, i], label=f"P(y >= {thr_val:g})")
plt.xlabel("Threshold")
plt.ylabel("Ordinal Probability")
plt.title(f"Ordinal Head Outputs Across Threshold Sweep — mem_len={case['mem_len']}")
plt.legend(fontsize=7, ncol=2)
plt.grid(True, which="both", alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, "04_ordinal_probabilities.png"), dpi=150)
plt.close()
print(f"Saved: {PLOT_DIR}/04_ordinal_probabilities.png")

# =========================
# Plot 5: absolute error by threshold
# =========================
plt.figure(figsize=(10, 6))
plt.plot(results["threshold"], results["abs_error"], label="Abs raw error", alpha=0.8)
plt.plot(results["threshold"], results["abs_log10_error"], label="Abs log10 error", alpha=0.8)
plt.yscale("log")
plt.xlabel("Threshold")
plt.ylabel("Error")
plt.title(f"Prediction Error Across Threshold Sweep — mem_len={case['mem_len']}")
plt.legend()
plt.grid(True, which="both", alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, "05_error_by_threshold.png"), dpi=150)
plt.close()
print(f"Saved: {PLOT_DIR}/05_error_by_threshold.png")

# =========================
# Plot 6: smoothing delta
# =========================
if USE_TEST_TIME_SMOOTHING:
    plt.figure(figsize=(10, 6))
    plt.plot(results["threshold"], results["smoothing_delta"], label="Smoothed - raw estimate")
    plt.axhline(0.0, linewidth=1.0)
    plt.xlabel("Threshold")
    plt.ylabel("BER delta")
    plt.title(f"Test-Time Smoothing Delta — mem_len={case['mem_len']}")
    plt.legend()
    plt.grid(True, which="both", alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(PLOT_DIR, "06_smoothing_delta.png"), dpi=150)
    plt.close()
    print(f"Saved: {PLOT_DIR}/06_smoothing_delta.png")


Generated physical scenario
radius    = 3.6485676254025527
distance  = 10.135731959475024
diffusion = 51.36802218605558
Ts        = 0.3990491521301547
N         = 388928
mem_len   = 17
P         = [0.03002163 0.03958956 0.02587469 0.01797425 0.01332206 0.01035728
 0.00834171 0.00690131 0.00583101 0.00501058 0.00436557 0.00384773
 0.0034246  0.0030736  0.00277866 0.002528   0.00231285]
P_scaled  = [11676.25240443 15397.4890797  10063.39179496  6990.68817612
  5181.32044315  4028.23443699  3244.32474935  2684.11384566
  2267.84291636  1948.75560929  1697.89363599  1496.49134765
  1331.92094457  1195.4103824   1080.6984328    983.21014921
   899.5338417 ]
variances = [11325.7122782  14787.90923469  9803.00464253  6865.03581585
  5112.29460409  3986.51290324  3217.26153179  2665.589936
  2254.61910246  1938.99120965  1690.48135712  1490.73324755
  1327.35965456  1191.73616511  1077.69553997   980.72459354
   897.45335089]
Threshold search interval: 0.0 to 72167.57219033288
sum(P_scaled) = 